# TribesTalk sample-only voice engine
Private research/noncommercial preparation using Breeze TTS 2 on Kaggle.

The final renderer uses only PCM samples, a pronunciation dictionary and the
Python standard library. No neural inference or GPU is required at playback.

Run stages separately. Automatic ASR, speaker comparisons and alignment output
are candidates, not reviews. Training contains several speakers. Stock expressive
filenames and generated event cues are not trusted labels. Do not generate a large
gap corpus before listening to the small pilot. No passage guarantees every
English diphone or username. All exports remain private and noncommercial.
This reusable source template is public; notebook runs and prepared audio are private.
The friend handles game integration. This workflow prepares standalone text-to-WAV data.


In [ ]:
from pathlib import Path
import hashlib, json, subprocess, sys
ROOT=Path('/kaggle/temp/tribes2-voice-lab')
ROOT.mkdir(parents=True,exist_ok=True)
FILES={'gap_jobs.py': '"""Plan only missing voice/style material; text coverage never certifies audio."""\nimport argparse\nfrom collections import Counter\nimport hashlib\nimport json\nfrom pathlib import Path\nfrom phonetics import Lexicon,UnknownWords,normalize,select_gaps,transitions\nfrom renderer import Renderer\nfrom prep.expression_support import EMOTIONS\n\nWARNING=\'Predicted material only. Validate generated words, actual phone alignment and acoustic quality; never promote review flags from text coverage.\'\n\ndef plan_incremental(root,bank,lex,speakers,emotions,max_sentences=4,candidate_coverage=False,carrier=\'\',sentences=None,workload_priority=False):\n    from build_reports import candidates\n    root=Path(root);bank=Path(bank)\n    if not 1<=max_sentences<=60:raise ValueError(\'Select 1–60 sentences per voice/style\')\n    if not speakers or not emotions or any(e not in EMOTIONS for e in emotions):raise ValueError(\'Explicit known voices/styles required\')\n    renderer=Renderer(bank,lex,audition_unverified=candidate_coverage,bank_pronunciations=True)\n    pool=list(dict.fromkeys(candidates() if sentences is None else sentences))\n    target=set();unknown=[];pair_weights=Counter()\n    for text in pool:\n        try:\n            pairs=set(transitions(lex.pronounce(text)[\'phones\']))\n            target.update(pairs);pair_weights.update(pairs)\n        except UnknownWords as exc:unknown.append({\'text\':text,\'unknown\':exc.words})\n    # Complete and failed-but-intact attempts both stay excluded: seed/context\n    # are deterministic, so silently repeating the same bad take buys nothing.\n    attempted={}\n    manifest=root/\'artifacts/pilot/pilot-jobs.json\'\n    for old in json.loads(manifest.read_text()) if manifest.exists() else []:\n        if old.get(\'kind\')!=\'speech\':continue\n        source=Path(old[\'source\'])\n        if not source.is_file() or hashlib.sha256(source.read_bytes()).hexdigest()!=old[\'sha256\']:\n            raise ValueError(\'Existing generated source missing/changed; restore before planning: \'+old[\'id\'])\n        attempted.setdefault((old[\'speaker_id\'],old[\'emotion\']),set()).add(normalize(old[\'text\'],lex.aliases))\n    jobs=[];reports={}\n    for speaker in speakers:\n        if not speaker or Path(speaker).name!=speaker:raise ValueError(\'Directory-safe speaker ID required\')\n        ref=root/\'artifacts/references-provisional\'/f\'{speaker}.wav\'\n        ref_sha=hashlib.sha256(ref.read_bytes()).hexdigest()\n        receipt=json.loads(ref.with_suffix(\'.json\').read_text())\n        if ref_sha!=receipt[\'sha256\']:raise ValueError(\'Reference checksum changed: \'+speaker)\n        ref_text=ref.with_suffix(\'.txt\').read_text()\n        if not ref_text.strip():raise ValueError(\'Empty reference transcript: \'+speaker)\n        for emotion in emotions:\n            existing=renderer.available.get((speaker,emotion),set())\n            missing_texts=[];supported=0;excluded=0\n            for text in pool:\n                renderer.warnings=[]\n                try:\n                    renderer.pronounce_for_bank(speaker,emotion,text);supported+=1\n                except UnknownWords:continue\n                except ValueError:\n                    spoken=(carrier.strip()+\' \'+text).strip()\n                    if normalize(spoken,lex.aliases) in attempted.get((speaker,emotion),set()):excluded+=1\n                    else:missing_texts.append(spoken)\n            selected=select_gaps(missing_texts,lex,target,existing,max_sentences=max_sentences,\n                                 pair_weights=pair_weights if workload_priority else None)\n            for item in selected[\'selected\']:\n                identity={\'speaker\':speaker,\'emotion\':emotion,\'text\':item[\'text\'],\n                          \'instruction\':EMOTIONS[emotion],\'reference_sha256\':ref_sha,\'ref_text\':ref_text}\n                key=speaker+\'_\'+emotion+\'_gap_\'+hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()[:16]\n                jobs.append({\'id\':key,\'speaker_id\':speaker,\'emotion\':emotion,\'kind\':\'speech\',\'text\':item[\'text\'],\n                    \'instruction\':EMOTIONS[emotion],\'predicted_phones\':item[\'phones\'],\n                    \'predicted_new_pairs\':item[\'new_pairs\'],\'ref_audio_path\':str(ref),\'ref_text\':ref_text,\n                    \'reference_sha256\':ref_sha,\'source\':str(root/\'artifacts/pilot\'/(key+\'.wav\')),\n                    \'verified\':False,\'pronunciation_verified\':False,\'reference_verified\':False})\n            reports[speaker+\'/\'+emotion]={\'workload_requests\':len(pool),\'supported_phone_paths\':supported,\n                \'selected_sentences\':len(selected[\'selected\']),\'existing_attempts_excluded\':excluded,\n                \'canonical_target_pairs\':len(target),\'canonical_missing_before\':[\'|\'.join(x) for x in sorted(target-existing)],\n                \'predicted_uncovered_after\':selected[\'uncovered\'],\'selection_rejections\':selected[\'rejected\'],\n                \'coverage_baseline\':\'UNREVIEWED CANDIDATES, private planning only\' if candidate_coverage else \'reviewed units only\',\n                \'audio_verified\':False}\n    return jobs,{\'schema\':1,\'warning\':WARNING,\'bank_digest\':renderer.digest,\'candidate_coverage\':candidate_coverage,\n                  \'carrier\':carrier,\'unknown_workload_items\':unknown,\'groups\':reports,\n                  \'selection\':\'workload_frequency\' if workload_priority else \'unique_pair_coverage\'}\n\ndef immutable_json(path,data):\n    path=Path(path);text=json.dumps(data,indent=2)\n    if path.exists() and path.read_text()!=text:raise ValueError(\'Refusing to overwrite changed job plan: \'+str(path))\n    if not path.exists():path.parent.mkdir(parents=True,exist_ok=True);path.write_text(text)\n\ndef main():\n    root=Path(__file__).resolve().parent\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--speaker\',nargs=\'+\',default=[\'Announcer\'])\n    p.add_argument(\'--emotions\',nargs=\'+\',choices=list(EMOTIONS),default=[\'neutral\'])\n    p.add_argument(\'--bank\',type=Path);p.add_argument(\'--candidate-coverage\',action=\'store_true\')\n    p.add_argument(\'--carrier\',default=\'\');p.add_argument(\'--max-sentences\',type=int,default=4)\n    p.add_argument(\'--workload-priority\',action=\'store_true\',help=\'Prioritize predicted pairs shared by more workload messages; still requires actual audio validation\')\n    p.add_argument(\'--out\',type=Path);a=p.parse_args()\n    lex=Lexicon.load(root/\'artifacts/reports/cmudict.json\',root/\'pronunciation-overrides.json\')\n    if a.bank:\n        if not a.out:p.error(\'Incremental planning requires an explicit NEW --out path\')\n        jobs,report=plan_incremental(root,a.bank,lex,a.speaker,a.emotions,a.max_sentences,a.candidate_coverage,a.carrier,\n                                     workload_priority=a.workload_priority)\n        immutable_json(a.out,jobs);immutable_json(a.out.with_suffix(\'.report.json\'),report)\n        print(json.dumps({\'jobs\':len(jobs),\'file\':str(a.out),\'groups\':report[\'groups\'],\'warning\':WARNING},indent=2))\n        return\n    if len(a.speaker)!=1 or a.emotions!=[\'neutral\'] or a.carrier or a.candidate_coverage or a.workload_priority:\n        p.error(\'Multi-voice/style, carrier and incremental coverage options require --bank\')\n    # Preserve the original neutral set-cover command and IDs for existing work.\n    speaker=a.speaker[0]\n    plan=json.loads((root/\'artifacts/reports\'/f\'gap-script-{speaker}.json\').read_text())\n    ref=root/\'artifacts/references-provisional\'/f\'{speaker}.wav\'\n    if not ref.is_file():raise ValueError(\'Missing reference: \'+speaker)\n    jobs=[]\n    for i,item in enumerate(plan[\'selected\']):\n        phones=lex.pronounce(item[\'text\'])[\'phones\']\n        if phones!=item[\'phones\']:raise ValueError(\'Pronunciation changed since set-cover selection\')\n        jobs.append({\'id\':f\'{speaker}_neutral_gap_{i:03d}\',\'speaker_id\':speaker,\'emotion\':\'neutral\',\'kind\':\'speech\',\n             \'text\':item[\'text\'],\'predicted_phones\':phones,\'instruction\':None,\'ref_audio_path\':str(ref),\n             \'ref_text\':ref.with_suffix(\'.txt\').read_text(),\'verified\':False,\'pronunciation_verified\':False})\n    out=a.out or root/\'artifacts\'/f\'gap-jobs-{speaker}.json\'\n    immutable_json(out,jobs)\n    print(json.dumps({\'jobs\':len(jobs),\'file\':str(out),\'target_pairs\':plan[\'target_transition_count\'],\'warning\':WARNING},indent=2))\n\nif __name__==\'__main__\':main()\n', 'bank.py': '"""Extract reviewed aligned phones into midpoint-to-midpoint PCM16 units.\n\nAlignment JSON contains source WAV, speaker_id, emotion, text, aligner metadata,\nreview flags and phones [{label,start,end,variant?,anchor?,protected_release?}].\nTimes are seconds. Stop anchors must be explicitly checked in stable closure,\nbefore protected release; unreviewed stops remain candidates, never production.\n"""\nfrom __future__ import annotations\nfrom array import array\nimport hashlib\nimport json\nfrom pathlib import Path\nimport sys\nimport wave\nfrom phonetics import base\n\nSTOPS = {\'P\',\'B\',\'T\',\'D\',\'K\',\'G\'}\n\ndef read_wav(path):\n    with wave.open(str(path),\'rb\') as w:\n        if (w.getnchannels(),w.getsampwidth(),w.getcomptype()) != (1,2,\'NONE\'):\n            raise ValueError(\'Expected mono PCM16 WAV\')\n        rate = w.getframerate()\n        audio = array(\'h\', w.readframes(w.getnframes()))\n        if sys.byteorder != \'little\':\n            audio.byteswap()\n        return rate,audio\n\ndef write_wav(path,rate,audio):\n    path=Path(path)\n    path.parent.mkdir(parents=True,exist_ok=True)\n    a=array(\'h\',audio)\n    if sys.byteorder != \'little\':\n        a.byteswap()\n    with wave.open(str(path),\'wb\') as w:\n        w.setnchannels(1); w.setsampwidth(2); w.setframerate(rate); w.writeframes(a.tobytes())\n\ndef safe_path(root,name):\n    root=Path(root).resolve()\n    p=(root/name).resolve()\n    if p != root and root not in p.parents:\n        raise ValueError(\'Path outside bank\')\n    return p\n\ndef extract(alignment, output, handle_ms=5, reviewed_only=False):\n    if not 2.5 <= handle_ms <= 5:\n        raise ValueError(\'Handles must support a 5–10 ms total crossfade\')\n    output=Path(output); output.mkdir(parents=True,exist_ok=True)\n    source=Path(alignment[\'source\'])\n    rate,audio=read_wav(source)\n    duration=len(audio)/rate\n    phones=alignment[\'phones\']\n    prev=0\n    for p in phones:\n        if not p[\'label\'] or not 0 <= p[\'start\'] < p[\'end\'] <= duration or p[\'start\'] < prev-0.000001:\n            raise ValueError(\'Invalid/overlapping phone intervals\')\n        if p[\'start\']-prev > 0.002:\n            raise ValueError(\'Alignment gap: represent silence explicitly\')\n        prev=p[\'end\']\n    anchors=[]\n    for p in phones:\n        anchor=p.get(\'anchor\',(p[\'start\']+p[\'end\'])/2)\n        if not p[\'start\'] <= anchor <= p[\'end\']:\n            raise ValueError(\'Anchor outside phone\')\n        protected=p.get(\'protected_release\')\n        if protected and (not p[\'start\'] <= protected[0] < protected[1] <= p[\'end\'] or anchor >= protected[0]):\n            raise ValueError(\'Stop anchor must precede protected release\')\n        anchors.append(round(anchor*rate))\n    h=round(rate*handle_ms/1000)\n    source_sha=hashlib.sha256(source.read_bytes()).hexdigest()\n    if alignment.get(\'source_sha256\') and alignment[\'source_sha256\']!=source_sha:\n        raise ValueError(\'Alignment source checksum changed\')\n    units=[]\n    for i,(left,right) in enumerate(zip(phones,phones[1:])):\n        a,b=anchors[i:i+2]\n        if b <= a:\n            raise ValueError(\'Non-increasing anchors\')\n        first,last=max(0,a-h),min(len(audio),b+h)\n        sample=audio[first:last]\n        if not sample:\n            continue\n        protected_ok=True\n        for p,anchor in ((left,a),(right,b)):\n            if base(p[\'label\']) in STOPS:\n                rel=p.get(\'protected_release\')\n                protected_ok &= bool(p.get(\'closure_verified\') and rel and\n                                     anchor+h <= round(rel[0]*rate))\n        verified=all(alignment.get(k) is True for k in (\n            \'speaker_verified\',\'transcript_verified\',\'alignment_verified\',\'emotion_verified\')) and protected_ok\n        if reviewed_only and not verified:\n            continue\n        key=f\'{source_sha[:16]}-{i:04d}\'\n        path=f\'units/{key}.wav\'\n        write_wav(output/path,rate,sample)\n        units.append({\'id\':key,\'phones\':[left[\'label\'],right[\'label\']],\n            \'variants\':[left.get(\'variant\',\'canonical\'),right.get(\'variant\',\'canonical\')],\n            \'context\':{\'previous\':phones[i-1][\'label\'] if i else \'SIL\',\n                       \'next\':phones[i+2][\'label\'] if i+2<len(phones) else \'SIL\'},\n            \'speaker_id\':alignment[\'speaker_id\'],\'emotion\':alignment.get(\'emotion\',\'neutral\'),\n            \'file\':path,\'rate\':rate,\'left_handle\':a-first,\'right_handle\':last-b,\n            \'core_samples\':b-a,\'source_sha256\':source_sha,\'source\':str(source),\n            \'start_seconds\':a/rate,\'end_seconds\':b/rate,\n            \'sha256\':hashlib.sha256((output/path).read_bytes()).hexdigest(),\n            \'verified\':verified,\'release_safe\':protected_ok,\n            \'quality\':{\'clipped_fraction\':sum(abs(x)>=32760 for x in sample)/len(sample),\n                       \'alignment_score\':alignment.get(\'alignment_score\'),\n                       \'aligner\':alignment.get(\'aligner\'), \'reviewer\':alignment.get(\'reviewer\')}})\n    return units\n\ndef write_index(output, units, events=()):\n    output=Path(output)\n    ids=[u[\'id\'] for u in units]\n    if len(set(ids)) != len(ids):\n        raise ValueError(\'Duplicate unit IDs\')\n    speakers=sorted({x[\'speaker_id\'] for x in [*units,*events]})\n    d={\'schema\':1,\'license\':\'Private development; Breeze-derived outputs research/noncommercial only.\',\n       \'speakers\':speakers,\'units\':units,\'events\':list(events)}\n    d[\'content_digest\']=hashlib.sha256(json.dumps(d,sort_keys=True).encode()).hexdigest()\n    (output/\'index.json\').write_text(json.dumps(d,indent=2))\n    return {\'unit_count\':len(units),\'verified_units\':sum(u[\'verified\'] for u in units),\n            \'bytes\':sum(p.stat().st_size for p in output.rglob(\'*\') if p.is_file())}\n', 'test_companion.py': '"""Remote mailbox fixtures. Fake render bytes are NOT acoustic/native evidence."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport tempfile\nimport unittest\nfrom companion import Companion,install,MAILBOX,MAX_TEXT\n\nclass FakeRenderer:\n    def __init__(self):self.calls=[];self.fail=False\n    def render(self,speaker,text,output):\n        self.calls.append((speaker,text))\n        if self.fail:raise ValueError(\'Unknown words (add pronunciation overrides): fixture\')\n        # Transport fixture only: deliberately not an actual playable waveform.\n        Path(output).write_bytes(b\'transport fixture, not native audio\')\n        return {\'audio_seconds\':0.1,\'warnings\':[],\n                \'output_sha256\':hashlib.sha256(Path(output).read_bytes()).hexdigest()}\n\nclass CompanionTests(unittest.TestCase):\n    def setUp(self):\n        self.temp=tempfile.TemporaryDirectory();self.root=Path(self.temp.name)\n        self.base=self.root/\'base\';self.base.mkdir()\n        self.bank=self.root/\'bank\';self.bank.mkdir()\n        (self.bank/\'index.json\').write_text(json.dumps({\'speakers\':[\'Announcer\'],\'units\':[],\'events\':[]}))\n        self.lexicon=self.root/\'lexicon.json\';self.lexicon.write_text(json.dumps({\'hello\':[[\'HH\',\'EH1\',\'L\',\'OW0\']]}))\n        install(self.base,self.bank,self.lexicon)\n        self.folder=self.base/MAILBOX;self.config=json.loads((self.folder/\'companion.json\').read_text())\n        self.fake=FakeRenderer();self.now=100.0\n        self.service=Companion(self.base,renderer=self.fake,clock=lambda:self.now)\n    def tearDown(self):self.temp.cleanup()\n    def send(self,seq=1,kind=\'S\',speaker=\'Announcer\',emotion=\'neutral\',text=\'hello\',**changes):\n        fields={\'version\':\'1\',\'token\':self.config[\'token\'],\'session\':self.config[\'session\'],\n                \'kind\':kind,\'seq\':str(seq),\'speaker\':speaker,\'emotion\':emotion,\'text\':text}\n        fields.update(changes)\n        with (self.folder/\'requests.tsv\').open(\'a\') as stream:stream.write(\'\\t\'.join(fields.values())+\'\\n\')\n        return self.service.process_once()\n    def response(self):return (self.folder/\'response.tsv\').read_text().strip().split(\'\\t\')\n    def ack(self,seq=1):return self.send(seq=seq,kind=\'A\',speaker=\'-\',emotion=\'-\',text=\'playback_timer_elapsed\')\n    def test_speech_then_ack_does_not_claim_audible_playback(self):\n        self.assertEqual(self.send(),\'ready\');self.assertEqual(self.response()[3],\'READY\')\n        self.assertEqual(self.fake.calls,[(\'Announcer\',\'hello\')])\n        self.assertFalse(json.loads((self.folder/\'last-render.json\').read_text())[\'native_playback_verified\'])\n        self.assertEqual(self.ack(),\'acknowledged\');self.assertIsNone(self.service.state[\'pending\'])\n    def test_bad_and_non_ascii_tokens_rejected_before_renderer(self):\n        self.assertEqual(self.send(token=\'0\'*64),\'authentication_failed\')\n        self.assertEqual(self.send(token=\'王\'*64),\'authentication_failed\')\n        self.assertEqual(self.fake.calls,[])\n        self.assertNotIn(self.config[\'token\'],(self.folder/\'last-error.json\').read_text())\n    def test_protocol_mismatch_explicit(self):\n        self.assertEqual(self.send(version=\'2\'),\'protocol_mismatch\')\n        self.assertEqual(self.response()[4],\'protocol_mismatch\');self.assertEqual(self.fake.calls,[])\n    def test_path_and_unsupported_voice_style_rejected(self):\n        self.assertEqual(self.send(speaker=\'../../outside\'),\'unsupported_voice_style\')\n        self.assertEqual(self.send(seq=2,emotion=\'unknown\'),\'unsupported_voice_style\')\n        self.assertEqual(self.fake.calls,[])\n    def test_markup_long_and_control_text_refused(self):\n        for seq,text in enumerate((\'[grunt]\',\'a\'*(MAX_TEXT+1),\'hello\\x00world\'),1):\n            self.assertEqual(self.send(seq=seq,text=text),\'invalid_text\')\n        self.assertEqual(self.fake.calls,[])\n    def test_busy_does_not_overwrite_unacknowledged_output(self):\n        self.send();before=(self.folder/\'response.tsv\').read_bytes()\n        self.assertEqual(self.send(seq=2),\'busy\')\n        self.assertEqual((self.folder/\'response.tsv\').read_bytes(),before)\n        self.assertEqual(len(self.fake.calls),1)\n    def test_duplicate_request_never_rerenders(self):\n        self.send();self.ack()\n        self.assertEqual(self.send(),\'duplicate_sequence\');self.assertEqual(len(self.fake.calls),1)\n    def test_restart_preserves_pending_and_offset(self):\n        self.send()\n        self.service=Companion(self.base,renderer=self.fake,clock=lambda:self.now)\n        self.assertEqual(self.service.process_once(),\'idle\')\n        self.assertEqual(self.ack(),\'acknowledged\');self.assertEqual(len(self.fake.calls),1)\n    def test_partial_line_waits_without_consuming(self):\n        row=\'\\t\'.join((\'1\',self.config[\'token\'],self.config[\'session\'],\'S\',\'1\',\'Announcer\',\'neutral\',\'hello\'))\n        (self.folder/\'requests.tsv\').write_text(row)\n        self.assertEqual(self.service.process_once(),\'partial\');self.assertEqual(self.service.state[\'offset\'],0)\n        with (self.folder/\'requests.tsv\').open(\'a\') as stream:stream.write(\'\\n\')\n        self.assertEqual(self.service.process_once(),\'ready\')\n    def test_truncation_is_not_silent_replay(self):\n        self.send();(self.folder/\'requests.tsv\').write_bytes(b\'\')\n        with self.assertRaisesRegex(ValueError,\'truncated\'):self.service.process_once()\n    def test_ack_timeout_is_explicit(self):\n        self.send();self.now+=32\n        self.assertEqual(self.service.process_once(),\'idle\')\n        self.assertEqual(self.response()[4],\'ack_timeout\');self.assertIsNone(self.service.state[\'pending\'])\n    def test_changed_dictionary_blocks_startup(self):\n        self.lexicon.write_text(\'{}\')\n        with self.assertRaisesRegex(ValueError,\'configuration changed\'):Companion(self.base,renderer=self.fake)\n    def test_existing_output_is_preserved(self):\n        output=self.base/\'audio/t2vl-live\'/f\'{self.config["session"]}-1.wav\';output.write_bytes(b\'keep\')\n        self.assertEqual(self.send(),\'output_exists\');self.assertEqual(output.read_bytes(),b\'keep\')\n    def test_renderer_refusal_preserves_detail(self):\n        self.fake.fail=True\n        self.assertEqual(self.send(),\'render_refused\')\n        self.assertIn(\'Unknown words\',(self.folder/\'last-error.json\').read_text())\n    def test_install_refuses_existing_configuration_or_client_script(self):\n        with self.assertRaisesRegex(ValueError,\'already exists\'):install(self.base,self.bank,self.lexicon)\n        other=self.root/\'other\';(other/\'scripts\').mkdir(parents=True)\n        script=other/\'scripts/t2vl_live_client.cs\';script.write_text(\'human script\')\n        with self.assertRaisesRegex(ValueError,\'overwrite\'):install(other,self.bank,self.lexicon)\n        self.assertEqual(script.read_text(),\'human script\');self.assertFalse((other/MAILBOX).exists())\n    def test_installed_server_bound_to_bank_and_has_validation(self):\n        server=(self.folder/\'server.cs\').read_text();client=(self.base/\'scripts/t2vl_live_client.cs\').read_text()\n        self.assertIn(self.config[\'bank_sha256\'],server)\n        self.assertNotIn(\'@BANK@\',server);self.assertNotIn(\'@VOICES@\',server)\n        self.assertIn(\'function T2VLLiveTextOK\',server)\n        self.assertIn(\'isSuperAdmin\',server)\n        self.assertNotIn(\'eval(\',server+client)\n    def test_symlink_mailbox_escape_refused(self):\n        other=self.root/\'symlink-base\';other.mkdir()\n        outside=self.root/\'outside\';outside.mkdir()\n        (other/\'prefs\').symlink_to(outside,target_is_directory=True)\n        with self.assertRaisesRegex(ValueError,\'escapes\'):install(other,self.bank,self.lexicon)\n\nif __name__==\'__main__\':\n    if not str(Path(__file__).resolve()).startswith(\'/kaggle/\'):\n        raise RuntimeError(\'Run transport fixtures on the authorized remote fixture lane, not the laptop\')\n    suite=unittest.defaultTestLoader.loadTestsFromTestCase(CompanionTests)\n    result=unittest.TextTestRunner(verbosity=2).run(suite)\n    Path(\'test-companion-results.json\').write_text(json.dumps({\'tests\':result.testsRun,\n         \'failures\':len(result.failures),\'errors\':len(result.errors),\n         \'scope\':\'Fake-render transport fixtures, NOT native/audio acceptance\'},indent=2))\n    raise SystemExit(0 if result.wasSuccessful() else 1)\n', 'speaker_audit.py': '"""Remote ONNX speaker comparisons, not automatic identity certification.\n\nWeSpeaker ResNet34-LM (CC-BY-4.0), pinned HF revision. Matching scores are\nuncalibrated cosine similarities, not probabilities or verified voice names.\n"""\nfrom __future__ import annotations\nimport hashlib\nimport json\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\n\nROOT=Path(__file__).resolve().parent\nMODEL=\'Wespeaker/wespeaker-voxceleb-resnet34-LM\'\nREVISION=\'f0c48c298fd835726c27956a5d617bad7115627e\'\n\ndef main():\n    import kaldi_native_fbank as knf\n    import numpy as np\n    import onnxruntime as ort\n    from huggingface_hub import hf_hub_download\n    model=hf_hub_download(MODEL,\'voxceleb_resnet34_LM.onnx\',revision=REVISION)\n    options=ort.SessionOptions();options.intra_op_num_threads=2;options.inter_op_num_threads=1\n    sess=ort.InferenceSession(model,sess_options=options,providers=[\'CPUExecutionProvider\'])\n    print(\'ONNX\',[(x.name,x.shape) for x in sess.get_inputs()],flush=True)\n    rows=json.loads((ROOT/\'artifacts/audio_inventory.json\').read_text())\n    manifest={r[\'path\']:r for r in rows}\n    out=ROOT/\'artifacts/speaker-audit\';out.mkdir(parents=True,exist_ok=True)\n    cache=out/\'embeddings\';cache.mkdir(exist_ok=True)\n    start=time.time()\n    def embedding(name):\n        file=ROOT/\'artifacts/pcm24k\'/name\n        digest=hashlib.sha256(file.read_bytes()).hexdigest()\n        cached=cache/(digest+\'.npy\')\n        if cached.exists():return np.load(cached)\n        raw=subprocess.check_output([\'ffmpeg\',\'-nostdin\',\'-v\',\'error\',\'-i\',str(file),\'-ac\',\'1\',\'-ar\',\'16000\',\n                                     \'-f\',\'s16le\',\'-c:a\',\'pcm_s16le\',\'-\'])\n        wav=np.frombuffer(raw,dtype=\'<i2\').astype(np.float32)\n        # Whole recordings: no assumption of common speaker from folder. Long recordings\n        # get separate windows; disagreement is retained for review, not silently averaged.\n        opts=knf.FbankOptions();opts.frame_opts.samp_freq=16000;opts.frame_opts.dither=0\n        opts.frame_opts.window_type=\'hamming\';opts.mel_opts.num_bins=80\n        extractor=knf.OnlineFbank(opts);extractor.accept_waveform(16000,wav.tolist());extractor.input_finished()\n        features=np.stack([extractor.get_frame(i) for i in range(extractor.num_frames_ready)])\n        embeddings=[]\n        for pos in range(0,len(features),300):\n            feat=features[pos:pos+300]\n            if len(feat)<100 and embeddings:continue\n            if len(feat)<25:raise ValueError(\'Too little audio for speaker embedding\')\n            feat=feat-feat.mean(axis=0,keepdims=True)\n            e=np.asarray(sess.run(None,{sess.get_inputs()[0].name:feat[None].astype(np.float32)})[0]).reshape(-1)\n            embeddings.append(e/max(float(np.linalg.norm(e)),0.00000001))\n        values=np.stack(embeddings);np.save(cached,values)\n        return values\n    packs=[*(f\'Male{i}\' for i in range(1,6)),*(f\'Fem{i}\' for i in range(1,6)),*(f\'Derm{i}\' for i in range(1,4)),\'Bot1\']\n    refs={}\n    for pack in packs:\n        possible=[r for r in rows if r[\'path\'].startswith(\'voice/\'+pack+\'/\') and 1.2<=r[\'seconds\']<=8\n                  and any(w in r[\'path\'].casefold() for w in (\'gbl.hi.\',\'gbl.thanks.\',\'def.base.\',\'att.base.\',\'slf.defend.\'))]\n        if not possible:\n            possible=[r for r in rows if r[\'path\'].startswith(\'voice/\'+pack+\'/\') and 1.2<=r[\'seconds\']<=8]\n        if not possible:raise ValueError(\'No usable enrollment candidate for \'+pack)\n        names=[r[\'path\'] for r in sorted(possible,key=lambda r:r[\'path\'])[:3]]\n        values=np.concatenate([embedding(n) for n in names])\n        centroid=values.mean(axis=0);centroid/=np.linalg.norm(centroid)\n        refs[pack]=(centroid,names)\n    for label,names in {\n        \'Announcer_candidate\':[\'voice/Announcer/ANN.60.WAV\',\'voice/Announcer/ANN.gameover.WAV\'],\n        \'TrainingFemale_candidate\':[\'voice/Training/Mission1/T1_01.WAV\'],\n        \'BriefingMale_candidate\':[\'voice/Training/Briefings/T1.brief02.WAV\'],\n    }.items():\n        values=np.concatenate([embedding(n) for n in names]);c=values.mean(axis=0);c/=np.linalg.norm(c)\n        refs[label]=(c,names)\n    results=[]\n    for i,row in enumerate(rows):\n        if not row[\'path\'].startswith((\'voice/Training/\',\'voice/Announcer/\')):continue\n        if row[\'seconds\']<1:\n            results.append({\'path\':row[\'path\'],\'status\':\'too_short\',\'speaker_verified\':False});continue\n        values=embedding(row[\'path\'])\n        scores=sorted(((label,float(values.mean(axis=0)@c)) for label,(c,_) in refs.items()),key=lambda x:-x[1])\n        nearest=[max(refs,key=lambda k:float(e@refs[k][0])) for e in values]\n        results.append({\'path\':row[\'path\'],\'top_matches\':scores[:4],\'margin\':scores[0][1]-scores[1][1],\n                        \'window_nearest\':nearest,\'window_disagreement\':len(set(nearest))>1,\n                        \'speaker_verified\':False,\'status\':\'candidate_only\'})\n        if (i+1)%50==0:print(\'speaker comparisons\',i+1,flush=True)\n    same=embedding(\'voice/Training/Mission1/T1_01.WAV\')[0]\n    pilot_comparisons=[]\n    pilots=ROOT/\'artifacts/pilot/pilot-jobs.json\'\n    if pilots.exists():\n        for job in json.loads(pilots.read_text()):\n            if job.get(\'kind\')!=\'speech\':continue\n            label=job[\'speaker_id\'] if job[\'speaker_id\'] in refs else job[\'speaker_id\']+\'_candidate\'\n            if label not in refs:continue\n            values=embedding(job[\'source\'])\n            pilot_comparisons.append({\'id\':job[\'id\'],\'reference_label\':label,\n                \'window_cosines\':[float(e@refs[label][0]) for e in values],\n                \'speaker_verified\':False,\'note\':\'Uncalibrated comparison, not a voice identity verdict.\'})\n    instrument={\'positive_same_clip_cosine\':float(same@same),\n                \'contrast_reference_cosine\':float(same@refs[\'BriefingMale_candidate\'][0]),\n                \'warning\':\'Contrast is not ground-truth different-speaker certification.\'}\n    receipt={\'model\':MODEL,\'revision\':REVISION,\'license\':\'CC-BY-4.0; attribution WeSpeaker authors\',\n        \'sha256\':hashlib.sha256(Path(model).read_bytes()).hexdigest(),\'seconds\':time.time()-start,\n        \'enrollment\':{k:v[1] for k,v in refs.items()},\'instrument\':instrument,\n        \'records\':results,\'pilot_comparisons\':pilot_comparisons,\n        \'warning\':\'Uncalibrated scores; no automatic speaker or gender verification. Processed/robotic voices may confuse embeddings.\'}\n    (out/\'speaker-comparisons.json\').write_text(json.dumps(receipt,indent=2))\n    print(json.dumps(instrument),flush=True)\n    print(\'========== DONE ==========\',flush=True)\n\nif __name__==\'__main__\':main()\n', 'archive_artifacts.py': '"""Durable Kaggle output archive; only explicit artifact and version records."""\nimport hashlib\nimport argparse\nimport json\nfrom pathlib import Path\nimport tarfile\n\nroot=Path(__file__).resolve().parent\np=argparse.ArgumentParser();p.add_argument(\'--snapshot\',type=Path);p.add_argument(\'--name\',default=\'tribes2-voice-artifacts\')\na=p.parse_args()\nif Path(a.name).name!=a.name:raise ValueError(\'Directory-safe archive name required\')\noutput=Path(\'/kaggle/working\')/(a.name+\'.tar.gz\')\nassert str(root).startswith(\'/kaggle/\'), \'Remote archive only\'\nfiles=[p for p in (root/\'artifacts\').rglob(\'*\') if p.is_file() and not p.name.endswith((\'.part\',\'.tmp\'))]\nif a.snapshot:\n    # Exclude the live writer\'s directory, then include only completed, hash-matched jobs.\n    files=[p for p in files if root/\'artifacts/pilot\' not in p.parents]\n    for job in json.loads(a.snapshot.read_text()):\n        path=Path(job[\'source\']).resolve()\n        if root/\'artifacts\' not in path.parents:raise ValueError(\'Snapshot source outside artifact directory\')\n        if hashlib.sha256(path.read_bytes()).hexdigest()!=job[\'sha256\']:\n            raise ValueError(\'Snapshot source changed: \'+job[\'id\'])\n        files.extend((path,path.with_suffix(\'.json\')))\nfiles.extend(p for p in root.glob(\'test*results.json\'))\nfor name in (\'asr-resolved.txt\',\'breeze-resolved.txt\',\'align-resolved.txt\',\'breeze-source-revision.txt\',\n             \'breeze-model-revision.json\',\'asr-setup.json\',\'breeze-setup.json\',\'align-setup.json\'):\n    p=root/\'environments\'/name\n    if p.exists():files.append(p)\nmanifest=[]\nfor path in sorted(files):\n    manifest.append({\'path\':str(path.relative_to(root)),\'bytes\':path.stat().st_size,\n                     \'sha256\':hashlib.sha256(path.read_bytes()).hexdigest()})\nmanifest_path=root/(a.name+\'-manifest.json\')\nmanifest_path.write_text(json.dumps(manifest,indent=2))\nwith tarfile.open(output.with_suffix(\'.gz.part\'),\'w:gz\') as tar:\n    for path in sorted(files):tar.add(path,arcname=str(path.relative_to(root)),recursive=False)\n    tar.add(manifest_path,arcname=\'artifact-manifest.json\')\noutput.with_suffix(\'.gz.part\').replace(output)\nreceipt={\'archive\':str(output),\'bytes\':output.stat().st_size,\'files\':len(files),\n         \'sha256\':hashlib.sha256(output.read_bytes()).hexdigest()}\noutput.with_suffix(\'.receipt.json\').write_text(json.dumps(receipt,indent=2))\nprint(json.dumps(receipt,indent=2))\nprint(\'========== DONE ==========\',flush=True)\n', 'prepare_sequence.py': '"""Serial preparation job, run by the existing VM-kit job control plane."""\nfrom pathlib import Path\nimport subprocess\nimport sys\nroot=Path(__file__).resolve().parent\nfor command in ([sys.executable,\'-u\',\'setup_remote.py\',\'asr\'],\n                [str(root/\'environments/asr/bin/python\'),\'-u\',\'asr_entry.py\',\'--asr\',\'--skip-prep\'],\n                [sys.executable,\'-u\',\'setup_remote.py\',\'breeze\']):\n    subprocess.run(command,cwd=root,check=True)\nprint(\'========== PREPARATION SEQUENCE DONE ==========\',flush=True)\n', 'renderer.py': '"""Sample-only, standard-library runtime. Missing speech or events raise by default."""\nfrom __future__ import annotations\nimport argparse\nfrom array import array\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\nimport time\nfrom bank import read_wav,write_wav,safe_path\nfrom phonetics import Lexicon,transitions,base\n\nEMOTIONS={\'neutral\',\'angry\',\'sad\',\'excited\',\'calm\'}\nEVENTS={\'laugh\',\'sigh\',\'grunt\',\'cheer\',\'cough\',\'clear_throat\',\'pain\',\'death_cry\'}\n\nclass MissingUnit(ValueError):\n    pass\n\ndef markup(text):\n    emotion=\'neutral\'; opened=None; out=[]\n    for part in re.split(r\'(\\[[^\\]]+\\])\',text):\n        if not part:\n            continue\n        if not part.startswith(\'[\'):\n            if part.strip():\n                out.append((\'speech\',emotion,part))\n            continue\n        tag=part[1:-1]\n        if tag in EMOTIONS:\n            if opened:\n                raise ValueError(\'Nested emotion tags are not supported\')\n            emotion=opened=tag\n        elif tag.startswith(\'/\') and tag[1:] in EMOTIONS:\n            if opened != tag[1:]:\n                raise ValueError(\'Mismatched emotion tag\')\n            emotion=\'neutral\'; opened=None\n        elif tag in EVENTS:\n            out.append((\'event\',emotion,tag))\n        else:\n            raise ValueError(\'Unknown expression tag: \'+tag)\n    if opened:\n        raise ValueError(\'Unclosed emotion tag\')\n    return out\n\nclass Renderer:\n    def __init__(self,bank_path,lexicon,crossfade_ms=10,emotion_fallback=\'error\',stress_fallback=False,audition_unverified=False,bank_pronunciations=False):\n        self.root=Path(bank_path); self.lexicon=lexicon\n        self.index=json.loads((self.root/\'index.json\').read_text())\n        if not 5 <= crossfade_ms <= 10:\n            raise ValueError(\'Crossfade must be 5–10 ms\')\n        if emotion_fallback not in (\'error\',\'neutral\'):\n            raise ValueError(\'Invalid emotion fallback\')\n        self.crossfade_ms=crossfade_ms; self.fallback=emotion_fallback\n        self.stress_fallback=stress_fallback; self.warnings=[]; self.rate=None\n        self.audition_unverified=audition_unverified\n        self.bank_pronunciations=bank_pronunciations\n        self.available={}\n        for unit in self.index[\'units\']:\n            if unit.get(\'verified\') is True or audition_unverified:\n                key=(unit[\'speaker_id\'],unit[\'emotion\'])\n                self.available.setdefault(key,set()).add(tuple(unit[\'phones\']))\n        self.digest=hashlib.sha256((self.root/\'index.json\').read_bytes()).hexdigest()\n    def has_pair(self,speaker,emotion,pair):\n        emotions=[emotion,\'neutral\'] if self.fallback==\'neutral\' and emotion!=\'neutral\' else [emotion]\n        for emo in emotions:\n            available=self.available.get((speaker,emo),set())\n            if tuple(pair) in available:return True\n            if self.stress_fallback and any(tuple(map(base,p))==tuple(map(base,pair)) for p in available):return True\n        return False\n    def pronounce_for_bank(self,speaker,emotion,text):\n        """Shortest supported dictionary/pause path; never invent or relabel audio.\n\n        Strong punctuation keeps a silence boundary. A comma may have no recorded\n        pause. Word boundaries may take existing silence transitions. Every departure\n        from the dictionary-first, punctuation-literal path is reported explicitly.\n        """\n        normalized,tokens=self.lexicon.pronunciation_tokens(text)\n        # Future feasibility depends only on the last phone, not the path history.\n        states={\'SIL\':(0,[\'SIL\'],[])}\n        for position,token in enumerate(tokens):\n            next_states={}\n            for last,(cost,phones,choices) in states.items():\n                options=[]\n                if \'punctuation\' in token:\n                    mark=token[\'punctuation\']\n                    options.append(([] if last==\'SIL\' else [\'SIL\'],0,[]))\n                    if mark==\',\' and last!=\'SIL\':\n                        options.append(([],1,[{\'pause_choice\':\'comma_without_recorded_silence\',\'token\':position}]))\n                else:\n                    for index,pron in enumerate(token[\'pronunciations\']):\n                        change=[] if index==0 else [{\'pronunciation_choice\':token[\'word\'],\'phones\':pron,\'token\':position}]\n                        options.append((pron,index,change))\n                        if last!=\'SIL\':\n                            options.append(([\'SIL\']+pron,index+2,change+[\n                                {\'pause_choice\':\'recorded_word_boundary\',\'before\':token[\'word\'],\'token\':position}]))\n                for tail,extra,changes in options:\n                    path=[last]+tail\n                    if not all(self.has_pair(speaker,emotion,pair) for pair in zip(path,path[1:])):continue\n                    end=path[-1];candidate=(cost+extra,phones+tail,choices+changes)\n                    if end not in next_states or candidate[0]<next_states[end][0]:next_states[end]=candidate\n            if not next_states:\n                label=token.get(\'word\',token.get(\'punctuation\'))\n                raise MissingUnit(f\'{speaker}/{emotion}: no supported dictionary/pause path at {label!r}\')\n            states=next_states\n        finished=[state for last,state in states.items() if last==\'SIL\' or self.has_pair(speaker,emotion,(last,\'SIL\'))]\n        if not finished:raise MissingUnit(f\'{speaker}/{emotion}: missing final silence transition\')\n        _,phones,choices=min(finished,key=lambda state:state[0])\n        if phones[-1]!=\'SIL\':phones=phones+[\'SIL\']\n        self.warnings.extend(choices)\n        return {\'normalized\':normalized,\'phones\':phones}\n    def select(self,speaker,emotion,pair,context=None):\n        for emo in ([emotion,\'neutral\'] if self.fallback==\'neutral\' and emotion!=\'neutral\' else [emotion]):\n            candidates=[u for u in self.index[\'units\'] if u[\'speaker_id\']==speaker and u[\'emotion\']==emo\n                        and (u.get(\'verified\') is True or self.audition_unverified) and tuple(u[\'phones\'])==tuple(pair)]\n            if not candidates and self.stress_fallback:\n                candidates=[u for u in self.index[\'units\'] if u[\'speaker_id\']==speaker and u[\'emotion\']==emo\n                    and (u.get(\'verified\') is True or self.audition_unverified) and tuple(map(base,u[\'phones\']))==tuple(map(base,pair))]\n                if candidates:\n                    self.warnings.append({\'stress_fallback\':pair})\n            if candidates:\n                if emo!=emotion:\n                    self.warnings.append({\'emotion_fallback\':emotion,\'to\':\'neutral\',\'pair\':pair})\n                # Acoustic allophones remain reviewed annotations. Prefer the\n                # same neighboring-phone context rather than blindly pooling them.\n                def context_penalty(u):\n                    if not context:return 0\n                    actual=u.get(\'context\',{})\n                    return sum(actual.get(k)!=value for k,value in context.items())\n                chosen=min(candidates,key=lambda u:(u[\'quality\'].get(\'clipped_fraction\',1),context_penalty(u),u[\'id\']))\n                if not chosen.get(\'verified\'):\n                    self.warnings.append({\'unverified_audition_unit\':chosen[\'id\'],\'pair\':pair})\n                return chosen\n        raise MissingUnit(f\'{speaker}/{emotion}: missing {pair[0]}|{pair[1]}\')\n    def load(self,u):\n        p=safe_path(self.root,u[\'file\'])\n        if hashlib.sha256(p.read_bytes()).hexdigest()!=u[\'sha256\']:\n            raise ValueError(\'Sample checksum mismatch: \'+u[\'file\'])\n        rate,audio=read_wav(p)\n        if self.rate is not None and rate!=self.rate:\n            raise ValueError(\'Mixed sample rates in bank\')\n        self.rate=rate\n        return audio\n    def speech(self,speaker,emotion,text):\n        p=self.pronounce_for_bank(speaker,emotion,text) if self.bank_pronunciations else self.lexicon.pronounce(text)\n        phones=p[\'phones\']\n        units=[self.select(speaker,emotion,pair,{\'previous\':phones[i-1] if i else \'SIL\',\n                    \'next\':phones[i+2] if i+2<len(phones) else \'SIL\'})\n               for i,pair in enumerate(zip(phones,phones[1:]))]\n        if not units:\n            return array(\'h\')\n        result=array(\'h\'); prev=None\n        for u in units:\n            audio=self.load(u)\n            h=round(self.rate*self.crossfade_ms/2000)\n            if u[\'left_handle\']<h or u[\'right_handle\']<h:\n                raise ValueError(\'Unit lacks requested overlap handles\')\n            # Read symmetric handles: the 2*h crossfade spans a shared phone anchor.\n            audio=audio[u[\'left_handle\']-h:len(audio)-u[\'right_handle\']+h]\n            if prev is None:\n                result.extend(audio[h:])\n            else:\n                n=2*h\n                if len(result)<n or len(audio)<n:\n                    raise ValueError(\'Unit too short for complementary crossfade\')\n                for j in range(n):\n                    alpha=j/(n-1) if n>1 else 1\n                    result[-n+j]=round(result[-n+j]*(1-alpha)+audio[j]*alpha)\n                result.extend(audio[n:])\n            prev=u\n        return result[:-h]\n    def event(self,speaker,emotion,name):\n        for emo in ([emotion,\'neutral\'] if self.fallback==\'neutral\' and emotion!=\'neutral\' else [emotion]):\n            candidates=[u for u in self.index[\'events\'] if u[\'speaker_id\']==speaker and\n                        u[\'emotion\']==emo and u[\'event\']==name and u.get(\'verified\') is True]\n            if candidates:\n                if emo!=emotion:\n                    self.warnings.append({\'event_emotion_fallback\':name,\'from\':emotion,\'to\':emo})\n                return self.load(candidates[0])\n        raise MissingUnit(f\'{speaker}/{emotion}: missing whole-sample event {name}\')\n    def render(self,speaker,text,output,cache=None):\n        self.warnings=[]\n        t0=time.perf_counter()\n        # Lexicon and aliases participate in cache identity, not just text and bank.\n        config={\'bank\':self.digest,\'speaker\':speaker,\'text\':text,\'crossfade\':self.crossfade_ms,\n            \'fallback\':self.fallback,\'stress\':self.stress_fallback,\'audition_unverified\':self.audition_unverified,\n             \'dictionary\':self.lexicon.dictionary_digest,\n             \'bank_pronunciations\':self.bank_pronunciations,\n            \'overrides\':self.lexicon.overrides,\'aliases\':self.lexicon.aliases}\n        key=hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()\n        cached=Path(cache)/(key+\'.wav\') if cache else None\n        if cached and cached.exists() and cached.with_suffix(\'.json\').exists():\n            metadata=json.loads(cached.with_suffix(\'.json\').read_text())\n            cached_bytes=cached.read_bytes()\n            if metadata.get(\'output_sha256\')==hashlib.sha256(cached_bytes).hexdigest():\n                Path(output).parent.mkdir(parents=True,exist_ok=True)\n                Path(output).write_bytes(cached_bytes)\n                return {**metadata,\'cache_hit\':True,\'wall_seconds\':time.perf_counter()-t0}\n        result=array(\'h\')\n        for kind,emotion,value in markup(text):\n            piece=self.speech(speaker,emotion,value) if kind==\'speech\' else self.event(speaker,emotion,value)\n            result.extend(piece)\n        if not result or not self.rate:\n            raise ValueError(\'Empty synthesis request\')\n        write_wav(output,self.rate,result)\n        metadata={\'cache_hit\':False,\'wall_seconds\':time.perf_counter()-t0,\'audio_seconds\':len(result)/self.rate,\n                  \'bytes\':Path(output).stat().st_size,\'warnings\':self.warnings,\'speaker\':speaker,\'cache_key\':key,\n                  \'output_sha256\':hashlib.sha256(Path(output).read_bytes()).hexdigest()}\n        if cached:\n            cached.parent.mkdir(parents=True,exist_ok=True)\n            tmp=cached.with_suffix(\'.wav.part\')\n            tmp.write_bytes(Path(output).read_bytes());tmp.replace(cached)\n            meta_tmp=cached.with_suffix(\'.json.part\')\n            meta_tmp.write_text(json.dumps(metadata,indent=2));meta_tmp.replace(cached.with_suffix(\'.json\'))\n        return metadata\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--bank\',required=True);p.add_argument(\'--lexicon\',required=True)\n    p.add_argument(\'--overrides\');p.add_argument(\'--speaker\',required=True)\n    p.add_argument(\'--text\',required=True);p.add_argument(\'--output\',required=True)\n    p.add_argument(\'--cache\');p.add_argument(\'--emotion-fallback\',choices=[\'error\',\'neutral\'],default=\'error\')\n    p.add_argument(\'--stress-fallback\',action=\'store_true\')\n    p.add_argument(\'--audition-unverified\',action=\'store_true\',help=\'Audition candidates only; never certifies a bank\')\n    p.add_argument(\'--bank-pronunciations\',action=\'store_true\',help=\'Use supported dictionary variants/recorded pauses and report every non-default choice\')\n    a=p.parse_args()\n    r=Renderer(a.bank,Lexicon.load(a.lexicon,a.overrides),emotion_fallback=a.emotion_fallback,stress_fallback=a.stress_fallback,\n               audition_unverified=a.audition_unverified,bank_pronunciations=a.bank_pronunciations)\n    print(json.dumps(r.render(a.speaker,a.text,a.output,a.cache),indent=2))\n\nif __name__==\'__main__\':main()\n', 'validate_pilots.py': '"""ASR pre-sort of generated words. Never certifies speaker, emotion or alignment."""\nfrom dataclasses import asdict\nimport hashlib\nimport argparse\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport sys\nfrom phonetics import normalize\n\nif not os.environ.get(\'T2_VALIDATION_LIBS_READY\'):\n    dirs=[str(p) for path in sys.path for p in Path(path).glob(\'nvidia/*/lib\') if Path(path).is_dir()]\n    env=dict(os.environ,T2_VALIDATION_LIBS_READY=\'1\')\n    env[\'LD_LIBRARY_PATH\']=\':\'.join(dirs+[env.get(\'LD_LIBRARY_PATH\',\'\')])\n    os.execve(sys.executable,[sys.executable,__file__,*sys.argv[1:]],env)\n\nfrom remote_prepare import load_asr_model\nroot=Path(__file__).resolve().parent\np=argparse.ArgumentParser();p.add_argument(\'--jobs\',type=Path,default=root/\'artifacts/pilot/pilot-jobs.json\')\np.add_argument(\'--output\',type=Path,default=root/\'artifacts/pilot/asr-validation.json\');a=p.parse_args()\njobs=json.loads(a.jobs.read_text())\nmodel=load_asr_model()\nrecords=[]\ndef words(s):return re.findall(r"[^\\W\\d_]+(?:\'[^\\W\\d_]+)?",normalize(s))\nfor job in jobs:\n    segments,_=model.transcribe(job[\'source\'],language=\'en\',beam_size=5,condition_on_previous_text=False)\n    segments=[asdict(s) for s in segments]\n    actual=\' \'.join(s[\'text\'].strip() for s in segments)\n    expected=re.sub(r\'\\([^)]*\\)\',\'\',job[\'text\'])\n    record={\'id\':job[\'id\'],\'expected\':expected,\'asr_text\':actual,\'exact_normalized_words\':words(actual)==words(expected),\n            \'segments\':segments,\'human_verified\':False,\'kind\':job[\'kind\'],\n            \'source_sha256\':hashlib.sha256(Path(job[\'source\']).read_bytes()).hexdigest()}\n    records.append(record);print(json.dumps({k:record[k] for k in (\'id\',\'asr_text\',\'exact_normalized_words\')}),flush=True)\na.output.write_text(json.dumps(records,indent=2))\nprint(\'========== DONE ==========\',flush=True)\n', 'remote_prepare.py': '"""Kaggle-only source download, decode, inventory and ASR candidate preparation."""\nfrom __future__ import annotations\nimport argparse\nimport csv\nfrom dataclasses import asdict\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\nimport wave\n\nROOT = Path(__file__).resolve().parent\nsys.path.insert(0,str(ROOT/\'prep\'))\nimport voice_lab as lab\nimport expression_support\n\ndef load_asr_model():\n    from faster_whisper import WhisperModel\n    lock_path=ROOT/\'toolchain-lock.json\'\n    if lock_path.exists():\n        from huggingface_hub import snapshot_download\n        spec=json.loads(lock_path.read_text())[\'asr_model\']\n        model=snapshot_download(spec[\'repo\'],revision=spec[\'revision\'])\n    else:\n        model=\'large-v3\'\n    return WhisperModel(model,device=\'cuda\',compute_type=\'float16\',device_index=0)\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--asr\', action=\'store_true\')\n    ap.add_argument(\'--all-asr\', action=\'store_true\')\n    ap.add_argument(\'--skip-prep\', action=\'store_true\', help=\'Reuse completed download/decode receipt\')\n    args = ap.parse_args()\n    if not str(ROOT).startswith(\'/kaggle/\'):\n        raise RuntimeError(\'Run preparation on the assigned Kaggle VM, not the laptop\')\n    out = ROOT/\'artifacts\'\n    out.mkdir(exist_ok=True)\n    raw, pcm = out/\'originals\', out/\'pcm24k\'\n    start = time.time()\n    if not args.skip_prep:\n        lab.download_assets(raw, workers=8)\n    rows = lab.build_review(out/\'review.csv\')\n    expression_support.build_expressive_review(lab,out/\'expressive_review.csv\')\n    details = []\n    for i,row in enumerate(rows if not args.skip_prep else []):\n        dest = pcm/row[\'path\']\n        if not dest.exists():\n            lab.decode_pcm(raw/row[\'path\'],dest)\n        with wave.open(str(dest)) as w:\n            details.append({\'path\':row[\'path\'],\'seconds\':w.getnframes()/w.getframerate(),\n                            \'pcm_bytes\':dest.stat().st_size,\'rate\':w.getframerate()})\n        if (i+1)%200 == 0:\n            print(\'decoded\',i+1,flush=True)\n    if not args.skip_prep:\n        (out/\'audio_inventory.json\').write_text(json.dumps(details,indent=2))\n        (out/\'download_decode_receipt.json\').write_text(json.dumps({\n            \'files\':len(details),\'original_bytes\':sum(p.stat().st_size for p in raw.rglob(\'*\') if p.is_file()),\n            \'pcm_bytes\':sum(d[\'pcm_bytes\'] for d in details),\'audio_seconds\':sum(d[\'seconds\'] for d in details),\n            \'seconds\':time.time()-start,\'revision\':lab.read_manifest()[\'revision\'],\n        },indent=2))\n    else:\n        receipt=json.loads((out/\'download_decode_receipt.json\').read_text())\n        assert receipt[\'files\']==len(rows), \'Incomplete source preparation\'\n    if args.asr:\n        model = load_asr_model()\n        targets = rows if args.all_asr else [r for r in rows if r[\'group\'] in (\'Announcer\',\'Training\')]\n        transcript_path = out/\'asr_candidates.jsonl\'\n        done = {json.loads(line)[\'path\'] for line in transcript_path.read_text().splitlines()} if transcript_path.exists() else set()\n        with transcript_path.open(\'a\',buffering=1) as f:\n            for row in targets:\n                if row[\'path\'] in done:\n                    continue\n                t0=time.time()\n                segments,info = model.transcribe(str(pcm/row[\'path\']),language=\'en\',beam_size=5,\n                    vad_filter=True,word_timestamps=True,condition_on_previous_text=False)\n                segments=[asdict(s) for s in segments]\n                result={\'path\':row[\'path\'],\'text\':\' \'.join(s[\'text\'].strip() for s in segments),\n                        \'segments\':segments,\'seconds\':time.time()-t0,\'speaker_id\':row[\'speaker_id\'],\n                        \'speaker_verified\':False,\'transcript_verified\':False,\n                        \'asr_model\':\'Systran/faster-whisper-large-v3\',\'language_probability\':info.language_probability}\n                f.write(json.dumps(result)+\'\\n\')\n                print(\'ASR\',row[\'path\'],result[\'text\'],flush=True)\n    print(\'========== DONE ==========\',flush=True)\n\nif __name__ == \'__main__\':\n    main()\n', 'benchmark_renderer.py': '"""Measure actual full-dictionary sample-only rendering, including cache behavior."""\nimport argparse\nimport cProfile\nimport json\nfrom pathlib import Path\nimport pstats\nimport time\nfrom phonetics import Lexicon\nfrom renderer import Renderer\n\nroot=Path(__file__).resolve().parent\np=argparse.ArgumentParser();p.add_argument(\'--bank\',type=Path,default=root/\'artifacts/stock-alignment/candidate-bank\')\np.add_argument(\'--speaker\',default=\'Announcer\');p.add_argument(\'--text\',default=\'This looks like a real battle.\')\np.add_argument(\'--tag\',default=\'baseline\');a=p.parse_args()\nassert str(root).startswith(\'/kaggle/\'), \'Remote numerical test only\'\nout=root/\'artifacts/renderer-benchmark\'/a.tag;out.mkdir(parents=True,exist_ok=True)\nt0=time.perf_counter()\nlex=Lexicon.load(root/\'artifacts/reports/cmudict.json\',root/\'pronunciation-overrides.json\')\nr=Renderer(a.bank,lex,audition_unverified=True)\nstartup=time.perf_counter()-t0\nprof=cProfile.Profile();prof.enable()\nfirst=r.render(a.speaker,a.text,out/\'profile.wav\')\nprof.disable();stats=pstats.Stats(prof)\ntop=sorted(stats.stats.items(),key=lambda x:x[1][3],reverse=True)[:12]\nuncached=[r.render(a.speaker,a.text,out/\'uncached.wav\')[\'wall_seconds\'] for _ in range(10)]\nr.render(a.speaker,a.text,out/\'cached.wav\',out/\'phrase-cache\')\ncached=[r.render(a.speaker,a.text,out/\'cached.wav\',out/\'phrase-cache\')[\'wall_seconds\'] for _ in range(10)]\nreport={\'text\':a.text,\'bank\':str(a.bank),\'voice_quality\':\'unverified candidate audition; not a production verdict\',\n        \'startup_seconds\':startup,\'audio_seconds\':first[\'audio_seconds\'],\'output_bytes\':first[\'bytes\'],\n        \'uncached_seconds\':uncached,\'cached_seconds\':cached,\'uncached_mean_seconds\':sum(uncached)/10,\n        \'cached_mean_seconds\':sum(cached)/10,\'dictionary_words\':len(lex.entries),\n        \'profile\':[{\'function\':f\'{key[0]}:{key[1]}:{key[2]}\',\'cumulative_seconds\':value[3]} for key,value in top]}\n(out/\'result.json\').write_text(json.dumps(report,indent=2))\nprint(json.dumps(report,indent=2))\nprint(\'========== DONE ==========\',flush=True)\n', 'render_auditions.py': '"""Real sample-only smoke with unverified units explicitly restricted to audition."""\nimport json\nimport argparse\nfrom pathlib import Path\nfrom renderer import Renderer,MissingUnit\nfrom phonetics import Lexicon,UnknownWords\n\nroot=Path(__file__).resolve().parent\np=argparse.ArgumentParser();p.add_argument(\'--bank\',type=Path,default=root/\'artifacts/stock-alignment/candidate-bank\')\np.add_argument(\'--speaker\',default=\'Announcer\')\np.add_argument(\'--bank-pronunciations\',action=\'store_true\')\np.add_argument(\'--tag\',default=\'stitch-auditions\');a=p.parse_args()\nlex=Lexicon.load(root/\'artifacts/reports/cmudict.json\',root/\'pronunciation-overrides.json\')\nif Path(a.tag).name!=a.tag:raise ValueError(\'Directory-safe tag required\')\nout=root/\'artifacts\'/a.tag;out.mkdir(parents=True,exist_ok=True)\nbank=a.bank\nrenderer=Renderer(bank,lex,audition_unverified=True,bank_pronunciations=a.bank_pronunciations)\nresults=[]\nfor i,text in enumerate((\'Game over. Prepare yourselves!\', \'Match begins now. Game over.\',\n                         \'You are out of bounds! Prepare yourselves.\', \'This looks like a real battle.\',\n                         \'Welcome to the server, Cody!\', \'Welcome to the server, Cody. Defend the generator!\')):\n    try:\n        path=out/f\'announcer_{i:02d}.wav\'\n        metadata=renderer.render(a.speaker,text,path,out/\'phrase-cache\')\n        metadata.update({\'text\':text,\'file\':str(path),\'verified\':False,\'mode\':\'unverified candidate audition only\'})\n        results.append(metadata)\n        print(json.dumps({\'text\':text,\'audio_seconds\':metadata[\'audio_seconds\'],\n                          \'wall_seconds\':metadata[\'wall_seconds\'],\'warnings\':len(metadata[\'warnings\'])}),flush=True)\n    except (MissingUnit,UnknownWords,ValueError) as e:\n        results.append({\'text\':text,\'error\':str(e),\'verified\':False})\n        print(\'EXPLICIT REFUSAL:\',text,str(e),flush=True)\n(out/\'results.json\').write_text(json.dumps(results,indent=2))\n(out/\'phrase-candidates.json\').write_text(json.dumps([\n    {\'file\':r[\'file\'],\'sha256\':r[\'output_sha256\'],\'text\':r[\'text\'],\'speaker_id\':a.speaker,\'verified\':False}\n    for r in results if \'file\' in r],indent=2))\nprint(\'========== DONE ==========\',flush=True)\n', 'test_preparation.py': '"""Remote fixture tests for alignment staging, stop preservation and game packaging."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport tempfile\nimport unittest\nfrom unittest.mock import patch\nimport zipfile\nimport pilot\nfrom alignment import stage,convert\nfrom bank import extract,write_wav,read_wav\nfrom events import export_events\nfrom phonetics import Lexicon,select_gaps,transitions\nfrom package_game import package\nfrom build_bank import merge_banks\nfrom gap_jobs import plan_incremental,immutable_json\n\nclass PreparationTests(unittest.TestCase):\n    def setUp(self):\n        self.temp=tempfile.TemporaryDirectory();self.root=Path(self.temp.name)\n        self.wav=self.root/\'test.wav\';write_wav(self.wav,16000,[0]*16000)\n    def tearDown(self):self.temp.cleanup()\n    def test_staging_unknown_refused(self):\n        with self.assertRaises(ValueError):stage([{\'text\':\'unknown\',\'source\':str(self.wav),\'speaker_id\':\'one\'}],Lexicon({}),self.root/\'corpus\')\n    def test_staging_preserves_speaker(self):\n        lex=Lexicon({\'one\':[[\'W\',\'AH1\',\'N\']]})\n        data=stage([{\'text\':\'one\',\'source\':str(self.wav),\'speaker_id\':x} for x in (\'female_candidate\',\'male_candidate\')],lex,self.root/\'corpus\')\n        self.assertEqual(len(data),2)\n    def test_stop_release_remains_intact(self):\n        samples=[0]*16000\n        samples[5000:5100]=[25000]*100\n        write_wav(self.wav,16000,samples)\n        d={\'source\':str(self.wav),\'speaker_id\':\'synthetic\',\'emotion\':\'neutral\',\n           **{k:True for k in (\'speaker_verified\',\'transcript_verified\',\'alignment_verified\',\'emotion_verified\')},\n           \'phones\':[{\'label\':\'SIL\',\'start\':0,\'end\':0.2},\n                     {\'label\':\'P\',\'start\':0.2,\'end\':0.4,\'anchor\':0.25,\'closure_verified\':True,\'protected_release\':[0.3,0.34]},\n                     {\'label\':\'AH1\',\'start\':0.4,\'end\':0.8},{\'label\':\'SIL\',\'start\':0.8,\'end\':1}]}\n        units=extract(d,self.root/\'bank\')\n        self.assertTrue(all(u[\'verified\'] for u in units))\n        release=next(u for u in units if u[\'phones\'][0]==\'P\')\n        self.assertEqual(sum(x==25000 for x in read_wav(self.root/\'bank\'/release[\'file\'])[1]),100)\n    def test_mfa_textgrid_candidate(self):\n        import textgrid\n        tg=textgrid.TextGrid(minTime=0,maxTime=1)\n        tier=textgrid.IntervalTier(name=\'phones\',minTime=0,maxTime=1)\n        tier.add(0,0.2,\'\');tier.add(0.2,0.8,\'AH1\');tier.add(0.8,1,\'\')\n        tg.append(tier);p=self.root/\'test.TextGrid\';tg.write(str(p))\n        data=convert(p,{\'source\':str(self.wav),\'speaker_id\':\'candidate\',\'emotion\':\'neutral\',\'text\':\'a\'})\n        self.assertEqual([x[\'label\'] for x in data[\'phones\']],[\'SIL\',\'AH1\',\'SIL\'])\n        self.assertFalse(data[\'alignment_verified\'])\n    def test_event_carrier_refused(self):\n        review={\'source\':str(self.wav),\'speaker_id\':\'one\',\'event\':\'laugh\',\'start\':0.1,\'end\':0.5,\n                \'speaker_verified\':True,\'event_verified\':True,\'boundaries_verified\':True,\'event_type\':\'wordless\',\n                \'source_sha256\':hashlib.sha256(self.wav.read_bytes()).hexdigest()}\n        with self.assertRaises(ValueError):export_events([review],self.root/\'bank\')\n        review[\'contains_carrier_speech\']=False\n        self.assertEqual(len(export_events([review],self.root/\'bank\')),1)\n    def test_unverified_event_skipped(self):\n        self.assertEqual(export_events([{\'source\':str(self.wav)}],self.root/\'bank\'),[])\n    def test_unverified_pack_refused(self):\n        with self.assertRaises(ValueError):package([{\'file\':str(self.wav),\'verified\':False}],self.root/\'test.vl2\')\n    def test_pack_local_wav_and_allowlist(self):\n        p=self.root/\'test.vl2\'\n        entries=package([{\'file\':str(self.wav),\'verified\':True,\'text\':\'synthetic fixture\',\'speaker_id\':\'one\',\n                          \'sha256\':hashlib.sha256(self.wav.read_bytes()).hexdigest()}],p)\n        with zipfile.ZipFile(p) as z:\n            client=z.read(\'scripts/t2vl_client.cs\').decode()\n            self.assertIn(\'alxCreateSource(AudioGui, %path)\',client)\n            self.assertEqual(z.read(entries[0][\'file\']),self.wav.read_bytes())\n            self.assertEqual(z.testzip(),None)\n        self.assertIn(\'isSuperAdmin\',p.with_suffix(\'.server.cs\').read_text())\n    def test_changed_reviewed_phrase_refused(self):\n        with self.assertRaisesRegex(ValueError,\'checksum\'):\n            package([{\'file\':str(self.wav),\'verified\':True,\'sha256\':\'0\'*64}],self.root/\'test.vl2\')\n    def assert_permission_rejected_before_model_import(self,permission):\n        if permission is not None:\n            path=self.root/\'artifacts/permissions/rights-attestation.json\'\n            path.parent.mkdir(parents=True,exist_ok=True)\n            path.write_text(json.dumps(permission))\n        original_import=__import__\n        def guarded_import(name,*args,**kwargs):\n            if name.split(\'.\')[0] in (\'numpy\',\'soundfile\',\'torch\',\'infer\',\'breeze_infer\',\'models\'):\n                self.fail(\'Model dependencies imported before permission rejection: \'+name)\n            return original_import(name,*args,**kwargs)\n        with patch.object(pilot,\'ROOT\',self.root),patch(\'builtins.__import__\',side_effect=guarded_import):\n            with self.assertRaises(PermissionError):pilot.generate([\'Announcer\'],allow_provisional=True)\n    def test_absent_permission_refused_before_model_import(self):\n        self.assert_permission_rejected_before_model_import(None)\n    def test_generation_output_tags_are_isolated(self):\n        with patch.object(pilot,\'ROOT\',self.root):\n            self.assertEqual(pilot.output_folder(),self.root/\'artifacts/pilot\')\n            self.assertNotEqual(pilot.output_folder(\'gpu0\'),pilot.output_folder(\'gpu1\'))\n            for tag in (\'\',\'..\',\'.\',\'../outside\',\'/absolute\'):\n                with self.assertRaises(ValueError):pilot.output_folder(tag)\n    def test_fast_generation_is_opt_in(self):\n        import inspect\n        self.assertIs(inspect.signature(pilot.generate).parameters[\'fast\'].default,False)\n    def test_generation_output_symlink_escape_refused(self):\n        artifacts=self.root/\'artifacts\';artifacts.mkdir()\n        (artifacts/\'escape\').symlink_to(self.root,target_is_directory=True)\n        with patch.object(pilot,\'ROOT\',self.root),self.assertRaises(ValueError):pilot.output_folder(\'escape\')\n    def test_unconfirmed_permission_refused_before_model_import(self):\n        self.assert_permission_rejected_before_model_import({\n            \'performer_consent_and_recording_rights_confirmed\':False,\'voice_ids\':[\'Announcer\']})\n    def test_wrong_voice_permission_refused_before_model_import(self):\n        self.assert_permission_rejected_before_model_import({\n            \'performer_consent_and_recording_rights_confirmed\':True,\'voice_ids\':[\'Male1\']})\n    def test_merge_preserves_conflicting_old_alignment_ids(self):\n        banks=[]\n        for i in range(2):\n            folder=self.root/f\'bank{i}\';folder.mkdir();write_wav(folder/\'unit.wav\',16000,[i]*100)\n            unit={\'id\':\'same-old-source-and-phone-index\',\'file\':\'unit.wav\',\'speaker_id\':\'one\',\n                  \'emotion\':\'neutral\',\'phones\':[\'SIL\',\'AH1\'],\'verified\':False,\n                  \'sha256\':hashlib.sha256((folder/\'unit.wav\').read_bytes()).hexdigest(),\n                  \'start_seconds\':i/100,\'end_seconds\':0.1,\'left_handle\':5,\'right_handle\':5}\n            (folder/\'index.json\').write_text(json.dumps({\'units\':[unit],\'events\':[]}));banks.append(folder)\n        self.assertEqual(merge_banks(banks,self.root/\'strict\'),[])\n        units=merge_banks([*banks,banks[0]],self.root/\'merged\',include_candidates=True)\n        self.assertEqual(len(units),2)\n        self.assertEqual(len({u[\'id\'] for u in units}),2)\n        self.assertTrue(all(u[\'verified\'] is False for u in units))\n        self.assertNotEqual((self.root/\'merged\'/units[0][\'file\']).read_bytes(),\n                            (self.root/\'merged\'/units[1][\'file\']).read_bytes())\n        (banks[0]/\'unit.wav\').write_bytes(b\'changed\')\n        with self.assertRaisesRegex(ValueError,\'checksum\'):\n            merge_banks(banks,self.root/\'broken\',include_candidates=True)\n    def gap_fixture(self):\n        import shutil\n        folder=self.root/\'artifacts/references-provisional\';folder.mkdir(parents=True,exist_ok=True)\n        for speaker in (\'Male1\',\'Male2\'):\n            ref=folder/(speaker+\'.wav\');shutil.copy2(self.wav,ref)\n            ref.with_suffix(\'.txt\').write_text(\'one\')\n            ref.with_suffix(\'.json\').write_text(json.dumps({\'sha256\':hashlib.sha256(ref.read_bytes()).hexdigest(),\n                                                          \'reference_verified\':False}))\n        bank=self.root/\'gap-bank\';bank.mkdir(exist_ok=True)\n        phones=[\'SIL\',\'W\',\'AH1\',\'N\',\'SIL\']\n        units=[{\'id\':str(i),\'speaker_id\':\'Male1\',\'emotion\':\'neutral\',\'phones\':list(pair),\'verified\':False}\n               for i,pair in enumerate(zip(phones,phones[1:]))]\n        (bank/\'index.json\').write_text(json.dumps({\'units\':units,\'events\':[]}))\n        return bank,Lexicon({\'one\':[[\'W\',\'AH1\',\'N\']],\'two\':[[\'T\',\'UW1\']]})\n    def test_gap_candidate_coverage_is_explicit(self):\n        bank,lex=self.gap_fixture()\n        strict,report=plan_incremental(self.root,bank,lex,[\'Male1\'],[\'neutral\'],2,sentences=[\'one\',\'two\'])\n        audition,private=plan_incremental(self.root,bank,lex,[\'Male1\'],[\'neutral\'],2,True,sentences=[\'one\',\'two\'])\n        self.assertEqual(len(strict),2);self.assertEqual([x[\'text\'] for x in audition],[\'two\'])\n        self.assertFalse(report[\'candidate_coverage\']);self.assertTrue(private[\'candidate_coverage\'])\n        self.assertTrue(all(x[\'verified\'] is False and x[\'reference_verified\'] is False for x in audition))\n    def test_gap_voice_style_separation(self):\n        bank,lex=self.gap_fixture()\n        jobs,report=plan_incremental(self.root,bank,lex,[\'Male1\',\'Male2\'],[\'neutral\',\'angry\'],2,True,sentences=[\'one\',\'two\'])\n        self.assertEqual(report[\'groups\'][\'Male1/neutral\'][\'supported_phone_paths\'],1)\n        for key in (\'Male2/neutral\',\'Male1/angry\',\'Male2/angry\'):\n            self.assertEqual(report[\'groups\'][key][\'supported_phone_paths\'],0)\n            self.assertEqual(report[\'groups\'][key][\'selected_sentences\'],2)\n        self.assertEqual(len({x[\'id\'] for x in jobs}),len(jobs))\n        self.assertTrue(all(x[\'instruction\'] for x in jobs if x[\'emotion\']==\'angry\'))\n    def test_gap_job_identity_and_source_are_stable(self):\n        bank,lex=self.gap_fixture()\n        args=(self.root,bank,lex,[\'Male1\'],[\'neutral\'],2,True)\n        first,_=plan_incremental(*args,sentences=[\'one\',\'two\'])\n        second,_=plan_incremental(*args,sentences=[\'one\',\'two\'])\n        self.assertEqual(first,second)\n        self.assertEqual(Path(first[0][\'source\']).name,first[0][\'id\']+\'.wav\')\n    def test_gap_previous_intact_attempt_is_not_repeated(self):\n        bank,lex=self.gap_fixture();folder=self.root/\'artifacts/pilot\';folder.mkdir()\n        old={\'id\':\'previous\',\'kind\':\'speech\',\'speaker_id\':\'Male1\',\'emotion\':\'neutral\',\'text\':\'two\',\n             \'source\':str(self.wav),\'sha256\':hashlib.sha256(self.wav.read_bytes()).hexdigest()}\n        (folder/\'pilot-jobs.json\').write_text(json.dumps([old]))\n        jobs,report=plan_incremental(self.root,bank,lex,[\'Male1\'],[\'neutral\'],2,True,sentences=[\'one\',\'two\'])\n        self.assertEqual(jobs,[])\n        self.assertEqual(report[\'groups\'][\'Male1/neutral\'][\'existing_attempts_excluded\'],1)\n        self.assertTrue(report[\'groups\'][\'Male1/neutral\'][\'predicted_uncovered_after\'])\n    def test_gap_changed_reference_is_refused(self):\n        bank,lex=self.gap_fixture();ref=self.root/\'artifacts/references-provisional/Male1.wav\'\n        ref.write_bytes(ref.read_bytes()+b\'changed\')\n        with self.assertRaisesRegex(ValueError,\'Reference checksum\'):\n            plan_incremental(self.root,bank,lex,[\'Male1\'],[\'neutral\'],sentences=[\'two\'])\n    def test_gap_unknown_workload_is_reported(self):\n        bank,lex=self.gap_fixture()\n        jobs,report=plan_incremental(self.root,bank,lex,[\'Male1\'],[\'neutral\'],sentences=[\'xzq\'])\n        self.assertEqual(jobs,[]);self.assertEqual(report[\'unknown_workload_items\'][0][\'unknown\'],[\'xzq\'])\n    def test_gap_plan_changed_bytes_are_not_overwritten(self):\n        path=self.root/\'plan.json\';immutable_json(path,[{\'id\':\'one\'}]);before=path.read_bytes()\n        immutable_json(path,[{\'id\':\'one\'}])\n        with self.assertRaisesRegex(ValueError,\'overwrite\'):immutable_json(path,[{\'id\':\'two\'}])\n        self.assertEqual(path.read_bytes(),before)\n    def test_workload_priority_preserves_default_but_favors_shared_pairs(self):\n        lex=Lexicon({\'rare\':[[\'R\',\'EH1\',\'Z\',\'UW1\',\'V\',\'OY1\',\'B\',\'IH0\']],\n                     \'common\':[[\'K\',\'AA1\',\'M\',\'AH0\',\'N\']]})\n        sentences=[\'rare\',\'common\']\n        target=set().union(*(set(transitions(lex.pronounce(s)[\'phones\'])) for s in sentences))\n        weights={pair:10 for pair in transitions(lex.pronounce(\'common\')[\'phones\'])}\n        original=select_gaps(sentences,lex,target,max_sentences=1)\n        explicit_default=select_gaps(sentences,lex,target,max_sentences=1,pair_weights=None)\n        weighted=select_gaps(sentences,lex,target,max_sentences=1,pair_weights=weights)\n        self.assertEqual(original,explicit_default)\n        self.assertEqual(original[\'selected\'][0][\'text\'],\'rare\')\n        self.assertEqual(weighted[\'selected\'][0][\'text\'],\'common\')\n        self.assertFalse(weighted[\'selected\'][0][\'audio_verified\'])\n    def test_workload_priority_excludes_old_attempts_and_keeps_style_separate(self):\n        bank,lex=self.gap_fixture();folder=self.root/\'artifacts/pilot\';folder.mkdir()\n        old={\'id\':\'previous\',\'kind\':\'speech\',\'speaker_id\':\'Male1\',\'emotion\':\'neutral\',\'text\':\'two\',\n             \'source\':str(self.wav),\'sha256\':hashlib.sha256(self.wav.read_bytes()).hexdigest()}\n        (folder/\'pilot-jobs.json\').write_text(json.dumps([old]))\n        jobs,report=plan_incremental(self.root,bank,lex,[\'Male1\'],[\'neutral\',\'angry\'],2,True,\n                                    sentences=[\'one\',\'two\'],workload_priority=True)\n        self.assertEqual(report[\'selection\'],\'workload_frequency\')\n        self.assertEqual(report[\'groups\'][\'Male1/neutral\'][\'selected_sentences\'],0)\n        self.assertEqual(report[\'groups\'][\'Male1/angry\'][\'selected_sentences\'],2)\n        self.assertTrue(all(j[\'emotion\']==\'angry\' and not j[\'verified\'] for j in jobs))\n\nif __name__==\'__main__\':\n    if not str(Path(__file__).resolve()).startswith(\'/kaggle/\'):\n        raise RuntimeError(\'Run tests on assigned remote compute\')\n    suite=unittest.defaultTestLoader.loadTestsFromTestCase(PreparationTests)\n    result=unittest.TextTestRunner(verbosity=2).run(suite)\n    Path(\'test-preparation-results.json\').write_text(json.dumps({\'tests\':result.testsRun,\n        \'failures\':len(result.failures),\'errors\':len(result.errors)},indent=2))\n    raise SystemExit(0 if result.wasSuccessful() else 1)\n', 'companion.py': '"""Opt-in local native-game mailbox. Sample-only; no network listener or eval.\n\nThis is an implementation awaiting remote fixtures and actual native acceptance.\nInstalling writes only into the explicitly supplied game base directory. Nothing\nis installed or started merely by importing this module.\n"""\nfrom __future__ import annotations\nimport argparse\nimport hashlib\nimport hmac\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport secrets\nimport signal\nimport threading\nimport time\nfrom phonetics import Lexicon\nfrom renderer import Renderer\n\nVERSION=\'1\'\nVOICES=[*(f\'Male{i}\' for i in range(1,6)),*(f\'Fem{i}\' for i in range(1,6)),\n        *(f\'Derm{i}\' for i in range(1,4)),\'Bot1\',\'Announcer\',\'TrainingFemale\',\'BriefingMale\']\nEMOTIONS={\'neutral\',\'angry\',\'sad\',\'excited\',\'calm\'}\nMAX_TEXT=240\nMAX_LOG=4194304\nMAX_OUTPUT=134217728\nMAILBOX=\'prefs/t2vl-live\'\n\ndef owned(base,relative):\n    base=Path(base).resolve();candidate=base/relative\n    if candidate.is_symlink():raise ValueError(\'Symlink mailbox/output refused\')\n    path=candidate.resolve()\n    if base not in path.parents:raise ValueError(\'Mailbox/output escapes game base\')\n    return path\n\ndef atomic(path,data):\n    path=Path(path)\n    if path.is_symlink():raise ValueError(\'Symlink destination refused\')\n    path.parent.mkdir(parents=True,exist_ok=True)\n    temporary=path.with_name(path.name+\'.\'+secrets.token_hex(8)+\'.part\')\n    try:\n        fd=os.open(temporary,os.O_WRONLY|os.O_CREAT|os.O_EXCL,0o600)\n        with os.fdopen(fd,\'wb\') as stream:\n            stream.write(data);stream.flush();os.fsync(stream.fileno())\n        temporary.replace(path)\n    finally:\n        temporary.unlink(missing_ok=True)\n\ndef save(path,data):atomic(path,json.dumps(data,indent=2).encode())\n\ndef digest(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\ndef install(base,bank,lexicon,overrides=None,audition_unverified=False,bank_pronunciations=False):\n    base=Path(base).resolve()\n    if not base.is_dir():raise ValueError(\'Supply an EXISTING native game base directory\')\n    mailbox=owned(base,MAILBOX)\n    if mailbox.exists():raise ValueError(\'Mailbox already exists; refusing to replace identity, pending requests or evidence\')\n    client_script=owned(base,\'scripts/t2vl_live_client.cs\')\n    if client_script.exists():raise ValueError(\'Client script already exists; refusing to overwrite it\')\n    index=Path(bank).resolve()/\'index.json\';inventory=json.loads(index.read_text())\n    voices=sorted(set(inventory[\'speakers\'])&set(VOICES))\n    if not voices:raise ValueError(\'No supported voices in bank\')\n    config={\'version\':VERSION,\'session\':secrets.token_hex(8),\'token\':secrets.token_hex(32),\n            \'bank\':str(Path(bank).resolve()),\'bank_sha256\':digest(index),\n            \'lexicon\':str(Path(lexicon).resolve()),\'lexicon_sha256\':digest(lexicon),\n            \'overrides\':str(Path(overrides).resolve()) if overrides else None,\n            \'overrides_sha256\':digest(overrides) if overrides else None,\'voices\':voices,\n            \'audition_unverified\':bool(audition_unverified),\'bank_pronunciations\':bool(bank_pronunciations)}\n    mailbox.mkdir(parents=True,mode=0o700)\n    owned(base,\'audio/t2vl-live\').mkdir(parents=True,exist_ok=True)\n    atomic(mailbox/\'requests.tsv\',b\'\')\n    atomic(mailbox/\'response.tsv\',b\'\')\n    save(mailbox/\'companion.json\',config)\n    save(mailbox/\'state.json\',{\'session\':config[\'session\'],\'offset\':0,\'last_seq\':0,\'pending\':None})\n    code=\'// PRIVATE local configuration. Never distribute the token.\\n\'\n    for name,value in [(\'Version\',VERSION),(\'Session\',config[\'session\']),(\'Token\',config[\'token\']),(\'Bank\',config[\'bank_sha256\'])]:\n        code+=f\'$T2VLLive::{name} = "{value}";\\n\'\n    for voice in voices:code+=f\'$T2VLLive::Voice["{voice}"] = true;\\n\'\n    atomic(mailbox/\'config.cs\',code.encode())\n    atomic(client_script,CLIENT.encode())\n    # Does not execute/install on a server; explicit server loading remains separate.\n    server=SERVER.replace(\'@BANK@\',config[\'bank_sha256\']).replace(\'@VOICES@\',\' \'.join(voices))\n    server+=\'\\n\'+CLIENT[CLIENT.index(\'function T2VLLiveTextOK\'):]\n    atomic(mailbox/\'server.cs\',server.encode())\n    save(mailbox/\'installation.json\',{\'version\':VERSION,\'bank_sha256\':config[\'bank_sha256\'],\n         \'candidate_audition_only\':audition_unverified,\'native_client_test\':\'NOT RUN\',\n         \'production_license_clearance\':\'NOT ESTABLISHED\'})\n    return {\'base\':str(base),\'version\':VERSION,\'bank_sha256\':config[\'bank_sha256\'],\n            \'candidate_audition_only\':audition_unverified,\'native_client_test\':\'NOT RUN\'}\n\nclass Companion:\n    def __init__(self,base,renderer=None,clock=time.time):\n        self.base=Path(base).resolve();self.folder=owned(self.base,MAILBOX);self.clock=clock\n        self.config=json.loads(owned(self.base,MAILBOX+\'/companion.json\').read_text())\n        self.state=json.loads(owned(self.base,MAILBOX+\'/state.json\').read_text())\n        c=self.config\n        if c[\'version\']!=VERSION or self.state[\'session\']!=c[\'session\']:raise ValueError(\'Protocol/session mismatch\')\n        if not re.fullmatch(r\'[0-9a-f]{16}\',c[\'session\']) or not re.fullmatch(r\'[0-9a-f]{64}\',c[\'token\']):\n            raise ValueError(\'Invalid local credential/session configuration\')\n        for path,expected in [(Path(c[\'bank\'])/\'index.json\',c[\'bank_sha256\']),\n                              (c[\'lexicon\'],c[\'lexicon_sha256\'])]+([(c[\'overrides\'],c[\'overrides_sha256\'])] if c[\'overrides\'] else []):\n            if digest(path)!=expected:raise ValueError(\'Bank/dictionary configuration changed; do not reuse this session\')\n        self.renderer=renderer if renderer is not None else Renderer(c[\'bank\'],Lexicon.load(c[\'lexicon\'],c[\'overrides\']),\n            audition_unverified=c[\'audition_unverified\'],bank_pronunciations=c[\'bank_pronunciations\'])\n    def persist(self):save(owned(self.base,MAILBOX+\'/state.json\'),self.state)\n    def publish(self,seq,status,reason,milliseconds=0):\n        row=\'\\t\'.join((VERSION,self.config[\'session\'],str(seq),status,reason,str(milliseconds)))+\'\\n\'\n        atomic(owned(self.base,MAILBOX+\'/response.tsv\'),row.encode())\n    def error(self,reason,detail=\'\'):\n        save(owned(self.base,MAILBOX+\'/last-error.json\'),{\'reason\':reason,\'detail\':str(detail)[:300],\n             \'at\':self.clock(),\'native_playback_verified\':False})\n        return reason\n    def process_once(self):\n        pending=self.state[\'pending\']\n        if pending and self.clock()>pending[\'deadline\']:\n            self.publish(pending[\'seq\'],\'ERROR\',\'ack_timeout\')\n            self.state[\'pending\']=None;self.persist();self.error(\'ack_timeout\')\n        path=owned(self.base,MAILBOX+\'/requests.tsv\');stat=path.stat()\n        if stat.st_size>MAX_LOG:raise ValueError(\'Request-log budget exhausted; preserve/archive before resetting\')\n        if stat.st_size<self.state[\'offset\']:raise ValueError(\'Request log truncated; refusing unsafe replay\')\n        identity=[stat.st_dev,stat.st_ino]\n        if self.state.get(\'log_identity\',identity)!=identity:raise ValueError(\'Request log replaced; refusing unsafe replay\')\n        self.state[\'log_identity\']=identity\n        with path.open(\'rb\') as stream:\n            stream.seek(self.state[\'offset\']);line=stream.readline(2049)\n            if not line:return \'idle\'\n            if len(line)>2048:raise ValueError(\'Oversized mailbox line; preserve log and inspect\')\n            if not line.endswith(b\'\\n\'):return \'partial\'\n            self.state[\'offset\']=stream.tell()\n        # Offset is committed before acting: interruption may lose a request, never\n        # silently replay speech. Recovery/timeout is explicit, not exactly-once audio.\n        self.persist()\n        try:fields=line.decode(\'utf-8\').rstrip(\'\\n\').split(\'\\t\')\n        except UnicodeDecodeError:return self.error(\'invalid_encoding\')\n        if len(fields)!=8:return self.error(\'invalid_fields\')\n        version,token,session,kind,seq,speaker,emotion,text=fields\n        if (not re.fullmatch(r\'[0-9a-f]{64}\',token) or not re.fullmatch(r\'[0-9a-f]{16}\',session) or\n                not hmac.compare_digest(token,self.config[\'token\']) or not hmac.compare_digest(session,self.config[\'session\'])):\n            return self.error(\'authentication_failed\')\n        if not re.fullmatch(r\'[1-9][0-9]{0,8}\',seq):return self.error(\'invalid_sequence\')\n        seq=int(seq);pending=self.state[\'pending\']\n        if version!=VERSION:\n            if not pending:self.publish(seq,\'ERROR\',\'protocol_mismatch\')\n            return self.error(\'protocol_mismatch\')\n        if kind==\'A\':\n            if not pending or seq!=pending[\'seq\']:return self.error(\'stale_ack\')\n            if text not in (\'playback_timer_elapsed\',\'engine_failed\',\'cancelled\'):return self.error(\'invalid_ack\')\n            self.publish(seq,\'DONE\',text);self.state[\'pending\']=None;self.persist()\n            return \'acknowledged\'\n        if kind!=\'S\':return self.error(\'invalid_operation\')\n        if pending:return self.error(\'busy\',\'One outstanding request; no response or speech was overwritten\')\n        if seq<=self.state[\'last_seq\']:return self.error(\'duplicate_sequence\')\n        self.state[\'last_seq\']=seq;self.persist()\n        reason=None\n        if speaker not in self.config[\'voices\'] or emotion not in EMOTIONS:reason=\'unsupported_voice_style\'\n        elif not text.strip() or len(text)>MAX_TEXT or any(ord(c)<32 for c in text) or \'[\' in text or \']\' in text:\n            reason=\'invalid_text\'\n        if reason:\n            self.publish(seq,\'ERROR\',reason);return self.error(reason)\n        output=owned(self.base,f\'audio/t2vl-live/{session}-{seq}.wav\')\n        temporary=output.with_suffix(\'.part.wav\')\n        if output.exists() or temporary.exists() or temporary.is_symlink():\n            self.publish(seq,\'ERROR\',\'output_exists\');return self.error(\'output_exists\',\'Existing bytes preserved\')\n        used=sum(p.stat().st_size for p in output.parent.glob(session+\'-*.wav\') if p.is_file())\n        try:\n            if used>=MAX_OUTPUT:raise ValueError(\'Output budget exhausted; archive owned outputs explicitly\')\n            rendered=text if emotion==\'neutral\' else f\'[{emotion}]{text}[/{emotion}]\'\n            result=self.renderer.render(speaker,rendered,temporary)\n            milliseconds=round(result[\'audio_seconds\']*1000)\n            if not 1<=milliseconds<=30000 or used+temporary.stat().st_size>MAX_OUTPUT:\n                raise ValueError(\'Rendered output exceeds duration/storage budget\')\n            temporary.replace(output)\n            self.state[\'pending\']={\'seq\':seq,\'deadline\':self.clock()+milliseconds/1000+30}\n            self.persist();self.publish(seq,\'READY\',\'none\',milliseconds)\n            # Warnings retain missing/fallback/candidate provenance; no token/text.\n            save(owned(self.base,MAILBOX+\'/last-render.json\'),{**result,\'seq\':seq,\'text_stored\':False,\n                 \'native_playback_verified\':False})\n            return \'ready\'\n        except ValueError as exc:\n            self.publish(seq,\'ERROR\',\'render_refused\');return self.error(\'render_refused\',exc)\n        finally:temporary.unlink(missing_ok=True)\n\nCLIENT=r\'\'\'// PRIVATE, opt-in local companion. Not native-tested. Never eval chat text.\nfunction T2VLLiveDigits(%value)\n{\n   if (strlen(%value) < 1 || strlen(%value) > 9) return false;\n   for (%i=0; %i<strlen(%value); %i++)\n      if (strstr("0123456789", getSubStr(%value,%i,1)) == -1) return false;\n   return (%value > 0);\n}\nfunction T2VLLiveAppend(%kind,%seq,%speaker,%emotion,%text)\n{\n   %file = new FileObject();\n   %ok = %file.openForAppend("prefs/t2vl-live/requests.tsv");\n   if (%ok) {\n      %file.writeLine($T2VLLive::Version TAB $T2VLLive::Token TAB $T2VLLive::Session TAB\n                      %kind TAB %seq TAB %speaker TAB %emotion TAB %text);\n      %file.close();\n   }\n   %file.delete();\n   return %ok;\n}\nfunction T2VLLiveStart()\n{\n   if ($T2VLLive::Enabled) {\n      if (isObject(ServerConnection) && ServerConnection.getId()==$T2VLLive::Connection) return;\n      T2VLLiveStop();\n   }\n   if (!isObject(ServerConnection)) { error("Join the intended server before enabling the companion."); return; }\n   exec("prefs/t2vl-live/config.cs"); // Locally generated config, NEVER server text.\n   if ($T2VLLive::Version !$= "1" || strlen($T2VLLive::Session)!=16 || strlen($T2VLLive::Token)!=64) return;\n   $T2VLLive::Sequence=0;\n   %file=new FileObject();\n   if (%file.openForRead("prefs/t2vl-live/requests.tsv")) {\n      while (!%file.isEOF()) {\n         %line=%file.readLine(); %seq=getField(%line,4);\n         if (getField(%line,2) $= $T2VLLive::Session && T2VLLiveDigits(%seq) && %seq>$T2VLLive::Sequence)\n            $T2VLLive::Sequence=%seq;\n      }\n      %file.close();\n   }\n   %file.delete();\n   $T2VLLive::Enabled=true;\n   $T2VLLive::Connection=ServerConnection.getId();\n   commandToServer(\'T2VLLiveReady\',"1",$T2VLLive::Bank);\n}\nfunction clientCmdT2VLLiveSpeak(%version,%speaker,%emotion,%text)\n{\n   if (!$T2VLLive::Enabled || %version !$= "1") return;\n   if (!isObject(ServerConnection) || ServerConnection.getId()!=$T2VLLive::Connection) { T2VLLiveStop(); return; }\n   if ($T2VLLive::Pending !$= "") { error("T2 voice companion busy; announcement refused."); return; }\n   if (!$T2VLLive::Voice[%speaker] || !T2VLLiveTextOK(%emotion,%text)) return;\n   $T2VLLive::Sequence++;\n   if (!T2VLLiveDigits($T2VLLive::Sequence)) { error("T2 voice sequence exhausted."); return; }\n   if (T2VLLiveAppend("S",$T2VLLive::Sequence,%speaker,%emotion,%text)) {\n      $T2VLLive::Pending=$T2VLLive::Sequence;\n      $T2VLLive::Deadline=getRealTime()+60000;\n      T2VLLivePoll();\n   }\n}\nfunction T2VLLivePoll()\n{\n   if (!$T2VLLive::Enabled || $T2VLLive::Pending $= "") return;\n   if (!isObject(ServerConnection) || ServerConnection.getId()!=$T2VLLive::Connection) { T2VLLiveStop(); return; }\n   if ($T2VLLive::Playing) {\n      if (getRealTime()>=$T2VLLive::Ends) T2VLLiveComplete($T2VLLive::Pending,"playback_timer_elapsed");\n      else $T2VLLive::Timer=schedule(200,0,"T2VLLivePoll");\n      return;\n   }\n   %file=new FileObject(); %line="";\n   if (%file.openForRead("prefs/t2vl-live/response.tsv")) { %line=%file.readLine(); %file.close(); }\n   %file.delete();\n   %seq=getField(%line,2); %status=getField(%line,3); %duration=getField(%line,5);\n   if (getField(%line,0) $= "1" && getField(%line,1) $= $T2VLLive::Session && %seq $= $T2VLLive::Pending) {\n      if (%status $= "ERROR") { error("T2 voice refusal: " @ getField(%line,4)); $T2VLLive::Pending=""; return; }\n      if (%status $= "READY" && T2VLLiveDigits(%duration) && %duration<=30000) {\n         %path="audio/t2vl-live/" @ $T2VLLive::Session @ "-" @ %seq @ ".wav";\n         if (!isFile(%path)) { T2VLLiveComplete(%seq,"engine_failed"); return; }\n         $T2VLLive::Handle=alxCreateSource(AudioGui,%path);\n         if (!$T2VLLive::Handle) { T2VLLiveComplete(%seq,"engine_failed"); return; }\n         alxPlay($T2VLLive::Handle);\n         $T2VLLive::Playing=true;\n         $T2VLLive::Ends=getRealTime()+%duration+50;\n         $T2VLLive::Timer=schedule(200,0,"T2VLLivePoll");\n         return;\n      }\n   }\n   if (getRealTime()>$T2VLLive::Deadline) { T2VLLiveComplete($T2VLLive::Pending,"cancelled"); return; }\n   $T2VLLive::Timer=schedule(200,0,"T2VLLivePoll");\n}\nfunction T2VLLiveComplete(%seq,%result)\n{\n   if (%seq !$= $T2VLLive::Pending) return;\n   T2VLLiveAppend("A",%seq,"-","-",%result);\n   $T2VLLive::Playing=false;\n   $T2VLLive::Pending="";\n}\nfunction T2VLLiveStop()\n{\n   cancel($T2VLLive::Timer);\n   if ($T2VLLive::Handle !$= "") alxStop($T2VLLive::Handle);\n   if ($T2VLLive::Pending !$= "") T2VLLiveComplete($T2VLLive::Pending,"cancelled");\n   $T2VLLive::Enabled=false;\n}\nfunction T2VLLiveTextOK(%emotion,%text)\n{\n   if (%emotion !$= "neutral" && %emotion !$= "angry" && %emotion !$= "sad" && %emotion !$= "excited" && %emotion !$= "calm") return false;\n   if (strlen(%text)<1 || strlen(%text)>240) return false;\n   return strstr(%text,"\\t")==-1 && strstr(%text,"\\n")==-1 && strstr(%text,"\\r")==-1 && strstr(%text,"[")==-1 && strstr(%text,"]")==-1;\n}\n\'\'\'\n\nSERVER=r\'\'\'// PRIVATE server opt-in. Client companion and actual native acceptance required.\n$T2VLLiveServer::ExpectedBank="@BANK@";\nfunction serverCmdT2VLLiveReady(%client,%version,%bank)\n{\n   %client.t2vlLiveReady=(%version $= "1" && %bank $= $T2VLLiveServer::ExpectedBank);\n}\nfunction serverCmdT2VLLiveAnnounce(%client,%speaker,%emotion,%text)\n{\n   if ((!%client.isAdmin && !%client.isSuperAdmin) || !T2VLLiveServerVoice(%speaker) || !T2VLLiveTextOK(%emotion,%text)) return;\n   if (%client.t2vlLiveLast !$= "" && getSimTime()-%client.t2vlLiveLast<1000) return;\n   %client.t2vlLiveLast=getSimTime();\n   for (%i=0;%i<ClientGroup.getCount();%i++) {\n      %target=ClientGroup.getObject(%i);\n      if (%target.t2vlLiveReady) commandToClient(%target,\'T2VLLiveSpeak\',"1",%speaker,%emotion,%text);\n      else messageClient(%target,\'\',"Custom voice needs the matching opt-in local companion.");\n   }\n}\nfunction T2VLLiveServerVoice(%voice)\n{\n   %voices="@VOICES@";\n   for (%i=0;%i<getWordCount(%voices);%i++)\n      if (%voice $= getWord(%voices,%i)) return true;\n   return false;\n}\n\'\'\'\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__);sub=p.add_subparsers(dest=\'action\',required=True)\n    i=sub.add_parser(\'install\');i.add_argument(\'--base\',type=Path,required=True)\n    i.add_argument(\'--bank\',type=Path,required=True);i.add_argument(\'--lexicon\',type=Path,required=True)\n    i.add_argument(\'--overrides\',type=Path);i.add_argument(\'--audition-unverified\',action=\'store_true\')\n    i.add_argument(\'--bank-pronunciations\',action=\'store_true\')\n    s=sub.add_parser(\'serve\');s.add_argument(\'--base\',type=Path,required=True);s.add_argument(\'--once\',action=\'store_true\')\n    a=p.parse_args()\n    if a.action==\'install\':\n        print(json.dumps(install(a.base,a.bank,a.lexicon,a.overrides,a.audition_unverified,a.bank_pronunciations),indent=2));return\n    service=Companion(a.base)\n    if a.once:print(service.process_once());return\n    stop=threading.Event()\n    for signum in (signal.SIGINT,signal.SIGTERM):signal.signal(signum,lambda *_:stop.set())\n    print(\'Local sample-only companion ready. Native audible playback is NOT certified.\',flush=True)\n    while not stop.is_set():\n        result=service.process_once()\n        if result not in (\'idle\',\'partial\'):\n            if result in (\'ready\',\'acknowledged\'):print(result,flush=True)\n            else:print((service.folder/\'last-error.json\').read_text(),flush=True)\n        stop.wait(0.2)\n\nif __name__==\'__main__\':main()\n', 'events.py': '"""Export whole reviewed vocal events; generated carrier speech is not an event."""\nimport hashlib\nfrom pathlib import Path\nfrom bank import read_wav,write_wav\nfrom renderer import EVENTS,EMOTIONS\n\ndef export_events(reviews,output):\n    output=Path(output);result=[]\n    for r in reviews:\n        if not all(r.get(k) is True for k in (\'speaker_verified\',\'event_verified\',\'boundaries_verified\')):\n            continue\n        if r[\'event\'] not in EVENTS or r.get(\'emotion\',\'neutral\') not in EMOTIONS:\n            raise ValueError(\'Invalid event or emotion\')\n        if r.get(\'contains_carrier_speech\') is not False:\n            raise ValueError(\'Explicitly exclude carrier speech before event export\')\n        if r.get(\'event_type\') not in (\'wordless\',\'spoken_cheer\',\'other_vocalization\'):\n            raise ValueError(\'Classify speech versus wordless sound; do not guess from cues\')\n        source_sha=hashlib.sha256(Path(r[\'source\']).read_bytes()).hexdigest()\n        if r.get(\'source_sha256\',r.get(\'sha256\'))!=source_sha:\n            raise ValueError(\'Event source lacks a matching review checksum\')\n        rate,audio=read_wav(r[\'source\'])\n        a,b=round(r[\'start\']*rate),round(r[\'end\']*rate)\n        if not 0<=a<b<=len(audio):raise ValueError(\'Invalid event boundaries\')\n        sample=audio[a:b]\n        digest=hashlib.sha256(sample.tobytes()).hexdigest()\n        path=\'events/\'+digest[:24]+\'.wav\';write_wav(output/path,rate,sample)\n        result.append({\'id\':digest[:24],\'speaker_id\':r[\'speaker_id\'],\'emotion\':r.get(\'emotion\',\'neutral\'),\n            \'event\':r[\'event\'],\'event_type\':r[\'event_type\'],\'file\':path,\'rate\':rate,\'verified\':True,\n            \'sha256\':hashlib.sha256((output/path).read_bytes()).hexdigest(),\n            \'source\':r[\'source\'],\'start\':r[\'start\'],\'end\':r[\'end\'],\'reviewer\':r.get(\'reviewer\')})\n    return result\n', 'phonetics.py': '"""Dictionary-only runtime frontend. No model, network, or GPU dependencies.\n\nCMUdict JSON is produced during preparation. ARPAbet stress is retained in\nvariant keys; base coverage can separately collapse vowel stress.\n"""\nfrom __future__ import annotations\nimport json\nimport hashlib\nimport re\nfrom collections import Counter\nfrom pathlib import Path\nfrom types import MappingProxyType\n\nSIL = \'SIL\'\nONES = \'zero one two three four five six seven eight nine ten eleven twelve thirteen fourteen fifteen sixteen seventeen eighteen nineteen\'.split()\nTENS = \'zero ten twenty thirty forty fifty sixty seventy eighty ninety\'.split()\nABBREVIATIONS = {\'tts\':\'tee tee ess\', \'afk\':\'ay eff kay\', \'gg\':\'good game\',\n                 \'ctf\':\'see tee eff\', \'fps\':\'eff pee ess\', \'admin\':\'administrator\',\n                 \'vs\':\'versus\', \'hp\':\'aitch pee\', \'id\':\'eye dee\'}\n\nclass UnknownWords(ValueError):\n    def __init__(self, words):\n        self.words = sorted(set(words))\n        super().__init__(\'Unknown words (add pronunciation overrides): \' + \', \'.join(self.words))\n\ndef number(n):\n    n = int(n)\n    if n < 0:\n        return \'minus \' + number(-n)\n    if n < 20:\n        return ONES[n]\n    if n < 100:\n        return TENS[n//10] + (\' \'+ONES[n%10] if n%10 else \'\')\n    if n < 1000:\n        return ONES[n//100]+\' hundred\'+(\' \'+number(n%100) if n%100 else \'\')\n    if n < 1000000:\n        return number(n//1000)+\' thousand\'+(\' \'+number(n%1000) if n%1000 else \'\')\n    return \' \'.join(ONES[int(c)] for c in str(n))\n\ndef normalize(text, aliases=None):\n    aliases = {k.casefold():v for k,v in (aliases or {}).items()}\n    # A whole username alias wins before punctuation, digits, and camel-case splitting.\n    for key in sorted(aliases, key=len, reverse=True):\n        text = re.sub(r\'(?<!\\w)\'+re.escape(key)+r\'(?!\\w)\', lambda _: aliases[key], text, flags=re.I)\n    text = re.sub(r\'(?<=[a-z])(?=[A-Z])\',\' \',text)\n    text = re.sub(r\'(?<=\\d)(?=[A-Za-z])|(?<=[A-Za-z])(?=\\d)\', \' \', text)\n    text = text.replace(\'_\',\' \').replace(\'/\',\' \').replace(\'&\',\' and \')\n    text = re.sub(r\'\\b(-?\\d+)\\.(\\d+)\\b\',lambda m:number(m[1])+\' point \'+\' \'.join(ONES[int(c)] for c in m[2]),text)\n    text = re.sub(r\'(?<!\\w)-?\\d+\', lambda m:number(m[0]),text)\n    # Remove command prefix only; never silently discard an unknown command word.\n    words = re.findall(r"[^\\W\\d_]+(?:\'[^\\W\\d_]+)?|[.!?;,:]",text.casefold())\n    return \' \'.join(ABBREVIATIONS.get(w,w) for w in words)\n\ndef base(phone):\n    return re.sub(r\'[012]$\',\'\',phone)\n\ndef transitions(phones, collapse_stress=False):\n    ps = [base(p) for p in phones] if collapse_stress else phones\n    return Counter(zip(ps,ps[1:]))\n\nclass Lexicon:\n    def __init__(self, entries, overrides=None, aliases=None):\n        frozen={k.casefold():(tuple(tuple(p) for p in v) if v and isinstance(v[0],(list,tuple)) else tuple(v))\n                for k,v in entries.items()}\n        self._entries=MappingProxyType(frozen)\n        self._dictionary_digest=hashlib.sha256(json.dumps(frozen,sort_keys=True).encode()).hexdigest()\n        self.overrides = {k.casefold():v for k,v in (overrides or {}).items()}\n        self.aliases = aliases or {}\n    @property\n    def entries(self):\n        return self._entries\n    @property\n    def dictionary_digest(self):\n        return self._dictionary_digest\n    @classmethod\n    def load(cls, path, overrides=None):\n        d = json.loads(Path(path).read_text())\n        o = json.loads(Path(overrides).read_text()) if overrides else {}\n        return cls(d,o.get(\'phones\'),o.get(\'aliases\'))\n    def pronunciation_tokens(self,text):\n        """Retain attested dictionary alternatives; explicit overrides replace them."""\n        normalized=normalize(text,self.aliases)\n        tokens=[];unknown=[]\n        for word in normalized.split():\n            if word in \'.!?;,:\':\n                tokens.append({\'punctuation\':word});continue\n            prons=self.overrides.get(word) or self.entries.get(word)\n            if not prons:\n                unknown.append(word);continue\n            if not isinstance(prons[0],(list,tuple)):prons=[prons]\n            tokens.append({\'word\':word,\'pronunciations\':[list(p) for p in prons]})\n        if unknown:raise UnknownWords(unknown)\n        return normalized,tokens\n    def pronounce(self,text):\n        normalized = normalize(text,self.aliases)\n        phones, unknown, words = [SIL], [], []\n        boundaries = []\n        for w in normalized.split():\n            if w in \'.!?;,:\':\n                if phones[-1] != SIL:\n                    phones.append(SIL)\n                continue\n            prons = self.overrides.get(w) or self.entries.get(w)\n            if not prons:\n                unknown.append(w)\n                continue\n            pron = prons[0] if isinstance(prons[0],(list,tuple)) else prons\n            if words and phones[-1] != SIL:\n                boundaries.append((phones[-1],pron[0]))\n            phones.extend(pron)\n            words.append(w)\n        if unknown:\n            raise UnknownWords(unknown)\n        if phones[-1] != SIL:\n            phones.append(SIL)\n        return {\'normalized\':normalized,\'phones\':phones,\'words\':words,\'cross_word\':boundaries}\n\ndef coverage(rows, lexicon, verified_units=()):\n    """Rows are never pooled across speaker or emotion. Unassigned rows stay explicit."""\n    reports = {}\n    for row in rows:\n        key = (row.get(\'speaker_id\') or \'UNASSIGNED\')+\'/\'+row.get(\'emotion\',\'neutral\')\n        r = reports.setdefault(key, {\'predicted\':Counter(),\'cross_word\':Counter(),\'unknown\':[],\n                                     \'rows\':0,\'unverified_transcript_rows\':0,\'usable\':set()})\n        r[\'rows\'] += 1\n        if str(row.get(\'transcript_verified\',\'false\')).lower() != \'true\':\n            r[\'unverified_transcript_rows\'] += 1\n        try:\n            p = lexicon.pronounce(row.get(\'text\',\'\'))\n        except UnknownWords as e:\n            r[\'unknown\'].extend(e.words)\n            continue\n        r[\'predicted\'].update(transitions(p[\'phones\']))\n        r[\'cross_word\'].update(p[\'cross_word\'])\n    for u in verified_units:\n        if u.get(\'verified\') is True:\n            key = u[\'speaker_id\']+\'/\'+u[\'emotion\']\n            reports.setdefault(key, {\'predicted\':Counter(),\'cross_word\':Counter(),\'unknown\':[],\n                                      \'rows\':0,\'unverified_transcript_rows\':0,\'usable\':set()})[\'usable\'].add(tuple(u[\'phones\']))\n    return {k:{**{n:v for n,v in r.items() if n not in (\'predicted\',\'cross_word\',\'usable\',\'unknown\')},\n               \'predicted_transitions\':{\'|\'.join(p):n for p,n in sorted(r[\'predicted\'].items())},\n               \'cross_word_transitions\':{\'|\'.join(p):n for p,n in sorted(r[\'cross_word\'].items())},\n               \'verified_usable_transitions\':[\'|\'.join(p) for p in sorted(r[\'usable\'])],\n               \'unknown_words\':sorted(set(r[\'unknown\']))} for k,r in reports.items()}\n\ndef select_gaps(sentences, lexicon, target_pairs, existing=(), max_sentences=60, pair_weights=None):\n    """Greedy set cover of a defined target, not a claim of all English coverage."""\n    missing = set(map(tuple,target_pairs))-set(map(tuple,existing))\n    candidates, rejected, chosen = [], [], []\n    for s in sentences:\n        try:\n            p = lexicon.pronounce(s)\n            candidates.append((s,p,set(transitions(p[\'phones\']))))\n        except UnknownWords as e:\n            rejected.append({\'text\':s,\'unknown\':e.words})\n    while missing and candidates and len(chosen) < max_sentences:\n        def score(candidate):\n            gain=candidate[2]&missing\n            value=sum(pair_weights.get(pair,1) for pair in gain) if pair_weights is not None else len(gain)\n            return value/max(1,len(candidate[1][\'phones\']))\n        best = max(candidates,key=score)\n        gain = best[2]&missing\n        if not gain:\n            break\n        chosen.append({\'text\':best[0],\'phones\':best[1][\'phones\'],\'new_pairs\':[\'|\'.join(p) for p in sorted(gain)],\n                       \'pronunciation_verified\':False,\'audio_verified\':False})\n        missing -= gain\n        candidates.remove(best)\n    return {\'selected\':chosen,\'uncovered\':[\'|\'.join(p) for p in sorted(missing)],\'rejected\':rejected,\n            \'scope\':\'Only the declared target transitions; predicted pronunciation pending audio validation.\'}\n', 'build_bank.py': '"""Build a strict reviewed bank, or an explicitly labeled candidate audition bank."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nfrom bank import extract,write_index,safe_path\nfrom events import export_events\n\ndef merge_banks(paths,output,include_candidates=False):\n    """Preserve distinct alignments of one source; old source/index IDs can collide."""\n    output=Path(output);units=[];seen=set()\n    for source in paths:\n        source=Path(source)\n        index=json.loads((source/\'index.json\').read_text())\n        if index.get(\'events\'):\n            raise ValueError(\'Speech-bank merge refuses banks containing events; preserve/export reviewed events separately\')\n        for original in index[\'units\']:\n            if not include_candidates and original.get(\'verified\') is not True:continue\n            audio=safe_path(source,original[\'file\']).read_bytes()\n            if hashlib.sha256(audio).hexdigest()!=original[\'sha256\']:\n                raise ValueError(\'Source-bank sample checksum mismatch\')\n            # Bind identity to actual bytes, phone labels, timing, review and voice.\n            metadata={k:v for k,v in original.items() if k not in (\'id\',\'file\')}\n            digest=hashlib.sha256(json.dumps(metadata,sort_keys=True).encode()).hexdigest()\n            if digest in seen:continue\n            seen.add(digest)\n            name=\'units/\'+digest+\'.wav\';destination=output/name\n            destination.parent.mkdir(parents=True,exist_ok=True)\n            if destination.exists() and destination.read_bytes()!=audio:\n                raise ValueError(\'Refusing conflicting merged sample overwrite\')\n            destination.write_bytes(audio)\n            units.append({**original,\'id\':digest,\'file\':name})\n    return units\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--alignments\',type=Path)\n    p.add_argument(\'--bank\',type=Path,action=\'append\',default=[],help=\'Merge existing banks without dropping alternate alignments\')\n    p.add_argument(\'--output\',type=Path,required=True)\n    p.add_argument(\'--events\',type=Path)\n    p.add_argument(\'--include-candidates\',action=\'store_true\')\n    a=p.parse_args();a.output.mkdir(parents=True,exist_ok=True)\n    if not a.alignments and not a.bank:p.error(\'Supply --alignments or one or more --bank paths\')\n    units=merge_banks(a.bank,a.output,a.include_candidates);skipped=[]\n    for path in sorted(a.alignments.rglob(\'*.json\')) if a.alignments else []:\n        data=json.loads(path.read_text())\n        if not a.include_candidates and not all(data.get(k) is True for k in (\n                \'speaker_verified\',\'transcript_verified\',\'alignment_verified\',\'emotion_verified\')):\n            skipped.append({\'file\':str(path),\'reason\':\'missing explicit audio/speaker/transcript/emotion reviews\'});continue\n        if not a.include_candidates and not data.get(\'source_sha256\'):\n            skipped.append({\'file\':str(path),\'reason\':\'review is not bound to a source checksum\'});continue\n        extracted=extract(data,a.output,reviewed_only=not a.include_candidates)\n        units.extend(extracted if a.include_candidates else [u for u in extracted if u[\'verified\']])\n    events=export_events(json.loads(a.events.read_text()),a.output) if a.events else []\n    receipt=write_index(a.output,units,events)\n    receipt.update({\'candidate_mode\':a.include_candidates,\'skipped\':skipped,\'events\':len(events)})\n    (a.output/\'build-receipt.json\').write_text(json.dumps(receipt,indent=2))\n    print(json.dumps(receipt,indent=2))\n    print(\'========== DONE ==========\',flush=True)\n\nif __name__==\'__main__\':main()\n', 'alignment.py': '"""MFA phone-alignment bridge. Automatic output stays unverified until reviewed."""\nfrom __future__ import annotations\nimport argparse\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport subprocess\nfrom bank import extract,read_wav,write_index\nfrom phonetics import Lexicon,normalize,base\n\ndef stage(jobs,lexicon,corpus):\n    corpus=Path(corpus);corpus.mkdir(parents=True,exist_ok=True)\n    metadata={}\n    for job in jobs:\n        if job.get(\'kind\',\'speech\')!=\'speech\':\n            continue\n        p=lexicon.pronounce(job[\'text\']) # refuse unknown words before spending alignment time\n        source=Path(job[\'source\'])\n        key=hashlib.sha256((str(source)+job[\'text\']).encode()).hexdigest()[:20]\n        # Each known/candidate voice has its own MFA speaker directory, never Training as one voice.\n        speaker=job[\'speaker_id\']\n        if not speaker or Path(speaker).name!=speaker:\n            raise ValueError(\'Directory-safe, explicit speaker ID required\')\n        folder=corpus/speaker;folder.mkdir(exist_ok=True)\n        shutil.copy2(source,folder/(key+\'.wav\'))\n        (folder/(key+\'.lab\')).write_text(\' \'.join(p[\'words\']))\n        metadata[speaker+\'/\'+key]={**job,\'source\':str(source.resolve()),\'normalized\':p[\'normalized\']}\n    (corpus/\'jobs.json\').write_text(json.dumps(metadata,indent=2))\n    return metadata\n\ndef export_dictionary(lexicon,path):\n    with Path(path).open(\'w\') as f:\n        for word,prons in {**lexicon.entries,**lexicon.overrides}.items():\n            if \' \' in word:continue\n            if not isinstance(prons[0],(list,tuple)):prons=[prons]\n            for pron in prons:\n                f.write(word+\'\\t\'+\' \'.join(pron)+\'\\n\')\n\ndef convert(textgrid_path,job):\n    import textgrid\n    tg=textgrid.TextGrid.fromFile(str(textgrid_path))\n    tier=next(t for t in tg.tiers if t.name==\'phones\' or t.name.endswith(\' - phones\'))\n    rate,audio=read_wav(job[\'source\']); duration=len(audio)/rate\n    phones=[];prev=0\n    for interval in tier.intervals:\n        start,end=float(interval.minTime),float(interval.maxTime)\n        label=interval.mark.strip()\n        if end-start<=0:continue\n        if start>prev+0.000001:phones.append({\'label\':\'SIL\',\'start\':prev,\'end\':start})\n        # spn is unknown/noise, not certified silence; keep it explicit.\n        label=\'SIL\' if label in (\'\',\'sil\',\'sp\') else (\'SPN\' if label==\'spn\' else label)\n        phones.append({\'label\':label,\'start\':start,\'end\':min(end,duration),\'variant\':\'canonical\',\n                       \'closure_verified\':False})\n        prev=end\n    if prev<duration:phones.append({\'label\':\'SIL\',\'start\':prev,\'end\':duration})\n    # Merge adjacent silence intervals; internal pauses remain explicit units.\n    merged=[]\n    for p in phones:\n        if merged and p[\'label\']==\'SIL\' and merged[-1][\'label\']==\'SIL\':merged[-1][\'end\']=p[\'end\']\n        else:merged.append(p)\n    return {**job,\'phones\':merged,\'source_sha256\':hashlib.sha256(Path(job[\'source\']).read_bytes()).hexdigest(),\n            \'aligner\':\'Montreal Forced Aligner 3.3.9 / english_us_arpa\',\n            \'alignment_verified\':False,\'emotion_verified\':False,\'speaker_verified\':False,\n            \'transcript_verified\':False,\'reviewer\':None,\n            \'notes\':\'Automatic candidate. Review boundaries, silence, allophones and stop closures/releases before promotion.\'}\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--jobs\',type=Path,required=True)\n    p.add_argument(\'--lexicon\',type=Path,required=True);p.add_argument(\'--overrides\',type=Path)\n    p.add_argument(\'--out\',type=Path,required=True);p.add_argument(\'--mfa\',type=Path,required=True)\n    p.add_argument(\'--model\',default=\'english_us_arpa\')\n    a=p.parse_args();a.out.mkdir(parents=True,exist_ok=True)\n    lex=Lexicon.load(a.lexicon,a.overrides); jobs=json.loads(a.jobs.read_text())\n    corpus=a.out/\'corpus\';meta=stage(jobs,lex,corpus)\n    dictionary=a.out/\'lexicon.dict\';export_dictionary(lex,dictionary)\n    aligned=a.out/\'aligned\'\n    env=dict(os.environ)\n    env[\'PATH\']=str(a.mfa.resolve().parent)+os.pathsep+env.get(\'PATH\',\'\')\n    subprocess.run([str(a.mfa.resolve()),\'align\',str(corpus.resolve()),str(dictionary.resolve()),a.model,str(aligned.resolve()),\n                    \'--clean\',\'--overwrite\',\'--num_jobs\',\'2\',\'--output_format\',\'long_textgrid\'],check=True,env=env)\n    units=[]\n    for name,job in meta.items():\n        alignment=convert(aligned/(name+\'.TextGrid\'),job)\n        dest=a.out/\'alignment-json\'/(name+\'.json\');dest.parent.mkdir(parents=True,exist_ok=True)\n        dest.write_text(json.dumps(alignment,indent=2))\n        units.extend(extract(alignment,a.out/\'candidate-bank\'))\n    receipt=write_index(a.out/\'candidate-bank\',units)\n    (a.out/\'receipt.json\').write_text(json.dumps(receipt,indent=2))\n    print(json.dumps(receipt,indent=2))\n    print(\'========== DONE ==========\',flush=True)\n\nif __name__==\'__main__\':main()\n', 'pilot.py': '"""Build transparent provisional references, then time Breeze\'s load-once pilot.\n\nProvisional reference mode is only for small auditions, not bank promotion or\nbulk generation. No machine transcript/speaker label is silently marked reviewed.\n"""\nfrom __future__ import annotations\nimport argparse\nfrom dataclasses import asdict\nimport hashlib\nimport importlib.metadata\nimport json\nfrom pathlib import Path\nimport subprocess\nimport shutil\nimport sys\nimport time\nfrom bank import read_wav,write_wav\n\nROOT=Path(__file__).resolve().parent\nSOURCES={\n    \'Announcer\':[\'voice/Announcer/ANN.60.WAV\',\'voice/Announcer/ANN.gameover.WAV\',\n                 \'voice/Announcer/ANN.bdscore.WAV\',\'voice/Announcer/ANN.bescore.WAV\',\n                 \'voice/Announcer/ANN.dsscore.WAV\'],\n    \'TrainingFemale\':[\'voice/Training/Mission1/T1_01.WAV\',\'voice/Training/Mission1/T1_01a.WAV\'],\n    \'BriefingMale\':[\'voice/Training/Briefings/T1.brief02.WAV\'],\n}\n\ndef prepare_references(character_packs=False):\n    rows={r[\'path\']:r for r in map(json.loads,(ROOT/\'artifacts/asr_candidates.jsonl\').read_text().splitlines())}\n    selection=dict(SOURCES)\n    if character_packs:\n        sys.path.insert(0,str(ROOT/\'prep\'))\n        from voice_lab import transcript_seeds\n        hints=transcript_seeds()\n        def comparable(text):\n            return \'\'.join(c for c in text.casefold() if c.isalnum() or c.isspace()).split()\n        # Use normal speech hints as candidate transcript only if ASR has actually run.\n        for pack in [*(f\'Male{i}\' for i in range(1,6)),*(f\'Fem{i}\' for i in range(1,6)),*(f\'Derm{i}\' for i in range(1,4)),\'Bot1\']:\n            actual={name.casefold():name for name in rows if name.startswith(\'voice/\'+pack+\'/\')}\n            durations={r[\'path\']:r[\'seconds\'] for r in json.loads((ROOT/\'artifacts/audio_inventory.json\').read_text())}\n            names=[];seconds=0\n            for stem in (\'gbl.hi\',\'gbl.thanks\',\'def.base\',\'att.base\',\'rep.generator\',\'slf.def.base\',\n                         \'need.support\',\'def.flag\',\'att.generator\',\'def.generator\',\'def.sensors\',\'def.objective\',\n                         \'rep.base\',\'slf.att.base\',\'gbl.bye\'):\n                name=actual.get(f\'voice/{pack}/{stem}.wav\'.casefold())\n                # Reject evident ASR/hint disagreement; do not feed "we\'re our generator"\n                # as if it were a complete, correct "repair our generator" reference.\n                agreed=name and stem in hints and comparable(rows[name][\'text\'])==comparable(hints[stem])\n                if agreed and rows[name][\'text\'].strip() and seconds+durations[name]+0.15<=15:\n                    names.append(name);seconds+=durations[name]+0.15\n                if seconds>=8:break\n            if seconds<3:raise ValueError(\'Insufficient complete ASR candidate reference for \'+pack)\n            selection[pack]=names\n    result=[]\n    for speaker,names in selection.items():\n        pieces=[];texts=[];used=[]\n        for name in names:\n            if name not in rows:raise ValueError(\'ASR incomplete: \'+name)\n            row=rows[name]\n            if not row[\'text\'].strip():raise ValueError(\'Empty transcript: \'+name)\n            rate,audio=read_wav(ROOT/\'artifacts/pcm24k\'/name)\n            if len(audio)/rate>20:raise ValueError(\'Reference too long; select another complete clip: \'+name)\n            pieces.extend(audio);pieces.extend([0]*round(rate*0.15))\n            texts.append(row[\'text\']);used.append({\'path\':name,\'asr_text\':row[\'text\']})\n        duration=len(pieces)/rate\n        if not 3<=duration<=20:raise ValueError(f\'{speaker}: reference {duration:.2f}s outside 3–20s\')\n        out=ROOT/\'artifacts/references-provisional\'/f\'{speaker}.wav\'\n        if out.exists():\n            history=out.parent/\'history\'/hashlib.sha256(out.read_bytes()).hexdigest()[:16]\n            history.mkdir(parents=True,exist_ok=True)\n            for p in (out,out.with_suffix(\'.txt\'),out.with_suffix(\'.json\')):\n                if p.exists():shutil.copy2(p,history/p.name)\n        write_wav(out,rate,pieces)\n        out.with_suffix(\'.txt\').write_text(\' \'.join(texts))\n        receipt={\'speaker_id\':speaker,\'sources\':used,\'seconds\':duration,\'speaker_verified\':False,\n                 \'transcript_verified\':False,\'reference_verified\':False,\n                 \'warning\':\'ASR-derived, unreviewed reference for small cloning audition only. Training role is a hypothesis until acoustic review.\',\n                 \'sha256\':hashlib.sha256(out.read_bytes()).hexdigest()}\n        out.with_suffix(\'.json\').write_text(json.dumps(receipt,indent=2))\n        result.append(receipt)\n    (ROOT/\'artifacts/references-provisional/references.json\').write_text(json.dumps(result,indent=2))\n    return result\n\ndef output_folder(tag=\'pilot\'):\n    if not tag or Path(tag).name!=tag or tag in (\'.\',\'..\'):\n        raise ValueError(\'Directory-safe output tag required\')\n    artifacts=(ROOT/\'artifacts\').resolve()\n    out=(artifacts/tag).resolve()\n    if not out.is_relative_to(artifacts):raise ValueError(\'Output tag escapes artifacts directory\')\n    return out\n\ndef generate(speakers,allow_provisional=False,expressions=False,text_jobs=None,include_events=True,output_tag=\'pilot\',fast=False):\n    permission_path=ROOT/\'artifacts/permissions/rights-attestation.json\'\n    permission=json.loads(permission_path.read_text()) if permission_path.exists() else {}\n    if (permission.get(\'performer_consent_and_recording_rights_confirmed\') is not True or\n            not set(speakers)<=set(permission.get(\'voice_ids\',[]))):\n        raise PermissionError(\'Breeze requires explicit performer consent and reference-recording rights. \'\n            \'Record confirmed permissions for these voice IDs in artifacts/permissions/rights-attestation.json; \'\n            \'do not infer consent from public game assets or noncommercial use.\')\n    if not allow_provisional:raise ValueError(\'Provisional refs require an explicit audition-only flag\')\n    out=output_folder(output_tag)\n    source=ROOT/\'environments/breeze-tts\'\n    sys.path.insert(0,str(source))\n    import numpy as np\n    import soundfile as sf\n    import torch\n    import infer as official\n    from breeze_infer.runtime import load_runtime,resolve_device,set_all_seeds,update_generation_config_for_breeze\n    from breeze_infer.templates import prepare_inputs,get_template,select_template_name\n    from models.fast_streaming import FastBreezeStreamingRuntime,FastStreamingConfig\n    sys.path.insert(0,str(ROOT/\'prep\'))\n    from expression_support import EMOTIONS,BREEZE_EVENTS\n    if not torch.cuda.is_available():raise RuntimeError(\'CUDA GPU required for preparation\')\n    out.mkdir(parents=True,exist_ok=True)\n    jobs=[]\n    for speaker in speakers:\n        ref=ROOT/\'artifacts/references-provisional\'/f\'{speaker}.wav\'\n        receipt=json.loads(ref.with_suffix(\'.json\').read_text())\n        assert hashlib.sha256(ref.read_bytes()).hexdigest()==receipt[\'sha256\']\n        for emotion,instruction in (EMOTIONS.items() if expressions else [(\'neutral\',None)]):\n            jobs.append({\'id\':speaker+\'_\'+emotion,\'speaker_id\':speaker,\'emotion\':emotion,\'kind\':\'speech\',\n                \'text\':\'Welcome to the server, Cody. Defend the generator!\',\'instruction\':instruction,\n                \'ref_audio_path\':str(ref),\'ref_text\':ref.with_suffix(\'.txt\').read_text()})\n        if expressions and include_events:\n            for event,text in BREEZE_EVENTS.items():\n                jobs.append({\'id\':speaker+\'_event_\'+event,\'speaker_id\':speaker,\'emotion\':\'neutral\',\'kind\':\'event_candidate\',\n                    \'event\':event,\'text\':text,\'instruction\':None,\'ref_audio_path\':str(ref),\'ref_text\':ref.with_suffix(\'.txt\').read_text()})\n    if text_jobs is not None:\n        for job in text_jobs:\n            speaker=job[\'speaker_id\']\n            ref=ROOT/\'artifacts/references-provisional\'/f\'{speaker}.wav\'\n            if (speaker not in speakers or Path(job[\'id\']).name!=job[\'id\'] or not job[\'text\'].strip()\n                    or job.get(\'emotion\') not in EMOTIONS or job.get(\'kind\')!=\'speech\'):\n                raise ValueError(\'Invalid corpus job identity or mode\')\n            if job[\'instruction\']!=EMOTIONS[job[\'emotion\']]:raise ValueError(\'Unselected voice direction\')\n            if job[\'ref_audio_path\']!=str(ref) or job[\'ref_text\']!=ref.with_suffix(\'.txt\').read_text():\n                raise ValueError(\'Corpus job reference changed\')\n        jobs=jobs+text_jobs if expressions else text_jobs\n    torch.cuda.reset_peak_memory_stats()\n    t0=time.perf_counter()\n    tokenizer,model,codec=load_runtime(ROOT/\'environments/breeze-model\',device=resolve_device(),attn_implementation=\'eager\')\n    update_generation_config_for_breeze(model)\n    cfg=FastStreamingConfig(max_new_tokens=official.MAX_NEW_TOKENS,max_seq_len=official.MAX_SEQ_LEN,\n        fast_all=fast,fast_text_encoder=False,fast_backbone_prefill=False,fast_backbone_decode=False,\n        fast_depth_decoder=False,fast_codec=False,repetition_penalty=official.REPETITION_PENALTY)\n    runtime=FastBreezeStreamingRuntime(model,codec,cfg,tokenizer=tokenizer)\n    torch.cuda.synchronize();load_seconds=time.perf_counter()-t0\n    manifest=out/\'pilot-jobs.json\'\n    records=json.loads(manifest.read_text()) if manifest.exists() else []\n    by_id={r[\'id\']:r for r in records}\n    previous=out/\'run-card.json\'\n    if previous.exists():\n        history=out/\'run-history\';history.mkdir(exist_ok=True)\n        old=previous.read_bytes()\n        (history/(hashlib.sha256(old).hexdigest()[:16]+\'.json\')).write_bytes(old)\n    for job in jobs:\n        path=out/(job[\'id\']+\'.wav\')\n        if path.exists():\n            old=json.loads(path.with_suffix(\'.json\').read_text())\n            if any(old.get(k)!=job.get(k) for k in (\'text\',\'instruction\',\'ref_audio_path\',\'ref_text\')):\n                raise ValueError(\'Refusing to overwrite changed pilot context: \'+job[\'id\'])\n            if hashlib.sha256(path.read_bytes()).hexdigest()!=old[\'sha256\']:\n                raise ValueError(\'Existing pilot checksum mismatch\')\n            by_id[job[\'id\']]=old\n            print(\'Reusing checksum-matched pilot:\',job[\'id\'],flush=True)\n            continue\n        start=time.perf_counter();set_all_seeds(42)\n        request={\'id\':job[\'id\'],\'speaker\':\'S0\',**{k:job[k] for k in (\'text\',\'ref_audio_path\',\'ref_text\')}}\n        if job[\'instruction\']:request[\'instruction\']=job[\'instruction\']\n        inputs=prepare_inputs(tokenizer,codec,model,[request],get_template(select_template_name(request)),\n            guidance_scale=4.0 if job[\'instruction\'] else 1.0,guidance_scale_ref=None,guidance_scale_ins=None)\n        chunks=[];steps=[0]\n        for chunk in runtime.iter_audio_chunks(inputs,request_id=job[\'id\'],seed=42,\n                token_observer=lambda _:steps.__setitem__(0,steps[0]+1)):\n            chunks.append(np.asarray(chunk.audio,dtype=np.float32).reshape(-1))\n        torch.cuda.synchronize()\n        audio=np.concatenate(chunks) if chunks else np.zeros(0,dtype=np.float32)\n        if not np.isfinite(audio).all() or not len(audio) or not np.any(audio):\n            raise RuntimeError(\'Invalid/silent audio; reject this pilot\')\n        complete=bool(len(audio)) and steps[0]<official.MAX_NEW_TOKENS\n        sf.write(path,audio,runtime.sample_rate,subtype=\'PCM_16\')\n        record={**job,\'source\':str(path),\'seed\':42,\'audio_seconds\':len(audio)/runtime.sample_rate,\n            \'wall_seconds\':time.perf_counter()-start,\'codec_steps\':steps[0],\'complete_by_frame_guard\':complete,\n            \'peak_allocated_bytes\':torch.cuda.max_memory_allocated(),\'verified\':False,\n            \'absolute_peak\':float(np.max(np.abs(audio))),\n            \'clipped_fraction\':float(np.mean(np.abs(audio)>=0.9999)),\n            \'speaker_verified\':False,\'transcript_verified\':False,\'reference_verified\':False,\n            \'sha256\':hashlib.sha256(path.read_bytes()).hexdigest()}\n        path.with_suffix(\'.json\').write_text(json.dumps(record,indent=2));by_id[job[\'id\']]=record\n        manifest.write_text(json.dumps(list(by_id.values()),indent=2))\n        print(json.dumps({k:record[k] for k in (\'id\',\'audio_seconds\',\'wall_seconds\',\'complete_by_frame_guard\',\'peak_allocated_bytes\')}),flush=True)\n        if not complete:raise RuntimeError(\'Incomplete generation; reject, never slice or train on this result\')\n    versions={}\n    for name in (\'torch\',\'torchaudio\',\'transformers\',\'qwen-tts\',\'numpy\',\'soundfile\'):\n        try:versions[name]=importlib.metadata.version(name)\n        except importlib.metadata.PackageNotFoundError:versions[name]=None\n    (out/\'run-card.json\').write_text(json.dumps({\'model\':json.loads((ROOT/\'environments/breeze-model-revision.json\').read_text()),\n        \'source_revision\':(ROOT/\'environments/breeze-source-revision.txt\').read_text().strip(),\n        \'load_seconds\':load_seconds,\'gpu\':torch.cuda.get_device_name(),\'gpu_total_bytes\':torch.cuda.get_device_properties(0).total_memory,\n        \'dependencies\':versions,\'jobs\':list(by_id.values()),\'this_run_job_ids\':[j[\'id\'] for j in jobs],\n        \'fast_path\':fast,\'large_corpus_authorized_by_pilot\':False},indent=2))\n    print(\'========== DONE ==========\',flush=True)\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'stage\',choices=[\'references\',\'generate\'])\n    p.add_argument(\'--speakers\',nargs=\'+\',default=[\'Announcer\']);p.add_argument(\'--allow-provisional-reference\',action=\'store_true\')\n    p.add_argument(\'--expressions\',action=\'store_true\');p.add_argument(\'--character-packs\',action=\'store_true\')\n    p.add_argument(\'--jobs\',type=Path,help=\'Explicit small corpus job list with the same reference and review status\')\n    p.add_argument(\'--no-events\',action=\'store_true\',help=\'Speech emotion auditions only, without event carriers\')\n    p.add_argument(\'--output-tag\',default=\'pilot\',help=\'Separate artifacts subdirectory for an independent generation shard; default unchanged\')\n    p.add_argument(\'--fast\',action=\'store_true\',help=\'Opt in to the official accelerated streaming stages; default remains eager\')\n    a=p.parse_args()\n    if a.stage==\'references\':print(json.dumps(prepare_references(a.character_packs),indent=2))\n    else:generate(a.speakers,a.allow_provisional_reference,a.expressions,\n                   json.loads(a.jobs.read_text()) if a.jobs else None,not a.no_events,a.output_tag,a.fast)\n\nif __name__==\'__main__\':\n    try:main()\n    except Exception as exc:\n        out=ROOT/\'artifacts/pilot\';out.mkdir(parents=True,exist_ok=True)\n        (out/\'last-failure.json\').write_text(json.dumps({\'type\':type(exc).__name__,\'message\':str(exc)},indent=2))\n        raise\n', 'package_game.py': '"""Build a PRIVATE preinstalled-phrase VL2, not a server-to-client WAV transfer.\n\nOnly reviewed rendered phrases may be packaged. Live arbitrary usernames are\nnot solved by this pack: they require new client content or a companion renderer.\n"""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport zipfile\nfrom bank import read_wav\n\ndef package(phrases,output):\n    output=Path(output);output.parent.mkdir(parents=True,exist_ok=True)\n    entries=[]\n    for phrase in phrases:\n        if phrase.get(\'verified\') is not True:\n            raise ValueError(\'Refusing to package unreviewed phrase\')\n        p=Path(phrase[\'file\']);rate,audio=read_wav(p)\n        if not audio:raise ValueError(\'Empty phrase\')\n        digest=hashlib.sha256(p.read_bytes()).hexdigest()\n        if phrase.get(\'sha256\',phrase.get(\'output_sha256\'))!=digest:\n            raise ValueError(\'Phrase lacks a matching review checksum: \'+str(p))\n        entries.append({\'id\':digest[:24],\'sha256\':digest,\'file\':\'audio/t2vl/\'+digest[:24]+\'.wav\',\n                        \'duration\':len(audio)/rate,\'source\':str(p),\'text\':phrase[\'text\'],\'speaker_id\':phrase[\'speaker_id\']})\n    digest=hashlib.sha256(json.dumps(entries,sort_keys=True).encode()).hexdigest()\n    client=\'// PRIVATE / research and noncommercial. Install on EVERY participating client.\\n\'\n    client+=\'$T2VL::Manifest = "\'+digest+\'";\\n\'\n    server=\'// Load on the server. This script transfers IDs, NOT audio bytes.\\n\'\n    server+=\'$T2VL::Manifest = "\'+digest+\'";\\n\'\n    for e in entries:\n        client+=\'$T2VL::Path["\'+e[\'id\']+\'"] = "\'+e[\'file\']+\'";\\n\'\n        server+=\'$T2VL::Allowed["\'+e[\'id\']+\'"] = true;\\n\'\n    client+=\'\'\'\nfunction clientCmdT2VLPlay(%id)\n{\n   %path = $T2VL::Path[%id];\n   if (%path $= "" || !isFile(%path))\n   {\n      error("T2 Voice Lab: missing preinstalled phrase " @ %id);\n      return;\n   }\n   if ($T2VL::Handle !$= "")\n      alxStop($T2VL::Handle);\n   $T2VL::Handle = alxCreateSource(AudioGui, %path);\n   alxPlay($T2VL::Handle);\n}\n// After connecting, call T2VLReady(); a successful handshake proves only that\n// this script loaded, NOT that an audio device played the WAV successfully.\nfunction T2VLReady()\n{\n   commandToServer(\'T2VLReady\', $T2VL::Manifest);\n}\n\'\'\'\n    server+=\'\'\'\nfunction serverCmdT2VLReady(%client, %manifest)\n{\n   %client.t2vlReady = (%manifest $= $T2VL::Manifest);\n}\nfunction serverCmdT2VLAnnounce(%client, %id)\n{\n   if ((!%client.isAdmin && !%client.isSuperAdmin) || !$T2VL::Allowed[%id])\n      return;\n   if (%client.t2vlLast !$= "" && getSimTime() - %client.t2vlLast < 1000)\n      return;\n   %client.t2vlLast = getSimTime();\n   for (%i = 0; %i < ClientGroup.getCount(); %i++)\n   {\n      %target = ClientGroup.getObject(%i);\n      if (%target.t2vlReady)\n         commandToClient(%target, \'T2VLPlay\', %id);\n      else\n         messageClient(%target, \'\', "A custom voice announcement requires the matching client voice pack.");\n   }\n}\n\'\'\'\n    with zipfile.ZipFile(output,\'w\',compression=zipfile.ZIP_DEFLATED) as z:\n        z.writestr(\'scripts/t2vl_client.cs\',client)\n        for e in entries:z.write(e[\'source\'],e[\'file\'])\n        z.writestr(\'t2vl-manifest.json\',json.dumps({\'digest\':digest,\'phrases\':entries,\n            \'live_client_test\':\'NOT RUN\',\'license\':\'Private development; research/noncommercial.\'},indent=2))\n    output.with_suffix(\'.server.cs\').write_text(server)\n    output.with_suffix(\'.manifest.json\').write_text(json.dumps({\'digest\':digest,\'phrases\':entries,\'bytes\':output.stat().st_size},indent=2))\n    return entries\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--phrases\',type=Path,required=True);p.add_argument(\'--out\',type=Path,required=True)\n    a=p.parse_args();print(json.dumps(package(json.loads(a.phrases.read_text()),a.out),indent=2))\n', 'build_reports.py': '"""Remote dictionary export, per-speaker predicted coverage and compact script selection."""\nfrom pathlib import Path\nimport argparse\nimport csv\nimport json\nimport importlib.metadata\nimport sys\nfrom phonetics import Lexicon,coverage,select_gaps,transitions,UnknownWords\n\nROOT=Path(__file__).resolve().parent\nEXPECTED=[*(f\'Male{i}\' for i in range(1,6)),*(f\'Fem{i}\' for i in range(1,6)),\n          *(f\'Derm{i}\' for i in range(1,4)),\'Bot1\',\'Announcer\',\'TrainingFemale\',\'BriefingMale\']\n\ndef candidates():\n    subjects=[\'Cody\',\'The blue team\',\'The red team\',\'Our heavy armor\',\'Your scout\',\'The enemy\',\n              \'A quiet warrior\',\'The administrator\',\'Five brave soldiers\',\'The commander\']\n    verbs=[\'will defend\',\'must repair\',\'should protect\',\'can attack\',\'will approach\',\'must watch\']\n    objects=[\'the generator\',\'our flag\',\'the eastern tower\',\'the base\',\'the vehicle station\',\'the bridge\',\n             \'the sensor\',\'the command center\',\'the launch pad\']\n    lines=[f\'{s} {v} {o}.\' for s in subjects for v in verbs for o in objects]\n    lines += [\'Welcome to the server, Cody!\',\'Prepare for battle.\',\'The match begins in sixty seconds.\',\n              \'Your username pronunciation requires an override.\',\'Please join the blue team.\',\n              \'Shields are down. Charge the hill!\',\'Watch the ridge, then cross the valley.\',\n              \'Five quick scouts jump past three heavy tanks.\',\'A veteran warrior quietly repairs the damaged vehicle.\',\n              \'Stop shooting. Check your targets.\',\'The administrator has banned player forty two.\',\n              \'Server maintenance begins in two minutes.\',\'The final score is twenty three to nineteen.\',\n              \'Cody, defend the generator!\',\'A fresh squad has reached the northern entrance.\']\n    return lines\n\ndef main():\n    ap=argparse.ArgumentParser();ap.add_argument(\'--bank\',type=Path,action=\'append\')\n    args=ap.parse_args()\n    import cmudict\n    out=ROOT/\'artifacts/reports\';out.mkdir(parents=True,exist_ok=True)\n    entries=cmudict.dict()\n    licenses=[]\n    dist=importlib.metadata.distribution(\'cmudict\')\n    for path in dist.files or []:\n        if \'license\' in str(path).casefold() and dist.locate_file(path).is_file():\n            licenses.append(str(path)+\'\\n\'+dist.locate_file(path).read_text())\n    if not licenses:raise RuntimeError(\'CMUdict license missing; do not export unattributed dictionary\')\n    (out/\'CMUDICT-LICENSE.txt\').write_text(\'\\n\\n\'.join(licenses))\n    (out/\'cmudict.json\').write_text(json.dumps(entries,separators=(\',\',\':\')))\n    lex=Lexicon.load(out/\'cmudict.json\',ROOT/\'pronunciation-overrides.json\')\n    with (ROOT/\'artifacts/review.csv\').open() as f:rows=list(csv.DictReader(f))\n    asr={}\n    path=ROOT/\'artifacts/asr_candidates.jsonl\'\n    if path.exists():asr={r[\'path\']:r for r in map(json.loads,path.read_text().splitlines())}\n    for r in rows:\n        if not r[\'text\'] and r[\'path\'] in asr:\n            r[\'text\']=asr[r[\'path\']][\'text\']\n        if r[\'group\']==\'Training\' and not r[\'speaker_id\']:\n            # Never collapse all 334 training recordings into a fictional single speaker.\n            r[\'speaker_id\']=\'UNASSIGNED:\'+r[\'path\']\n        if r[\'group\']==\'Announcer\' and not Path(r[\'path\']).name.startswith(\'ANN.\'):\n            r[\'speaker_id\']=\'UNASSIGNED:\'+r[\'path\']\n    bank_paths=args.bank or [ROOT/\'artifacts/stock-alignment/candidate-bank/index.json\',\n                            ROOT/\'artifacts/phone-alignment/candidate-bank/index.json\']\n    units={}\n    for path in bank_paths:\n        if path.exists():\n            for u in json.loads(path.read_text())[\'units\']:\n                units[(u[\'speaker_id\'],u[\'emotion\'],u[\'id\'])]=u\n    report=coverage(rows,lex,units.values())\n    (out/\'coverage-predicted.json\').write_text(json.dumps(report,indent=2))\n    status=[]\n    for speaker in EXPECTED:\n        r=report.get(speaker+\'/neutral\',{})\n        status.append({\'speaker_id\':speaker,\'display_name_verified\':False,\n            \'transcript_predicted_transitions\':len(r.get(\'predicted_transitions\',{})),\n            \'verified_usable_transitions\':len(r.get(\'verified_usable_transitions\',[])),\n            \'emotions\':{e:\'not yet validated\' for e in (\'neutral\',\'angry\',\'sad\',\'excited\',\'calm\')},\n            \'assignment\':\'pending acoustic/source review\' if speaker in (\'TrainingFemale\',\'BriefingMale\') else \'asset pack id, not certified display-name mapping\'})\n    (out/\'voice-status.json\').write_text(json.dumps(status,indent=2))\n    sentences=candidates()\n    # Target = declared Tribes announcement workload. This is NOT all English diphones.\n    target=set()\n    for s in sentences:\n        try:target.update(transitions(lex.pronounce(s)[\'phones\']))\n        except UnknownWords:pass\n    candidate_coverage={}\n    for unit in units.values():\n        key=unit[\'speaker_id\']+\'/\'+unit[\'emotion\']\n        r=candidate_coverage.setdefault(key,{\'unit_count\':0,\'pairs\':set(),\'verified_pairs\':set(),\n                                             \'unreviewed_stop_units\':0})\n        r[\'unit_count\']+=1;r[\'pairs\'].add(tuple(unit[\'phones\']))\n        if unit.get(\'verified\') is True:r[\'verified_pairs\'].add(tuple(unit[\'phones\']))\n        if unit.get(\'release_safe\') is not True:r[\'unreviewed_stop_units\']+=1\n    audio_report={k:{\'unit_count\':r[\'unit_count\'],\'candidate_transition_count\':len(r[\'pairs\']),\n                     \'candidate_transitions\':[\'|\'.join(p) for p in sorted(r[\'pairs\'])],\n                     \'verified_transition_count\':len(r[\'verified_pairs\']),\n                     \'candidate_target_covered\':len(r[\'pairs\']&target),\n                     \'candidate_target_missing\':[\'|\'.join(p) for p in sorted(target-r[\'pairs\'])],\n                     \'unreviewed_stop_units\':r[\'unreviewed_stop_units\']} for k,r in candidate_coverage.items()}\n    (out/\'coverage-audio-candidates.json\').write_text(json.dumps(audio_report,indent=2))\n    for speaker in EXPECTED:\n        existing=[tuple(x.split(\'|\')) for x in report.get(speaker+\'/neutral\',{}).get(\'verified_usable_transitions\',[])]\n        r=select_gaps(sentences,lex,target,existing,max_sentences=60)\n        r.update({\'speaker_id\':speaker,\'target_transition_count\':len(target),\n                  \'baseline\':\'verified usable audio only; hints do not reduce gaps\'})\n        (out/f\'gap-script-{speaker}.json\').write_text(json.dumps(r,indent=2))\n    (out/\'gap-candidates.txt\').write_text(\'\\n\'.join(sentences)+\'\\n\')\n    print(json.dumps({\'rows\':len(rows),\'asr_candidates\':len(asr),\'reports\':len(report),\n                      \'voices\':len(status),\'target_transition_count\':len(target)},indent=2))\n    print(\'========== DONE ==========\',flush=True)\n\nif __name__==\'__main__\':main()\n', 'export_review.py': '"""Side-by-side review candidates; never overwrite the human source-review table."""\nimport csv\nimport json\nfrom pathlib import Path\nimport re\n\nroot=Path(__file__).resolve().parent\nout=root/\'artifacts/reports\';out.mkdir(parents=True,exist_ok=True)\nwith (root/\'artifacts/review.csv\').open() as f:original=list(csv.DictReader(f))\nasr={r[\'path\']:r for r in map(json.loads,(root/\'artifacts/asr_candidates.jsonl\').read_text().splitlines())}\naudit_path=root/\'artifacts/speaker-audit/speaker-comparisons.json\'\naudit={r[\'path\']:r for r in json.loads(audit_path.read_text())[\'records\']} if audit_path.exists() else {}\ndef comparable(s):return re.findall(r"[^\\W\\d_]+(?:\'[^\\W\\d_]+)?",s.casefold())\nrows=[]\nfor row in original:\n    machine=asr.get(row[\'path\'],{})\n    speaker=audit.get(row[\'path\'],{})\n    top=speaker.get(\'top_matches\',[])\n    rows.append({\'path\':row[\'path\'],\'pack_id\':row[\'group\'],\'reviewed_speaker_id\':row[\'speaker_id\'] if row[\'speaker_verified\']==\'true\' else \'\',\n        \'speaker_verified\':row[\'speaker_verified\'],\'speaker_match_candidate\':top[0][0] if top else \'\',\n        \'speaker_cosine_not_probability\':top[0][1] if top else \'\',\n        \'speaker_window_disagreement\':speaker.get(\'window_disagreement\',\'\'),\n        \'stock_transcript_hint\':row[\'text\'],\'ASR_candidate\':machine.get(\'text\',\'\'),\n        \'hint_ASR_disagree\':bool(row[\'text\'] and comparable(row[\'text\'])!=comparable(machine.get(\'text\',\'\'))),\n        \'ASR_empty\':not bool(machine.get(\'text\',\'\').strip()),\'transcript_verified\':row[\'transcript_verified\'],\n        \'notes\':row[\'notes\']})\nwith (out/\'transcript-and-speaker-candidates.csv\').open(\'w\',newline=\'\') as f:\n    w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)\nregistry=out/\'voice-registry.json\'\nif not registry.exists():\n    ids=[*(f\'Male{i}\' for i in range(1,6)),*(f\'Fem{i}\' for i in range(1,6)),*(f\'Derm{i}\' for i in range(1,4)),\n         \'Bot1\',\'Announcer\',\'TrainingFemale\',\'BriefingMale\']\n    values=[]\n    for speaker in ids:\n        values.append({\'id\':speaker,\'in_game_display_name\':None,\'display_mapping_verified\':False,\n            \'speaker_acoustic_verified\':False,\n            \'canonical_role_name_in_mission_source\':{\'TrainingFemale\':\'Kenzie\',\'BriefingMale\':\'Akanruth\'}.get(speaker),\n            \'role_source\':\'Training1.mis briefing text; TrainingGui.cs associates briefing WAV sequence\' if speaker in (\'TrainingFemale\',\'BriefingMale\') else None,\n            \'mapping_evidence\':[],\'notes\':\'Numbered asset-pack ID is not an authenticated in-game display-name mapping.\'})\n    registry.write_text(json.dumps({\'voices\':values,\n        \'secondary_display_name_list_source\':\'https://tribes.fandom.com/wiki/Voice_Packs\',\n        \'secondary_list_not_a_numbered_mapping\':True,\n        \'search_limitations\':\'SearXNG queries had blocked/rate-limited Brave, DuckDuckGo and Wikipedia engines. Search was not exhaustive.\',\n        \'warning\':\'Do not map names from list order or assign all 334 training WAVs to one voice.\'},indent=2))\nprint(json.dumps({\'rows\':len(rows),\'ASR_empty\':sum(r[\'ASR_empty\'] for r in rows),\n                  \'hint_ASR_disagreements\':sum(r[\'hint_ASR_disagree\'] for r in rows),\n                  \'human_source_review_unchanged\':True,\'display_mappings_verified\':0},indent=2))\nprint(\'========== DONE ==========\',flush=True)\n', 'make_alignment_jobs.py': '"""Select a small, explicit phone-alignment smoke; reject unknown pronunciations."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\nimport sys\nfrom phonetics import Lexicon,UnknownWords\n\nroot=Path(__file__).resolve().parent\np=argparse.ArgumentParser();p.add_argument(\'--pilots\',action=\'store_true\');p.add_argument(\'--stock-announcer\',action=\'store_true\')\np.add_argument(\'--stock-character-packs\',action=\'store_true\')\np.add_argument(\'--validation\',type=Path,action=\'append\',default=[])\np.add_argument(\'--out\',type=Path,default=root/\'artifacts/alignment-jobs.json\')\na=p.parse_args();jobs=[];rejected=[]\nlex=Lexicon.load(root/\'artifacts/reports/cmudict.json\',root/\'pronunciation-overrides.json\')\nif a.stock_announcer:\n    rows={r[\'path\']:r for r in map(json.loads,(root/\'artifacts/asr_candidates.jsonl\').read_text().splitlines())}\n    for name in (\'ANN.10sec.WAV\',\'ANN.15sec.WAV\',\'ANN.30.WAV\',\'ANN.60.WAV\',\'ANN.gameover.WAV\',\n                 \'ANN.closegame_01.WAV\',\'ANN.closegame_03.WAV\',\'ANN.closegame_04.WAV\',\n                 \'ANN.match_begins.WAV\',\'ANN.prepare.WAV\',\'ANN.ib.WAV\',\'ANN.oob.WAV\'):\n        path=\'voice/Announcer/\'+name\n        jobs.append({\'id\':name,\'source\':str(root/\'artifacts/pcm24k\'/path),\'text\':rows[path][\'text\'],\n            \'speaker_id\':\'Announcer\',\'emotion\':\'neutral\',\'kind\':\'speech\',\'verified\':False,\n             \'speaker_verified\':False,\'transcript_verified\':False})\nif a.stock_character_packs:\n    sys.path.insert(0,str(root/\'prep\'))\n    from voice_lab import transcript_seeds\n    from expression_support import STOCK_CANDIDATES\n    hints=transcript_seeds()\n    rows=map(json.loads,(root/\'artifacts/asr_candidates.jsonl\').read_text().splitlines())\n    packs={*(f\'Male{i}\' for i in range(1,6)),*(f\'Fem{i}\' for i in range(1,6)),\n           *(f\'Derm{i}\' for i in range(1,4)),\'Bot1\'}\n    def words(text):return re.findall(r"[^\\W\\d_]+(?:\'[^\\W\\d_]+)?",text.casefold())\n    for row in rows:\n        path=Path(row[\'path\']);parts=path.parts\n        if len(parts)!=3 or parts[0]!=\'voice\' or parts[1] not in packs:continue\n        stem=path.stem.casefold()\n        # Expressive/nonverbal stock files stay in their review lane, not neutral speech.\n        if stem in STOCK_CANDIDATES or stem.startswith(\'avo.\'):continue\n        hint=hints.get(stem)\n        if not hint or not words(row[\'text\']) or words(hint)!=words(row[\'text\']):continue\n        jobs.append({\'id\':\'stock_\'+parts[1]+\'_\'+stem,\'source\':str(root/\'artifacts/pcm24k\'/path),\n                     \'text\':row[\'text\'],\'speaker_id\':parts[1],\'emotion\':\'neutral\',\'kind\':\'speech\',\n                     \'verified\':False,\'speaker_verified\':False,\'transcript_verified\':False,\n                     \'selection\':\'stock hint and machine words agree; acoustic/emotion review still required\'})\nif a.pilots:\n    validations={r[\'id\']:r for r in json.loads((root/\'artifacts/pilot/asr-validation.json\').read_text())}\n    for path in a.validation:\n        for record in json.loads(path.read_text()):\n            if record[\'id\'] in validations and validations[record[\'id\']]!=record:\n                raise ValueError(\'Conflicting validation for \'+record[\'id\'])\n            validations[record[\'id\']]=record\n    for job in json.loads((root/\'artifacts/pilot/pilot-jobs.json\').read_text()):\n        if job[\'kind\']!=\'speech\':continue\n        checked=validations.get(job[\'id\'],{})\n        digest=hashlib.sha256(Path(job[\'source\']).read_bytes()).hexdigest()\n        if (not job.get(\'complete_by_frame_guard\') or checked.get(\'exact_normalized_words\') is not True\n                or checked.get(\'source_sha256\')!=digest):\n            rejected.append({\'id\':job[\'id\'],\'reason\':\'missing/mismatched/failed generated-word validation\'})\n            continue\n        jobs.append(job)\naccepted=[]\nfor job in jobs:\n    try:lex.pronounce(job[\'text\']);accepted.append(job)\n    except UnknownWords as e:rejected.append({\'id\':job[\'id\'],\'unknown\':e.words})\na.out.parent.mkdir(parents=True,exist_ok=True);a.out.write_text(json.dumps(accepted,indent=2))\na.out.with_suffix(\'.rejected.json\').write_text(json.dumps(rejected,indent=2))\nif not accepted:raise ValueError(\'No alignable jobs\')\nprint(json.dumps({\'accepted\':len(accepted),\'rejected\':rejected,\'jobs\':str(a.out)},indent=2))\n', 'build_event_auditions.py': '"""Playable stock/event-cue review set; filename and ASR labels remain hypotheses."""\nimport csv\nimport hashlib\nimport html\nimport json\nfrom pathlib import Path\nimport re\nimport shutil\n\nroot=Path(__file__).resolve().parent\nout=root/\'artifacts/event-auditions\';out.mkdir(parents=True,exist_ok=True)\nwith (root/\'artifacts/expressive_review.csv\').open() as f:stock=list(csv.DictReader(f))\nasr={r[\'path\']:r for r in map(json.loads,(root/\'artifacts/asr_candidates.jsonl\').read_text().splitlines())}\nseen={r[\'source_path\'] for r in stock}\nextra=[]\nfor name,row in asr.items():\n    if name not in seen and re.search(r\'\\b(?:ha[ ,!.-]*){3,}\',row[\'text\'],re.I):\n        extra.append({\'source_path\':name,\'speaker_id\':name.split(\'/\')[1] if \'/Training/\' not in name else \'\',\n                      \'candidate_label\':\'possible_laugh\',\'candidate_type\':\'ASR hint, not classification\'})\nentries=[]\nfor row in [*stock,*extra]:\n    source=root/\'artifacts/pcm24k\'/row[\'source_path\']\n    digest=hashlib.sha256(source.read_bytes()).hexdigest()\n    file=\'original_\'+digest[:20]+\'.wav\';shutil.copy2(source,out/file)\n    entries.append({\'source\':row[\'source_path\'],\'file\':file,\'sha256\':digest,\'speaker_candidate\':row[\'speaker_id\'],\n        \'label_candidate\':row[\'candidate_label\'],\'kind\':\'stock_candidate\',\'hint_type\':row[\'candidate_type\'],\n        \'asr_text\':asr.get(row[\'source_path\'],{}).get(\'text\',\'\'),\'speaker_verified\':\'false\',\'event_verified\':\'false\',\n        \'boundaries_verified\':\'false\',\'emotion\':\'\',\'event_type\':\'\',\'start_seconds\':\'\',\'end_seconds\':\'\',\'contains_carrier_speech\':\'\',\'notes\':\'\'})\nfor event in (\'laugh\',\'sigh\',\'cough\',\'clear_throat\'):\n    source=root/\'artifacts/pilot\'/f\'Announcer_event_{event}.wav\'\n    if not source.exists():continue\n    file=\'generated_\'+source.name;shutil.copy2(source,out/file)\n    entries.append({\'source\':str(source),\'file\':file,\'sha256\':hashlib.sha256(source.read_bytes()).hexdigest(),\n        \'speaker_candidate\':\'Announcer\',\'label_candidate\':event,\'kind\':\'generated_carrier_sentence\',\n        \'hint_type\':\'documented cue; actual event not verified\',\'asr_text\':\'\',\'speaker_verified\':\'false\',\'event_verified\':\'false\',\n        \'boundaries_verified\':\'false\',\'emotion\':\'\',\'event_type\':\'\',\'start_seconds\':\'\',\'end_seconds\':\'\',\'contains_carrier_speech\':\'true\',\'notes\':\'Audition and isolate; never export the whole carrier as an event.\'})\nreview=out/\'review.csv\'\nif review.exists():\n    old_bytes=review.read_bytes();history=out/\'review-history\';history.mkdir(exist_ok=True)\n    (history/(hashlib.sha256(old_bytes).hexdigest()[:16]+\'.csv\')).write_bytes(old_bytes)\n    with review.open() as f:old={(r[\'file\'],r[\'sha256\']):r for r in csv.DictReader(f)}\n    for row in entries:\n        previous=old.get((row[\'file\'],row[\'sha256\']),{})\n        for key in (\'speaker_verified\',\'event_verified\',\'boundaries_verified\',\'emotion\',\'event_type\',\'start_seconds\',\'end_seconds\',\'contains_carrier_speech\',\'notes\'):\n            if key in previous:row[key]=previous[key]\nwith (out/\'review.csv\').open(\'w\',newline=\'\') as f:\n    w=csv.DictWriter(f,fieldnames=list(entries[0]));w.writeheader();w.writerows(entries)\n(out/\'manifest.json\').write_text(json.dumps(entries,indent=2))\nparts=[\'<!doctype html><meta charset="utf-8"><title>Vocal event review</title><h1>Unverified vocal-event candidates</h1>\',\n    \'<p>A spoken aww is not crying. Spoken cheers are not wordless sounds. A cue does not prove an event occurred. \'\n    \'Generated carrier sentences must be listened to and bounded before export.</p>\']\nfor e in entries:\n    parts.append(\'<h3>\'+html.escape(e[\'speaker_candidate\']+\' / \'+e[\'label_candidate\']+\' / \'+e[\'kind\'])+\'</h3>\'+\n                 \'<audio controls preload="none" src="\'+e[\'file\']+\'"></audio><p>ASR hint: \'+html.escape(e[\'asr_text\'])+\'</p>\')\n(out/\'index.html\').write_text(\'\\n\'.join(parts))\nprint(json.dumps({\'stock_candidates\':len(stock),\'additional_ASR_hints\':len(extra),\'generated_carriers\':len(entries)-len(stock)-len(extra),\n                  \'verified_events\':0,\'bytes\':sum(p.stat().st_size for p in out.iterdir() if p.is_file())},indent=2))\nprint(\'========== DONE ==========\',flush=True)\n', 'completed_snapshot.py': '"""Freeze completed job receipts without reading a live aggregate manifest."""\nimport argparse\nimport fnmatch\nimport hashlib\nimport json\nfrom pathlib import Path\n\nroot=Path(__file__).resolve().parent\np=argparse.ArgumentParser();p.add_argument(\'--id-pattern\',action=\'append\',default=[])\np.add_argument(\'--output\',type=Path,required=True);p.add_argument(\'--expected\',type=int)\na=p.parse_args();jobs=[]\nfor path in sorted((root/\'artifacts/pilot\').glob(\'*.json\')):\n    if a.id_pattern and not any(fnmatch.fnmatchcase(path.stem,pattern) for pattern in a.id_pattern):continue\n    job=json.loads(path.read_text())\n    if not isinstance(job,dict) or job.get(\'id\')!=path.stem:continue\n    source=path.with_suffix(\'.wav\')\n    if Path(job[\'source\']).resolve()!=source.resolve():raise ValueError(\'Job audio path changed\')\n    if not job.get(\'complete_by_frame_guard\'):raise ValueError(\'Incomplete generation: \'+job[\'id\'])\n    if hashlib.sha256(source.read_bytes()).hexdigest()!=job[\'sha256\']:raise ValueError(\'Changed audio: \'+job[\'id\'])\n    jobs.append(job)\nif a.expected is not None and len(jobs)!=a.expected:raise ValueError(f\'Expected {a.expected} jobs, found {len(jobs)}\')\na.output.parent.mkdir(parents=True,exist_ok=True);a.output.write_text(json.dumps(jobs,indent=2))\nprint(json.dumps({\'completed_jobs\':len(jobs),\'snapshot\':str(a.output)},indent=2))\nprint(\'========== DONE ==========\',flush=True)\n', 'test_engine.py': '"""Remote CPU regression/benchmark suite. Synthetic fixtures are NOT voice coverage."""\nfrom array import array\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport tempfile\nimport time\nimport unittest\nfrom bank import extract,write_wav,write_index,read_wav,safe_path\nfrom phonetics import Lexicon,UnknownWords,number,normalize,coverage,select_gaps\nfrom renderer import Renderer,MissingUnit,markup\n\nclass EngineTests(unittest.TestCase):\n    def setUp(self):\n        self.tmp=tempfile.TemporaryDirectory()\n        self.root=Path(self.tmp.name)\n        self.lex=Lexicon({\'hello\':[[\'HH\',\'AH0\',\'L\',\'OW1\']], \'world\':[[\'W\',\'ER1\',\'L\',\'D\']]})\n    def tearDown(self):self.tmp.cleanup()\n    def make_bank(self,emotion=\'neutral\'):\n        src=self.root/\'source.wav\'; rate=16000\n        write_wav(src,rate,array(\'h\',(round(9000*math.sin(2*math.pi*180*i/rate)) for i in range(rate))))\n        labels=[\'SIL\',\'HH\',\'AH0\',\'L\',\'OW1\',\'SIL\']\n        alignment={\'source\':str(src),\'speaker_id\':\'synthetic\',\'emotion\':emotion,\n                   \'speaker_verified\':True,\'transcript_verified\':True,\'alignment_verified\':True,\n                   \'emotion_verified\':True,\'phones\':[{\'label\':p,\'start\':i/6,\'end\':(i+1)/6} for i,p in enumerate(labels)]}\n        bank=self.root/\'bank\'\n        units=extract(alignment,bank)\n        write_index(bank,units)\n        return bank,units\n    def test_numbers(self):\n        self.assertEqual(number(2042),\'two thousand forty two\')\n        self.assertEqual(normalize(\'/kick Cody_42 in 2.5 seconds\'), \'kick cody forty two in two point five seconds\')\n    def test_username_override(self):\n        self.assertEqual(normalize(\'CodeMasterCody3D\',{\'CodeMasterCody3D\':\'Cody\'}),\'cody\')\n    def test_cross_word(self):\n        p=self.lex.pronounce(\'hello world\')\n        self.assertEqual(p[\'cross_word\'],[(\'OW1\',\'W\')])\n        self.assertEqual(p[\'phones\'][0],\'SIL\'); self.assertEqual(p[\'phones\'][-1],\'SIL\')\n    def test_unknown_is_explicit(self):\n        with self.assertRaises(UnknownWords) as cm:self.lex.pronounce(\'hello xzq\')\n        self.assertEqual(cm.exception.words,[\'xzq\'])\n    def test_unicode_username_is_not_silently_removed(self):\n        with self.assertRaises(UnknownWords) as cm:self.lex.pronounce(\'hello 王\')\n        self.assertEqual(cm.exception.words,[\'王\'])\n    def test_dictionary_is_immutable(self):\n        with self.assertRaises(TypeError):self.lex.entries[\'hello\']=[[\'P\']]\n        with self.assertRaises(TypeError):self.lex.entries[\'hello\'][0][0]=\'P\'\n        with self.assertRaises(AttributeError):self.lex.entries={}\n    def path_renderer(self,lex,phones,**kwargs):\n        bank=self.root/\'paths\';bank.mkdir(exist_ok=True)\n        units=[{\'speaker_id\':\'synthetic\',\'emotion\':\'neutral\',\'phones\':list(pair),\'verified\':True}\n               for pair in zip(phones,phones[1:])]\n        (bank/\'index.json\').write_text(json.dumps({\'units\':units,\'events\':[]}))\n        return Renderer(bank,lex,bank_pronunciations=True,**kwargs)\n    def test_bank_dictionary_alternative_is_explicit(self):\n        lex=Lexicon({\'to\':[[\'T\',\'UW1\'],[\'T\',\'AH0\']],\'the\':[[\'DH\',\'AH0\']]})\n        phones=[\'SIL\',\'T\',\'AH0\',\'DH\',\'AH0\',\'SIL\']\n        r=self.path_renderer(lex,phones)\n        self.assertEqual(r.pronounce_for_bank(\'synthetic\',\'neutral\',\'to the\')[\'phones\'],phones)\n        self.assertEqual(r.warnings[0][\'pronunciation_choice\'],\'to\')\n        self.assertEqual(lex.pronounce(\'to the\')[\'phones\'][2],\'UW1\')\n    def test_bank_paths_look_ahead_without_greedy_failure(self):\n        lex=Lexicon({\'a\':[[\'AH0\'],[\'AH1\']],\'word\':[[\'W\']]})\n        r=self.path_renderer(lex,[\'SIL\',\'AH1\',\'W\',\'SIL\'])\n        r.available[(\'synthetic\',\'neutral\')].add((\'SIL\',\'AH0\'))\n        self.assertEqual(r.pronounce_for_bank(\'synthetic\',\'neutral\',\'a word\')[\'phones\'],[\'SIL\',\'AH1\',\'W\',\'SIL\'])\n    def test_bank_path_respects_explicit_override(self):\n        lex=Lexicon({\'to\':[[\'T\',\'UW1\'],[\'T\',\'AH0\']]},{\'to\':[\'T\',\'UW1\']})\n        r=self.path_renderer(lex,[\'SIL\',\'T\',\'AH0\',\'SIL\'])\n        with self.assertRaises(MissingUnit):r.pronounce_for_bank(\'synthetic\',\'neutral\',\'to\')\n    def test_bank_path_comma_choice_reported(self):\n        lex=Lexicon({\'one\':[[\'W\',\'AH1\',\'N\']],\'two\':[[\'T\',\'UW1\']]})\n        phones=[\'SIL\',\'W\',\'AH1\',\'N\',\'T\',\'UW1\',\'SIL\']\n        r=self.path_renderer(lex,phones)\n        self.assertEqual(r.pronounce_for_bank(\'synthetic\',\'neutral\',\'one, two\')[\'phones\'],phones)\n        self.assertTrue(any(x.get(\'pause_choice\')==\'comma_without_recorded_silence\' for x in r.warnings))\n        with self.assertRaises(MissingUnit):r.pronounce_for_bank(\'synthetic\',\'neutral\',\'one. two\')\n    def test_bank_path_recorded_word_pause(self):\n        lex=Lexicon({\'one\':[[\'W\',\'AH1\',\'N\']],\'two\':[[\'T\',\'UW1\']]})\n        phones=[\'SIL\',\'W\',\'AH1\',\'N\',\'SIL\',\'T\',\'UW1\',\'SIL\']\n        r=self.path_renderer(lex,phones)\n        self.assertEqual(r.pronounce_for_bank(\'synthetic\',\'neutral\',\'one two\')[\'phones\'],phones)\n        self.assertTrue(any(x.get(\'pause_choice\')==\'recorded_word_boundary\' for x in r.warnings))\n    def test_bank_paths_never_borrow_voice_style_or_review(self):\n        lex=Lexicon({\'one\':[[\'W\',\'AH1\',\'N\']]})\n        r=self.path_renderer(lex,[\'SIL\',\'W\',\'AH1\',\'N\',\'SIL\'])\n        for speaker,emotion in ((\'other\',\'neutral\'),(\'synthetic\',\'angry\')):\n            with self.assertRaises(MissingUnit):r.pronounce_for_bank(speaker,emotion,\'one\')\n        data=json.loads((r.root/\'index.json\').read_text())\n        for unit in data[\'units\']:unit[\'verified\']=False\n        (r.root/\'index.json\').write_text(json.dumps(data))\n        r=Renderer(r.root,lex,bank_pronunciations=True)\n        with self.assertRaises(MissingUnit):r.pronounce_for_bank(\'synthetic\',\'neutral\',\'one\')\n    def test_bank_paths_unknown_and_missing_remain_errors(self):\n        r=self.path_renderer(self.lex,[\'SIL\',\'HH\',\'AH0\',\'L\',\'OW1\',\'SIL\'])\n        with self.assertRaises(UnknownWords):r.pronounce_for_bank(\'synthetic\',\'neutral\',\'hello xzq\')\n        with self.assertRaises(MissingUnit):r.pronounce_for_bank(\'synthetic\',\'neutral\',\'world\')\n    def test_bank_mode_cache_isolated_from_strict(self):\n        bank,_=self.make_bank();out=self.root/\'out.wav\'\n        strict=Renderer(bank,self.lex).render(\'synthetic\',\'hello\',out,self.root/\'cache\')\n        bank_result=Renderer(bank,self.lex,bank_pronunciations=True).render(\'synthetic\',\'hello\',out,self.root/\'cache\')\n        self.assertNotEqual(strict[\'cache_key\'],bank_result[\'cache_key\'])\n        self.assertEqual(strict[\'output_sha256\'],bank_result[\'output_sha256\'])\n    def test_context_selects_reviewed_variant(self):\n        bank,units=self.make_bank()\n        original=units[1]\n        alternate={**original,\'id\':\'alternate\',\'variants\':[\'reviewed_allophone\',\'canonical\'],\n                   \'context\':{\'previous\':\'K\',\'next\':\'AH0\'}}\n        write_index(bank,[*units,alternate])\n        selected=Renderer(bank,self.lex).select(\'synthetic\',\'neutral\',original[\'phones\'],\n            context={\'previous\':\'K\',\'next\':\'AH0\'})\n        self.assertEqual(selected[\'id\'],\'alternate\')\n    def test_markup(self):\n        self.assertEqual(markup(\'[angry]hello[/angry] [grunt]\'),[(\'speech\',\'angry\',\'hello\'),(\'event\',\'neutral\',\'grunt\')])\n    def test_bad_markup(self):\n        for text in (\'[angry]hello\',\'[angry][sad]hello[/sad][/angry]\',\'[foo]\',\'[/sad]\'):\n            with self.assertRaises(ValueError):markup(text)\n    def test_missing_speaker(self):\n        bank,_=self.make_bank()\n        with self.assertRaises(MissingUnit):Renderer(bank,self.lex).render(\'other\',\'hello\',self.root/\'out.wav\')\n    def test_missing_phone(self):\n        bank,_=self.make_bank()\n        with self.assertRaises(MissingUnit):Renderer(bank,self.lex).render(\'synthetic\',\'world\',self.root/\'out.wav\')\n    def test_missing_event(self):\n        bank,_=self.make_bank()\n        with self.assertRaises(MissingUnit):Renderer(bank,self.lex).render(\'synthetic\',\'[grunt]\',self.root/\'out.wav\')\n    def test_emotion_fallback_explicit(self):\n        bank,_=self.make_bank()\n        with self.assertRaises(MissingUnit):Renderer(bank,self.lex).render(\'synthetic\',\'[angry]hello[/angry]\',self.root/\'out.wav\')\n        r=Renderer(bank,self.lex,emotion_fallback=\'neutral\').render(\'synthetic\',\'[angry]hello[/angry]\',self.root/\'out.wav\')\n        self.assertTrue(r[\'warnings\'])\n    def test_complementary_crossfade_duration(self):\n        bank,units=self.make_bank()\n        out=self.root/\'out.wav\'\n        Renderer(bank,self.lex).render(\'synthetic\',\'hello\',out)\n        rate,audio=read_wav(out)\n        self.assertEqual(len(audio),sum(u[\'core_samples\'] for u in units))\n        # Same-source adjacent units reconstruct the original samples exactly.\n        _,original=read_wav(self.root/\'source.wav\')\n        a=round(units[0][\'start_seconds\']*rate)\n        self.assertEqual(audio,original[a:a+len(audio)])\n    def test_cache(self):\n        bank,_=self.make_bank();r=Renderer(bank,self.lex); out=self.root/\'out.wav\'\n        first=r.render(\'synthetic\',\'hello\',out,self.root/\'cache\')\n        second=r.render(\'synthetic\',\'hello\',out,self.root/\'cache\')\n        self.assertFalse(first[\'cache_hit\']);self.assertTrue(second[\'cache_hit\'])\n        self.assertEqual(first[\'cache_key\'],second[\'cache_key\'])\n        r.lexicon.aliases={\'greeting\':\'hello\'}\n        self.assertNotEqual(r.render(\'synthetic\',\'hello\',out,self.root/\'cache\')[\'cache_key\'],first[\'cache_key\'])\n    def test_corrupt_sample(self):\n        bank,units=self.make_bank();p=bank/units[0][\'file\'];p.write_bytes(p.read_bytes()+b\'x\')\n        with self.assertRaisesRegex(ValueError,\'checksum\'):Renderer(bank,self.lex).render(\'synthetic\',\'hello\',self.root/\'out.wav\')\n    def test_unverified_refused(self):\n        bank,units=self.make_bank();units[0][\'verified\']=False;write_index(bank,units)\n        with self.assertRaises(MissingUnit):Renderer(bank,self.lex).render(\'synthetic\',\'hello\',self.root/\'out.wav\')\n    def test_candidate_audition_is_explicit(self):\n        bank,units=self.make_bank();units[0][\'verified\']=False;write_index(bank,units)\n        result=Renderer(bank,self.lex,audition_unverified=True).render(\'synthetic\',\'hello\',self.root/\'out.wav\')\n        self.assertTrue(any(\'unverified_audition_unit\' in w for w in result[\'warnings\']))\n    def test_corrupt_cache_rebuilt(self):\n        bank,_=self.make_bank();r=Renderer(bank,self.lex);out=self.root/\'out.wav\'\n        result=r.render(\'synthetic\',\'hello\',out,self.root/\'cache\')\n        cached=self.root/\'cache\'/(result[\'cache_key\']+\'.wav\');cached.write_bytes(b\'partial\')\n        rebuilt=r.render(\'synthetic\',\'hello\',out,self.root/\'cache\')\n        self.assertFalse(rebuilt[\'cache_hit\']);self.assertGreater(out.stat().st_size,44)\n    def test_path_traversal(self):\n        with self.assertRaises(ValueError):safe_path(self.root,\'../../outside.wav\')\n    def test_predicted_not_verified(self):\n        r=coverage([{\'speaker_id\':\'synthetic\',\'text\':\'hello\'}],self.lex)[\'synthetic/neutral\']\n        self.assertTrue(r[\'predicted_transitions\']);self.assertFalse(r[\'verified_usable_transitions\'])\n        self.assertEqual(r[\'unverified_transcript_rows\'],1)\n    def test_speaker_coverage_isolated(self):\n        r=coverage([{\'speaker_id\':\'one\',\'text\':\'hello\'},{\'speaker_id\':\'two\',\'text\':\'world\'}],self.lex)\n        self.assertNotEqual(r[\'one/neutral\'][\'predicted_transitions\'],r[\'two/neutral\'][\'predicted_transitions\'])\n    def test_gap_selector(self):\n        r=select_gaps([\'hello world\',\'hello\',\'xzq\'],self.lex,[(\'OW1\',\'W\'),(\'P\',\'Z\')])\n        self.assertEqual(len(r[\'selected\']),1); self.assertEqual(r[\'uncovered\'],[\'P|Z\']);self.assertEqual(len(r[\'rejected\']),1)\n    def test_bad_alignment(self):\n        src=self.root/\'source.wav\';write_wav(src,16000,[0]*16000)\n        with self.assertRaises(ValueError):extract({\'source\':str(src),\'phones\':[{\'label\':\'AH0\',\'start\':0,\'end\':2}]},self.root/\'bank\')\n    def test_stop_not_verified_without_release(self):\n        src=self.root/\'source.wav\';write_wav(src,16000,[0]*16000)\n        units=extract({\'source\':str(src),\'speaker_id\':\'synthetic\',\'speaker_verified\':True,\n            \'transcript_verified\':True,\'alignment_verified\':True,\'emotion_verified\':True,\n            \'phones\':[{\'label\':\'P\',\'start\':0,\'end\':0.5},{\'label\':\'AH0\',\'start\':0.5,\'end\':1}]},self.root/\'bank\')\n        self.assertFalse(units[0][\'verified\']);self.assertFalse(units[0][\'release_safe\'])\n    def test_event_whole_sample(self):\n        bank,units=self.make_bank();p=bank/\'event.wav\';write_wav(p,16000,[500]*8000)\n        write_index(bank,units,[{\'id\':\'event\',\'speaker_id\':\'synthetic\',\'emotion\':\'neutral\',\'event\':\'grunt\',\n            \'verified\':True,\'file\':\'event.wav\',\'sha256\':hashlib.sha256(p.read_bytes()).hexdigest()}])\n        out=self.root/\'out.wav\';r=Renderer(bank,self.lex).render(\'synthetic\',\'[grunt]\',out)\n        self.assertEqual(read_wav(out)[1],array(\'h\',[500]*8000))\n\ndef benchmark():\n    t=EngineTests();t.setUp()\n    try:\n        bank,units=t.make_bank();r=Renderer(bank,t.lex);out=t.root/\'bench.wav\'\n        uncached=[];cached=[]\n        for i in range(30):\n            uncached.append(r.render(\'synthetic\',\'hello\',out)[\'wall_seconds\'])\n        r.render(\'synthetic\',\'hello\',out,t.root/\'cache\')\n        for i in range(30):\n            cached.append(r.render(\'synthetic\',\'hello\',out,t.root/\'cache\')[\'wall_seconds\'])\n        return {\'fixture\':\'synthetic same-source sine wave, not speech quality evidence\',\'host\':\'assigned Kaggle CPU\',\n            \'unit_count\':len(units),\'bank_bytes\':sum(p.stat().st_size for p in bank.rglob(\'*\') if p.is_file()),\n            \'audio_seconds\':read_wav(out)[1].__len__()/16000,\'uncached_mean_seconds\':sum(uncached)/30,\n            \'cached_mean_seconds\':sum(cached)/30,\'runs_per_condition\':30}\n    finally:t.tearDown()\n\nif __name__==\'__main__\':\n    if not str(Path(__file__).resolve()).startswith(\'/kaggle/\'):\n        raise RuntimeError(\'Run tests on remote compute\')\n    suite=unittest.defaultTestLoader.loadTestsFromTestCase(EngineTests)\n    result=unittest.TextTestRunner(verbosity=2).run(suite)\n    Path(\'test-results.json\').write_text(json.dumps({\'tests\':result.testsRun,\'failures\':len(result.failures),\n        \'errors\':len(result.errors),\'benchmark\':benchmark() if result.wasSuccessful() else None},indent=2))\n    raise SystemExit(0 if result.wasSuccessful() else 1)\n', 'setup_remote.py': '"""Isolated remote dependencies; records exact resolved source/model revisions."""\nimport argparse\nimport importlib.metadata\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\nimport urllib.request\n\ndef run(*args,**kwargs):\n    subprocess.run(list(map(str,args)),check=True,**kwargs)\n\ndef create_env(path, system=False):\n    if not (path/\'bin/python\').exists():\n        run(sys.executable,\'-m\',\'venv\',\'--without-pip\',*([\'--system-site-packages\'] if system else []),path)\n    # Kaggle\'s Python 3.13 omits ensurepip. The host pip can seed an isolated env.\n    run(sys.executable,\'-m\',\'pip\',\'--python\',path/\'bin/python\',\'install\',\'pip\')\n\ndef main():\n    ap=argparse.ArgumentParser();ap.add_argument(\'stage\',choices=[\'asr\',\'breeze\',\'align\'])\n    a=ap.parse_args()\n    root=Path(__file__).resolve().parent\n    assert str(root).startswith(\'/kaggle/\'), \'Remote only\'\n    env=root/\'environments\';env.mkdir(exist_ok=True)\n    lock=json.loads((root/\'toolchain-lock.json\').read_text()) if (root/\'toolchain-lock.json\').exists() else {}\n    start=time.time()\n    if a.stage==\'asr\':\n        venv=env/\'asr\'\n        create_env(venv,system=True)\n        run(venv/\'bin/python\',\'-m\',\'pip\',\'install\',\'faster-whisper==1.2.1\',\'av==16.0.1\',\'cmudict==1.1.1\',\'textgrid==1.6.1\',\n            \'nvidia-cublas-cu12==12.8.4.1\',\'nvidia-cudnn-cu12==9.10.2.21\')\n        freeze=subprocess.check_output([str(venv/\'bin/python\'),\'-m\',\'pip\',\'freeze\'],text=True)\n    elif a.stage==\'breeze\':\n        source=env/\'breeze-tts\'\n        if not source.exists():\n            if lock.get(\'breeze_source_revision\'):\n                run(\'git\',\'init\',source)\n                run(\'git\',\'-C\',source,\'remote\',\'add\',\'origin\',\'https://github.com/breezeblue-ai/breeze-tts.git\')\n                run(\'git\',\'-C\',source,\'fetch\',\'--depth\',\'1\',\'origin\',lock[\'breeze_source_revision\'])\n                run(\'git\',\'-C\',source,\'checkout\',\'--detach\',\'FETCH_HEAD\')\n            else:\n                run(\'git\',\'clone\',\'--depth\',\'1\',\'https://github.com/breezeblue-ai/breeze-tts.git\',source)\n        rev=subprocess.check_output([\'git\',\'-C\',str(source),\'rev-parse\',\'HEAD\'],text=True).strip()\n        if lock.get(\'breeze_source_revision\') and rev!=lock[\'breeze_source_revision\']:\n            raise RuntimeError(\'Existing Breeze source differs from lock; preserve it and use a separate environment\')\n        (env/\'breeze-source-revision.txt\').write_text(rev+\'\\n\')\n        venv=env/\'breeze\'\n        create_env(venv)\n        run(venv/\'bin/python\',\'-m\',\'pip\',\'install\',\'--upgrade\',\'pip\')\n        requirements=root/\'requirements-breeze.lock.txt\'\n        run(venv/\'bin/python\',\'-m\',\'pip\',\'install\',\'-r\',requirements if requirements.exists() else source/\'requirements.txt\')\n        # A model revision pin is resolved once and reused on every later invocation.\n        pin=env/\'breeze-model-revision.json\'\n        if not pin.exists():\n            if lock.get(\'breeze_model\'):\n                pin.write_text(json.dumps(lock[\'breeze_model\'],indent=2))\n            else:\n                with urllib.request.urlopen(\'https://huggingface.co/api/models/BreezeBlue/Breeze-TTS-2\',timeout=60) as r:\n                    d=json.load(r)\n                pin.write_text(json.dumps({\'repo\':\'BreezeBlue/Breeze-TTS-2\',\'revision\':d[\'sha\']},indent=2))\n        if lock.get(\'breeze_model\') and json.loads(pin.read_text())!=lock[\'breeze_model\']:\n            raise RuntimeError(\'Existing model revision differs from lock; refusing to replace it\')\n        run(venv/\'bin/python\',\'-c\',\n            \'import json,sys; from huggingface_hub import snapshot_download; d=json.load(open(sys.argv[1])); snapshot_download(d["repo"], revision=d["revision"],local_dir=sys.argv[2])\',pin,env/\'breeze-model\')\n        freeze=subprocess.check_output([str(venv/\'bin/python\'),\'-m\',\'pip\',\'freeze\'],text=True)\n    else:\n        # MFA is isolated from both Torch environments and supplies its own Kaldi dependencies.\n        micromamba=env/\'bin/micromamba\'\n        if not micromamba.exists():\n            archive=env/\'micromamba.tar.bz2\'\n            urllib.request.urlretrieve(\'https://micro.mamba.pm/api/micromamba/linux-64/latest\',archive)\n            run(\'tar\',\'-xjf\',archive,\'-C\',env,\'bin/micromamba\')\n        prefix=env/\'mfa\'\n        run(micromamba,\'create\',\'-y\',\'-p\',prefix,\'-c\',\'conda-forge\',\'montreal-forced-aligner=3.3.9\',\'python=3.11\')\n        for kind,name in [(\'acoustic\',\'english_us_arpa\'),(\'dictionary\',\'english_us_arpa\')]:\n            run(micromamba,\'run\',\'-p\',prefix,\'mfa\',\'model\',\'download\',kind,name)\n        base=Path.home()/\'Documents/MFA/pretrained_models\'\n        for key,path in ((\'mfa_acoustic_sha256\',base/\'acoustic/english_us_arpa.zip\'),\n                         (\'mfa_dictionary_sha256\',base/\'dictionary/english_us_arpa.dict\')):\n            if lock.get(key) and hashlib.sha256(path.read_bytes()).hexdigest()!=lock[key]:\n                raise RuntimeError(\'Downloaded MFA model differs from tested lock: \'+str(path))\n        freeze=subprocess.check_output([str(micromamba),\'list\',\'-p\',str(prefix),\'--json\'],text=True)\n    (env/(a.stage+\'-resolved.txt\')).write_text(freeze)\n    (env/(a.stage+\'-setup.json\')).write_text(json.dumps({\'stage\':a.stage,\'seconds\':time.time()-start},indent=2))\n    print(\'========== DONE ==========\',flush=True)\n\nif __name__==\'__main__\':main()\n', 'record_environment.py': '"""Record non-secret dependency and model provenance for the remote preparation."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport subprocess\nimport shutil\nimport sys\n\nroot=Path(__file__).resolve().parent\nout=root/\'artifacts/provenance\';out.mkdir(parents=True,exist_ok=True)\ndata={\'host_python\':sys.version,\'source_revision\':json.loads((root/\'prep/assets.json\').read_text())[\'revision\'],\n      \'scripts\':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in root.glob(\'*.py\')},\n      \'hardware\':subprocess.check_output([\'nvidia-smi\',\'--query-gpu=name,driver_version,memory.total\',\'--format=csv\'],text=True)}\ncache=Path.home()/\'.cache/huggingface/hub/models--Systran--faster-whisper-large-v3/snapshots\'\ndata[\'asr_model\']={\'repo\':\'Systran/faster-whisper-large-v3\',\'cached_snapshots\':sorted(p.name for p in cache.iterdir()) if cache.exists() else [],\n                   \'note\':\'Snapshot inventory retained; alias resolution was performed by faster-whisper.\'}\nfor name in (\'breeze-model-revision.json\',\'breeze-source-revision.txt\',\'asr-setup.json\',\'breeze-setup.json\',\'align-setup.json\'):\n    path=root/\'environments\'/name\n    if path.exists():data[name]=path.read_text()\nfor stage in (\'asr\',\'breeze\'):\n    python=root/\'environments\'/stage/\'bin/python\'\n    if python.exists():\n        (out/(stage+\'-final-dependencies.txt\')).write_text(subprocess.check_output([str(python),\'-m\',\'pip\',\'freeze\'],text=True))\nmfa_root=Path.home()/\'Documents/MFA/pretrained_models\'\ndata[\'mfa_models\']=[{\'path\':str(p),\'bytes\':p.stat().st_size,\'sha256\':hashlib.sha256(p.read_bytes()).hexdigest()}\n                    for p in mfa_root.rglob(\'*\') if p.is_file() and \'english_us_arpa\' in p.name]\nlicense_source=root/\'environments/breeze-model/LICENSE\'\nif license_source.exists():\n    licenses=out/\'licenses\';licenses.mkdir(exist_ok=True)\n    shutil.copy2(license_source,licenses/\'Breeze-TTS-2-LICENSE.txt\')\n    (licenses/\'NOTICE.txt\').write_text(\'Breeze TTS 2 is licensed under the BreezeBlue Research and Non-Commercial \'\n        \'License Agreement. Copyright (c) 2026 RESONIA, INC. All Rights Reserved.\\n\'\n        \'Derived from Breeze TTS 2 by BreezeBlue and licensed for research and non-commercial use only.\\n\'\n        \'This notice does not grant rights in third-party game recordings or performances.\\n\')\n    data[\'breeze_license_sha256\']=hashlib.sha256(license_source.read_bytes()).hexdigest()\n(out/\'environment.json\').write_text(json.dumps(data,indent=2))\nprint(json.dumps({\'scripts\':len(data[\'scripts\']),\'asr_snapshots\':data[\'asr_model\'][\'cached_snapshots\'],\n                  \'mfa_files\':len(data[\'mfa_models\'])},indent=2))\nprint(\'========== DONE ==========\',flush=True)\n', 'asr_entry.py': '"""Expose pip CUDA libraries to CTranslate2 before process startup."""\nimport os\nfrom pathlib import Path\nimport runpy\nimport sys\n\nif not os.environ.get(\'T2_ASR_LIBS_READY\'):\n    dirs=[str(p) for path in sys.path for p in Path(path).glob(\'nvidia/*/lib\') if Path(path).is_dir()]\n    env=dict(os.environ,T2_ASR_LIBS_READY=\'1\')\n    env[\'LD_LIBRARY_PATH\']=\':\'.join(dirs+[env.get(\'LD_LIBRARY_PATH\',\'\')])\n    os.execve(sys.executable,[sys.executable,__file__,*sys.argv[1:]],env)\nsys.argv=[str(Path(__file__).with_name(\'remote_prepare.py\')),*sys.argv[1:]]\nrunpy.run_path(sys.argv[0],run_name=\'__main__\')\n', 'build_auditions.py': '"""Export playable reference/clone comparisons with explicit review status."""\nimport csv\nimport argparse\nimport hashlib\nimport html\nimport json\nfrom pathlib import Path\nimport shutil\n\nroot=Path(__file__).resolve().parent\np=argparse.ArgumentParser();p.add_argument(\'--jobs\',type=Path);a=p.parse_args()\nallowed={j[\'id\'] for j in json.loads(a.jobs.read_text())} if a.jobs else None\nout=root/\'artifacts/auditions\';out.mkdir(parents=True,exist_ok=True)\nrecords=[];parts=[\'<!doctype html><meta charset="utf-8"><title>Tribes 2 voice auditions</title>\',\n    \'<h1>Tribes 2 voice auditions</h1><p>Private research/noncommercial. Originals and generated clones below. \'\n    \'All references, transcripts and speaker labels remain provisional until reviewed.</p>\']\nfor ref in sorted((root/\'artifacts/references-provisional\').glob(\'*.wav\')):\n    speaker=ref.stem;parts.append(\'<h2>\'+html.escape(speaker)+\'</h2>\')\n    sources=[(\'Original reference\',ref,ref.with_suffix(\'.txt\').read_text())]\n    for p in sorted((root/\'artifacts/pilot\').glob(speaker+\'_*.wav\')):\n        if \'_gap_\' in p.stem:continue\n        if allowed is not None and p.stem not in allowed:continue\n        meta=json.loads(p.with_suffix(\'.json\').read_text())\n        sources.append((\'Breeze generated \'+p.stem,p,meta[\'text\']))\n    for label,path,text in sources:\n        name=(\'reference_\' if label==\'Original reference\' else \'generated_\')+path.name\n        shutil.copy2(path,out/name)\n        digest=hashlib.sha256(path.read_bytes()).hexdigest()\n        parts.append(\'<h3>\'+html.escape(label)+\'</h3><audio controls preload="none" src="\'+html.escape(name)+\'"></audio><p>\'+html.escape(text)+\'</p>\')\n        records.append({\'speaker_candidate\':speaker,\'label\':label,\'file\':name,\'sha256\':digest,\'candidate_text\':text,\n                        \'speaker_verified\':\'false\',\'transcript_verified\':\'false\',\'emotion_verified\':\'false\',\n                        \'event_verified\':\'false\',\'corrected_transcript\':\'\',\'notes\':\'\'})\n(out/\'index.html\').write_text(\'\\n\'.join(parts))\nreview=out/\'review.csv\'\nif review.exists():\n    old_bytes=review.read_bytes();history=out/\'review-history\';history.mkdir(exist_ok=True)\n    (history/(hashlib.sha256(old_bytes).hexdigest()[:16]+\'.csv\')).write_bytes(old_bytes)\n    with review.open() as f:old={(r[\'file\'],r[\'sha256\']):r for r in csv.DictReader(f)}\n    for row in records:\n        previous=old.get((row[\'file\'],row[\'sha256\']),{})\n        for key in (\'speaker_verified\',\'transcript_verified\',\'emotion_verified\',\'event_verified\',\'corrected_transcript\',\'notes\'):\n            if key in previous:row[key]=previous[key]\nwith (out/\'review.csv\').open(\'w\',newline=\'\') as f:\n    writer=csv.DictWriter(f,fieldnames=list(records[0]));writer.writeheader();writer.writerows(records)\n(out/\'comparison.m3u\').write_text(\'#EXTM3U\\n\'+\'\\n\'.join(r[\'file\'] for r in records)+\'\\n\')\n(out/\'manifest.json\').write_text(json.dumps(records,indent=2))\nprint(json.dumps({\'clips\':len(records),\'directory\':str(out),\'bytes\':sum(p.stat().st_size for p in out.iterdir() if p.is_file())},indent=2))\nprint(\'========== DONE ==========\',flush=True)\n', 'render_voice_matrix.py': '"""Audition every voice/style using samples only; refusals are part of the report."""\nimport argparse\nimport html\nimport json\nfrom pathlib import Path\nfrom phonetics import Lexicon\nfrom renderer import Renderer\n\n\ndef main():\n    root=Path(__file__).resolve().parent\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--bank\',type=Path,default=root/\'artifacts/phone-alignment/candidate-bank\')\n    p.add_argument(\'--text\',default=\'Defend the generator!\')\n    p.add_argument(\'--bank-pronunciations\',action=\'store_true\')\n    p.add_argument(\'--tag\',default=\'voice-matrix\')\n    p.add_argument(\'--check-workload\',action=\'store_true\',help=\'Check declared workload phone paths only, not audio quality\')\n    a=p.parse_args()\n    lex=Lexicon.load(root/\'artifacts/reports/cmudict.json\',root/\'pronunciation-overrides.json\')\n    if Path(a.tag).name!=a.tag:raise ValueError(\'Directory-safe tag required\')\n    renderer=Renderer(a.bank,lex,audition_unverified=True,bank_pronunciations=a.bank_pronunciations)\n    voices=[*(f\'Male{i}\' for i in range(1,6)),*(f\'Fem{i}\' for i in range(1,6)),\n            *(f\'Derm{i}\' for i in range(1,4)),\'Bot1\',\'Announcer\',\'TrainingFemale\',\'BriefingMale\']\n    emotions=(\'neutral\',\'angry\',\'sad\',\'excited\',\'calm\')\n    out=root/\'artifacts\'/a.tag;out.mkdir(parents=True,exist_ok=True)\n    if a.check_workload:\n        from build_reports import candidates\n        sentences=candidates();groups={}\n        for voice in voices:\n            for emotion in emotions:\n                supported=[];missing=[]\n                for text in sentences:\n                    renderer.warnings=[]\n                    try:\n                        renderer.pronounce_for_bank(voice,emotion,text)\n                        supported.append(text)\n                    except ValueError as exc:\n                        missing.append({\'text\':text,\'error\':str(exc)})\n                groups[voice+\'/\'+emotion]={\'requests\':len(sentences),\'supported_phone_paths\':len(supported),\n                    \'supported\':supported,\'missing\':missing}\n        report={\'scope\':\'Declared announcement workload only; phone-path feasibility, NOT audio/identity/emotion/native-playback review\',\n                \'bank\':str(a.bank),\'bank_digest\':renderer.digest,\'groups\':groups,\'verified\':False}\n        (out/\'results.json\').write_text(json.dumps(report,indent=2))\n        print(json.dumps({key:{\'requests\':x[\'requests\'],\'supported_phone_paths\':x[\'supported_phone_paths\']}\n                          for key,x in groups.items()},indent=2),flush=True)\n        print(\'========== DONE ==========\',flush=True)\n        return\n    rows=[]\n    page=[\'<!doctype html><meta charset="utf-8"><title>Sample-only voice matrix</title>\',\n          \'<h1>Sample-only voice matrix</h1><p>Private candidate auditions. No neural runtime. \'\n          \'Speaker identities, emotions and boundaries are NOT certified. Missing units are refused, \'\n          \'not replaced by a different speaker, emotion or pronunciation.</p>\',\n          \'<p>Text: \'+html.escape(a.text)+\'</p>\']\n    for voice in voices:\n        page.append(\'<h2>\'+html.escape(voice)+\'</h2>\')\n        for emotion in emotions:\n            key=voice+\'_\'+emotion\n            row={\'speaker_id\':voice,\'emotion\':emotion,\'text\':a.text,\'verified\':False,\n                 \'bank_pronunciations\':a.bank_pronunciations}\n            page.append(\'<h3>\'+html.escape(emotion)+\'</h3>\')\n            try:\n                text=a.text if emotion==\'neutral\' else f\'[{emotion}]{a.text}[/{emotion}]\'\n                result=renderer.render(voice,text,out/(key+\'.wav\'))\n                row.update(result,file=key+\'.wav\',status=\'unverified_audition\')\n                page.append(\'<audio controls preload="none" src="\'+key+\'.wav"></audio>\')\n            except ValueError as exc:\n                row.update(status=\'refused\',error=str(exc))\n                page.append(\'<p>REFUSED: \'+html.escape(str(exc))+\'</p>\')\n            rows.append(row)\n    (out/\'results.json\').write_text(json.dumps(rows,indent=2))\n    (out/\'index.html\').write_text(\'\\n\'.join(page))\n    print(json.dumps({\'requests\':len(rows),\'rendered\':sum(r[\'status\']==\'unverified_audition\' for r in rows),\n                      \'refused\':[{\'speaker_id\':r[\'speaker_id\'],\'emotion\':r[\'emotion\'],\'error\':r[\'error\']}\n                                 for r in rows if r[\'status\']==\'refused\'],\n                      \'verified\':0,\'directory\':str(out)},indent=2),flush=True)\n    print(\'========== DONE ==========\',flush=True)\n\n\nif __name__==\'__main__\':main()\n', 'prep/expression_support.py': '"""Prepare speaker-specific expression variants and nonverbal event candidates."""\nfrom __future__ import annotations\nimport json\nfrom pathlib import Path\n\nEMOTIONS = {\n    \'neutral\': None,\n    \'angry\': \'Keep the reference speaker identity. Speak with controlled anger: firm, tense, forceful, and clearly articulated. Do not scream or distort the words.\',\n    \'sad\': \'Keep the reference speaker identity. Speak sadly, with subdued energy, a restrained sorrowful tone, and slightly slower delivery. Keep every word intelligible.\',\n    \'excited\': \'Keep the reference speaker identity. Speak with excitement and enthusiasm, bright energy, and lively emphasis. Keep every word intelligible.\',\n    \'calm\': \'Keep the reference speaker identity. Speak calmly and reassuringly, with steady pacing and clear articulation.\',\n}\n# These English parenthesized event cues are documented by Breeze. Generation\n# remains a candidate until listened to and its event boundaries are marked.\nBREEZE_EVENTS = {\n    \'laugh\': \'(laugh) Welcome to the server.\',\n    \'sigh\': \'(sigh) We lost the objective.\',\n    \'cough\': \'(cough) Communications restored.\',\n    \'clear_throat\': \'(clears throat) Attention, warriors.\',\n}\n# Grunts and death cries exist as stock files. Awoo/woohoo is spoken cheering,\n# not evidence of a wordless cheer; aww is not evidence of crying.\nSTOCK_CANDIDATES = {\n    \'avo.grunt\': (\'grunt\', \'nonverbal_candidate\'),\n    \'avo.pain\': (\'pain\', \'nonverbal_candidate\'),\n    \'avo.deathcry_01\': (\'death_cry\', \'nonverbal_candidate\'),\n    \'avo.deathcry_02\': (\'death_cry\', \'nonverbal_candidate\'),\n    \'gbl.woohoo\': (\'cheer\', \'speech_candidate\'),\n    \'gbl.aww\': (\'disappointment\', \'speech_candidate\'),\n    \'gbl.quiet\': (\'angry_delivery\', \'speech_candidate\'),\n    \'gbl.brag\': (\'confident_delivery\', \'speech_candidate\'),\n    \'gbl.sarcasm\': (\'sarcastic_delivery\', \'speech_candidate\'),\n}\n\ndef build_expressive_review(lab, output):\n    import csv\n    output = Path(output)\n    previous = {}\n    if output.exists():\n        with output.open(newline=\'\', encoding=\'utf-8\') as f:\n            previous = {r[\'source_path\']:r for r in csv.DictReader(f)}\n    rows = []\n    for source in lab.read_manifest()[\'files\']:\n        p = Path(source[\'path\'])\n        guess = STOCK_CANDIDATES.get(p.stem.casefold())\n        if guess is None:\n            continue\n        group = p.parts[1]\n        row = {\'source_path\':source[\'path\'], \'speaker_id\':\'\' if group == \'Training\' else group,\n               \'candidate_label\':guess[0], \'candidate_type\':guess[1],\n               \'speaker_verified\':\'false\', \'event_verified\':\'false\',\n               \'start_seconds\':\'\', \'end_seconds\':\'\', \'notes\':\'\'}\n        row.update(previous.get(source[\'path\'], {}))\n        rows.append(row)\n    if rows:\n        lab.write_csv(output, rows, list(rows[0]))\n    return rows\n\ndef make_jobs(reference, speaker_id, output_root, texts, emotions=None, include_events=True):\n    reference, output_root = Path(reference).resolve(), Path(output_root).resolve()\n    transcript = reference.with_suffix(\'.txt\').read_text().strip()\n    if not reference.is_file() or not transcript:\n        raise ValueError(\'A reviewed reference WAV and its complete transcript are required.\')\n    if not speaker_id or Path(speaker_id).name != speaker_id:\n        raise ValueError(\'Use a single directory-safe speaker ID.\')\n    jobs = []\n    for emotion in (emotions or list(EMOTIONS)):\n        if emotion not in EMOTIONS:\n            raise ValueError(\'Unknown emotion: \'+emotion)\n        for i, text in enumerate(texts):\n            if not text.strip():\n                continue\n            jobs.append({\'id\':f\'{speaker_id}_{emotion}_{i:03d}\', \'speaker_id\':speaker_id,\n                         \'emotion\':emotion, \'kind\':\'speech\', \'text\':text,\n                         \'instruction\':EMOTIONS[emotion], \'ref_audio_path\':str(reference),\n                         \'ref_text\':transcript, \'seed\':42,\n                         \'output\':str(output_root/speaker_id/emotion/f\'phrase_{i:03d}.wav\'),\n                         \'verified\':False})\n    if include_events:\n        for event, text in BREEZE_EVENTS.items():\n            jobs.append({\'id\':f\'{speaker_id}_event_{event}\', \'speaker_id\':speaker_id,\n                         \'emotion\':\'neutral\', \'kind\':\'event_candidate\', \'event\':event,\n                         \'text\':text, \'instruction\':None, \'ref_audio_path\':str(reference),\n                         \'ref_text\':transcript, \'seed\':42,\n                         \'output\':str(output_root/speaker_id/\'events\'/f\'{event}_with_carrier.wav\'),\n                         \'verified\':False})\n    return jobs\n\ndef export_reviewed_events(lab, review_csv, raw_root, output_root, sample_rate=16000):\n    """Export only reviewed nonverbal sounds, preserving each speaker identity."""\n    import csv\n    import wave\n    with Path(review_csv).open(newline=\'\',encoding=\'utf-8\') as f:\n        rows = list(csv.DictReader(f))\n    index = []\n    for row in rows:\n        if row[\'speaker_verified\'].lower() != \'true\' or row[\'event_verified\'].lower() != \'true\':\n            continue\n        speaker = row[\'speaker_id\']\n        event = row[\'candidate_label\']\n        if any(not x or Path(x).name != x for x in (speaker,event)):\n            raise ValueError(\'Invalid speaker or event ID\')\n        source = Path(raw_root)/row[\'source_path\']\n        temporary = Path(output_root)/\'_decoded\'/row[\'source_path\']\n        lab.decode_pcm(source,temporary,sample_rate)\n        with wave.open(str(temporary),\'rb\') as w:\n            duration = w.getnframes()/w.getframerate()\n            start = float(row[\'start_seconds\'] or 0)\n            end = float(row[\'end_seconds\'] or duration)\n            if not 0 <= start < end <= duration:\n                raise ValueError(\'Invalid event boundaries: \'+row[\'source_path\'])\n            w.setpos(round(start*sample_rate))\n            data = w.readframes(round((end-start)*sample_rate))\n        destination = Path(output_root)/speaker/\'events\'/f\'{event}_{len(index):03d}.wav\'\n        destination.parent.mkdir(parents=True,exist_ok=True)\n        with wave.open(str(destination),\'wb\') as w:\n            w.setnchannels(1); w.setsampwidth(2); w.setframerate(sample_rate); w.writeframes(data)\n        index.append({\'speaker_id\':speaker,\'event\':event,\'file\':str(destination.relative_to(output_root)),\n                      \'source\':row[\'source_path\'],\'sample_rate\':sample_rate,\'verified\':True})\n    Path(output_root).mkdir(parents=True,exist_ok=True)\n    (Path(output_root)/\'events.json\').write_text(json.dumps(index,indent=2))\n    return index\n', 'prep/cannedChatItems.cs': '//--------------------------------------------------------------------------\n// \n// cannedChatItems.cs\n// \n//--------------------------------------------------------------------------\n\n$MinChatItemId = 0;\n$MaxChatItemId = 0;\n\nif ( !isObject( CannedChatItems ) )\n   new SimGroup( CannedChatItems );\n\n//--------------------------------------------------------------------------\nfunction installChatItem( %command, %text, %audioFile, %animCel, %teamOnly, %defaultkeys, %play3D )\n{\n   %cmdId = getSubStr( %command, 1, strlen( %command ) - 1 );\n\t%name = getTaggedString(%command);\n   //echo( "** cmdId = " @ %cmdId @ " **" );\n   if ( !isObject( $ChatTable[%cmdId] ) )\n   {\n      if ( %animCel == 0 )\n         %animation = "";\n      else\n         %animation = "cel" @ %animCel;\n\n\t\t//error("defvoicebinds="@$defaultVoiceBinds@",keyPress="@%keyPress@",keyCmd="@%keyCmd);\n      $ChatTable[%cmdId] = new CannedChatItem()\n      {\n         name = %name;\n         text = %text;\n         audioFile = %audioFile;\n         animation = %animation;\n         teamOnly = %teamOnly;\n         defaultKeys = %defaultkeys;\n\t\t\tplay3D = %play3D;\n      };\n      CannedChatItems.add( $ChatTable[%cmdId] );\n\n      if ( $MinChatItemId == 0 || %cmdId < $MinChatItemId )\n         $MinChatItemId = %cmdId;\n      if ( %cmdId > $MaxChatItemId )\n         $MaxChatItemId = %cmdId;\n   }\n}\n\n//--------------------------------------------------------------------------\nfunction installChatItemCallback( %command, %callback )\n{\n   %cmdId = getSubStr( %command, 1, strlen( %command ) - 1 );\n\n   // make sure there is a chat item created\n   if(isObject($ChatTable[%cmdId]))\n   {\n      for(%i = 0; (%aCallback = $ChatCallbacks[%cmdId, %i]) !$= ""; %i++)\n      {\n         // dont allow multiple instances\n         if(%aCallback == %callback)\n            return;\n      }\n\n      $ChatCallbacks[%cmdId, %i] = %callback;\n   }\n}\n\nfunction processChatItemCallbacks( %command )\n{\n   %cmdId = getSubStr( %command, 1, strlen( %command ) - 1 );\n\n   // make sure an actual chat item\n   if(isObject($ChatTable[%cmdId]))\n      for(%i = 0; (%callback = $ChatCallbacks[%cmdId, %i]) !$= ""; %i++)\n         call(%callback, $ChatTable[%cmdId]);\n}\n\n//--------------------------------------------------------------------------\n// ANIMATIONS\ninstallChatItem( \'ChatAnimAnnoyed\', "", "vqk.move", 4, false, "VGAA", true );\ninstallChatItem( \'ChatAnimGetSome\', "", "gbl.brag", 3, false, "VGAG", true );\ninstallChatItem( \'ChatAnimDance\', "", "gbl.woohoo", 5, false, "VGAD", true );\ninstallChatItem( \'ChatAnimSalute\', "", "slf.tsk.generic", 1, false, "VGAS", true );\ninstallChatItem( \'ChatAnimWave\', "", "gbl.hi", 2, false, "VGAW", true );\ninstallChatItem( \'ChatAnimSpec1\', "", "gbl.obnoxious", 6, false, "VGAZ", true );\ninstallChatItem( \'ChatAnimSpec2\', "", "gbl.aww", 7, false, "VGAX", true );\ninstallChatItem( \'ChatAnimSpec3\', "", "gbl.awesome", 8, false, "VGAC", true );\n\n//--------------------------------------------------------------------------\n// ATTACK\ninstallChatItem( \'ChatCmdAttack\', "Attack!", "att.attack", 0, true, "VAA", false );\ninstallChatItem( \'ChatCmdAttackBase\', "Attack the enemy base!", "att.base", 0, true, "VAB", false );\ninstallChatItem( \'ChatCmdAttackChase\', "Recover our flag!", "att.chase", 0, true, "VAC", false );\ninstallChatItem( \'ChatCmdAttackDistract\', "Disrupt the enemy defense!", "att.distract", 0, true, "VAD", false );\ninstallChatItem( \'ChatCmdAttackFlag\', "Get the enemy flag!", "att.flag", 0, true, "VAF", false );\ninstallChatItem( \'ChatCmdAttackGenerator\', "Destroy the enemy generator!", "att.generator", 0, true, "VAG", false );\ninstallChatItem( \'ChatCmdAttackObjective\', "Attack the objective!", "att.objective", 0, true, "VAO", false );\ninstallChatItem( \'ChatCmdAttackReinforce\', "Reinforce the offense!", "att.reinforcements", 0, true, "VAR", false );\ninstallChatItem( \'ChatCmdAttackSensors\', "Destroy enemy sensors!", "att.sensors", 0, true, "VAS", false );\ninstallChatItem( \'ChatCmdAttackTurrets\', "Destroy enemy turrets!", "att.turrets", 0, true, "VAT", false );\ninstallChatItem( \'ChatCmdAttackWait\', "Wait for my signal before attacking!", "att.wait", 0, true, "VAW", false );\ninstallChatItem( \'ChatCmdAttackVehicle\', "Destroy the enemy vehicle!", "att.vehicle", 0, true, "VAV", false );\n\n//--------------------------------------------------------------------------\n// BASE\ninstallChatItem( \'ChatBaseTaken\', "Our base is taken.", "bas.taken", 0, true, "VBT", false );\ninstallChatItem( \'ChatEnemyInBase\', "The enemy\'s in our base.", "bas.enemy", 0, true, "VBE", false );\ninstallChatItem( \'ChatBaseClear\', "Our base is clear.", "bas.clear", 0, true, "VBC", false );\ninstallChatItem( \'ChatCmdRetakeBase\', "Retake our base!", "bas.retake", 0, true, "VBR", false );\ninstallChatItem( \'ChatBaseSecure\', "Our base is secure.", "bas.secure", 0, true, "VBS", false );\n\n//--------------------------------------------------------------------------\n// DEFENSE\ninstallChatItem( \'ChatCmdDefendBase\', "Defend our base!", "def.base", 0, true, "VDB", false );\ninstallChatItem( \'ChatCmdDefendCarrier\', "Cover our flag carrier!", "def.carrier", 0, true, "VDC", false );\ninstallChatItem( \'ChatCmdDefendEntrances\', "Defend the entrances!", "def.entrances", 0, true, "VDE", false );\ninstallChatItem( \'ChatCmdDefendFlag\', "Defend our flag!", "def.flag", 0, true, "VDF", false );\ninstallChatItem( \'ChatCmdDefendGenerator\', "Protect the generator!", "def.generator", 0, true, "VDG", false );\ninstallChatItem( \'ChatCmdDefendMe\', "Cover me!", "def.me", 0, true, "VDM", false );\ninstallChatItem( \'ChatCmdDefendObjective\', "Defend the objective!", "def.objective", 0, true, "VDO", false );\ninstallChatItem( \'ChatCmdDefendReinforce\', "Reinforce our defense!", "def.reinforce", 0, true, "VDR", false );\ninstallChatItem( \'ChatCmdDefendSensors\', "Defend our sensors!", "def.sensors", 0, true, "VDS", false );\ninstallChatItem( \'ChatCmdDefendTurrets\', "Defend our turrets!", "def.turrets", 0, true, "VDT", false );\ninstallChatItem( \'ChatCmdDefendVehicle\', "Defend our vehicle!", "def.vehicle", 0, true, "VDV", false );\ninstallChatItem( \'ChatCmdDefendNexus\', "Defend the nexus!", "def.nexus", 0, true, "VDN", false );\n\n//--------------------------------------------------------------------------\n// COMMAND RESPONSE\ninstallChatItem( \'ChatCmdAcknowledged\', "Command acknowledged.", "cmd.acknowledge", 0, true, "VCA", false );\ninstallChatItem( \'ChatCmdWhat\', "What\'s your assignment?", "cmd.bot", 0, true, "VCW", false );\ninstallChatItem( \'ChatCmdCompleted\', "Command completed.", "cmd.completed", 0, true, "VCC", false );\ninstallChatItem( \'ChatCmdDeclined\', "Command declined.", "cmd.decline", 0, true, "VCD", false );\n\n//--------------------------------------------------------------------------\n// ENEMY STATUS\ninstallChatItem( \'ChatEnemyBaseDisabled\', "Enemy base is disabled.", "ene.base", 0, true, "VEB", false );\ninstallChatItem( \'ChatEnemyDisarray\', "The enemy is disrupted. Attack!", "ene.disarray", 0, true, "VED", false );\ninstallChatItem( \'ChatEnemyGeneratorDestroyed\', "Enemy generator destroyed.", "ene.generator", 0, true, "VEG", false );\ninstallChatItem( \'ChatEnemyRemotesDestroyed\', "Enemy remote equipment destroyed.", "ene.remotes", 0, true, "VER", false );\ninstallChatItem( \'ChatEnemySensorsDestroyed\', "Enemy sensors destroyed.", "ene.sensors", 0, true, "VES", false );\ninstallChatItem( \'ChatEnemyTurretsDestroyed\', "Enemy turrets destroyed.", "ene.turrets", 0, true, "VET", false );\ninstallChatItem( \'ChatEnemyVehicleDestroyed\', "Enemy vehicle station destroyed.", "ene.vehicle", 0, true, "VEV", false );\n\n//--------------------------------------------------------------------------\n// FLAG\ninstallChatItem( \'ChatFlagGotIt\', "I have the enemy flag!", "flg.flag", 0, true, "VFF", false );\ninstallChatItem( \'ChatCmdGiveMeFlag\', "Give me the flag!", "flg.give", 0, true, "VFG", false );\ninstallChatItem( \'ChatCmdReturnFlag\', "Retrieve our flag!", "flg.retrieve", 0, true, "VFR", false );\ninstallChatItem( \'ChatFlagSecure\', "Our flag is secure.", "flg.secure", 0, true, "VFS", false );\ninstallChatItem( \'ChatCmdTakeFlag\', "Take the flag from me!", "flg.take", 0, true, "VFT", false );\ninstallChatItem( \'ChatCmdHunterGiveFlags\', "Give your flags to me!", "flg.huntergive", 0, true, "VFO", false );\ninstallChatItem( \'ChatCmdHunterTakeFlags\', "Take my flags!", "flg.huntertake", 0, true, "VFP", false );\n\n//--------------------------------------------------------------------------\n// GLOBAL COMPLIMENTS\ninstallChatItem( \'ChatAwesome\', "Awesome!", "gbl.awesome", 0, false, "VGCA", false );\ninstallChatItem( \'ChatGoodGame\', "Good game!", "gbl.goodgame", 0, false, "VGCG", false );\ninstallChatItem( \'ChatNice\', "Nice move!", "gbl.nice", 0, false, "VGCN", false );\ninstallChatItem( \'ChatYouRock\', "You rock!", "gbl.rock", 0, false, "VGCR", false );\ninstallChatItem( \'ChatGreatShot\', "Great shot!", "gbl.shooting", 0, false , "VGCS");\n\n//--------------------------------------------------------------------------\n// GLOBAL\ninstallChatItem( \'ChatHi\', "Hi.", "gbl.hi", 0, false, "VGH", false );\ninstallChatItem( \'ChatBye\', "Bye.", "gbl.bye", 0, false, "VGB", false );\ninstallChatItem( \'ChatGlobalYes\', "Yes.", "gbl.yes", 0, false, "VGY", false );\ninstallChatItem( \'ChatGlobalNo\', "No.", "gbl.no", 0, false, "VGN", false );\ninstallChatItem( \'ChatAnyTime\', "Any time.", "gbl.anytime", 0, false, "VGRA", false );\ninstallChatItem( \'ChatDontKnow\', "I don\'t know.", "gbl.dunno", 0, false, "VGRD", false );\ninstallChatItem( \'ChatOops\', "Oops!", "gbl.oops", 0, false, "VGO", false );\ninstallChatItem( \'ChatQuiet\', "Quiet!", "gbl.quiet", 0, false, "VGQ", false );\ninstallChatItem( \'ChatShazbot\', "Shazbot!", "gbl.shazbot", 0, false, "VGS", false );\ninstallChatItem( \'ChatCheer\', "Woohoo!", "gbl.woohoo", 0, false, "VGW", false );\ninstallChatItem( \'ChatThanks\', "Thanks.", "gbl.thanks", 0, false, "VGRT", false );\ninstallChatItem( \'ChatWait\', "Wait a sec.", "gbl.wait", 0, false, "VGRW", false );\n\n//--------------------------------------------------------------------------\n// TRASH TALK\ninstallChatItem( \'ChatAww\', "Aww, that\'s too bad!", "gbl.aww", 0, false, "VGTA", false );\ninstallChatItem( \'ChatBrag\', "I am the greatest!", "gbl.brag", 0, false, "VGTG", false );\ninstallChatItem( \'ChatObnoxious\', "That\'s the best you can do?", "gbl.obnoxious", 0, false, "VGTB", false );\ninstallChatItem( \'ChatSarcasm\', "THAT was graceful!", "gbl.sarcasm", 0, false, "VGTT", false );\ninstallChatItem( \'ChatLearn\', "When ya gonna learn?", "gbl.when", 0, false, "VGTW", false );\n\n//--------------------------------------------------------------------------\n// NEED\ninstallChatItem( \'ChatNeedBombardier\', "Need a bombardier.", "need.bombardier", 0, true, "VNB", false );\ninstallChatItem( \'ChatNeedCover\', "Need covering fire.", "need.cover", 0, true, "VNC", false );\ninstallChatItem( \'ChatNeedDriver\', "Need driver for ground vehicle.", "need.driver", 0, true, "VND", false );\ninstallChatItem( \'ChatNeedEscort\', "Vehicle needs escort.", "need.escort", 0, true, "VNE", false );\ninstallChatItem( \'ChatNeedPilot\', "Need pilot for turbograv.", "need.flyer", 0, true, "VNP", false );\ninstallChatItem( \'ChatNeedPassengers\', "Gunship ready! Need a ride?", "need.gunship", 0, true, "VNG", false );\ninstallChatItem( \'ChatNeedHold\', "Hold that vehicle! I\'m coming!", "need.hold", 0, true, "VNH", false );\ninstallChatItem( \'ChatNeedRide\', "I need a ride!", "need.ride", 0, true, "VNR", false );\ninstallChatItem( \'ChatNeedSupport\', "Need vehicle support!", "need.support", 0, true, "VNS", false );\ninstallChatItem( \'ChatNeedTailgunner\', "Need a tailgunner.", "need.tailgunner", 0, true, "VNT", false );\ninstallChatItem( \'ChatNeedDestination\', "Where to?", "need.where", 0, true, "VNW", false );\n\n//--------------------------------------------------------------------------\n// REPAIR\ninstallChatItem( \'ChatRepairBase\', "Repair our base!", "rep.base", 0, true, "VRB", false );\ninstallChatItem( \'ChatRepairGenerator\', "Repair our generator!", "rep.generator", 0, true, "VRG", false );\ninstallChatItem( \'ChatRepairMe\', "Repair me!", "rep.me", 0, true, "VRM", false );\ninstallChatItem( \'ChatRepairSensors\', "Repair our sensors!", "rep.sensors", 0, true, "VRS", false );\ninstallChatItem( \'ChatRepairTurrets\', "Repair our turrets!", "rep.turrets", 0, true, "VRT", false );\ninstallChatItem( \'ChatRepairVehicle\', "Repair our vehicle station!", "rep.vehicle", 0, true, "VRV", false );\n\n//--------------------------------------------------------------------------\n// SELF ATTACK\ninstallChatItem( \'ChatSelfAttack\', "I will attack.", "slf.att.attack", 0, true, "VSAA", false );\ninstallChatItem( \'ChatSelfAttackBase\', "I\'ll attack the enemy base.", "slf.att.base", 0, true, "VSAB", false );\ninstallChatItem( \'ChatSelfAttackFlag\', "I\'ll go for the enemy flag.", "slf.att.flag", 0, true, "VSAF", false );\ninstallChatItem( \'ChatSelfAttackGenerator\', "I\'ll attack the enemy generator.", "slf.att.generator", 0, true, "VSAG", false );\ninstallChatItem( \'ChatSelfAttackSensors\', "I\'ll attack the enemy sensors.", "slf.att.sensors", 0, true, "VSAS", false );\ninstallChatItem( \'ChatSelfAttackTurrets\', "I\'ll attack the enemy turrets.", "slf.att.turrets", 0, true, "VSAT", false );\ninstallChatItem( \'ChatSelfAttackVehicle\', "I\'ll attack the enemy vehicle station.", "slf.att.vehicle", 0, true, "VSAV", false );\n\n//--------------------------------------------------------------------------\n// SELF DEFEND\ninstallChatItem( \'ChatSelfDefendBase\', "I\'ll defend our base.", "slf.def.base", 0, true, "VSDB", false );\ninstallChatItem( \'ChatSelfDefend\', "I\'m defending.", "slf.def.defend", 0, true, "VSDD", false );\ninstallChatItem( \'ChatSelfDefendFlag\', "I\'ll defend our flag.", "slf.def.flag", 0, true, "VSDF", false );\ninstallChatItem( \'ChatSelfDefendGenerator\', "I\'ll defend our generator.", "slf.def.generator", 0, true, "VSDG", false );\ninstallChatItem( \'ChatSelfDefendNexus\', "I\'ll defend the nexus.", "slf.def.nexus", 0, true, "VSDN", false );\ninstallChatItem( \'ChatSelfDefendSensors\', "I\'ll defend our sensors.", "slf.def.sensors", 0, true, "VSDS", false );\ninstallChatItem( \'ChatSelfDefendTurrets\', "I\'ll defend our turrets.", "slf.def.turrets", 0, true, "VSDT", false );\ninstallChatItem( \'ChatSelfDefendVehicle\', "I\'ll defend our vehicle bay.", "slf.def.vehicle", 0, true, "VSDV", false );\n\n//--------------------------------------------------------------------------\n// SELF REPAIR\ninstallChatItem( \'ChatSelfRepairBase\', "I\'ll repair our base.", "slf.rep.base", 0, true, "VSRB", false );\ninstallChatItem( \'ChatSelfRepairEquipment\', "I\'ll repair our equipment.", "slf.rep.equipment", 0, true, "VSRE", false );\ninstallChatItem( \'ChatSelfRepairGenerator\', "I\'ll repair our generator.", "slf.rep.generator", 0, true, "VSRG", false );\ninstallChatItem( \'ChatSelfRepair\', "I\'m on repairs.", "slf.rep.repairing", 0, true, "VSRR", false );\ninstallChatItem( \'ChatSelfRepairSensors\', "I\'ll repair our sensors.", "slf.rep.sensors", 0, true, "VSRS", false );\ninstallChatItem( \'ChatSelfRepairTurrets\', "I\'ll repair our turrets.", "slf.rep.turrets", 0, true, "VSRT", false );\ninstallChatItem( \'ChatSelfRepairVehicle\', "I\'ll repair our vehicle station.", "slf.rep.vehicle", 0, true, "VSRV", false );\n\n//--------------------------------------------------------------------------\n// SELF TASK\ninstallChatItem( \'ChatTaskCover\', "I\'ll cover you.", "slf.tsk.cover", 0, true, "VSTC", false );\ninstallChatItem( \'ChatTaskSetupD\', "I\'ll set up defenses.", "slf.tsk.defense", 0, true, "VSTD", false );\ninstallChatItem( \'ChatTaskOnIt\', "I\'m on it.", "slf.tsk.generic", 0, true, "VSTO", false );\ninstallChatItem( \'ChatTaskSetupRemote\', "I\'ll deploy remote equipment.", "slf.tsk.remotes", 0, true, "VSTR", false );\ninstallChatItem( \'ChatTaskSetupSensors\', "I\'ll deploy sensors.", "slf.tsk.sensors", 0, true, "VSTS", false );\ninstallChatItem( \'ChatTaskSetupTurrets\', "I\'ll deploy turrets.", "slf.tsk.turrets", 0, true, "VSTT", false );\ninstallChatItem( \'ChatTaskVehicle\', "I\'ll get a vehicle ready.", "slf.tsk.vehicle", 0, true, "VSTV", false );\n\n//--------------------------------------------------------------------------\n// TARGET\ninstallChatItem( \'ChatTargetAcquired\', "Target acquired.", "tgt.acquired", 0, true, "VTA", false );\ninstallChatItem( \'ChatCmdTargetBase\', "Target the enemy base! I\'m in position.", "tgt.base", 0, true, "VTB", false );\ninstallChatItem( \'ChatTargetDestroyed\', "Target destroyed!", "tgt.destroyed", 0, true, "VTD", false );\ninstallChatItem( \'ChatCmdTargetFlag\', "Target their flag! I\'m in position.", "tgt.flag", 0, true, "VTF", false );\ninstallChatItem( \'ChatTargetFire\', "Fire on my target!", "tgt.my", 0, true, "VTM", false );\ninstallChatItem( \'ChatTargetNeed\', "Need a target painted!", "tgt.need", 0, true, "VTN", false );\ninstallChatItem( \'ChatCmdTargetSensors\', "Target their sensors! I\'m in position.", "tgt.sensors", 0, true, "VTS", false );\ninstallChatItem( \'ChatCmdTargetTurret\', "Target their turret! I\'m in position.", "tgt.turret", 0, true, "VTT", false );\ninstallChatItem( \'ChatCmdTargetWait\', "Wait! I\'ll be in range soon.", "tgt.wait", 0, true, "VTW", false );\n\n//--------------------------------------------------------------------------\n// WARNING\ninstallChatItem( \'ChatWarnBomber\', "Incoming bomber!", "wrn.bomber", 0, true, "VWB", false );\ninstallChatItem( \'ChatWarnEnemies\', "Incoming hostiles!", "wrn.enemy", 0, true, "VWE", false );\ninstallChatItem( \'ChatWarnVehicles\', "Incoming vehicles!", "wrn.vehicles", 0, true, "VWV", false );\ninstallChatItem( \'ChatWarnShoot\', "Watch where you\'re shooting!", "wrn.watchit", 0, true, "VWW", false );\n\n//--------------------------------------------------------------------------\n// VERY QUICK\ninstallChatItem( \'ChatWelcome\', "Any time.", "vqk.anytime", 0, true, "VVA", false );\ninstallChatItem( \'ChatIsBaseSecure\', "Is our base secure?", "vqk.base", 0, true, "VVB", false );\ninstallChatItem( \'ChatCeaseFire\', "Cease fire!", "vqk.ceasefire", 0, true, "VVC", false );\ninstallChatItem( \'ChatDunno\', "I don\'t know.", "vqk.dunno", 0, true, "VVD", false );\ninstallChatItem( \'ChatHelp\', "HELP!", "vqk.help", 0, true, "VVH", false );\ninstallChatItem( \'ChatMove\', "Move, please!", "vqk.move", 0, true, "VVM", false );\ninstallChatItem( \'ChatTeamNo\', "No.", "vqk.no", 0, true, "VVN", false );\ninstallChatItem( \'ChatSorry\', "Sorry.", "vqk.sorry", 0, true, "VVS", false );\ninstallChatItem( \'ChatTeamThanks\', "Thanks.", "vqk.thanks", 0, true, "VVT", false );\ninstallChatItem( \'ChatTeamWait\', "Wait, please.", "vqk.wait", 0, true, "VVW", false );\ninstallChatItem( \'ChatTeamYes\', "Yes.", "vqk.yes", 0, true, "VVY", false );\n\n', 'prep/breeze_worker.py': '"""Kaggle-only batch driver using Breeze\'s documented inference modules.\nModel loaded once; jobs rendered sequentially to limit peak GPU memory.\n"""\nfrom __future__ import annotations\nimport argparse\nimport json\nimport sys\nfrom pathlib import Path\n\ndef main():\n    p=argparse.ArgumentParser()\n    p.add_argument(\'--source\',type=Path,required=True)\n    p.add_argument(\'--model\',type=Path,required=True)\n    p.add_argument(\'--jobs\',type=Path,required=True)\n    args=p.parse_args()\n    sys.path.insert(0,str(args.source.resolve()))\n    import torch\n    import soundfile as sf\n    from breeze_infer.runtime import load_runtime, resolve_device, set_all_seeds, update_generation_config_for_breeze\n    from breeze_infer.templates import get_template, prepare_inputs, select_template_name\n    from models.fast_streaming import FastBreezeStreamingRuntime, FastStreamingConfig\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'Enable a Kaggle CUDA GPU before generating expression variants.\')\n    tokenizer, model, codec = load_runtime(args.model, device=resolve_device(), attn_implementation=\'eager\')\n    update_generation_config_for_breeze(model)\n    config=FastStreamingConfig(max_new_tokens=1500,max_seq_len=2048,fast_all=False,\n        fast_text_encoder=False,fast_backbone_prefill=False,fast_backbone_decode=False,\n        fast_depth_decoder=False,fast_codec=False,repetition_penalty=1.1)\n    runtime=FastBreezeStreamingRuntime(model,codec,config,tokenizer=tokenizer)\n    jobs=json.loads(args.jobs.read_text())\n    for i,job in enumerate(jobs,1):\n        output=Path(job[\'output\'])\n        output.parent.mkdir(parents=True,exist_ok=True)\n        request={\'id\':job[\'id\'],\'speaker\':\'S0\',\'text\':job[\'text\'],\n                 \'ref_audio_path\':job[\'ref_audio_path\'],\'ref_text\':job[\'ref_text\']}\n        instruction=job.get(\'instruction\')\n        if instruction:\n            request[\'instruction\']=instruction\n        scale=4.0 if instruction else 1.0\n        seed=int(job.get(\'seed\',42))\n        set_all_seeds(seed)\n        inputs=prepare_inputs(tokenizer,codec,model,[request],get_template(select_template_name(request)),\n                              guidance_scale=scale,guidance_scale_ref=None,guidance_scale_ins=None)\n        temporary=output.with_name(output.stem+\'.part.wav\')\n        with sf.SoundFile(temporary,mode=\'w\',samplerate=runtime.sample_rate,channels=1,subtype=\'PCM_16\') as w:\n            for chunk in runtime.iter_audio_chunks(inputs,request_id=job[\'id\'],seed=seed):\n                w.write(chunk.audio)\n        temporary.replace(output)\n        output.with_suffix(\'.json\').write_text(json.dumps(job,indent=2))\n        print(f\'{i}/{len(jobs)}: {output}\',flush=True)\n        del inputs\n    print(\'Done. Listen and mark candidates before using them in a runtime voice bank.\')\n\nif __name__==\'__main__\':\n    main()\n', 'prep/assets.json': '{"repository":"exogen/t2-mapper","revision":"b9dda9eaf153d706e94fc71f242866d2f2709394","voice_root":"docs/base/@vl2/voice.vl2/audio","files":[{"path":"voice/Announcer/ANN.10sec.WAV","sha":"aa69b19c271a8bdd8669d0408f5ff665de6f9b0e","bytes":10524},{"path":"voice/Announcer/ANN.15sec.WAV","sha":"f1caa5f1619bbf9b2c516b5c8da60c30a37a3cb9","bytes":15448},{"path":"voice/Announcer/ANN.30.WAV","sha":"896b3f8d7772844c27a2c5695874394cb4df911d","bytes":11324},{"path":"voice/Announcer/ANN.60.WAV","sha":"622a420b883e49836798ccc240d55973da672955","bytes":12004},{"path":"voice/Announcer/ANN.bdscore.WAV","sha":"8f8bc9f84a9f7156dedf475b8c9d5fca60bc0067","bytes":21520},{"path":"voice/Announcer/ANN.bdwin.WAV","sha":"cfa5ca22f46bcab91a39aed5fc46172b263cd753","bytes":23002},{"path":"voice/Announcer/ANN.bescore.WAV","sha":"86c98e002a158daf04bf7297f3bcd42159fa29f4","bytes":22166},{"path":"voice/Announcer/ANN.bewin.WAV","sha":"ecf641318e7ca293662506c7c1b19a39cfa7e881","bytes":23326},{"path":"voice/Announcer/ANN.closegame_01.WAV","sha":"640db10dab7d66a0c9097b9bc2f2d7df4d09b275","bytes":18118},{"path":"voice/Announcer/ANN.closegame_03.WAV","sha":"11db157985e769362ee0dff805381af03071758f","bytes":23320},{"path":"voice/Announcer/ANN.closegame_04.WAV","sha":"4f2fe4b790b67b897417a2d78e39d8ea618ee67d","bytes":17376},{"path":"voice/Announcer/ANN.dsscore.WAV","sha":"9e2dac4bc1f602f3158c7a140468549b423c9b0b","bytes":24420},{"path":"voice/Announcer/ANN.dswin.WAV","sha":"ec1bee16461fbd377cf541efbf5d29bae63b68f9","bytes":26096},{"path":"voice/Announcer/ANN.gameover.WAV","sha":"af2da0657b48dcad39f73a812b22340e87bedc81","bytes":23058},{"path":"voice/Announcer/ANN.gamepoint_imminent.WAV","sha":"55849d82d6904d14cf8c2811102060c5ed8e299d","bytes":16980},{"path":"voice/Announcer/ANN.greed.WAV","sha":"050e4b564cb278ba9801e94680ce76e5044fd169","bytes":11250},{"path":"voice/Announcer/ANN.hoard.WAV","sha":"ffc4927c3104102323462f0ca4f5125faeb8ea31","bytes":9954},{"path":"voice/Announcer/ANN.holdflags.WAV","sha":"d2808146d2d6db40f7bf61fb056107f958a34edc","bytes":26464},{"path":"voice/Announcer/ANN.ib.WAV","sha":"864f8e0a4af2efa7ac2f2b21ec8f61d1327e89a8","bytes":20168},{"path":"voice/Announcer/ANN.infdenied.WAV","sha":"5a27afd695e7ebdd0943fdf691993c94559f609a","bytes":25638},{"path":"voice/Announcer/ANN.infscores.WAV","sha":"d4e568dc75f18fc6a48efbd33d690c4332efd1b6","bytes":22228},{"path":"voice/Announcer/ANN.infwins.WAV","sha":"15f3079baaa1eba63e8ad4d86b1f170cc3d39801","bytes":25964},{"path":"voice/Announcer/ANN.match_begins.WAV","sha":"0ade6c64c8160e5790f62d024d5c432480efd984","bytes":25718},{"path":"voice/Announcer/ANN.oob.WAV","sha":"d1096b45c9d7687459c94309350e553c17a4e727","bytes":16816},{"path":"voice/Announcer/ANN.oobh.WAV","sha":"1de348d4654c56b68ad179c67b182f6ef5a4e077","bytes":23002},{"path":"voice/Announcer/ANN.oobv.WAV","sha":"b902bd774b6f6e1b373b4e1bdf5598e726a6b306","bytes":24356},{"path":"voice/Announcer/ANN.prepare.WAV","sha":"3998c7e68f51392481cb4883450cfb1f87c4d916","bytes":25698},{"path":"voice/Announcer/ANN.pxscore.WAV","sha":"8694e25a9bad5f8e2c5333948b2c095a64b23923","bytes":20488},{"path":"voice/Announcer/ANN.pxwin.WAV","sha":"fe016ec14ce8be40ce2db7a27f7b56991f32fe4a","bytes":23518},{"path":"voice/Announcer/ANN.siege.WAV","sha":"45b0492acf92aadea945fcc7f62feb9be0de4f77","bytes":19042},{"path":"voice/Announcer/ANN.stodenied.WAV","sha":"c67a8a4db6f99b1b9d463ae6797e8089883f44ca","bytes":24344},{"path":"voice/Announcer/ANN.stoscores.WAV","sha":"4e7e3f0d81697fa77a79fe4f5824b59e37254aaf","bytes":26544},{"path":"voice/Announcer/ANN.stowins.WAV","sha":"b86b63675c404715614aa942fff8da84845ef6d3","bytes":28656},{"path":"voice/Announcer/ANN.swscore.WAV","sha":"cd6c22b70ed918e30b0f947839747ad0fc79277b","bytes":27514},{"path":"voice/Announcer/ANN.swwin.WAV","sha":"fd8deca36e090df9f8d8e86217cb4bf6fd4ae902","bytes":33444},{"path":"voice/Announcer/ANN.t1score.WAV","sha":"a8310203e774091bf413996f94b14c6ead6af42d","bytes":25064},{"path":"voice/Announcer/ANN.t1win.WAV","sha":"57d46c7a2f0aba6554229f16bb80b4e30be4502c","bytes":24808},{"path":"voice/Announcer/ANN.t2score.WAV","sha":"da7f1a81f8ce9676fee638de09c61ff77d843242","bytes":22422},{"path":"voice/Announcer/ANN.t2win.WAV","sha":"07353c387ba38dd34e99e4f6c054702bcb11a271","bytes":22618},{"path":"voice/Announcer/ANN.votefail.WAV","sha":"617258f780862a62b8e5f6db9344d3e38188b5dc","bytes":14014},{"path":"voice/Announcer/ANN.voteinit.WAV","sha":"4928a42e64d4d2224f67629b82cca14cdec0a63d","bytes":14770},{"path":"voice/Announcer/ANN.votepass.WAV","sha":"3957567c9c764bf26e35d8d7fbbfc2e9ed67bf06","bytes":13836},{"path":"voice/Announcer/ANN.yardsale.WAV","sha":"9b24f01a2f4de8dc32d2d88603ceb0103fc55a9f","bytes":10292},{"path":"voice/Announcer/flag_minedFem.WAV","sha":"64ea14bc5d2366501da6f3f011a72fcbba8597bb","bytes":17588},{"path":"voice/Announcer/flag_minedMale.WAV","sha":"438fefa902cfd81ec984d19e80e47ad0333eb80c","bytes":18492},{"path":"voice/Announcer/switching_sides.WAV","sha":"d6dbab6bf6fe136145b9bb5fe3f617404703872e","bytes":16156},{"path":"voice/Bot1/att.attack.wav","sha":"af76d311a123f8a1fc089b906db60e6ff8f8e0e5","bytes":7024},{"path":"voice/Bot1/att.base.wav","sha":"f89fdcb9796ad4df667c917aac53ea596d6e0bc8","bytes":16612},{"path":"voice/Bot1/att.chase.wav","sha":"ddfc58cd039bb06759d2e7f67b2a1934f6164015","bytes":15748},{"path":"voice/Bot1/att.distract.wav","sha":"88840e9e0351281f167b0cbebb30caf8f34e7d04","bytes":22522},{"path":"voice/Bot1/att.flag.wav","sha":"a927cd66dfd3930c51d055a7ef1736bdee88da8d","bytes":15400},{"path":"voice/Bot1/att.generator.wav","sha":"4573ce3c95f9471194abd69d955de7b3eb83842a","bytes":18892},{"path":"voice/Bot1/att.objective.wav","sha":"e166faae52fac78223528252108113755609e5a2","bytes":16532},{"path":"voice/Bot1/att.reinforcements.wav","sha":"66c527115e280c6913d19c722742998272b653d9","bytes":17648},{"path":"voice/Bot1/att.sensors.wav","sha":"4440d181fc56d9ec805b12663e2b9bd734842ae3","bytes":20186},{"path":"voice/Bot1/att.turrets.wav","sha":"63de9447626b4300403d150b04ed817de8fd13e5","bytes":19798},{"path":"voice/Bot1/att.vehicle.wav","sha":"ccc453b6fd9099035b1a4722711175a200a6b164","bytes":20132},{"path":"voice/Bot1/att.wait.wav","sha":"8e2572bad203d446c8f4890aee2c37873cafbbcc","bytes":22400},{"path":"voice/Bot1/avo.deathcry_01.wav","sha":"1174f670084214378e1cbcc100554b51e8ac7e11","bytes":11324},{"path":"voice/Bot1/avo.deathcry_02.wav","sha":"ac3deaa3397490d9e13e4cea50b186bc31ba4fc6","bytes":12860},{"path":"voice/Bot1/avo.grunt.wav","sha":"5976ae61577b2dedc71a4258a2995325a40ccd3e","bytes":5312},{"path":"voice/Bot1/avo.pain.wav","sha":"6e85f9161b16f146dea9f6a0596f62d404631eee","bytes":10432},{"path":"voice/Bot1/bas.clear.wav","sha":"6cc69a568b2daa3811b1af71dffa7a6484403efd","bytes":15034},{"path":"voice/Bot1/bas.enemy.wav","sha":"eff4fed986c1e109838e52b507ac9ca55468e4f6","bytes":21192},{"path":"voice/Bot1/bas.retake.wav","sha":"78a28ed9f44b8f5d7d9eb44dfed32c6ff1882337","bytes":14530},{"path":"voice/Bot1/bas.secure.wav","sha":"7d2df05209738e207683c16d7dccb11e1a0314a0","bytes":18596},{"path":"voice/Bot1/bas.taken.wav","sha":"1e8cb71afa57a4f8d406a8989f3caa64c6d5aad4","bytes":15894},{"path":"voice/Bot1/cmd.acknowledge.wav","sha":"5a57c6703c04efaa55e21c7bfaae4369f793ac7c","bytes":15856},{"path":"voice/Bot1/cmd.bot.WAV","sha":"6c4c40110c499d8582329dc85668901c38d11fe5","bytes":13148},{"path":"voice/Bot1/cmd.completed.wav","sha":"c0565a2d52be4f4987faad40a5e79deb604c014e","bytes":14524},{"path":"voice/Bot1/cmd.decline.wav","sha":"5cd6a4aa4bcbf8caad0faac49abd03f3d4b4add4","bytes":17938},{"path":"voice/Bot1/def.base.wav","sha":"235ea608b17fa65f0ad9d28161cfcdbe3cdbf853","bytes":15458},{"path":"voice/Bot1/def.carrier.wav","sha":"dd3a94fed5fae9ab6a5b046097c241b60293a2df","bytes":18698},{"path":"voice/Bot1/def.entrances.wav","sha":"b4399fc979a2e1cab830d7dca15ddff610b86659","bytes":17950},{"path":"voice/Bot1/def.flag.wav","sha":"7abfe4ab7d2c24008c6f5a5d74b75dc3070e7cf5","bytes":17622},{"path":"voice/Bot1/def.generator.wav","sha":"0f0c81d0459099adcff74b6d38a6aca9a6b6ee63","bytes":16018},{"path":"voice/Bot1/def.me.wav","sha":"933de4d8a9eefd51c9c8d93fa7ffc1e815c986ad","bytes":7110},{"path":"voice/Bot1/def.nexus.wav","sha":"563db5a89f565c878204c54ffe6f0181479810e0","bytes":15152},{"path":"voice/Bot1/def.objective.wav","sha":"883e13d6723e4923ec4bc2dd1b5493f983a7d3d4","bytes":14630},{"path":"voice/Bot1/def.reinforce.wav","sha":"3d4f8ef7ce314a517bad7bd8af3dce9f5a79c2c6","bytes":18230},{"path":"voice/Bot1/def.sensors.wav","sha":"5ee38257614babf425b788ef443d4e957a6c9b23","bytes":17444},{"path":"voice/Bot1/def.turrets.wav","sha":"d6261a59e8bdf113ed16aa785f0bea1a9d45734c","bytes":16910},{"path":"voice/Bot1/def.vehicle.wav","sha":"6786f121acb862d8af878d17cabeec38f327c2fa","bytes":14856},{"path":"voice/Bot1/ene.base.wav","sha":"947befdf1fa1d1b73892cfe09c66c83df4a20a46","bytes":19534},{"path":"voice/Bot1/ene.disarray.wav","sha":"d9ed88e10469d0fcacb7d42d0cf2599e4d389083","bytes":23966},{"path":"voice/Bot1/ene.generator.wav","sha":"c877452a68bb35c534338a7679439a8f655c333d","bytes":18928},{"path":"voice/Bot1/ene.remotes.wav","sha":"7220ffaccc071c2adadf027527ba446c08762204","bytes":23878},{"path":"voice/Bot1/ene.sensors.wav","sha":"f0ad43ea33c783dd13327cfddb2aee15d44bc683","bytes":20300},{"path":"voice/Bot1/ene.turrets.wav","sha":"7681262546acb0295ad051824ac1c1ca032d2a09","bytes":17026},{"path":"voice/Bot1/ene.vehicle.wav","sha":"a86747bc44713a1236d0244a796155e58afabf20","bytes":23236},{"path":"voice/Bot1/flg.flag.WAV","sha":"45deb3471ad12610d891e40d863e8bc08e0594b9","bytes":18218},{"path":"voice/Bot1/flg.give.wav","sha":"bd27d477998f9eeb0cc12381c30b3f95915676b1","bytes":11080},{"path":"voice/Bot1/flg.huntergive.wav","sha":"3b85348eccf1229f9d84e9f00f30933b5ad8c95c","bytes":16132},{"path":"voice/Bot1/flg.huntertake.wav","sha":"7f7608e46dfe987af7695bf8f233ba726a136c95","bytes":14088},{"path":"voice/Bot1/flg.secure.wav","sha":"33f4b33d54834f1026c2bffbb801fbe5f1926c40","bytes":20632},{"path":"voice/Bot1/flg.take.wav","sha":"ee9434142f9b40defb03cd2af5f12d97f3182685","bytes":17612},{"path":"voice/Bot1/gbl.anytime.wav","sha":"4e5407c7e487fde0c613f5d328bc669a7b1041b4","bytes":8622},{"path":"voice/Bot1/gbl.awesome.wav","sha":"c31d3e4225fc2e9c17ff8512f47013febb43b7f0","bytes":11850},{"path":"voice/Bot1/gbl.aww.wav","sha":"a062d62a04307c1228274438a1e0af1f4ba6c348","bytes":8648},{"path":"voice/Bot1/gbl.brag.wav","sha":"2259d00f2162a9724c1d5be30c66771fd07fd6c8","bytes":15456},{"path":"voice/Bot1/gbl.bye.wav","sha":"cd53d9a6765b49fc4fb6e19f25a6d0766b7b3eb8","bytes":6914},{"path":"voice/Bot1/gbl.doh.wav","sha":"c04f8cd39488008954e29c2586939769c2068006","bytes":12992},{"path":"voice/Bot1/gbl.dunno.wav","sha":"394e91f7cefb4da7df2ae93fc0c01f003e73292f","bytes":11796},{"path":"voice/Bot1/gbl.goodgame.wav","sha":"b4a88c63ad8b8da58d5ed428052ae7e85a004fe7","bytes":13102},{"path":"voice/Bot1/gbl.hi.wav","sha":"777fe6c7e44bad81d9bafa128f863f39c79c826e","bytes":5186},{"path":"voice/Bot1/gbl.nice.wav","sha":"9759b4988bdbc8107d2af1c26b060824a559dce5","bytes":9838},{"path":"voice/Bot1/gbl.no.wav","sha":"d0a6cafe92f3a0a8a4f7bd5df4e31a32beb38a85","bytes":5856},{"path":"voice/Bot1/gbl.obnoxious.wav","sha":"c8439b1d206423c9f91e7a5c6025b61b326693cb","bytes":18236},{"path":"voice/Bot1/gbl.oops.wav","sha":"d1d269d5ff2fc1851296883362e31a1cf2fb4d25","bytes":6706},{"path":"voice/Bot1/gbl.quiet.WAV","sha":"a0ad47cdf0a9deafa3c7989ec81083b697b206ba","bytes":11596},{"path":"voice/Bot1/gbl.rock.WAV","sha":"1861bf7bb211efefd072e24b01a8669ac25af79d","bytes":13420},{"path":"voice/Bot1/gbl.sarcasm.wav","sha":"266db8fb0dd6e15212bb8bb74f3d9774949a9e7e","bytes":25424},{"path":"voice/Bot1/gbl.shazbot.wav","sha":"c12bca791762a885e34c7675b8d02b64c444899a","bytes":10234},{"path":"voice/Bot1/gbl.shooting.WAV","sha":"ec24fdffbf27c5da19243596ddcee3e2202e4dd7","bytes":12456},{"path":"voice/Bot1/gbl.thanks.wav","sha":"b12c5f60802c0a2d258665ff3c283765e89eeb0c","bytes":5546},{"path":"voice/Bot1/gbl.wait.wav","sha":"5d6f70f5123a07b73ca2a3da297bef23e421f13f","bytes":4344},{"path":"voice/Bot1/gbl.when.wav","sha":"08ed511d4f6c3bd61e76f2247b9fffde472b241c","bytes":21928},{"path":"voice/Bot1/gbl.yes.wav","sha":"12e026dc52c3e28e63f3c3fa786c047bfa9cb0dd","bytes":7472},{"path":"voice/Bot1/need.bombardier.wav","sha":"93df55c6401e59d9dee161423f5ff0a243d08c10","bytes":12884},{"path":"voice/Bot1/need.cover.WAV","sha":"926763a0e9306fab0531f30b2b00dfa32d75a83b","bytes":16298},{"path":"voice/Bot1/need.driver.wav","sha":"5b544a64956c00331cd6863edfdb2d3963a01039","bytes":23706},{"path":"voice/Bot1/need.escort.wav","sha":"01539d957dba4b2f21a0050127afb42199a720ad","bytes":18864},{"path":"voice/Bot1/need.flyer.WAV","sha":"5d50c3d867227814d7c51ab6714ddff08afc1fd9","bytes":19724},{"path":"voice/Bot1/need.gunship.WAV","sha":"00cb136e274140cb8edd2b99999ef463230f19fa","bytes":13544},{"path":"voice/Bot1/need.hold.wav","sha":"dae826761806085f3545ef0b8c72d900eafb3ff4","bytes":26360},{"path":"voice/Bot1/need.ride.WAV","sha":"87b1f0525c7abe6366e45591e7992239d1972503","bytes":14882},{"path":"voice/Bot1/need.support.wav","sha":"15c137620ae90176a173f4b82f017c30bf805c82","bytes":18572},{"path":"voice/Bot1/need.tailgunner.wav","sha":"1b5588346fdb259aa5c512231ee9adca5081d368","bytes":14740},{"path":"voice/Bot1/need.where.wav","sha":"24fc7ae581470bb16732d8d11e980494b38defd2","bytes":9306},{"path":"voice/Bot1/rep.base.wav","sha":"9725d875f287686e14992419ab47c66e498c68da","bytes":14668},{"path":"voice/Bot1/rep.generator.wav","sha":"c6c185df951f9201d5263662fa6efec360f87bc1","bytes":16358},{"path":"voice/Bot1/rep.me.wav","sha":"aa86d77b13261992b74c000eea927466d0696640","bytes":11228},{"path":"voice/Bot1/rep.sensors.wav","sha":"f21d1be65ba0b9dd231f7a1f2a727c0e00c20b2e","bytes":18686},{"path":"voice/Bot1/rep.turrets.wav","sha":"7206c81593f741773b51872a252cea199b745cd7","bytes":15602},{"path":"voice/Bot1/rep.vehicle.wav","sha":"61479bcaabef7badb7dcb1575196ade1c8d327a8","bytes":22398},{"path":"voice/Bot1/slf.att.attack.wav","sha":"ad7971429e31ac3327a234b8ef99710ab040fd2c","bytes":12068},{"path":"voice/Bot1/slf.att.base.wav","sha":"44c3e7b73bb814056f8c67e5faa0ed7af665bb31","bytes":23346},{"path":"voice/Bot1/slf.att.flag.wav","sha":"75bc9f9707d5611aabb5384d283e9b709dfd3f26","bytes":22868},{"path":"voice/Bot1/slf.att.generator.wav","sha":"5c4a66677a7d047b43c7dc4665bc79f85c94d318","bytes":22120},{"path":"voice/Bot1/slf.att.sensors.wav","sha":"0af5ba3ff309e32cd16d7be4866ddf8e12fb8d92","bytes":23832},{"path":"voice/Bot1/slf.att.turrets.wav","sha":"30b2391862b0ea8bdc91f7868fe3315f8c77972e","bytes":20754},{"path":"voice/Bot1/slf.att.vehicle.wav","sha":"96b56454a4c3e53aaa100c5d489b933e92e0984d","bytes":27018},{"path":"voice/Bot1/slf.def.base.wav","sha":"dc24f21b1b6d8ed0ffc23824c96f6648bdb90309","bytes":19566},{"path":"voice/Bot1/slf.def.defend.wav","sha":"a75966f9e5c91471e0ed0c547f7dd09cfdec64f6","bytes":14450},{"path":"voice/Bot1/slf.def.flag.wav","sha":"55e7addbae6a19bad379839d513374a3f9a648a6","bytes":17328},{"path":"voice/Bot1/slf.def.generator.wav","sha":"3d8ee9c5bedb064ca9508f2a02c57ca491727759","bytes":21486},{"path":"voice/Bot1/slf.def.nexus.wav","sha":"70f5f2ef0581d27f40196c61f1b66df8a77c2b5e","bytes":21804},{"path":"voice/Bot1/slf.def.sensors.wav","sha":"451b3924baa4c8c3c9a497d0a0e2672e1cf98525","bytes":21276},{"path":"voice/Bot1/slf.def.turrets.wav","sha":"8a42ed28ba489db3fb19f1b310a28a1f57fff37f","bytes":19264},{"path":"voice/Bot1/slf.def.vehicle.wav","sha":"5b32eb9092853f7f1349b4966df7e70e5cf283bb","bytes":21758},{"path":"voice/Bot1/slf.rep.base.wav","sha":"6650923364b119b1687e7ba8bbe78f9776b63026","bytes":17722},{"path":"voice/Bot1/slf.rep.equipment.wav","sha":"712d080f33de3cabfbbe1953f213cffec1d444ef","bytes":19804},{"path":"voice/Bot1/slf.rep.generator.wav","sha":"fd568af7d98a999964392e3f81aae0c34e51891d","bytes":20836},{"path":"voice/Bot1/slf.rep.repairing.wav","sha":"f8da133ebb8be31a949e206b3bcf5a39e6383811","bytes":16148},{"path":"voice/Bot1/slf.rep.sensors.wav","sha":"13f4b4f9e8386a35c09edeb8ccad6775d27d92a1","bytes":18578},{"path":"voice/Bot1/slf.rep.turrets.wav","sha":"fc987ae4028c27fb157e63b73ec4fc5cb6b540d8","bytes":18268},{"path":"voice/Bot1/slf.rep.vehicle.wav","sha":"b234e75347fdf64d96f8c9bf4d25586f3904f5fd","bytes":21978},{"path":"voice/Bot1/slf.tsk.cover.WAV","sha":"7652f73a493c6c7b0bf6c301673e00170f37514e","bytes":11906},{"path":"voice/Bot1/slf.tsk.defense.wav","sha":"3419b3d29a9695bcfc81676a76c913e7a58ded3b","bytes":17902},{"path":"voice/Bot1/slf.tsk.generic.WAV","sha":"703a28353be76867a30435444a51cf43a4270e39","bytes":9836},{"path":"voice/Bot1/slf.tsk.remotes.wav","sha":"081d63e29d318dff6d365f447b930f1a56494064","bytes":22030},{"path":"voice/Bot1/slf.tsk.sensors.wav","sha":"b2cdd9e17d6045bc4c973390b602e524eea119f9","bytes":19986},{"path":"voice/Bot1/slf.tsk.turrets.wav","sha":"a439609a15958a4cc30c3345adce59f6c165b812","bytes":16030},{"path":"voice/Bot1/slf.tsk.vehicle.WAV","sha":"b196515b7e1826104d85501c72deefd08f2f1fea","bytes":20314},{"path":"voice/Bot1/tgt.acquired.wav","sha":"55b425c5aa0c9d9a729f8e90b0dbe721cb14f5df","bytes":14388},{"path":"voice/Bot1/tgt.base.wav","sha":"00c2864c3b6948d69a9b90790ea0262c7fb40b65","bytes":31400},{"path":"voice/Bot1/tgt.destroyed.wav","sha":"98bc46c910b08ebdd31990c2396df36eeee4f4e0","bytes":16322},{"path":"voice/Bot1/tgt.flag.wav","sha":"65379d3ffbaa1c3a98f565c56e7647a2a8c2690c","bytes":28806},{"path":"voice/Bot1/tgt.my.WAV","sha":"947d5be216eb224413fab4335ab669c30c2c963f","bytes":15048},{"path":"voice/Bot1/tgt.need.wav","sha":"07fcce198be6e68444c3abbbd84ecce316d87987","bytes":16278},{"path":"voice/Bot1/tgt.sensors.wav","sha":"8a62c440e8d6398fa576e292f339bb0ec969f1c6","bytes":28168},{"path":"voice/Bot1/tgt.turret.wav","sha":"b861550cfcd1ba693768f4f829eb00d5f443e8f4","bytes":27188},{"path":"voice/Bot1/tgt.wait.wav","sha":"96df4c7b268dc983b7d2428f2808c5e7ad647d2b","bytes":27844},{"path":"voice/Bot1/vqk.anytime.wav","sha":"e1f2f1b8e2461a672120e53bd2a5f47630c068a3","bytes":12982},{"path":"voice/Bot1/vqk.ceasefire.wav","sha":"6ce2fe756d768d2146fc2e285843b9ae7618e54a","bytes":8794},{"path":"voice/Bot1/vqk.dunno.wav","sha":"6f1fbdc6b940c0e6a4212de4d7ba1179f407007a","bytes":12402},{"path":"voice/Bot1/vqk.move.wav","sha":"9c6e59ecd11208fc8abf799703eb5ae2f585e70d","bytes":13152},{"path":"voice/Bot1/vqk.no.wav","sha":"cf0cb2af9e57964a72e9386a123acaa69abb487c","bytes":7764},{"path":"voice/Bot1/vqk.quiet.WAV","sha":"29ab4e19edd03d6f29f8a5a683850cb2f0032fb3","bytes":13968},{"path":"voice/Bot1/vqk.sorry.wav","sha":"195cb587130774fe115409ad2aabe58d113d0f33","bytes":6560},{"path":"voice/Bot1/vqk.thanks.wav","sha":"0da848b79c25c1c7fe60e8c8dffabfc60c0c9620","bytes":7878},{"path":"voice/Bot1/vqk.wait.wav","sha":"beb88fb2ef57b9fbcdecf8ce918b6b560d1d7b6e","bytes":12130},{"path":"voice/Bot1/vqk.yes.wav","sha":"78a48eae9224e5cc7c268e5a1c7fa589c117102f","bytes":4862},{"path":"voice/Bot1/wrn.bomber.wav","sha":"579fb4f8be9453cd7c3c172fe2497689cb2beb45","bytes":11870},{"path":"voice/Bot1/wrn.enemy.wav","sha":"1af9bd9c84630d4bb5111fa7dd6353094ad1795e","bytes":20192},{"path":"voice/Bot1/wrn.vehicles.wav","sha":"451c650fb4ffe29f3beae1934ed340a3e979272c","bytes":18430},{"path":"voice/Bot1/wrn.watchit.wav","sha":"b206dab58ff6518cc4def8b7cb9198a519d9b09b","bytes":17798},{"path":"voice/Derm1/att.attack.wav","sha":"102d175437e22830acb8c96d0b098568925512a2","bytes":16516},{"path":"voice/Derm1/att.base.wav","sha":"f7166c9ad5a8e53da9d6339670890860f17097f0","bytes":28148},{"path":"voice/Derm1/att.chase.wav","sha":"a28c8f9df9fc3f0affb46c2bb1ed139e685195c5","bytes":23270},{"path":"voice/Derm1/att.distract.wav","sha":"f98d7685383d7a73e6f0f518bc522f75dca6f323","bytes":28902},{"path":"voice/Derm1/att.flag.wav","sha":"984ab447e627837a7e459b1f931057df3f818492","bytes":24108},{"path":"voice/Derm1/att.generator.wav","sha":"0791dedccde1b0ebac3aa29a8cae7b22d78310b5","bytes":30804},{"path":"voice/Derm1/att.objective.wav","sha":"47c0af345dc95a5435eea903eba0276438ad362c","bytes":25590},{"path":"voice/Derm1/att.reinforcements.wav","sha":"39334a5df627d36dd07d7ab4ba7af0bfffeff786","bytes":27328},{"path":"voice/Derm1/att.sensors.wav","sha":"f7f27371ca7b0974c8201ef87f07c615ce8f87c3","bytes":29628},{"path":"voice/Derm1/att.turrets.wav","sha":"47d2a4927487b1a851df141fc10a2bababd940d6","bytes":31012},{"path":"voice/Derm1/att.vehicle.wav","sha":"935002bc553500cd5f50df36ec6c1b726a39c92e","bytes":23122},{"path":"voice/Derm1/att.wait.wav","sha":"b18979f550d2e80fda2a965463d59d0e2fafc4af","bytes":31936},{"path":"voice/Derm1/avo.deathcry_01.wav","sha":"2cbcd22b6ccd7c2d1936e2a8a6ff36ebe33eb834","bytes":18492},{"path":"voice/Derm1/avo.deathcry_02.wav","sha":"d5517bdb88fa56e638485b9b9b9206609ed9e90c","bytes":17980},{"path":"voice/Derm1/avo.grunt.wav","sha":"c2117cd457b4ab22d63bcea9785d7e1dac6f4e6e","bytes":6194},{"path":"voice/Derm1/avo.pain.wav","sha":"d3b17a0247d4c44eadd681b4712d413deeefeeb2","bytes":7404},{"path":"voice/Derm1/bas.clear.wav","sha":"c514cadf83780e8357dba1a5e5ea0c88e0350d48","bytes":19376},{"path":"voice/Derm1/bas.enemy.wav","sha":"29fed966c518ad7b7ecaae0a61f9f8687c6794df","bytes":25412},{"path":"voice/Derm1/bas.retake.wav","sha":"26b7b71f72b17cd8d7fff2bc9042e6c03f999405","bytes":26544},{"path":"voice/Derm1/bas.secure.wav","sha":"8ae2b5dceb6b5126164238790183d990b75cb330","bytes":25694},{"path":"voice/Derm1/bas.taken.wav","sha":"f9735f54482449e055b58ff207bef9a39fb94804","bytes":24584},{"path":"voice/Derm1/cmd.acknowledge.wav","sha":"90123f80090637750e6ba5bdc057835f670bd336","bytes":23788},{"path":"voice/Derm1/cmd.bot.WAV","sha":"73d383bb1ce82ef0911eb000759cbd771fbebe52","bytes":19376},{"path":"voice/Derm1/cmd.completed.wav","sha":"15ce5a624fb5dbefbfcb8a64459b9bda96a944ce","bytes":20254},{"path":"voice/Derm1/cmd.decline.wav","sha":"ee845280361e2acbf56ddc2172455f6610d7f5ba","bytes":22950},{"path":"voice/Derm1/def.base.wav","sha":"6547e789502d4aa2247976e6caf949707e1f7f72","bytes":26086},{"path":"voice/Derm1/def.carrier.wav","sha":"d3308afc1bb890d1273101106d2554ac6fc25467","bytes":22326},{"path":"voice/Derm1/def.entrances.wav","sha":"f05632bf00b3252e95a199f66a088f8192dca91c","bytes":22058},{"path":"voice/Derm1/def.flag.wav","sha":"f5e8088f38f449d620c4238d06b0bcbcf4e31a65","bytes":21282},{"path":"voice/Derm1/def.generator.wav","sha":"7e985b007221b0c5584f3c264c2fbeb3753279b4","bytes":20334},{"path":"voice/Derm1/def.me.wav","sha":"cbd0333f67091ef12e8f1db9a6ce3a92827762d6","bytes":11848},{"path":"voice/Derm1/def.nexus.wav","sha":"a70b9621856909c03ceb3470126dddfb483ef08b","bytes":20980},{"path":"voice/Derm1/def.objective.wav","sha":"69795f5ab9351a37b48b8c4e127e6c67a78459a7","bytes":21422},{"path":"voice/Derm1/def.reinforce.wav","sha":"f6b9ec06c27c5a728ab5f0ea5ee965f223cee670","bytes":22516},{"path":"voice/Derm1/def.sensors.wav","sha":"42e032768d144ff3d61abdf5421e1cc29f4ab6d8","bytes":24312},{"path":"voice/Derm1/def.turrets.wav","sha":"14100d6770f84dd2ea049ce7316ea4daa9164ad6","bytes":20248},{"path":"voice/Derm1/def.vehicle.wav","sha":"70bfbadd1e68da298cc843782bb46e416d03ffe9","bytes":20508},{"path":"voice/Derm1/ene.base.wav","sha":"28bd8363cbd3caf5149c6979e2e624cdb3019e2d","bytes":24016},{"path":"voice/Derm1/ene.disarray.wav","sha":"727eadfc633e2c3860bab17d1a7b8aaa96f28439","bytes":27564},{"path":"voice/Derm1/ene.generator.wav","sha":"e9f0956a4c2b4b26683b00d8d004fd10bfd245d5","bytes":26280},{"path":"voice/Derm1/ene.remotes.wav","sha":"21715259089c18857040bfe036fa0d4500afec0b","bytes":29774},{"path":"voice/Derm1/ene.sensors.wav","sha":"a66f5a4989f46ed356d46caa8e9eeb9373acda9a","bytes":25606},{"path":"voice/Derm1/ene.turrets.wav","sha":"9fe6cddea25b4107da97be10b8e79f2b8d7f13d9","bytes":26420},{"path":"voice/Derm1/ene.vehicle.wav","sha":"eccd74b7d533a502f2e884cd1cda3414a9209f27","bytes":30620},{"path":"voice/Derm1/flg.flag.WAV","sha":"d0fa3ecbc5c3859079036772b37ee6d2281d7514","bytes":24520},{"path":"voice/Derm1/flg.give.wav","sha":"7c2411327b3708ccdb8c623eb70047126758954e","bytes":19220},{"path":"voice/Derm1/flg.huntergive.wav","sha":"6592b8ba9bd43d04e7e6ff0710d1b73a350fad6f","bytes":20434},{"path":"voice/Derm1/flg.huntertake.wav","sha":"7d16baac3f5feeee459ebe6989af4a5b6808b33d","bytes":18934},{"path":"voice/Derm1/flg.retrieve.wav","sha":"4530d259cc00b7a4e6ad584e9d7ef010adb64d45","bytes":23378},{"path":"voice/Derm1/flg.secure.wav","sha":"2e248b3fa4a3d2a655da035d830220a040100cd4","bytes":24692},{"path":"voice/Derm1/flg.take.wav","sha":"74a2f0cbf5accd1e08afeadb88ee256b441cf144","bytes":19986},{"path":"voice/Derm1/gbl.anytime.wav","sha":"2606dd2ad8ae72c85d86e49fcec67dd807b214ec","bytes":15676},{"path":"voice/Derm1/gbl.awesome.wav","sha":"22bd628e20213d11cb01b9beb31024a3bc63d50b","bytes":31850},{"path":"voice/Derm1/gbl.aww.wav","sha":"eb27f1a5e8294e4a6bf63bf45e11f60ea0cb48fe","bytes":30560},{"path":"voice/Derm1/gbl.brag.wav","sha":"97a252b00ff937bb8b1595a7cb3bed5530e8012d","bytes":37942},{"path":"voice/Derm1/gbl.bye.wav","sha":"c4558ae282c005ccaaaf43ad6684c1e6009b946b","bytes":13120},{"path":"voice/Derm1/gbl.dunno.wav","sha":"fb0c2650ca701977a07f99ff1a3e3d3e535e4bf5","bytes":15902},{"path":"voice/Derm1/gbl.goodgame.wav","sha":"e8dfda784ca14736d05756731ddc9ffa67560024","bytes":17018},{"path":"voice/Derm1/gbl.hi.wav","sha":"ea366151b0714992dfa7254e6bb89c723463d3ac","bytes":19856},{"path":"voice/Derm1/gbl.nice.wav","sha":"273b134f2bb3387e3ee7a62d34c7c571f308ff7e","bytes":17988},{"path":"voice/Derm1/gbl.no.wav","sha":"b4bfab905cbddc6e370692a61b1859867db18596","bytes":12420},{"path":"voice/Derm1/gbl.obnoxious.wav","sha":"18427414fbf544f5d1eca3b6991599e47ea7e7fa","bytes":34364},{"path":"voice/Derm1/gbl.oops.wav","sha":"6291a4c41505d44392e237d8b2343a5fe7c8d8ea","bytes":12516},{"path":"voice/Derm1/gbl.quiet.WAV","sha":"34e6fe9bb27ee25337afd6a1ed2e99c811db0494","bytes":14756},{"path":"voice/Derm1/gbl.rock.WAV","sha":"99d5fc3bf1c806028d9dfd36b867f6ea28599709","bytes":30238},{"path":"voice/Derm1/gbl.sarcasm.WAV","sha":"556999bae0c60e60fd4dc1cefbcaaa0f1fb1187e","bytes":39538},{"path":"voice/Derm1/gbl.shazbot.wav","sha":"c5edc1798089dae141862ad52f2251613006fcd0","bytes":29832},{"path":"voice/Derm1/gbl.shooting.WAV","sha":"7666a73d1bc0e7f060728c771004538e97dfda3c","bytes":14374},{"path":"voice/Derm1/gbl.thanks.wav","sha":"ef6b2b70d0347d353c0688a538918215410fc824","bytes":13128},{"path":"voice/Derm1/gbl.wait.wav","sha":"20d5de393c10939ddd732589700480e3358fc34e","bytes":11614},{"path":"voice/Derm1/gbl.when.wav","sha":"3e3a1382bb5bf863e90f14728ccb3ebede70d520","bytes":33252},{"path":"voice/Derm1/gbl.woohoo.wav","sha":"e645b1ceec250e7ec87dddf2995fddd3ed5552e2","bytes":36908},{"path":"voice/Derm1/gbl.yes.wav","sha":"50f8ca65ffd0dca53a31315818f94cc3e8f86d15","bytes":13050},{"path":"voice/Derm1/need.bombardier.wav","sha":"191f2e5b8823182661c596807a26f07de4cee6b5","bytes":20494},{"path":"voice/Derm1/need.cover.WAV","sha":"41c7752c4809ec13c607b434affa453063a66d5b","bytes":23206},{"path":"voice/Derm1/need.driver.wav","sha":"93236b51d5a4e4fee7f6e9e6c9a19dc4444436df","bytes":28982},{"path":"voice/Derm1/need.escort.wav","sha":"6127a208c486bc53fdcc64c2271045684adc7c0e","bytes":21624},{"path":"voice/Derm1/need.flyer.WAV","sha":"67cd30a55ef304c8b733c4ed49bd48980b5a3c6e","bytes":26162},{"path":"voice/Derm1/need.gunship.WAV","sha":"7e28ff0bd4d6962d4cdd873c6154cb7f5f4ceb50","bytes":30838},{"path":"voice/Derm1/need.hold.wav","sha":"4701e8fc79a3686ae7c1a0a41bb87936f917222a","bytes":27474},{"path":"voice/Derm1/need.ride.WAV","sha":"7b9855befdbf601c57b903c53c06f5cab402a87d","bytes":18132},{"path":"voice/Derm1/need.support.wav","sha":"07bb0d9083c088e012a258a994bd8d59cbd37a43","bytes":22954},{"path":"voice/Derm1/need.tailgunner.wav","sha":"7a0a664cfc3ae26c0bd91359d34f01f08c92fe8b","bytes":18054},{"path":"voice/Derm1/need.where.wav","sha":"b9b34ceb5e076e2dd70e35284649a17e73f79ae1","bytes":13078},{"path":"voice/Derm1/rep.base.wav","sha":"31bd4ecc718a9d8c44c198136b6f54c192df942b","bytes":23932},{"path":"voice/Derm1/rep.generator.wav","sha":"83103ddc911fbd4abd3cf0d0ea456f33f0396f5d","bytes":22422},{"path":"voice/Derm1/rep.me.wav","sha":"c7d2b356e296b7ead63ab4977799a2415763b0b8","bytes":13816},{"path":"voice/Derm1/rep.sensors.wav","sha":"6105ba4c779d20055e9beda3a1ae5f67c4a43912","bytes":22412},{"path":"voice/Derm1/rep.turrets.wav","sha":"3e1ecabb86eee0cd55ff16910b49ad50bc678681","bytes":19092},{"path":"voice/Derm1/rep.vehicle.wav","sha":"45efe77b302049fa8e4288c33932f2c9ac42b7db","bytes":26436},{"path":"voice/Derm1/slf.att.attack.wav","sha":"cd228e6dae25ba8068951bfa4b3215e1f8ac27e1","bytes":18982},{"path":"voice/Derm1/slf.att.base.wav","sha":"7f1e56e044a81901380bc79374a402062865d0b2","bytes":29890},{"path":"voice/Derm1/slf.att.flag.wav","sha":"30346582afdb227f876a00b7fec650120d1f0269","bytes":25552},{"path":"voice/Derm1/slf.att.generator.wav","sha":"1cb10163598d187cee0697197b7e91d24bc47c29","bytes":25386},{"path":"voice/Derm1/slf.att.sensors.wav","sha":"12c2aa44a5c6d83dbf64b9bead08e06da86ec939","bytes":26452},{"path":"voice/Derm1/slf.att.turrets.wav","sha":"4871e4328f31ed938c735c8ad3b3fea9c2706694","bytes":28168},{"path":"voice/Derm1/slf.att.vehicle.wav","sha":"adc71326ef721708515e1fb126d223f54ba9b291","bytes":27996},{"path":"voice/Derm1/slf.def.base.wav","sha":"145ea203ff2c083c3928192299d9bedeb0ffb6fd","bytes":23858},{"path":"voice/Derm1/slf.def.defend.wav","sha":"4ae0a78d5a88da524eb7eceb62b1674e59bbf622","bytes":18808},{"path":"voice/Derm1/slf.def.flag.wav","sha":"64397b7f8d13d0e50950cfd06c071a3e1825ce38","bytes":26678},{"path":"voice/Derm1/slf.def.generator.wav","sha":"9753fc790d46575dd47bf1a15edf0312fb429b33","bytes":24816},{"path":"voice/Derm1/slf.def.nexus.wav","sha":"fe831312d65df6a346e5ee526212d11adf2dc9e9","bytes":25984},{"path":"voice/Derm1/slf.def.sensors.wav","sha":"077a8400a86ab1a6ee277ec405e2ed2293b86683","bytes":33206},{"path":"voice/Derm1/slf.def.turrets.wav","sha":"17e30d054dd175e567851db1bd527ecf4a4e5a42","bytes":28512},{"path":"voice/Derm1/slf.def.vehicle.wav","sha":"76854a1c4e00d7059f8870d3c3b6b2d03a4d689b","bytes":25472},{"path":"voice/Derm1/slf.rep.base.wav","sha":"85e087482e573e8917df6e3279dd7995f59f8dd8","bytes":20970},{"path":"voice/Derm1/slf.rep.equipment.wav","sha":"a517742e9eb3820620f1edcb6bf6c40785a15d41","bytes":23974},{"path":"voice/Derm1/slf.rep.generator.wav","sha":"0ebafcdee2e9ce0be1103e1571e04f2da4bb625a","bytes":26960},{"path":"voice/Derm1/slf.rep.repairing.wav","sha":"aa486d2240ae97d772c5d6ece384fcce8410d261","bytes":19014},{"path":"voice/Derm1/slf.rep.sensors.wav","sha":"94c048755a10e837659a7192fe5285e07b0e4314","bytes":25278},{"path":"voice/Derm1/slf.rep.turrets.wav","sha":"437aae47e540f0396208ca00d2d505cc9c460a20","bytes":22072},{"path":"voice/Derm1/slf.rep.vehicle.wav","sha":"bca13f4c364a96ef0679ccb24154f8a61010c1b8","bytes":22972},{"path":"voice/Derm1/slf.tsk.cover.WAV","sha":"d8262cb70c28d5c06fa57a2a6a2f9be6ff43d01f","bytes":15312},{"path":"voice/Derm1/slf.tsk.defense.wav","sha":"d3e5f9168c1e20c96c5e91e878cdf0b22adec711","bytes":20110},{"path":"voice/Derm1/slf.tsk.generic.WAV","sha":"06574bd53fae30030a033e765365c31cdb5bf9bf","bytes":12020},{"path":"voice/Derm1/slf.tsk.remotes.wav","sha":"d6769843b8d3bf9f501940bf64a1a8ce7a63ef75","bytes":23206},{"path":"voice/Derm1/slf.tsk.sensors.wav","sha":"c2f638bdcd6f8f2aeab5930820ca59d97b00671c","bytes":25454},{"path":"voice/Derm1/slf.tsk.turrets.wav","sha":"fb6f886c49206d1d72d407db52b6190c24be018d","bytes":21890},{"path":"voice/Derm1/slf.tsk.vehicle.WAV","sha":"44224cce184a3e40485498139a6b5dc397c4c29d","bytes":22030},{"path":"voice/Derm1/tgt.acquired.wav","sha":"b20d06069d10f4a99fd3cda03859ba38225d649c","bytes":20674},{"path":"voice/Derm1/tgt.base.wav","sha":"c9e4bef185def46d3dae46865acff9cfaeb06cdb","bytes":30672},{"path":"voice/Derm1/tgt.destroyed.wav","sha":"7adca1de0dddf47aca48ae57fda578fb1a7d2ca9","bytes":19568},{"path":"voice/Derm1/tgt.flag.wav","sha":"0bf2af88b7129a659ce30d10a41dafb5afcee014","bytes":35808},{"path":"voice/Derm1/tgt.my.WAV","sha":"64429c0bf69bb076f6289e398664b75d40fba866","bytes":22550},{"path":"voice/Derm1/tgt.need.wav","sha":"d4048603485f367939678bceef64c16c25fc3128","bytes":21036},{"path":"voice/Derm1/tgt.sensors.wav","sha":"c23f106163a07e84e0daf77364fc9348c043792d","bytes":35700},{"path":"voice/Derm1/tgt.turret.wav","sha":"fccc325b3effbcbef2749a9d4cd41d7ce28bd1bf","bytes":34220},{"path":"voice/Derm1/tgt.wait.wav","sha":"a6b0a583bd19b5e044e5c7a9c0633efb08119de9","bytes":27406},{"path":"voice/Derm1/vqk.anytime.wav","sha":"727a2c49317c5c514478119abca452a2d53d4ed9","bytes":12804},{"path":"voice/Derm1/vqk.base.wav","sha":"048ef7d48c26848cfabadf9d222faf03473e4d67","bytes":24292},{"path":"voice/Derm1/vqk.ceasefire.wav","sha":"aa859ff00b879beb08095cd97fe7f99189ef71a5","bytes":18798},{"path":"voice/Derm1/vqk.dunno.wav","sha":"8d2bdfb21e6bae6edf4e3800aec02a64cf61a4c6","bytes":16316},{"path":"voice/Derm1/vqk.help.wav","sha":"b37b9e27bd66ebfcdb98ce679284df4993c4790c","bytes":11960},{"path":"voice/Derm1/vqk.move.wav","sha":"a3f4cc64901b07cf310c5c8d847350aa2162a9c4","bytes":18602},{"path":"voice/Derm1/vqk.no.wav","sha":"44eb2bc6e741385c181380498e88117d64379b1c","bytes":12010},{"path":"voice/Derm1/vqk.quiet.WAV","sha":"5ad0aa0350863ef5000545866fcf5bfde1f4818a","bytes":12284},{"path":"voice/Derm1/vqk.sorry.wav","sha":"c682714d71053a3a0a7ee89b7a49368d32a1b2a8","bytes":15996},{"path":"voice/Derm1/vqk.thanks.wav","sha":"e469a69d4c3c1166d067043ab5bf6f3c47555b50","bytes":26112},{"path":"voice/Derm1/vqk.wait.wav","sha":"98a4a51419556ecd94133693955548ae72633edd","bytes":7686},{"path":"voice/Derm1/vqk.yes.wav","sha":"ae6f4502ce97565a66c0d7f47cf466218a90e8e1","bytes":15962},{"path":"voice/Derm1/wrn.bomber.wav","sha":"afba8eefe69250b73ac6db00900059c04ecd40c5","bytes":16814},{"path":"voice/Derm1/wrn.enemy.wav","sha":"eed827f057227cbd3b749885afa645f912d15099","bytes":19898},{"path":"voice/Derm1/wrn.vehicles.wav","sha":"a18c04ddaec66b0a6308fd0295f85d5a9c984bd4","bytes":19290},{"path":"voice/Derm1/wrn.watchit.wav","sha":"0bfccf505a1f1fabed084a0ae5e7bea7e590821c","bytes":21074},{"path":"voice/Derm2/att.attack.WAV","sha":"27487c4cb4a593b365fecf6906ab3bd4c5f44598","bytes":12642},{"path":"voice/Derm2/att.base.WAV","sha":"d49d10bd039b439b82b317388044b0fbe8f6244a","bytes":21044},{"path":"voice/Derm2/att.chase.WAV","sha":"1803ecb04ac8fc71e332fad98841e82d39adc15b","bytes":22360},{"path":"voice/Derm2/att.distract.WAV","sha":"3b617814d6e4bfab49d11aad0f1158e2db7505a4","bytes":28830},{"path":"voice/Derm2/att.flag.WAV","sha":"64f1b5e8271e7691e39e6964e43d9efcfd3b5d82","bytes":20226},{"path":"voice/Derm2/att.generator.WAV","sha":"92ba8088de8e1477779378bd6fb3584d3426a3a2","bytes":27856},{"path":"voice/Derm2/att.objective.WAV","sha":"b2515c60350b7218eab5a6cea88601dcdd6bdc14","bytes":26800},{"path":"voice/Derm2/att.reinforcements.WAV","sha":"b2a88bd72e8d72807258214f0e3b236ca549d2f6","bytes":26808},{"path":"voice/Derm2/att.sensors.WAV","sha":"95bc46e73780ee9b82dbbdbb4dd9b5c4b00a14f4","bytes":25652},{"path":"voice/Derm2/att.turrets.WAV","sha":"215ecb92b9da543242516561f689f702753b4eb0","bytes":31678},{"path":"voice/Derm2/att.vehicle.WAV","sha":"dd3aa88ff9f61cab54825739b531d4b60255f39c","bytes":24536},{"path":"voice/Derm2/att.wait.WAV","sha":"177562a1c2ddd028a81af84944c38670a289cd9c","bytes":31342},{"path":"voice/Derm2/avo.deathcry_01.WAV","sha":"33667aeb9645df29d8ac502f35efd36a0dc7c89e","bytes":19516},{"path":"voice/Derm2/avo.deathcry_02.WAV","sha":"ee7ac6cea50266437d7aada2e755bca38dc43c1e","bytes":15420},{"path":"voice/Derm2/avo.grunt.WAV","sha":"2fadc9c361de5c9da64312c8b90c0405075572a3","bytes":11320},{"path":"voice/Derm2/avo.pain.WAV","sha":"b2a2f7e8fc2fb2e3ad9a3a52c8b982a878b1ff30","bytes":7228},{"path":"voice/Derm2/bas.clear.WAV","sha":"2bef3b36a3ea95d23acfbeb736981df64af45c7c","bytes":23650},{"path":"voice/Derm2/bas.enemy.WAV","sha":"35f4781715e3c29a20ec72b9624e7079560555a9","bytes":23058},{"path":"voice/Derm2/bas.retake.WAV","sha":"1db8bc5ab012e16481930a7e7b47252f057c7a97","bytes":31470},{"path":"voice/Derm2/bas.secure.WAV","sha":"782f2b0eec9f3205446d304c8307116286f223d3","bytes":22674},{"path":"voice/Derm2/bas.taken.WAV","sha":"013e05e9564422880f279859efb2c7ac4bc0b6ba","bytes":20384},{"path":"voice/Derm2/cmd.acknowledge.WAV","sha":"ad36b186869c0e5458e7fc0c2d5619626b329c24","bytes":21490},{"path":"voice/Derm2/cmd.bot.WAV","sha":"6cae891ad1c05ddc429c5c5ba9d5bac2904d61ea","bytes":20220},{"path":"voice/Derm2/cmd.completed.WAV","sha":"c7029b818beed4c6926ed71e11d0e67773dc4fa1","bytes":19844},{"path":"voice/Derm2/cmd.decline.WAV","sha":"01d326e0a8fc3944ebb11d0da676dfa8660097f9","bytes":21642},{"path":"voice/Derm2/def.base.WAV","sha":"c78a2ade72330a8f59e086e54dc009f3a92c7cc9","bytes":22156},{"path":"voice/Derm2/def.carrier.WAV","sha":"a3f2867b6c6790fa7868807be5a88426e6f487f4","bytes":22308},{"path":"voice/Derm2/def.entrances.WAV","sha":"da2f17514ee76920471979d6b6f26cad1ddf4c9e","bytes":25548},{"path":"voice/Derm2/def.flag.WAV","sha":"9ece79c2c786d1ba598420f03f45947952041d7c","bytes":24922},{"path":"voice/Derm2/def.generator.WAV","sha":"3f1dae1a458f3183c69dc4579c12889beeda3a52","bytes":20822},{"path":"voice/Derm2/def.me.WAV","sha":"8435c208b8310f814108ccbdbd32a8e71cd0f403","bytes":16418},{"path":"voice/Derm2/def.nexus.WAV","sha":"aac2e00b4106437cf32d50560a81e97a8caba3c9","bytes":23472},{"path":"voice/Derm2/def.objective.WAV","sha":"3142d20f0e9c9a6b787305a5a437541c7ecd9b2f","bytes":26294},{"path":"voice/Derm2/def.reinforce.WAV","sha":"e95e21da4229deed4c7e8862d9b6834a5b7a7dd1","bytes":27574},{"path":"voice/Derm2/def.sensors.WAV","sha":"6c9b5ad9d2ce9059ec51685463facad6f73c3e6a","bytes":25782},{"path":"voice/Derm2/def.turrets.WAV","sha":"00a59a425fa2641770b816e9b088badad9d104d4","bytes":26576},{"path":"voice/Derm2/def.vehicle.WAV","sha":"23187825078fce85c06e48219e9f763b4c028291","bytes":22134},{"path":"voice/Derm2/ene.base.WAV","sha":"e793d528948fb2198b5e610896f734a91667276d","bytes":26768},{"path":"voice/Derm2/ene.disarray.WAV","sha":"4219fdcbc39eabb60fd64be163298291572055c0","bytes":33048},{"path":"voice/Derm2/ene.generator.WAV","sha":"1e2b83087783fe8d1bd60b52b8a2a35631d61e25","bytes":31182},{"path":"voice/Derm2/ene.remotes.WAV","sha":"6edc2e60a68bad03d97210d602c8a5a965514984","bytes":32720},{"path":"voice/Derm2/ene.sensors.WAV","sha":"7af89c450c8eb12f120461c958aa3fc2dc389fef","bytes":29530},{"path":"voice/Derm2/ene.turrets.WAV","sha":"c426c5d509a53598928f39df15f4d64e0437fd12","bytes":27378},{"path":"voice/Derm2/ene.vehicle.WAV","sha":"6096a4f0c9c59ef1b5ee41720bd0bca137a8ce7e","bytes":33138},{"path":"voice/Derm2/flg.flag.WAV","sha":"f512b5d62c340d03b771045ab5973cb612d31f51","bytes":24902},{"path":"voice/Derm2/flg.give.WAV","sha":"3329cf9467cfdf9ad168eb3d1ca79f8836fcd471","bytes":20868},{"path":"voice/Derm2/flg.huntergive.WAV","sha":"08d7c0e95f48ea031388da986f2eeb7454cb5eb2","bytes":25082},{"path":"voice/Derm2/flg.huntertake.WAV","sha":"b0fc8ddee799e3475a02eca0ae3138a92d78622b","bytes":18648},{"path":"voice/Derm2/flg.retrieve.WAV","sha":"a9aabd806adc2a452cc8e628d954fbe6ace11f5d","bytes":24012},{"path":"voice/Derm2/flg.secure.WAV","sha":"cf2c3ff6de27693d4fb4fc70f1658d337b2398a8","bytes":23952},{"path":"voice/Derm2/flg.take.WAV","sha":"df3751d747a78804099a93ddb92d11e85943fa6d","bytes":21126},{"path":"voice/Derm2/gbl.anytime.WAV","sha":"409bd8d68ea90bae37a46f130cc35e772d126b1f","bytes":23668},{"path":"voice/Derm2/gbl.awesome.WAV","sha":"48b231ba6414e2c3480e23682d5922962fd0d681","bytes":26038},{"path":"voice/Derm2/gbl.aww.WAV","sha":"74b076a6ab284b2d6048a3c86659b8bf54f6b09c","bytes":33360},{"path":"voice/Derm2/gbl.brag.WAV","sha":"bba6970ee0028da2eeebff6db65ccd4919deb0fc","bytes":28150},{"path":"voice/Derm2/gbl.bye.WAV","sha":"ce0fe1b6ff41c1063024f6c8a678b6f77cc4fd05","bytes":16556},{"path":"voice/Derm2/gbl.dunno.WAV","sha":"efb6fdb1ad53cbfb2cd7ce971b659cb2a3584aed","bytes":17824},{"path":"voice/Derm2/gbl.goodgame.WAV","sha":"46e598f588d5ee6a260ebda166d03dafc6c58aa2","bytes":18592},{"path":"voice/Derm2/gbl.hi.WAV","sha":"fb188156c18b19ba03789c145f13d19bb1335404","bytes":12768},{"path":"voice/Derm2/gbl.nice.WAV","sha":"ff55fd56dd84e480d7c108c48f376da0e631ce71","bytes":25818},{"path":"voice/Derm2/gbl.no.WAV","sha":"685cb7a59e981521f8d5dde7417d236a11700851","bytes":14548},{"path":"voice/Derm2/gbl.obnoxious.WAV","sha":"f853fb25451e0c241626e134e14d0bc48b67b66a","bytes":37412},{"path":"voice/Derm2/gbl.oops.WAV","sha":"4c6510a74c6406cda5d2d9cab8f6e6eaf6e13c7f","bytes":13610},{"path":"voice/Derm2/gbl.quiet.WAV","sha":"00c46b2bba954507cf8e06ee77a1b4f75ee1f6c0","bytes":13734},{"path":"voice/Derm2/gbl.rock.WAV","sha":"ff90bf9b708b2b2c383ff2abc14235acb7cdab68","bytes":32820},{"path":"voice/Derm2/gbl.sarcasm.WAV","sha":"91eabbb4f6e90f28674b8632e8c098827d613257","bytes":52910},{"path":"voice/Derm2/gbl.shazbot.WAV","sha":"6936be13832e40e58281284f7ce5a1131529e864","bytes":25098},{"path":"voice/Derm2/gbl.shooting.WAV","sha":"99f735fc08010cec444ac2a8675f895b8d5923a2","bytes":14822},{"path":"voice/Derm2/gbl.thanks.WAV","sha":"6c37664edba28918938c2ae1150df9a8e50922e8","bytes":15906},{"path":"voice/Derm2/gbl.wait.WAV","sha":"fa0fe2cfd953d7e822bcb9fe02c7bd4b89f210d0","bytes":28102},{"path":"voice/Derm2/gbl.when.WAV","sha":"0e11a44cc0f3e5886fbe36581becf74ecb5cb9a5","bytes":38730},{"path":"voice/Derm2/gbl.woohoo.WAV","sha":"9d90defea17a44164214eeb841dccc638d29ea45","bytes":30328},{"path":"voice/Derm2/gbl.yes.WAV","sha":"2e57057e783c12817d2bb4fc1254d84de4dca1f9","bytes":14488},{"path":"voice/Derm2/need.bombardier.WAV","sha":"8328213a5ea6b237707ac06c326f68d532117e74","bytes":22466},{"path":"voice/Derm2/need.cover.WAV","sha":"30877dd9b36fd7f22c6c9c327e8f4d355619cb8a","bytes":21982},{"path":"voice/Derm2/need.driver.WAV","sha":"e16bb282affc7aaf4c09ca7523d0a08c0c9bf65b","bytes":32688},{"path":"voice/Derm2/need.escort.WAV","sha":"3c751e641ef320acddf51076e08ec665ea1368c4","bytes":28472},{"path":"voice/Derm2/need.flyer.WAV","sha":"be1b8ea5e69a08664aaff95146a692370ebfab29","bytes":30174},{"path":"voice/Derm2/need.gunship.WAV","sha":"ee63596ebf3bc9335c2901d10ce90db263c63d44","bytes":34136},{"path":"voice/Derm2/need.hold.WAV","sha":"a34d4b401e2717b56ea409af2790fbe61529a4a6","bytes":33480},{"path":"voice/Derm2/need.ride.WAV","sha":"44563a691ed7a05ba7619207c3e25e2dd7914a1e","bytes":20292},{"path":"voice/Derm2/need.support.WAV","sha":"180c4c325cd6cafe8dcec6f9c173db633d90b74b","bytes":23666},{"path":"voice/Derm2/need.tailgunner.WAV","sha":"3b8f19e805d4d1977f218c78670103c31ee09940","bytes":25916},{"path":"voice/Derm2/need.where.WAV","sha":"d6aa783484b8960b524a67b671343e0ae35ffc9c","bytes":20390},{"path":"voice/Derm2/rep.base.WAV","sha":"13dce699afa8d3656ffba3650384bf916ec527b4","bytes":25568},{"path":"voice/Derm2/rep.generator.WAV","sha":"0adda003c5a0c03b3f3ed98a958e029dea246dd3","bytes":26266},{"path":"voice/Derm2/rep.me.WAV","sha":"12549aac14c5916ff1f8bcaaa678682f42009378","bytes":22872},{"path":"voice/Derm2/rep.sensors.WAV","sha":"ae392646f3bc4d3e4d52e33ab655b604e7f90984","bytes":27320},{"path":"voice/Derm2/rep.turrets.WAV","sha":"946ee1e7751f30ec967fae5f658b37187b172069","bytes":27130},{"path":"voice/Derm2/rep.vehicle.WAV","sha":"393f88a023801aff2f7b0827f566e355fe87c718","bytes":28288},{"path":"voice/Derm2/slf.att.attack.WAV","sha":"8ef2435c5a3fa7a150dc769cf568fcf81fa6d3a8","bytes":16938},{"path":"voice/Derm2/slf.att.base.WAV","sha":"10049e6b51e338dac0d6c618800e31a9e3e93e07","bytes":26378},{"path":"voice/Derm2/slf.att.flag.WAV","sha":"a419c14bb59175b1811a6be39eb65d6dea05ef7c","bytes":31424},{"path":"voice/Derm2/slf.att.generator.WAV","sha":"203b0877cb8d7ef257633c2a1ad888dd177b5ed2","bytes":30260},{"path":"voice/Derm2/slf.att.sensors.WAV","sha":"3a6eae68c02a4985b5252e9f52e85bdc3c8facaa","bytes":28288},{"path":"voice/Derm2/slf.att.turrets.WAV","sha":"5e9dbc059c9ea8f449980dde989f57f28e153e77","bytes":30544},{"path":"voice/Derm2/slf.att.vehicle.WAV","sha":"4c13181a7e13890414604d1189e7521c28f88dce","bytes":36346},{"path":"voice/Derm2/slf.def.base.WAV","sha":"25840aeb526cb007853edd3f25c00bea58475432","bytes":25382},{"path":"voice/Derm2/slf.def.defend.WAV","sha":"b96343debbbee6c4d3d4bdd9ad3c83357a80b74b","bytes":21762},{"path":"voice/Derm2/slf.def.flag.WAV","sha":"45f0453d9603c585731e55ca6b6cf48c38f3b70c","bytes":24470},{"path":"voice/Derm2/slf.def.generator.WAV","sha":"9dd4689331fe5c12036eef04f4accab9a909a21d","bytes":27614},{"path":"voice/Derm2/slf.def.nexus.WAV","sha":"2a35860dcc6746430fda30e2e956b3c431b84003","bytes":29228},{"path":"voice/Derm2/slf.def.sensors.WAV","sha":"a1fced364071f8a14a3e94f5e16e9dbc93f7624f","bytes":26228},{"path":"voice/Derm2/slf.def.turrets.WAV","sha":"49b3da90aa6309656f60067bc97b0c782da2e8c9","bytes":23072},{"path":"voice/Derm2/slf.def.vehicle.WAV","sha":"a10ef9e83a3b46bd69abc00eabdb05a1f067c2e8","bytes":30004},{"path":"voice/Derm2/slf.rep.base.WAV","sha":"4e01da0438da13c48b431ccc0dc54bdb88094c05","bytes":25012},{"path":"voice/Derm2/slf.rep.equipment.WAV","sha":"6f11e89a4f5758cc88675e94c1d10ba60a4d98bf","bytes":25348},{"path":"voice/Derm2/slf.rep.generator.WAV","sha":"e2a85ccdb0903188d078e29dbe25aa4550654022","bytes":27902},{"path":"voice/Derm2/slf.rep.repairing.WAV","sha":"9ebe1cde29e60d2bf1ff87b4dc701023bf0964ed","bytes":22070},{"path":"voice/Derm2/slf.rep.sensors.WAV","sha":"f84dc8ae0970d234b0456436f1b26b0a3bd9286f","bytes":23150},{"path":"voice/Derm2/slf.rep.turrets.WAV","sha":"1197c3ffa09dbd0b91068b5288ba44b043688461","bytes":24152},{"path":"voice/Derm2/slf.rep.vehicle.WAV","sha":"97dfd10b7a0aced33b90ff2f6b83720e89963093","bytes":29086},{"path":"voice/Derm2/slf.tsk.cover.WAV","sha":"9b4deb35ddaf387b7d4c95bc406f339db6a47d7c","bytes":14958},{"path":"voice/Derm2/slf.tsk.defense.WAV","sha":"de0ef6dd62266f64e8f9222f79ef8c0ff0b35155","bytes":22744},{"path":"voice/Derm2/slf.tsk.generic.WAV","sha":"b1116225b66a79645cc8cf19f4a3b5b8ff545e32","bytes":20710},{"path":"voice/Derm2/slf.tsk.remotes.WAV","sha":"f7469bdeb7589eff8315176355674ace543b2aa1","bytes":27478},{"path":"voice/Derm2/slf.tsk.sensors.WAV","sha":"00f36a1b070f7f13834146453b9bba685fae2b97","bytes":23776},{"path":"voice/Derm2/slf.tsk.turrets.WAV","sha":"8a1338fb9f4173e38ee4442c7311d4188246b4e6","bytes":23588},{"path":"voice/Derm2/slf.tsk.vehicle.WAV","sha":"b400fb00e8cbfebdd9bdc44c935afca53e2a1267","bytes":27394},{"path":"voice/Derm2/tgt.acquired.WAV","sha":"9f0242a4d4de0e862b69fe016aa973376b348dcd","bytes":20428},{"path":"voice/Derm2/tgt.base.WAV","sha":"80d6bef9095891f27599c0eeb2fb9542f23b5e97","bytes":38972},{"path":"voice/Derm2/tgt.destroyed.WAV","sha":"ddc2b71fe6b26d2f125d793426cd4f0d9df7dbb1","bytes":36340},{"path":"voice/Derm2/tgt.flag.WAV","sha":"bb936a9a3410acedf4805d51a5c93b586b282b5f","bytes":34976},{"path":"voice/Derm2/tgt.my.WAV","sha":"1e3223038f0c796a3382aaffa7c2c74d081645f5","bytes":24584},{"path":"voice/Derm2/tgt.need.WAV","sha":"562ba05c72a65e9c0b3991da35771680537f6879","bytes":24914},{"path":"voice/Derm2/tgt.sensors.WAV","sha":"9aef473a07059671a24c1d98d19ed8283912b4ae","bytes":34398},{"path":"voice/Derm2/tgt.turret.WAV","sha":"c9c0e88cd81704b0d4be0fbab00bab00f37d8bff","bytes":30662},{"path":"voice/Derm2/tgt.wait.WAV","sha":"c651fd01617addfb4564c7a2f337518ac758249c","bytes":34994},{"path":"voice/Derm2/vqk.anytime.WAV","sha":"2bb7ba16a1660141a1b16cce313899edb87e3c45","bytes":20436},{"path":"voice/Derm2/vqk.base.WAV","sha":"6f8a66119cc2f4320801d3a6eb3e64a2f8b30058","bytes":22588},{"path":"voice/Derm2/vqk.ceasefire.WAV","sha":"350f14ba6853724554eeed2974359748dbcf979a","bytes":17286},{"path":"voice/Derm2/vqk.dunno.WAV","sha":"961c0221526206c7c7ef7bb0c07464edbc10579c","bytes":15660},{"path":"voice/Derm2/vqk.help.WAV","sha":"67fe1c43852c52f4871c82731b3f61413b825174","bytes":11874},{"path":"voice/Derm2/vqk.move.WAV","sha":"68ebd09671d5983060cc33351e5678f402fcbefa","bytes":36830},{"path":"voice/Derm2/vqk.no.WAV","sha":"e8655a27142e7f56b7ba9a1325f8343c9894688e","bytes":9692},{"path":"voice/Derm2/vqk.quiet.WAV","sha":"9b38ffe1d413904b2501b4ff086ce83097460640","bytes":14592},{"path":"voice/Derm2/vqk.sorry.WAV","sha":"64305acfbdb7a5cdfd090ff30ded5698c5863247","bytes":16820},{"path":"voice/Derm2/vqk.thanks.WAV","sha":"db35534dd9f2b0a394886d41b3b48ecb86229abc","bytes":17174},{"path":"voice/Derm2/vqk.wait.WAV","sha":"3d08378bdde99010da2d64984eba52de18f72b16","bytes":16102},{"path":"voice/Derm2/vqk.yes.WAV","sha":"142aa31815bc5eb1502535e46ace761fb333c844","bytes":20316},{"path":"voice/Derm2/wrn.bomber.WAV","sha":"6661ae6e32728371fb35d29882f46c312c7fdf20","bytes":20188},{"path":"voice/Derm2/wrn.enemy.WAV","sha":"298336be1af738a4e08b8f1e51e94f516da1ebe8","bytes":23668},{"path":"voice/Derm2/wrn.vehicles.WAV","sha":"534427630a1ab166b1a955b7874b5f758c57da20","bytes":24140},{"path":"voice/Derm2/wrn.watchit.WAV","sha":"2b17364ee8ae0677fec4cbab204505ca16b19333","bytes":27152},{"path":"voice/Derm3/att.attack.WAV","sha":"a92b02e4c7db1904711e80974c70fd4b515eafbf","bytes":13840},{"path":"voice/Derm3/att.base.WAV","sha":"8087ee17a452cbae120a716f1c98c83152e76215","bytes":21840},{"path":"voice/Derm3/att.chase.WAV","sha":"a15842a92e7c6e78f785c8bcd284614db5e73e69","bytes":23322},{"path":"voice/Derm3/att.distract.WAV","sha":"84fd1035905147af30f95392aa7643dea0cebe7a","bytes":27444},{"path":"voice/Derm3/att.flag.WAV","sha":"32e84bf10447aee9c7b639c5b78ad00e6207cf94","bytes":24860},{"path":"voice/Derm3/att.generator.WAV","sha":"bd0ed1f03a7dc890aac2ac05637fb09c4fb45e5e","bytes":25878},{"path":"voice/Derm3/att.objective.WAV","sha":"a4f75f3c2ba2fcb605a3f16628bc71be2e2527da","bytes":22672},{"path":"voice/Derm3/att.reinforcements.WAV","sha":"d1b761db3257e92d49174a69c5a687d06d76f3de","bytes":26728},{"path":"voice/Derm3/att.sensors.WAV","sha":"e3f3644af30fe21242ee4e04dad49bc4b65041dc","bytes":28116},{"path":"voice/Derm3/att.turrets.WAV","sha":"fe72cffd527fec442b91c8a63674a8e47ceb4170","bytes":29036},{"path":"voice/Derm3/att.vehicle.WAV","sha":"04d8793dc3e7eab251813658e0c2086d0b01be4e","bytes":24820},{"path":"voice/Derm3/att.wait.WAV","sha":"ea6221b79702fd1bad25eeeee48adf81bd31e7f3","bytes":33214},{"path":"voice/Derm3/avo.deathcry_01.WAV","sha":"d01774221787ad5f05e87b4462f40418b20f6aad","bytes":17980},{"path":"voice/Derm3/avo.deathcry_02.WAV","sha":"98ab6a88896b8a68f6cae758815fc906008e9008","bytes":17468},{"path":"voice/Derm3/avo.grunt.WAV","sha":"8cf013d59dc69b69a015c287d3e6dd1252cce654","bytes":7306},{"path":"voice/Derm3/avo.pain.WAV","sha":"fba5745c2dd6880a817abe250e45ad90c39a2378","bytes":6440},{"path":"voice/Derm3/bas.clear.WAV","sha":"796af5f51d3529f8479e553e0e27cc995e4d49b3","bytes":26550},{"path":"voice/Derm3/bas.enemy.WAV","sha":"051757c5cd8b8e57ee0401a212bd93cde799defb","bytes":28226},{"path":"voice/Derm3/bas.retake.WAV","sha":"a2a10ff19bfe2a71134049fd75c82551c5a08e88","bytes":21714},{"path":"voice/Derm3/bas.secure.WAV","sha":"20164892930ef75d7e909465cf6ac6fb90c350b8","bytes":26674},{"path":"voice/Derm3/bas.taken.WAV","sha":"8ffb6614f123f5ce5e863bf6d68b9f208d82278f","bytes":25364},{"path":"voice/Derm3/cmd.acknowledge.WAV","sha":"13ee09f266dc6eae4b6bc2f9d7a09d6de93755a4","bytes":17344},{"path":"voice/Derm3/cmd.bot.WAV","sha":"c9dd1737de688707a7616e780c0bd3f73e9e9dc1","bytes":20080},{"path":"voice/Derm3/cmd.completed.WAV","sha":"1d481845f874264f140dea8d0f8dbf354a026423","bytes":17996},{"path":"voice/Derm3/cmd.decline.WAV","sha":"2b8775e9d25fbd1ab4e3fcd2e1d3791624680955","bytes":20698},{"path":"voice/Derm3/def.base.WAV","sha":"75e280f72536fce9866b8f8608d2f4acf9b28f7a","bytes":22654},{"path":"voice/Derm3/def.carrier.WAV","sha":"3685ae52ffd03562782507093630b938bad7a246","bytes":21630},{"path":"voice/Derm3/def.entrances.WAV","sha":"c5174f6f06c13b0c045478e3ae56d4a86aceb443","bytes":22460},{"path":"voice/Derm3/def.flag.WAV","sha":"22891b973441ac58cbf456e389db726d2c86b687","bytes":23918},{"path":"voice/Derm3/def.generator.WAV","sha":"c88a7908aea54f2d881f42e086b1ee2e9630c89a","bytes":19718},{"path":"voice/Derm3/def.me.WAV","sha":"3be683b569e28e0eb8ac8b9f53d76cba298b9b57","bytes":14654},{"path":"voice/Derm3/def.nexus.WAV","sha":"ee5dc9218466a346f947f7135910a9c48de8ddcf","bytes":21310},{"path":"voice/Derm3/def.objective.WAV","sha":"2f1fe9c0bf802a3d0446ef651b9f2b923e88c19f","bytes":20328},{"path":"voice/Derm3/def.reinforce.WAV","sha":"c22f9ad0a1beaae658390d4df4e2cb20f4557d5b","bytes":25876},{"path":"voice/Derm3/def.sensors.WAV","sha":"92cc21f6c5fbb07d35e68d7122a508dad089ff44","bytes":24282},{"path":"voice/Derm3/def.turrets.WAV","sha":"15d9d1b2633912f3911ade5b550a34fb530db84e","bytes":22942},{"path":"voice/Derm3/def.vehicle.WAV","sha":"28f5a3868cde3d05cae12e851aa4b6f1b99eeadf","bytes":22718},{"path":"voice/Derm3/ene.base.WAV","sha":"eff074e13c8e146178c0aea208ae7e015b8c5f43","bytes":22698},{"path":"voice/Derm3/ene.disarray.WAV","sha":"1fded9f0ce5e85e53080e35bbbd6fc492e1775bc","bytes":28770},{"path":"voice/Derm3/ene.generator.WAV","sha":"b54e257a0b71c45e51882458b6b38c357bd7ae48","bytes":25530},{"path":"voice/Derm3/ene.remotes.WAV","sha":"4c8e3e1d9efa60723a94f5daf541709f1b34b638","bytes":31598},{"path":"voice/Derm3/ene.sensors.WAV","sha":"cdffd7e786524a60c97f9fec424d02ecdb6ff933","bytes":27932},{"path":"voice/Derm3/ene.turrets.WAV","sha":"a923d3958a141d098dace1f2fdadef130c6c7ef5","bytes":28582},{"path":"voice/Derm3/ene.vehicle.WAV","sha":"54d62392d50be4a31eb86c412ef59986f189f3bb","bytes":31938},{"path":"voice/Derm3/flg.flag.WAV","sha":"eb03a982a3253f0070f67f077124b4087195b825","bytes":28300},{"path":"voice/Derm3/flg.give.WAV","sha":"0339c91c783ee170383cd54fa2124be1dc115731","bytes":19478},{"path":"voice/Derm3/flg.huntergive.WAV","sha":"affb54f2d62194b0576a70ffa3ce1ec787aa74d3","bytes":24310},{"path":"voice/Derm3/flg.huntertake.WAV","sha":"46bcf1578191134d5d8ad849c1100ae4c92c82ba","bytes":20866},{"path":"voice/Derm3/flg.retrieve.WAV","sha":"b97f069159a5b10c178afc88fe37eb3aeeb3f3b9","bytes":31304},{"path":"voice/Derm3/flg.secure.WAV","sha":"7958ca0f0caa9800637dc125c74bf793d2cf0796","bytes":21632},{"path":"voice/Derm3/flg.take.WAV","sha":"909fefaa424bf8e12659759ec0940bec37300d66","bytes":21328},{"path":"voice/Derm3/gbl.anytime.WAV","sha":"bb73f2668f198068aa0ff45c8933fb8b4b6c1ffe","bytes":19398},{"path":"voice/Derm3/gbl.awesome.WAV","sha":"a8f36476a7badf39d29fa5ebfc7d3390a794e74a","bytes":19120},{"path":"voice/Derm3/gbl.aww.WAV","sha":"e8b0c86f030c3868f043803ab0655f167e12a190","bytes":41154},{"path":"voice/Derm3/gbl.brag.WAV","sha":"92c087626ef9f8e00b3b5891850aace7657c09f3","bytes":28606},{"path":"voice/Derm3/gbl.bye.WAV","sha":"71f23761bc0138d784d613458239fffd807e8d6a","bytes":16856},{"path":"voice/Derm3/gbl.dunno.WAV","sha":"b5925b3011013cbdb6150eea6037156ff708f5cf","bytes":20226},{"path":"voice/Derm3/gbl.goodgame.WAV","sha":"c5965d0dc76bd74ee1d556c3df0468bf64cbf2e1","bytes":20928},{"path":"voice/Derm3/gbl.hi.WAV","sha":"aa4032a72dbc0ac59b9691d2c60d6ffcaa1a519f","bytes":15458},{"path":"voice/Derm3/gbl.nice.WAV","sha":"f138888c3b74662e0960cebb1192a7c0e190d638","bytes":28514},{"path":"voice/Derm3/gbl.no.WAV","sha":"7081847eadf7bb5775a800b49b6cdbd8238a0e6b","bytes":12732},{"path":"voice/Derm3/gbl.obnoxious.WAV","sha":"66414b6d839903cf291d5b19e0dfb9b243a63801","bytes":34888},{"path":"voice/Derm3/gbl.oops.WAV","sha":"4cbc3c754f695798c1108064fc321288ec59b202","bytes":15442},{"path":"voice/Derm3/gbl.quiet.WAV","sha":"547311468b4d5e1ca766e752af68d8f39c84795a","bytes":13156},{"path":"voice/Derm3/gbl.rock.WAV","sha":"ed89a382d59f6b98340d8cfff265cde2cdea3afe","bytes":22514},{"path":"voice/Derm3/gbl.sarcasm.WAV","sha":"b4063760bc0bf818af4d6bf590c05740875822e7","bytes":23540},{"path":"voice/Derm3/gbl.shazbot.WAV","sha":"b2e896329dfc3d6c80e5dafdcf587a206633164b","bytes":15850},{"path":"voice/Derm3/gbl.shooting.WAV","sha":"fb296332f8e308bb0f9f1a80f1cc86f83017f0ed","bytes":18504},{"path":"voice/Derm3/gbl.thanks.WAV","sha":"5c8163ab20e2137dc5aae695116a9ffa45e22b2c","bytes":15616},{"path":"voice/Derm3/gbl.wait.WAV","sha":"a922f307edd149f89a99790d123113b2711870de","bytes":13522},{"path":"voice/Derm3/gbl.when.WAV","sha":"b33c06acd96a19d802bd1ec5b22b26538aa5a1c3","bytes":35238},{"path":"voice/Derm3/gbl.woohoo.WAV","sha":"e1378ec258a43ecd88549586b8d67a27e95bb53d","bytes":25022},{"path":"voice/Derm3/gbl.yes.WAV","sha":"99722b3d252ffe0d9dad3bcda06b25ca6e00cde0","bytes":11930},{"path":"voice/Derm3/need.bombardier.WAV","sha":"2232bda0bc68170d05bef3150086e1c18edf9eeb","bytes":18730},{"path":"voice/Derm3/need.cover.WAV","sha":"bead9a36e53d5543c9adcb02d657d0e16d67df48","bytes":24806},{"path":"voice/Derm3/need.driver.WAV","sha":"d7ca001e611d9021b7b49d001864b3ce376359b7","bytes":27754},{"path":"voice/Derm3/need.escort.WAV","sha":"32366ea93e4c5fdf33734baef0ad2c537aac9d4d","bytes":21738},{"path":"voice/Derm3/need.flyer.WAV","sha":"8094f5f2e11772c04e40225bded2af9bd507f238","bytes":26382},{"path":"voice/Derm3/need.gunship.WAV","sha":"693979fcd2883488b020ec3016129b573dc978ef","bytes":31328},{"path":"voice/Derm3/need.hold.WAV","sha":"9543e2293d096be4d90d2b1d80f25051ed450b2e","bytes":25240},{"path":"voice/Derm3/need.ride.WAV","sha":"221845e42269de283f5e730c5930771fe21a7e31","bytes":20538},{"path":"voice/Derm3/need.support.WAV","sha":"38abee43f0addbe8ff71cdba7132996927586315","bytes":23746},{"path":"voice/Derm3/need.tailgunner.WAV","sha":"6cb8feb22c780333b1c2c2247222a497134ea58c","bytes":20498},{"path":"voice/Derm3/need.where.WAV","sha":"5bb2ec8f6d60d666bf006baebdc14674cb5d403d","bytes":16948},{"path":"voice/Derm3/rep.base.WAV","sha":"baa5d6ada3d0f7c6bb8406f17fdddfea19501f15","bytes":24666},{"path":"voice/Derm3/rep.generator.WAV","sha":"d813a1dfdda81ae69f14755e1e08dd7909a421d8","bytes":24632},{"path":"voice/Derm3/rep.me.WAV","sha":"ab427b3e4521445c62e41bdac48ed4c984aab93e","bytes":19994},{"path":"voice/Derm3/rep.sensors.WAV","sha":"b28f61732d22b8d96e622992fe26e1a80e67a6fc","bytes":26864},{"path":"voice/Derm3/rep.turrets.WAV","sha":"52286b16ece3e04745c0d0e2ce1efef2be273f58","bytes":24710},{"path":"voice/Derm3/rep.vehicle.WAV","sha":"36868c451177587a02e79307fdb688b7b4094d76","bytes":26292},{"path":"voice/Derm3/slf.att.attack.WAV","sha":"94b961489926c246ff4c1af3a833aa31abbc94be","bytes":17108},{"path":"voice/Derm3/slf.att.base.WAV","sha":"69571f096901e014035d94f74ffa3bc037cb5210","bytes":25012},{"path":"voice/Derm3/slf.att.flag.WAV","sha":"62487ffbdfd1a5666d2d0d461b4849e30ed45439","bytes":25334},{"path":"voice/Derm3/slf.att.generator.WAV","sha":"f67a815264fe6ff51a0edcf6a2737342458191af","bytes":25960},{"path":"voice/Derm3/slf.att.sensors.WAV","sha":"ccde4cf4f116fd426c6c5124c8b8917b0408310e","bytes":24984},{"path":"voice/Derm3/slf.att.turrets.WAV","sha":"9109b5d669aa227b2352db9e7b274bab1a66483f","bytes":24012},{"path":"voice/Derm3/slf.att.vehicle.WAV","sha":"f0b6847085bc9e1357f944b05e3d325e75882e2c","bytes":28932},{"path":"voice/Derm3/slf.def.base.WAV","sha":"ec41bf7d84490647e0a4d66ff328efdec7b3cef8","bytes":26032},{"path":"voice/Derm3/slf.def.defend.WAV","sha":"2fcf07dc40f131069cc9e5c3c6b2aec273882c19","bytes":18850},{"path":"voice/Derm3/slf.def.flag.WAV","sha":"ae0b80964ad614346f3ac1e448d5b4a31b8ded3e","bytes":24218},{"path":"voice/Derm3/slf.def.generator.WAV","sha":"c5c325a7dd4472a211231bcad5efe30ae0eeb3ce","bytes":25702},{"path":"voice/Derm3/slf.def.nexus.WAV","sha":"ae8f5b56768c5ca23167ac24ea085cbfbd03e2dc","bytes":22850},{"path":"voice/Derm3/slf.def.sensors.WAV","sha":"ca6e2a88d35210277a7b16b52eef8cb3f757edad","bytes":24606},{"path":"voice/Derm3/slf.def.turrets.WAV","sha":"45ab6f7029cfedc97ddaceb44f660a50faddb3f5","bytes":23644},{"path":"voice/Derm3/slf.def.vehicle.WAV","sha":"f2131becd4984c2d80104634cc08e380363ac92c","bytes":26256},{"path":"voice/Derm3/slf.rep.base.WAV","sha":"10eb9c0148f391151e659079535fece1aea5cc23","bytes":25892},{"path":"voice/Derm3/slf.rep.equipment.WAV","sha":"34f5162b65dcb5d73fb243e8c07b3e44b4ea3fb6","bytes":26090},{"path":"voice/Derm3/slf.rep.generator.WAV","sha":"d65a3cd965c697b3fae4a82341b45b8407756c03","bytes":25254},{"path":"voice/Derm3/slf.rep.repairing.WAV","sha":"ce8b30c246385c5f6c3b0438523b70afc587cdab","bytes":23400},{"path":"voice/Derm3/slf.rep.sensors.WAV","sha":"4315014d2d39013a11935c65825cfa9985ab770a","bytes":23160},{"path":"voice/Derm3/slf.rep.turrets.WAV","sha":"a5e434492cb30a8bd03791a89183cb50a10a1574","bytes":23964},{"path":"voice/Derm3/slf.rep.vehicle.WAV","sha":"d5c55551a023fb765df64832663dfdb0d7398e10","bytes":27264},{"path":"voice/Derm3/slf.tsk.cover.WAV","sha":"64bcf3fa75e6bda648ebabf8e34b17a01932bf65","bytes":14640},{"path":"voice/Derm3/slf.tsk.defense.WAV","sha":"ae7bdd3acf14caf4a60754b02bc437dc3626c093","bytes":20088},{"path":"voice/Derm3/slf.tsk.generic.WAV","sha":"f8762992061b3b2a517f47941386fd1c30b63c68","bytes":12912},{"path":"voice/Derm3/slf.tsk.remotes.WAV","sha":"910220d03e2d6082882dfc5fcc57b060a47122ff","bytes":21356},{"path":"voice/Derm3/slf.tsk.sensors.WAV","sha":"6e1e8d9084e19e45f65156e3ce0f765efc608f57","bytes":23436},{"path":"voice/Derm3/slf.tsk.turrets.WAV","sha":"629c52dc50872c7a61a8838674de3b33af074e8d","bytes":23686},{"path":"voice/Derm3/slf.tsk.vehicle.WAV","sha":"007ede6e02a82f78590e941211712e90ea67234c","bytes":22450},{"path":"voice/Derm3/tgt.acquired.WAV","sha":"8cbcb01574784a400076b2192374ef75c682f6b5","bytes":21336},{"path":"voice/Derm3/tgt.base.WAV","sha":"75759d65bf89552690f4e91ef84f5f21743810ee","bytes":32974},{"path":"voice/Derm3/tgt.destroyed.WAV","sha":"dac3a3f3289f1ad79b804e989c9df2fd1ef98715","bytes":22894},{"path":"voice/Derm3/tgt.flag.WAV","sha":"89ec2c1a0f090034ff1c509b9e74b347a5ae3845","bytes":31814},{"path":"voice/Derm3/tgt.my.WAV","sha":"031121922cddf920d2f04c8d4008b68cba68130e","bytes":28778},{"path":"voice/Derm3/tgt.need.WAV","sha":"bf3bb675d9928597dffef74210e2ff8dab1a9807","bytes":26148},{"path":"voice/Derm3/tgt.sensors.WAV","sha":"c99f79d5c869160bb1c6687d250706cf5cc7e25e","bytes":40242},{"path":"voice/Derm3/tgt.turret.WAV","sha":"137c8d5f7019faeb729bcbec596898cd022cfc9c","bytes":35388},{"path":"voice/Derm3/tgt.wait.WAV","sha":"fecb9be44433b74e8e07498a706f67455ac11a65","bytes":32548},{"path":"voice/Derm3/vqk.anytime.WAV","sha":"3469ca5f2ba73e64e76ca6ae8a12a9404b9fdc53","bytes":16208},{"path":"voice/Derm3/vqk.base.WAV","sha":"fa5c02b96c5e1b1a9b3b0e326d3bbd1561fae6ae","bytes":18156},{"path":"voice/Derm3/vqk.ceasefire.WAV","sha":"e1d4c3990edd35b3a65180a6e63d961001d9f860","bytes":15794},{"path":"voice/Derm3/vqk.dunno.WAV","sha":"013b58f52a7df32db81496a66b710c282c8b2f43","bytes":12244},{"path":"voice/Derm3/vqk.help.WAV","sha":"73c81e664b3ba6fa0dd2ff56f1bfca6c76f307b6","bytes":16746},{"path":"voice/Derm3/vqk.move.WAV","sha":"1fdd61e0737a256e66caa41cb6b5b5390f395aa8","bytes":18112},{"path":"voice/Derm3/vqk.no.WAV","sha":"31ff740cd1733c02b8981ac2f4948459eef66ab2","bytes":10424},{"path":"voice/Derm3/vqk.quiet.WAV","sha":"44a83d02eed73168e03f22b2122262f87fb60f4f","bytes":19726},{"path":"voice/Derm3/vqk.sorry.WAV","sha":"2693ff14d1a00105fa4c94045b91f314a602625f","bytes":10432},{"path":"voice/Derm3/vqk.thanks.WAV","sha":"75eb809ed6d56f215538b3d4078ce8730ee76731","bytes":12532},{"path":"voice/Derm3/vqk.wait.WAV","sha":"68b6e0d0904c49c6360e118af83d2a7e4f160ccf","bytes":10456},{"path":"voice/Derm3/vqk.yes.WAV","sha":"462559de2cf52e3899d0ec132954819d74dfd9b1","bytes":11964},{"path":"voice/Derm3/wrn.bomber.WAV","sha":"f9baf0ebc5ed85fee61fe11b5a81b1956d604db4","bytes":20316},{"path":"voice/Derm3/wrn.enemy.WAV","sha":"c47c32452a6ff101766674eec0fa3ea598769970","bytes":25832},{"path":"voice/Derm3/wrn.vehicles.WAV","sha":"b56eb53644ddc8c421f426c51826541d6647b983","bytes":27100},{"path":"voice/Derm3/wrn.watchit.WAV","sha":"a868a77f3eeecfb3538b7ae6135c406b87909bce","bytes":24072},{"path":"voice/Fem1/att.attack.WAV","sha":"2001db94e132a58b06ca3fa9d6e5cc1409625f6e","bytes":11874},{"path":"voice/Fem1/att.base.WAV","sha":"80d221578ba18298ad40086ebafdd23b12323b2a","bytes":20346},{"path":"voice/Fem1/att.chase.WAV","sha":"8d3a5468d57a6048bc16565d0561f5e8dc30dc28","bytes":18888},{"path":"voice/Fem1/att.distract.WAV","sha":"af8c8dcfb93cf02827f430034ed3d264c60ac826","bytes":20996},{"path":"voice/Fem1/att.flag.WAV","sha":"7ae2d3990d03972326f16527ede60fd33a5f1e3b","bytes":24328},{"path":"voice/Fem1/att.generator.WAV","sha":"6b9649f600483b140feacd4cfef405509c18266d","bytes":27638},{"path":"voice/Fem1/att.objective.WAV","sha":"73dbbaf83b267e5c02d1140e0560066c5d646014","bytes":22410},{"path":"voice/Fem1/att.reinforcements.WAV","sha":"7830422829dcc8906942f3d0119163cfaca6e4b2","bytes":19110},{"path":"voice/Fem1/att.sensors.WAV","sha":"ea3cb520adf2e4aee5eca478a701dc5d53765521","bytes":23808},{"path":"voice/Fem1/att.turrets.WAV","sha":"3b3a01aad3fd5ee51511e04ca64f6936f948c5c1","bytes":22774},{"path":"voice/Fem1/att.vehicle.WAV","sha":"763edb8b610d3486e0d782e7da7f08cfe9c42534","bytes":24154},{"path":"voice/Fem1/att.wait.WAV","sha":"185b566c5425bb295b914aab730903b09c16a41d","bytes":25690},{"path":"voice/Fem1/avo.deathcry_01.WAV","sha":"f789cc13436862a077f1c85ac9652d50c1192dd9","bytes":18492},{"path":"voice/Fem1/avo.deathcry_02.WAV","sha":"04a02523bbae0896a9adacb83d7b4d6b9f064226","bytes":12600},{"path":"voice/Fem1/avo.grunt.WAV","sha":"42d24667121d1fef43791fad80410a57dfdad463","bytes":4902},{"path":"voice/Fem1/avo.pain.WAV","sha":"17184d52f59009dd35191c555529147838e15b43","bytes":5418},{"path":"voice/Fem1/bas.clear.WAV","sha":"05b0127c8d3ad46ee20e95e94bc7b0c8ce6afdd5","bytes":20982},{"path":"voice/Fem1/bas.enemy.WAV","sha":"56453d4b7364ddae56441438819e04ef08aec0d9","bytes":22532},{"path":"voice/Fem1/bas.retake.WAV","sha":"df36da272cbfb58441387ce9e8af17689e869c82","bytes":18586},{"path":"voice/Fem1/bas.secure.WAV","sha":"cde51a7864484a0422da5411ed4c67d8a044c8bb","bytes":19110},{"path":"voice/Fem1/bas.taken.WAV","sha":"974d54cfca9bd3653cbb33ba0db5ae6a12587bf9","bytes":19278},{"path":"voice/Fem1/cmd.acknowledge.WAV","sha":"b9de452f244347be45d732d0013972d476d2d891","bytes":17014},{"path":"voice/Fem1/cmd.bot.WAV","sha":"a467182feafd23c82415b4e4f67a2bf6efd12570","bytes":17314},{"path":"voice/Fem1/cmd.completed.WAV","sha":"81a03d4fb932085e86c188bd48f083178fb5b125","bytes":17848},{"path":"voice/Fem1/cmd.decline.WAV","sha":"ba35b2a6e9e900c716a7ba52d34ae4049ff34449","bytes":18612},{"path":"voice/Fem1/def.base.WAV","sha":"f0ffdcc3707bbce04916bda65e7ac45ff0026e42","bytes":17382},{"path":"voice/Fem1/def.carrier.WAV","sha":"edc1c853fba11658f343276825e986344c4b15e9","bytes":19760},{"path":"voice/Fem1/def.entrances.WAV","sha":"655382c1b98fdcd8bc8487ce2f5c4bc2ce92659f","bytes":18396},{"path":"voice/Fem1/def.flag.WAV","sha":"03ab7ed0c7cd250d382e35469e7d5a2b02a1bad6","bytes":20394},{"path":"voice/Fem1/def.generator.WAV","sha":"29208e80d77b8db2068295715ade87e7b5d57a0f","bytes":17734},{"path":"voice/Fem1/def.me.WAV","sha":"0b043cc177d700562f2743fc616e298db8a0947e","bytes":11922},{"path":"voice/Fem1/def.nexus.WAV","sha":"567d8a731206d8661ef07561a5102aa3e61c73a8","bytes":18356},{"path":"voice/Fem1/def.objective.WAV","sha":"11e98d3d119c7acfe85e80449549e5b459c7e04c","bytes":17886},{"path":"voice/Fem1/def.reinforce.WAV","sha":"60754746600bb2340b6f5c55d7a6ef3f6a72fd3b","bytes":20810},{"path":"voice/Fem1/def.sensors.WAV","sha":"121753e7f33d9de0fb4e8ab5af22876f7eb3afec","bytes":20886},{"path":"voice/Fem1/def.turrets.WAV","sha":"f66d4a5c3fba0399fe3923275c61061e89d6472f","bytes":17808},{"path":"voice/Fem1/def.vehicle.WAV","sha":"c4c8be687f6106f0537febd56fd7e1f70544ec77","bytes":17788},{"path":"voice/Fem1/ene.base.WAV","sha":"843bc0a3fd7fcc0571526b7688b76bc522450fa7","bytes":19770},{"path":"voice/Fem1/ene.disarray.WAV","sha":"f5523b7f674e29f170f0934217a6139e49bc836f","bytes":26090},{"path":"voice/Fem1/ene.generator.WAV","sha":"201ef35b319a8556c6f129992c6ee2708866ab11","bytes":22644},{"path":"voice/Fem1/ene.remotes.WAV","sha":"c7edbc15ee27c9558189b3c3ac078914dd74f382","bytes":26522},{"path":"voice/Fem1/ene.sensors.WAV","sha":"e5b9057cf5bedd43129e6d91c4c97c90c8a80faf","bytes":22088},{"path":"voice/Fem1/ene.turrets.WAV","sha":"fbd7fb090b8ac3db22d5ce0282fd7dd9d1eb10d7","bytes":22344},{"path":"voice/Fem1/ene.vehicle.WAV","sha":"59cf94e6e9d05b2dc6242fe37acec77b2fd826e7","bytes":28326},{"path":"voice/Fem1/flg.flag.WAV","sha":"450bc190315cea6254b603375cebef794c486429","bytes":22202},{"path":"voice/Fem1/flg.give.WAV","sha":"a833e41e6accfcda8636d24050689098598e0434","bytes":17898},{"path":"voice/Fem1/flg.huntergive.WAV","sha":"0b926d9a13b366fd088dd1d39e552018ddeee9fa","bytes":22410},{"path":"voice/Fem1/flg.huntertake.WAV","sha":"008ad212fe21b01b5851e3cbd056d141215c71ce","bytes":18560},{"path":"voice/Fem1/flg.retrieve.WAV","sha":"ea2c72f4f657d71b32a75b2c6717682942bd3659","bytes":18256},{"path":"voice/Fem1/flg.secure.WAV","sha":"65543acb5c4237951c4beaa995165cbbbedb12ca","bytes":18768},{"path":"voice/Fem1/flg.take.WAV","sha":"4eb27cee8961505aa19c11ff50859823d9ffba14","bytes":20492},{"path":"voice/Fem1/gbl.anytime.WAV","sha":"7f36f7176015b6b412a1e6052937bef002fdf7d2","bytes":14210},{"path":"voice/Fem1/gbl.awesome.WAV","sha":"ec74ce023ea17c42dd4d809971adfd0330667df8","bytes":13140},{"path":"voice/Fem1/gbl.aww.WAV","sha":"2250c38d75f4af88434bcb7c79dfcc783090f4ca","bytes":32086},{"path":"voice/Fem1/gbl.brag.WAV","sha":"132849e0457957ee407c65ed8c3537afefd7bf8c","bytes":25830},{"path":"voice/Fem1/gbl.bye.WAV","sha":"1fc9417c7a8a366fa7644a2e479e66b1625ee222","bytes":9968},{"path":"voice/Fem1/gbl.dunno.WAV","sha":"e5663ac73f49d282610af34eeb6bb9363bf391cc","bytes":14468},{"path":"voice/Fem1/gbl.goodgame.WAV","sha":"9d9c9e14a8d3b1b9ceb2e3888c31681c72171cfb","bytes":16584},{"path":"voice/Fem1/gbl.hi.WAV","sha":"55b5f0bf5a14712a71517b263ceb6a892bf16e4c","bytes":10002},{"path":"voice/Fem1/gbl.nice.WAV","sha":"680aa288a49292e1ae1f02f4e138c3d2ab69d8af","bytes":20566},{"path":"voice/Fem1/gbl.no.WAV","sha":"c0be07ed6ab3bf439d4e9672790723932d2e63a5","bytes":10268},{"path":"voice/Fem1/gbl.obnoxious.WAV","sha":"9578a8b8200966d0eeb05fc5f43c25d4168ebc03","bytes":25430},{"path":"voice/Fem1/gbl.oops.WAV","sha":"0cd18249f159f68931ff0d18d410dc7b1ac743d2","bytes":9934},{"path":"voice/Fem1/gbl.quiet.WAV","sha":"1e634b52d04127a9969eafdd1add4c6f75b181c5","bytes":12722},{"path":"voice/Fem1/gbl.rock.WAV","sha":"15ee3d393c48520b3ef79e9f1ad8f4de403f37ef","bytes":17126},{"path":"voice/Fem1/gbl.sarcasm.WAV","sha":"7fceb26f5525c4e464c37b7b05edeb31493a2d15","bytes":26642},{"path":"voice/Fem1/gbl.shazbot.WAV","sha":"fa8b86ea159c644c773b36dd903a655a625af2a3","bytes":15390},{"path":"voice/Fem1/gbl.shooting.WAV","sha":"9fe35f2d3d1d945b36e1d7f5ede0acde75dae4b2","bytes":20262},{"path":"voice/Fem1/gbl.thanks.WAV","sha":"7e229549505a70b9223e38a134c731552e51cffb","bytes":11974},{"path":"voice/Fem1/gbl.wait.WAV","sha":"f852dc599a191e8d89a0b564523d60d1ac719208","bytes":14016},{"path":"voice/Fem1/gbl.when.WAV","sha":"df73d90ae81aa4cf7de1cba734bf4cd7cc3c40fb","bytes":20966},{"path":"voice/Fem1/gbl.woohoo.WAV","sha":"691928c9dc0152044182156c66d545f62db70627","bytes":24066},{"path":"voice/Fem1/gbl.yes.WAV","sha":"f2338b2af9e4c18f9ec30b5433ee370a0fe10157","bytes":13568},{"path":"voice/Fem1/need.bombardier.WAV","sha":"9a09d643cc9897be1d5f22e9271a74b7a193ef43","bytes":17192},{"path":"voice/Fem1/need.cover.WAV","sha":"6837dbeed1bb6257cde3968ff50abcaa2e3c35c9","bytes":19918},{"path":"voice/Fem1/need.driver.WAV","sha":"3e69dd73a0849c07a0e58ed4941c2b731ae53040","bytes":21856},{"path":"voice/Fem1/need.escort.WAV","sha":"be30f8685137dd3cfd05a183726a2826da172ebf","bytes":18836},{"path":"voice/Fem1/need.flyer.WAV","sha":"c595d7043f3bd8af174cfc7c9bd43d0d22c996b3","bytes":24032},{"path":"voice/Fem1/need.gunship.WAV","sha":"9120de80e6573b62a825ea74ffadd4c02bf3e622","bytes":26208},{"path":"voice/Fem1/need.hold.WAV","sha":"95d1e202ba4629d12dc6df1c5eb1d8343ccdeda7","bytes":26454},{"path":"voice/Fem1/need.ride.WAV","sha":"571addba21da12dedcf8b07d9d24dee030dbfe37","bytes":14726},{"path":"voice/Fem1/need.support.WAV","sha":"f349ab07c6720d014848434eddd053c8d774126c","bytes":20402},{"path":"voice/Fem1/need.tailgunner.WAV","sha":"6be887e0975cc17ee17e68e18402a7b86784c192","bytes":20280},{"path":"voice/Fem1/need.where.WAV","sha":"2122ea51d8d8da7c82c651e55b436d6567e8aaf1","bytes":12366},{"path":"voice/Fem1/rep.base.WAV","sha":"a4e96b1f5f2c9e3d47a201ece4cba296d8ee6de5","bytes":18950},{"path":"voice/Fem1/rep.generator.WAV","sha":"3f577a9eaa637420d71ced793c4177b5a26a1da6","bytes":18114},{"path":"voice/Fem1/rep.me.WAV","sha":"53a4bc8530f7f669ab407b22366661d7cce0f198","bytes":15582},{"path":"voice/Fem1/rep.sensors.WAV","sha":"02b3a3d036e10117856e6c0d90579d61c704dd4c","bytes":21724},{"path":"voice/Fem1/rep.turrets.WAV","sha":"f3559df49f2294493fca7f1425fbede50830c721","bytes":20816},{"path":"voice/Fem1/rep.vehicle.WAV","sha":"7f0b26ec636882ba0aec30fa648edb6e87a8ad7c","bytes":23776},{"path":"voice/Fem1/slf.att.attack.WAV","sha":"335f45c623574590da9dd95f9dcab351f27dd4bd","bytes":14760},{"path":"voice/Fem1/slf.att.base.WAV","sha":"7ca6e7655e24e238bdf6dc9d4eb8d1de09608874","bytes":22072},{"path":"voice/Fem1/slf.att.flag.WAV","sha":"5518ebfe64c615818b30d1e1865889d64cc70526","bytes":20786},{"path":"voice/Fem1/slf.att.generator.WAV","sha":"dbdcadc5fe85cfd5c15622be3ac8b30bf0769f6e","bytes":24942},{"path":"voice/Fem1/slf.att.sensors.WAV","sha":"2a8f19e1520b9fbe587a343161527a8183fa3b02","bytes":21500},{"path":"voice/Fem1/slf.att.turrets.WAV","sha":"5bdaa77f1e1c8f04c672d62b6de4a9222acb046e","bytes":25174},{"path":"voice/Fem1/slf.att.vehicle.WAV","sha":"ca3037d4c3650646de1a6d758375ad2aba698113","bytes":27800},{"path":"voice/Fem1/slf.def.base.WAV","sha":"00c55fd699f62f35951888902c5f242f281f72a7","bytes":21402},{"path":"voice/Fem1/slf.def.defend.WAV","sha":"7e724b39402732f86ef28ab81cee33c40c3083e3","bytes":15254},{"path":"voice/Fem1/slf.def.flag.WAV","sha":"7bf30f333a2965e958b1da8af9d2f74a39336583","bytes":20904},{"path":"voice/Fem1/slf.def.generator.WAV","sha":"efdcc622b79732a8eb28e58151adf71677e1eeb9","bytes":21082},{"path":"voice/Fem1/slf.def.nexus.WAV","sha":"167c1171c53b0937979b1b1ce53f7d72e3ef4c48","bytes":20548},{"path":"voice/Fem1/slf.def.sensors.WAV","sha":"a7bb29b2116a375dbf02f2cf2ac427edfa61e012","bytes":25136},{"path":"voice/Fem1/slf.def.turrets.WAV","sha":"0a74bf047a36fa2a01bb6585ef5a73ebf2cb19fa","bytes":23998},{"path":"voice/Fem1/slf.def.vehicle.WAV","sha":"f72560e6187f5d7e1e347c7c01c0685440a58d9e","bytes":23700},{"path":"voice/Fem1/slf.rep.base.WAV","sha":"de23fd014b7680ef3821e83a0c26ba4f71163314","bytes":19656},{"path":"voice/Fem1/slf.rep.equipment.WAV","sha":"244a32a981384d66bd0fe36f035c761f161f1b66","bytes":20684},{"path":"voice/Fem1/slf.rep.generator.WAV","sha":"3778ec44d5131f8cb5ad424d0eb2a183abd60420","bytes":19052},{"path":"voice/Fem1/slf.rep.repairing.WAV","sha":"fa7089550fa91cdb0696bb67f5b8dc620adf6d0f","bytes":15510},{"path":"voice/Fem1/slf.rep.sensors.WAV","sha":"d6a4239431d7951b78dc9fc5ceffe1267ac5814d","bytes":20570},{"path":"voice/Fem1/slf.rep.turrets.WAV","sha":"a6fda238a7335fe279668c0a466298a562746a52","bytes":18596},{"path":"voice/Fem1/slf.rep.vehicle.WAV","sha":"1a7e2b771078ddce3b8998609db3ed27c8024d73","bytes":22926},{"path":"voice/Fem1/slf.tsk.cover.WAV","sha":"067365b4b43c03c9e683cab279cb716c2acc0b03","bytes":13662},{"path":"voice/Fem1/slf.tsk.defense.WAV","sha":"953ec9acdb4be00ad404ba06e1cc22b576b8d923","bytes":19964},{"path":"voice/Fem1/slf.tsk.generic.WAV","sha":"56c929637696ea4f2abaa3bebe979dd9902f392c","bytes":11828},{"path":"voice/Fem1/slf.tsk.remotes.WAV","sha":"58804cad9572d8d3d753530e177fd85a4d5b2b10","bytes":22844},{"path":"voice/Fem1/slf.tsk.sensors.WAV","sha":"8e61f659875b5af1f9442d98255fdc5b43587b1c","bytes":18058},{"path":"voice/Fem1/slf.tsk.turrets.WAV","sha":"805f122db7619e30fa76ee15b182c0a4015bbaf3","bytes":18472},{"path":"voice/Fem1/slf.tsk.vehicle.WAV","sha":"bc90a78a517cccd11e9aff0f8f45f313aa40fc68","bytes":17828},{"path":"voice/Fem1/tgt.acquired.WAV","sha":"173f5a5c9519d691fc39e610f80fc1952259be3d","bytes":14890},{"path":"voice/Fem1/tgt.base.WAV","sha":"76841f6124738eff6a9602c98ef4687991a2f53a","bytes":28664},{"path":"voice/Fem1/tgt.destroyed.WAV","sha":"b91ec9a3fb0edcdcb5204e70706fb5924993a946","bytes":20834},{"path":"voice/Fem1/tgt.flag.WAV","sha":"50662718022341146e41bab212f5d1a50bc3cf73","bytes":25022},{"path":"voice/Fem1/tgt.my.WAV","sha":"7a7f337a3c41c7667e6e1228568c6cb2c1c20a73","bytes":18614},{"path":"voice/Fem1/tgt.need.WAV","sha":"cc5ba4a20df3f6d4ab71ce1b69f46cb09336075e","bytes":18534},{"path":"voice/Fem1/tgt.sensors.WAV","sha":"c159da891a931bf2ce03ea785ee8317a382699a7","bytes":27504},{"path":"voice/Fem1/tgt.turret.WAV","sha":"c52cc6fe9313857727e1c8d7ef7ac61c8ffeb3c5","bytes":27138},{"path":"voice/Fem1/tgt.wait.WAV","sha":"7681d56e0c6089b85857d6081da3ca2994004759","bytes":25586},{"path":"voice/Fem1/vqk.anytime.WAV","sha":"df94daf4377cb995a3fed98532328189889c6978","bytes":9260},{"path":"voice/Fem1/vqk.base.WAV","sha":"784d11aa6bf93f09d2dce15c8e77376578c5d5e9","bytes":13226},{"path":"voice/Fem1/vqk.ceasefire.WAV","sha":"f2f84f65635ac2dfb13a6a4efa98667fee583979","bytes":9556},{"path":"voice/Fem1/vqk.dunno.WAV","sha":"bdb3980e2253a4d982799dcd4f2e8ed3d76ad48a","bytes":8136},{"path":"voice/Fem1/vqk.help.WAV","sha":"9467d2f5267cf2c1a1dae18c7886ed020f1d5522","bytes":6630},{"path":"voice/Fem1/vqk.move.WAV","sha":"83407a0bcf0c55d0c13b0323f6158aadaf625bb1","bytes":11894},{"path":"voice/Fem1/vqk.no.WAV","sha":"53f03cd318e1cbe077f2685d56c57b44751a520c","bytes":7074},{"path":"voice/Fem1/vqk.quiet.WAV","sha":"54802f09f09e030915c3f1d08d01c40fc1491014","bytes":13776},{"path":"voice/Fem1/vqk.sorry.WAV","sha":"83494dda05533970b44f94dcf2b5a818906975ce","bytes":7078},{"path":"voice/Fem1/vqk.thanks.WAV","sha":"ac1e944cb83108094c4db50fae7db8be766a768b","bytes":5466},{"path":"voice/Fem1/vqk.wait.WAV","sha":"0d863ce478526760b309118e7d203842382f231b","bytes":8154},{"path":"voice/Fem1/vqk.yes.WAV","sha":"ca978246b6a94047aedc5904c37eb53995f1cc0c","bytes":8684},{"path":"voice/Fem1/wrn.bomber.WAV","sha":"d2686dd4c0b92a39b1c4a6a7a9af036784030f0d","bytes":20198},{"path":"voice/Fem1/wrn.enemy.WAV","sha":"dfabb64715e1b5fe867f68892755e1bc5cbce987","bytes":19752},{"path":"voice/Fem1/wrn.vehicles.WAV","sha":"e529514cd18c61978f254f1d54b17b52630f51ce","bytes":19608},{"path":"voice/Fem1/wrn.watchit.WAV","sha":"ab2044c4def7865271df75c8ae724f1a8d3d8d2c","bytes":19262},{"path":"voice/Fem2/att.attack.wav","sha":"373ad62ed94581a001f251a120adb32069fab808","bytes":11650},{"path":"voice/Fem2/att.base.wav","sha":"65c3a4e41c7848eebec24af20f81be02196d9011","bytes":20726},{"path":"voice/Fem2/att.chase.wav","sha":"3e893cca8159bbd499ec80af29aec787c1912d6d","bytes":19680},{"path":"voice/Fem2/att.distract.wav","sha":"75d3f064a160e8333ebe95378e5079897a6abfed","bytes":21762},{"path":"voice/Fem2/att.flag.wav","sha":"baebebaa96143592d517fb4839f7b05ee7ae8b7e","bytes":20744},{"path":"voice/Fem2/att.generator.wav","sha":"17a0a7c1a78bb36d30ad12345066e537198a31f4","bytes":22210},{"path":"voice/Fem2/att.objective.wav","sha":"41c0fa3de4a58ceed3b6cd79f89df0bc48694919","bytes":19602},{"path":"voice/Fem2/att.reinforcements.wav","sha":"56dfc19376b56ef46b726a12256e4157387c430b","bytes":22252},{"path":"voice/Fem2/att.sensors.wav","sha":"f459da505c1365cf5939618673d6e468cbcb89d6","bytes":23546},{"path":"voice/Fem2/att.turrets.wav","sha":"45c96e92cbe0d1f0a535728b8bec12c28672a269","bytes":22534},{"path":"voice/Fem2/att.vehicle.wav","sha":"150618c320dd6b78afd6548a87496ddbcf867061","bytes":21658},{"path":"voice/Fem2/att.wait.wav","sha":"605acf13b3eb5b0f0d9ecadab7d0dedc0803ee1f","bytes":26676},{"path":"voice/Fem2/avo.deathcry_01.wav","sha":"9a85d2ecb021d01255d03b3bbbece41a76b3d798","bytes":7758},{"path":"voice/Fem2/avo.deathcry_02.wav","sha":"b5324a43eeb1a69a1dfc96f1f2a4da169fe851ea","bytes":10742},{"path":"voice/Fem2/avo.grunt.wav","sha":"0c7862532cfb229955d9eacbbd7f6366c3ba4bb3","bytes":4702},{"path":"voice/Fem2/avo.pain.wav","sha":"5cedf805852ac98d0da8aae18ede5c846ad558c3","bytes":5574},{"path":"voice/Fem2/bas.clear.wav","sha":"b3ea3d7c03ad4960f163b678f64e9320332397d6","bytes":16658},{"path":"voice/Fem2/bas.enemy.wav","sha":"b1494d33d4b7273afb0cd4cdbe1993db11bd44d1","bytes":23202},{"path":"voice/Fem2/bas.retake.wav","sha":"e3b2977f39e57c1ca9c6106a239cb5822b92a6a4","bytes":20098},{"path":"voice/Fem2/bas.secure.wav","sha":"dc19e76dd601b76286896354c02af4901bb2fd63","bytes":18658},{"path":"voice/Fem2/bas.taken.wav","sha":"7f5a1241623032bbdb7439ef594c017842102236","bytes":20226},{"path":"voice/Fem2/cmd.acknowledge.wav","sha":"15826f4f9e164ee5c213b3071e2d0286bbd6d0ae","bytes":16510},{"path":"voice/Fem2/cmd.bot.WAV","sha":"5f3f9769a0a3325f1a32de3d62a2166990bc3658","bytes":21324},{"path":"voice/Fem2/cmd.completed.wav","sha":"fa57cbf637aa393e6b5eea604a2a02e9e0462cd0","bytes":15634},{"path":"voice/Fem2/cmd.decline.wav","sha":"4e43284312bc47f9c273232f17c73cee57ed5e56","bytes":18744},{"path":"voice/Fem2/def.base.wav","sha":"28b9d5d053852e392dc017bd8700428ac0a3e2b4","bytes":18594},{"path":"voice/Fem2/def.carrier.wav","sha":"531ec146c252cfcc7ebe63ed25c8d26070909c50","bytes":24118},{"path":"voice/Fem2/def.entrances.wav","sha":"0944c149043e581eb44893e86ca5c8388997d5dc","bytes":23088},{"path":"voice/Fem2/def.flag.wav","sha":"5fd4fd12bee926aa17329ecca6e74946b5ccccf6","bytes":21814},{"path":"voice/Fem2/def.generator.wav","sha":"4ea3cfe6225c6d80f1cd35a691039a26b55ff258","bytes":18764},{"path":"voice/Fem2/def.me.wav","sha":"4ab8c3fe1d1897bec5801e06a37caf3dd97ea098","bytes":15988},{"path":"voice/Fem2/def.nexus.wav","sha":"ba9d827f2cda7c2b36236179a086d0922bfbed25","bytes":21476},{"path":"voice/Fem2/def.objective.wav","sha":"d2f01914dfabc51d836ef42ea6811995ee3e6acd","bytes":20790},{"path":"voice/Fem2/def.reinforce.wav","sha":"2922e594c0289a314d39389a400cc1c60a893c03","bytes":24096},{"path":"voice/Fem2/def.sensors.wav","sha":"6ee2b379da2ef12eb5bb47a01c7c69f811dc1acb","bytes":22296},{"path":"voice/Fem2/def.turrets.wav","sha":"197fe8b653afd571f6368c226c2c2d1e57b9c2ca","bytes":18468},{"path":"voice/Fem2/def.vehicle.wav","sha":"756fc8ef9414f042b736cd42c089ef54f1b7d30e","bytes":18240},{"path":"voice/Fem2/ene.base.wav","sha":"203a51874738c611cf1d57056bcb603d45a72151","bytes":24112},{"path":"voice/Fem2/ene.disarray.wav","sha":"afa154dc67d4c025e7943f95b3f1eef7635edc93","bytes":28674},{"path":"voice/Fem2/ene.generator.wav","sha":"8c870d2f3c0cb48428c62fdd8ad6c56628513f1e","bytes":24430},{"path":"voice/Fem2/ene.remotes.wav","sha":"81bef5f0800ec0632969f5275fc839a241a2e8fc","bytes":28080},{"path":"voice/Fem2/ene.sensors.wav","sha":"e81e325a292e3b59bf517836b1718c5a6e40ab68","bytes":24614},{"path":"voice/Fem2/ene.turrets.wav","sha":"a8748189b299d6854d6073fa9d3bf918e681b83b","bytes":21492},{"path":"voice/Fem2/ene.vehicle.wav","sha":"4e178b16a3e95f010d89939d429b29dbcd35d47b","bytes":28430},{"path":"voice/Fem2/flg.flag.WAV","sha":"212eeb63d5f507ce04ba2c20075a3c4b911ab022","bytes":24400},{"path":"voice/Fem2/flg.give.wav","sha":"8f8991fd9c6cc325876711c2f64e4da412721201","bytes":14712},{"path":"voice/Fem2/flg.huntergive.wav","sha":"56ebdd9cf16269338db3de9442a185e89b23bb97","bytes":22880},{"path":"voice/Fem2/flg.huntertake.wav","sha":"d4849fc78f452a0e80672a5777f0370758482dd8","bytes":16788},{"path":"voice/Fem2/flg.retrieve.wav","sha":"2a6b7e02b8c1d92ad9e8bf6c3a8183e5f16e46a4","bytes":23272},{"path":"voice/Fem2/flg.secure.wav","sha":"82508c6ed2a50a9ea32983e69adbb66c3f66799a","bytes":23818},{"path":"voice/Fem2/flg.take.wav","sha":"87bb3804960cd6736e28711f5427124670594f1b","bytes":20006},{"path":"voice/Fem2/gbl.anytime.wav","sha":"2515e74a5847400ce2f8f3ce9bef9a161a2e6423","bytes":13318},{"path":"voice/Fem2/gbl.awesome.wav","sha":"9f5f491b76573152d3db3e7ff03d46a6f1e3e2ed","bytes":20376},{"path":"voice/Fem2/gbl.aww.wav","sha":"9497eeb960bd477d01a830c8ddf97f26fa02fb45","bytes":29806},{"path":"voice/Fem2/gbl.brag.wav","sha":"89ede2295ba0ba15760bdb00b843aab67d2e01ea","bytes":29840},{"path":"voice/Fem2/gbl.bye.wav","sha":"bbaff571670801bde8b95b52dd854317013bc871","bytes":15486},{"path":"voice/Fem2/gbl.dunno.wav","sha":"582cb9a9fa2f299a69f269c39956838880be7fc7","bytes":21166},{"path":"voice/Fem2/gbl.goodgame.wav","sha":"a0f4ae50a0fe707caf25d2f661e6ff169dc885fe","bytes":13918},{"path":"voice/Fem2/gbl.hi.wav","sha":"e86694b8c72571df6a40c686f0e0ba615328e5b5","bytes":14372},{"path":"voice/Fem2/gbl.nice.wav","sha":"700aeb5c0a24b0cd7f7b5223a2dc9c74df951012","bytes":16216},{"path":"voice/Fem2/gbl.no.wav","sha":"e50f8b44ef7c50788820b79aba966be7ba7fcaf2","bytes":12984},{"path":"voice/Fem2/gbl.obnoxious.wav","sha":"4aea42a788d41c74a337806547b4f6ccf30c3371","bytes":16930},{"path":"voice/Fem2/gbl.oops.wav","sha":"dade2e01b0867e732c659f314a0286b0a323acab","bytes":12306},{"path":"voice/Fem2/gbl.quiet.WAV","sha":"4ed8d58f4990fb9ed457b42fb107b984f3621420","bytes":20892},{"path":"voice/Fem2/gbl.rock.WAV","sha":"6d94b79b13081604bc727c47776415855fd2c767","bytes":23892},{"path":"voice/Fem2/gbl.sarcasm.wav","sha":"0566084ad31a85c8393d3e05005eb98b70bcb427","bytes":32588},{"path":"voice/Fem2/gbl.shazbot.wav","sha":"afa66b0d05813d68548f95f034b144c20c0064db","bytes":14450},{"path":"voice/Fem2/gbl.shooting.WAV","sha":"130cce3beda190a02e66f336ffb3e01ab7d6e1a8","bytes":16444},{"path":"voice/Fem2/gbl.thanks.wav","sha":"4e4392d31ec18fc9c949a6b0bfe15f31c49b3235","bytes":16884},{"path":"voice/Fem2/gbl.wait.wav","sha":"9ae329ebfb423fc4bb8929568d003a141ae6901b","bytes":13098},{"path":"voice/Fem2/gbl.when.wav","sha":"0c9c90396651cb2690655f73572a996b80d225df","bytes":24028},{"path":"voice/Fem2/gbl.woohoo.wav","sha":"9f7545654b662e6ca14da33062b886c2defd48c5","bytes":18814},{"path":"voice/Fem2/gbl.yes.wav","sha":"a2a91aa66f5272c604e898e08149c3b4b5176bee","bytes":12214},{"path":"voice/Fem2/need.bombardier.wav","sha":"7c3519a336a176c18c15712d895f8bc1863019a3","bytes":17512},{"path":"voice/Fem2/need.cover.WAV","sha":"1e4e671eebc642f6d68843a6acc12d0926f6c407","bytes":18654},{"path":"voice/Fem2/need.driver.wav","sha":"1d66330b46f5b19f6920209088508e56af611410","bytes":26396},{"path":"voice/Fem2/need.escort.wav","sha":"02e76dae6322df6179b0ac025f27ebd68f113fc3","bytes":23398},{"path":"voice/Fem2/need.flyer.WAV","sha":"2134feadba6144c064f60772fe704884f0e94cb0","bytes":27188},{"path":"voice/Fem2/need.gunship.WAV","sha":"649b364bf93d7e59e880396659f688691a7e13da","bytes":23848},{"path":"voice/Fem2/need.hold.wav","sha":"25e79a1b42cf9c17b1d12e5fc2c527c29567e643","bytes":24566},{"path":"voice/Fem2/need.ride.WAV","sha":"6d6259568d284057ec229e92c86396af6d03ba48","bytes":19062},{"path":"voice/Fem2/need.support.wav","sha":"97dc15407e7bbb2de1a8775d01c4984be621e630","bytes":18508},{"path":"voice/Fem2/need.tailgunner.wav","sha":"f11b05657772c8d822aefaab3eb9ef2ba087a442","bytes":19250},{"path":"voice/Fem2/need.where.wav","sha":"85dde1e5baabcb526c7a78deb6a09b0955f5d632","bytes":11220},{"path":"voice/Fem2/rep.base.wav","sha":"36bd1363c9e1a6b53f1a3c9d3f75f6dcb2f7b43e","bytes":18258},{"path":"voice/Fem2/rep.generator.wav","sha":"0cf77fdd1c0e60b2e8b53453831a13e60a7715b9","bytes":19474},{"path":"voice/Fem2/rep.me.wav","sha":"63c00654955ebf6cf758f85b0d54907b4f181305","bytes":16604},{"path":"voice/Fem2/rep.sensors.wav","sha":"9a578c196b98e33294bad1c718caefa61c91dbfe","bytes":19330},{"path":"voice/Fem2/rep.turrets.wav","sha":"9131b0e5705b5dbc1fe2f8f81073c37d2925205c","bytes":18980},{"path":"voice/Fem2/rep.vehicle.wav","sha":"758dac831802bfda89cc26cbbe9865ddf101879f","bytes":24112},{"path":"voice/Fem2/slf.att.attack.wav","sha":"60c38488cec956cb7227067eca643c534b64ab78","bytes":16160},{"path":"voice/Fem2/slf.att.base.wav","sha":"d9c3c765a74e0736a7d324bf09d4320fadc1c25c","bytes":23790},{"path":"voice/Fem2/slf.att.flag.wav","sha":"b2db83312613c524cbafd6c725da9de29d8065f3","bytes":24056},{"path":"voice/Fem2/slf.att.generator.wav","sha":"25a974a7b531ae14f4e61b646409410a2d8a9551","bytes":23980},{"path":"voice/Fem2/slf.att.sensors.wav","sha":"2ece3a5a123d290c5e73ac35bb953eafd0ade4e6","bytes":25284},{"path":"voice/Fem2/slf.att.turrets.wav","sha":"e73e500120520020e33c9d28c7bd6d9e8d204fd7","bytes":25190},{"path":"voice/Fem2/slf.att.vehicle.wav","sha":"0763d3e75fbfe25629e431bca5d5401c49428469","bytes":26554},{"path":"voice/Fem2/slf.def.base.wav","sha":"6dc7651a2b3adab9ead33b8a1b5eca78b7b42aaf","bytes":22372},{"path":"voice/Fem2/slf.def.defend.wav","sha":"dba4121ef04a17dfc0e04ed768d1fab5a1703d38","bytes":16828},{"path":"voice/Fem2/slf.def.flag.wav","sha":"0cd6038b40489169a8d8211fd0beafb2905ec205","bytes":22540},{"path":"voice/Fem2/slf.def.generator.wav","sha":"8d67f359545e1c1d53d590d36917470961c8beca","bytes":20420},{"path":"voice/Fem2/slf.def.nexus.wav","sha":"eb9185670184b183dd1cd1b0f9d7d2792106a086","bytes":20568},{"path":"voice/Fem2/slf.def.sensors.wav","sha":"c8e05adf98f1b2838b9198b878b08c43493f4fd0","bytes":22620},{"path":"voice/Fem2/slf.def.turrets.wav","sha":"93abcdd4b1ffe7f143847c55c94ea2975350508b","bytes":22938},{"path":"voice/Fem2/slf.def.vehicle.wav","sha":"626a5753359d019813efb28fd1b6d7836740ebe7","bytes":22052},{"path":"voice/Fem2/slf.rep.base.wav","sha":"012b3358dd7f3c54902b7c66aa89b2bdc1339b23","bytes":19000},{"path":"voice/Fem2/slf.rep.equipment.wav","sha":"a446985dc739ebc26c8c1b2274ecd29080fdb5eb","bytes":21272},{"path":"voice/Fem2/slf.rep.generator.wav","sha":"8d86650140b4ade3aaffa20c9965ee27228c9ce6","bytes":23676},{"path":"voice/Fem2/slf.rep.repairing.wav","sha":"6eb14f569ea8063714ff4be4114525ed069e995b","bytes":19168},{"path":"voice/Fem2/slf.rep.sensors.wav","sha":"11496dfca4b82329c5919eaff46a81f4f2ddcbe2","bytes":21686},{"path":"voice/Fem2/slf.rep.turrets.wav","sha":"8bedf70e774eb1678ce7ff1078a5284fee1b394a","bytes":21106},{"path":"voice/Fem2/slf.rep.vehicle.wav","sha":"4cf6a121ed771ef594460791467e5fb3f1a6733d","bytes":25742},{"path":"voice/Fem2/slf.tsk.cover.WAV","sha":"2f2d0c89948d1a99d2282d2d7ee78b9036f3f83a","bytes":15462},{"path":"voice/Fem2/slf.tsk.defense.wav","sha":"1e28f4d65f16aa057d12006967f66ef661b3e777","bytes":20532},{"path":"voice/Fem2/slf.tsk.generic.WAV","sha":"142bd709cc991138629a4b2113314c54146c4299","bytes":12146},{"path":"voice/Fem2/slf.tsk.remotes.wav","sha":"d50e40dfa6ff07f7ac5319a6ef942ce27b1c063f","bytes":23100},{"path":"voice/Fem2/slf.tsk.sensors.wav","sha":"bf556ff18f76399307986deab94b278e6997682b","bytes":19414},{"path":"voice/Fem2/slf.tsk.turrets.wav","sha":"0fb883c5247c913994a0e2db74607abaccc12ddf","bytes":19694},{"path":"voice/Fem2/slf.tsk.vehicle.WAV","sha":"854aef85c1c7b9cdc57ccbf57f928f974526722e","bytes":18474},{"path":"voice/Fem2/tgt.acquired.wav","sha":"8f478d93a21f10ea95013a55ea66684ec839efc0","bytes":16582},{"path":"voice/Fem2/tgt.base.wav","sha":"859ec840b1e0793f659b6fb64a1e7d460b97c65d","bytes":31686},{"path":"voice/Fem2/tgt.destroyed.wav","sha":"dfd5e8d954ed3df6fffba0cb61531409cd4dd606","bytes":19434},{"path":"voice/Fem2/tgt.flag.wav","sha":"28f1911aa72ce356dd9ec98173dba18d7d10b02c","bytes":29048},{"path":"voice/Fem2/tgt.my.WAV","sha":"3ba7ec1286c7de2aa06b6caeb460093666f8e199","bytes":21892},{"path":"voice/Fem2/tgt.need.wav","sha":"46200d4c03c43fd4826fc232736af0e9f543c4bf","bytes":19936},{"path":"voice/Fem2/tgt.sensors.wav","sha":"04e98b1c0f3d1f11d74f147d0ba0250afeb98ca2","bytes":31722},{"path":"voice/Fem2/tgt.turret.wav","sha":"e78ac118e290b75580f7d5c1800f5c2435df0eac","bytes":27836},{"path":"voice/Fem2/tgt.wait.wav","sha":"cc1128ab2138f4e6ae082f4a378b150b6f0c56fd","bytes":26382},{"path":"voice/Fem2/vqk.anytime.wav","sha":"5339d0f9e6687d226d3d74291129d63f7b45f87e","bytes":9460},{"path":"voice/Fem2/vqk.base.wav","sha":"f5dbc7f33ce806047b626c3ba531ca5f3c487ed3","bytes":12834},{"path":"voice/Fem2/vqk.ceasefire.wav","sha":"82ed70a0b12590d8a83e77ec5ab227c7f8b65d90","bytes":11106},{"path":"voice/Fem2/vqk.dunno.wav","sha":"05f0baa5fffd8ed69aad74869a9a96f7d79d10c6","bytes":11790},{"path":"voice/Fem2/vqk.help.wav","sha":"e44eb1a323b14c1b36bcb06617840414dfdf6327","bytes":7986},{"path":"voice/Fem2/vqk.move.wav","sha":"4d77697c4309b1316d4821e529992e324077c997","bytes":8720},{"path":"voice/Fem2/vqk.no.wav","sha":"24d91da4d6778b6a44c8640bda07babe64be045e","bytes":11926},{"path":"voice/Fem2/vqk.quiet.WAV","sha":"50ded9ffde98d90ecd766a6d8eba47c81c53b777","bytes":8360},{"path":"voice/Fem2/vqk.sorry.wav","sha":"e236887f539ca39969e78ae04b7ec61dd386b08f","bytes":12530},{"path":"voice/Fem2/vqk.thanks.wav","sha":"4dad53885dfdd195b7d97a28df501e67122fcf95","bytes":8954},{"path":"voice/Fem2/vqk.wait.wav","sha":"4c180cdc630ef63fcb78fe15de52d83b93bda9da","bytes":7802},{"path":"voice/Fem2/vqk.yes.wav","sha":"65670b4e1eca06910b52f15dbc0a7e87fc88545e","bytes":11866},{"path":"voice/Fem2/wrn.bomber.wav","sha":"fc3511068d90fca446d81deb1bc27e9249f9e48b","bytes":15922},{"path":"voice/Fem2/wrn.enemy.wav","sha":"0e228ffca5445859450f30d22f494548328053d6","bytes":18144},{"path":"voice/Fem2/wrn.vehicles.wav","sha":"d423180871fea7e992cc9a226350b5aef94f9824","bytes":18908},{"path":"voice/Fem2/wrn.watchit.wav","sha":"a7e0cd03a6b7281151d2db6b47786c552549d476","bytes":23138},{"path":"voice/Fem3/att.attack.WAV","sha":"f60bc43bac891ba5080115443dc050c595721713","bytes":13568},{"path":"voice/Fem3/att.base.WAV","sha":"f1778e86bbbebbbaee641b7fe5009ec124a45d9e","bytes":21634},{"path":"voice/Fem3/att.chase.WAV","sha":"5abdce60f163e87db453179e0f4b136f2f91f43d","bytes":18726},{"path":"voice/Fem3/att.distract.WAV","sha":"923fe46a73b2c898ffc8df3ba888d001cfd9a121","bytes":22212},{"path":"voice/Fem3/att.flag.WAV","sha":"18ac20b1f6bb63a3d68d418eb2f3aa3565a27430","bytes":20098},{"path":"voice/Fem3/att.generator.WAV","sha":"6644fa0c3e2d34de49720ac84cd5285a56fd1831","bytes":27348},{"path":"voice/Fem3/att.objective.WAV","sha":"87fe279f635cb442de69c3f204c95d9dda8ec255","bytes":22778},{"path":"voice/Fem3/att.reinforcements.WAV","sha":"f439e45b7ac1ab3c0df5049aab69fa8e914a2b90","bytes":20856},{"path":"voice/Fem3/att.sensors.WAV","sha":"fd78ec6bc747008cd2ab05485c4de04c0a93dd58","bytes":26208},{"path":"voice/Fem3/att.turrets.WAV","sha":"8677e34edfd906a0d9e36be52a770c8b9916d7eb","bytes":25904},{"path":"voice/Fem3/att.vehicle.WAV","sha":"62ca91d60456937aad0b4b1e352539797771158b","bytes":26538},{"path":"voice/Fem3/att.wait.WAV","sha":"58d5b74fe483cbd8b460d674be1f3998353b6100","bytes":28190},{"path":"voice/Fem3/avo.deathcry_01.WAV","sha":"70d4101e8c7df99a47b72994bd9b19699d2dee4f","bytes":11324},{"path":"voice/Fem3/avo.deathcry_02.WAV","sha":"d02f33ec0187e83d794a2a94c3c565f626342a5c","bytes":12084},{"path":"voice/Fem3/avo.grunt.WAV","sha":"c00dd9d726cd0537ce722800b8074b2c050bf06a","bytes":5054},{"path":"voice/Fem3/avo.pain.WAV","sha":"6bff2ab517c241bb28a6440a22401977009af46e","bytes":6730},{"path":"voice/Fem3/bas.clear.WAV","sha":"4f0999b489d72e6a49dcaa752d179bdc42d1fb36","bytes":18170},{"path":"voice/Fem3/bas.enemy.WAV","sha":"b1ea074cb4e0705e99b524003194af4a4898032c","bytes":24740},{"path":"voice/Fem3/bas.retake.WAV","sha":"a621e74c5c9b214e6ac18e2b004cbb39f4712da9","bytes":23078},{"path":"voice/Fem3/bas.secure.WAV","sha":"4a755d981dc3d598a8575965cbbd942da8819a13","bytes":19410},{"path":"voice/Fem3/bas.taken.WAV","sha":"c2613f87c23e4f4c42dc858533aa2d565b11cb5d","bytes":20758},{"path":"voice/Fem3/cmd.acknowledge.WAV","sha":"9d37767d9423c6c0af9bf6a575aa4a97b26aa9aa","bytes":21788},{"path":"voice/Fem3/cmd.bot.WAV","sha":"ca937eca927e0c46a8d53ab46a4385a688fee12f","bytes":16386},{"path":"voice/Fem3/cmd.completed.WAV","sha":"3c4bff147719f28db964e3e6071328e3a0e8d4cc","bytes":15490},{"path":"voice/Fem3/cmd.decline.WAV","sha":"e9a12f2dce20b90a52099dabadfbe691d66cc1a9","bytes":16700},{"path":"voice/Fem3/def.base.WAV","sha":"7eec570bc17ac12bf30d1de8bbb143afd52898aa","bytes":23390},{"path":"voice/Fem3/def.carrier.WAV","sha":"badb614b42874c5456e387202f5d12a8165d395c","bytes":23188},{"path":"voice/Fem3/def.entrances.WAV","sha":"85279f0dc52f936051ba0dacd2f17cc250669a67","bytes":22794},{"path":"voice/Fem3/def.flag.WAV","sha":"e91b2ce04610ae9f40fee68b12a21b1a336d55fb","bytes":19890},{"path":"voice/Fem3/def.generator.WAV","sha":"da36d495b9a1fe11c7a2a7ffd7c3d264eecea597","bytes":21640},{"path":"voice/Fem3/def.me.WAV","sha":"63fd921635f29333319ea330a26f7844d973cfa0","bytes":15684},{"path":"voice/Fem3/def.nexus.WAV","sha":"3f18dde0cc59c7e61347ba4071d859da521d835c","bytes":20704},{"path":"voice/Fem3/def.objective.WAV","sha":"06404daa4f09066577eb7ae927c49bcae863fd5a","bytes":22150},{"path":"voice/Fem3/def.reinforce.WAV","sha":"b95a01c9ec6619a27701dfda8adaa90e382e0251","bytes":24320},{"path":"voice/Fem3/def.sensors.WAV","sha":"6b25109a8222cd27d1d1ad119cf7bfaf3060d624","bytes":21638},{"path":"voice/Fem3/def.turrets.WAV","sha":"702b6bedbdc459f4a295aa030a1b94b3803467e4","bytes":18138},{"path":"voice/Fem3/def.vehicle.WAV","sha":"289133a11a5ed146bbe9ebc42872a076e2c786e3","bytes":17198},{"path":"voice/Fem3/ene.base.WAV","sha":"e54ac8b4a426d1182221865035f3f37f70fda793","bytes":24570},{"path":"voice/Fem3/ene.disarray.WAV","sha":"05b60a0a56a2bbd68f637093ed07429e6e653e16","bytes":33588},{"path":"voice/Fem3/ene.generator.WAV","sha":"28bde35afda474e97db46009b547b64ce3a1b550","bytes":26050},{"path":"voice/Fem3/ene.remotes.WAV","sha":"e335627ca5d0877b16f3595a7c5e1860e1ba0e99","bytes":30922},{"path":"voice/Fem3/ene.sensors.WAV","sha":"033877d0a9da2eb092c1483ff74f979cf5f0042b","bytes":26588},{"path":"voice/Fem3/ene.turrets.WAV","sha":"8a132c38b891f198596ea51c97ee7024cf1f29e1","bytes":26682},{"path":"voice/Fem3/ene.vehicle.WAV","sha":"190e657ed8dcf460ccdd9909d937d3d7506b1f46","bytes":31268},{"path":"voice/Fem3/flg.flag.WAV","sha":"6cee85f2442ca3482d47077a550b670f1eecb73e","bytes":23080},{"path":"voice/Fem3/flg.give.WAV","sha":"b712d589047fc3a55bfe4b40a86e75f7c585e4f8","bytes":18518},{"path":"voice/Fem3/flg.huntergive.WAV","sha":"9747ae0369063e59f89397ed5075ecd1198386b9","bytes":18342},{"path":"voice/Fem3/flg.huntertake.WAV","sha":"9e42a49e9de94c8c277e3be4cd4639dac751ff00","bytes":18520},{"path":"voice/Fem3/flg.retrieve.WAV","sha":"9dedc4dcd6dd525a221885ef0372e4b8d8179d76","bytes":19166},{"path":"voice/Fem3/flg.secure.WAV","sha":"d3acc8052a3fd89191564ecf739c9deb29744f69","bytes":19058},{"path":"voice/Fem3/flg.take.WAV","sha":"a2dd3551d0a9ba0d5ac2f90d3753e79656c3d535","bytes":18780},{"path":"voice/Fem3/gbl.anytime.WAV","sha":"4f29c1e05625098c482fe655e3da49251c2a9568","bytes":17136},{"path":"voice/Fem3/gbl.awesome.WAV","sha":"3398f867b27358697c43317981ed74fd9e403925","bytes":13058},{"path":"voice/Fem3/gbl.aww.WAV","sha":"416279dde4c41acf7126bba06d5f75dfcf21e8e1","bytes":24240},{"path":"voice/Fem3/gbl.brag.WAV","sha":"a914ba8482ce531ff8e2c998a43b1a4469f23b84","bytes":23376},{"path":"voice/Fem3/gbl.bye.WAV","sha":"66fb93b2ff68d1e96958f60b130e383a1f8e93f5","bytes":9200},{"path":"voice/Fem3/gbl.dunno.WAV","sha":"d5c7bba112bc3c2f7ddd5b3fce46d0d78fd2dd0b","bytes":12264},{"path":"voice/Fem3/gbl.goodgame.WAV","sha":"603efc810add90635ab31636503e21f7ed56aad7","bytes":16012},{"path":"voice/Fem3/gbl.hi.WAV","sha":"d923f2c0d36e4cf25e8b5e807ecc28a1329e097d","bytes":12318},{"path":"voice/Fem3/gbl.nice.WAV","sha":"31f89d6def28bd08583e49dd5326ff725732b068","bytes":23204},{"path":"voice/Fem3/gbl.no.WAV","sha":"d32eb5843ab8fa5712393e4ab27a0927d5faa3b5","bytes":10104},{"path":"voice/Fem3/gbl.obnoxious.WAV","sha":"d634b392cfd48faa08e907993dcf0deaf53b9fb2","bytes":15076},{"path":"voice/Fem3/gbl.oops.WAV","sha":"9303f9d936d927ee655123f19cc2b5cc9edbb331","bytes":9094},{"path":"voice/Fem3/gbl.quiet.WAV","sha":"d8bb533b04df0375b8f4b0c5f4de9402049c63c3","bytes":14020},{"path":"voice/Fem3/gbl.rock.WAV","sha":"3151c0a9b6497a5862d455ad5e77fffa27c37caf","bytes":21956},{"path":"voice/Fem3/gbl.sarcasm.WAV","sha":"b893a5f909bd7935086c775eb4ea5f236c23bb87","bytes":28918},{"path":"voice/Fem3/gbl.shazbot.WAV","sha":"0152de167d5c7c9feba2bcc11f0f3e3e52ff7701","bytes":15436},{"path":"voice/Fem3/gbl.shooting.WAV","sha":"a5d129fecccfea0ba84ddee9d96f034de71b2568","bytes":18038},{"path":"voice/Fem3/gbl.thanks.WAV","sha":"29ea9aeb602176bba382307e53706c83fb24fd9f","bytes":10972},{"path":"voice/Fem3/gbl.wait.WAV","sha":"b184b138e7be2ae6e2d2bd150a07827c5120e394","bytes":12076},{"path":"voice/Fem3/gbl.when.WAV","sha":"e638ef4f729b6dd901baa7b884c8cfbc15dfbf79","bytes":21446},{"path":"voice/Fem3/gbl.woohoo.WAV","sha":"2d4a277527dbafffcb1861f3c2eeca208e173b3c","bytes":15668},{"path":"voice/Fem3/gbl.yes.WAV","sha":"3344d9c7c57cdcfb5b734591897f1603f15b0bac","bytes":10048},{"path":"voice/Fem3/need.bombardier.WAV","sha":"71254e107ffcd9f062f4d4afc2859b4876bd30a9","bytes":17372},{"path":"voice/Fem3/need.cover.WAV","sha":"fcf0a3533f2fd2bafa60997a88ca4bcce3c40b13","bytes":17716},{"path":"voice/Fem3/need.driver.WAV","sha":"e2b87539d5c6a72d7360eb805ae45051f60acf87","bytes":20388},{"path":"voice/Fem3/need.escort.WAV","sha":"8de78d0e3d8218a6f8c9ab646aff700179728f04","bytes":17244},{"path":"voice/Fem3/need.flyer.WAV","sha":"13fd4b44605f224cd17568eb55b715fc7481eccc","bytes":21676},{"path":"voice/Fem3/need.gunship.WAV","sha":"5b9f8a5e0a1302ab25955f37ed794c622733bb9e","bytes":25862},{"path":"voice/Fem3/need.hold.WAV","sha":"11ff6397fdc0cfc1e3c640bba63752bf7590ac82","bytes":17672},{"path":"voice/Fem3/need.ride.WAV","sha":"e043a66fa2bb38e1d0900aec89c18216abae6b53","bytes":15778},{"path":"voice/Fem3/need.support.WAV","sha":"319e5e765b2005f7a92bf16e00d8a4644b87bb14","bytes":17376},{"path":"voice/Fem3/need.tailgunner.WAV","sha":"9f63745fa29aa20bc838488018e131510d7f80dc","bytes":16370},{"path":"voice/Fem3/need.where.WAV","sha":"99232b4b7135c9e846044af364f6ebb6682963d3","bytes":13718},{"path":"voice/Fem3/rep.base.WAV","sha":"cb69fbaa1f35cdd968694a325c6891a7e7098835","bytes":18828},{"path":"voice/Fem3/rep.generator.WAV","sha":"4a3fbdf4b9db8a880eeaab552d27fd571e61ce28","bytes":18292},{"path":"voice/Fem3/rep.me.WAV","sha":"315a6deb84cce74b318b1362dff88eb82091040a","bytes":14230},{"path":"voice/Fem3/rep.sensors.WAV","sha":"454cdf02c6c0cd933b5c811493ebf76d1af7444b","bytes":19926},{"path":"voice/Fem3/rep.turrets.WAV","sha":"1a4bcfb7e1a18c7563296e7a9ceb51c2b682314d","bytes":18596},{"path":"voice/Fem3/rep.vehicle.WAV","sha":"87c4a7169ea68c80601ed2b001af306ac1fac3be","bytes":20262},{"path":"voice/Fem3/slf.att.attack.WAV","sha":"1161f81fdd4f444032328e97ea7ed060f182c38f","bytes":13882},{"path":"voice/Fem3/slf.att.base.WAV","sha":"faf776049584be4f17b1d28d0c76a77248394a72","bytes":21230},{"path":"voice/Fem3/slf.att.flag.WAV","sha":"5f777df802dea254abcdac2ce40cacf64b460770","bytes":20344},{"path":"voice/Fem3/slf.att.generator.WAV","sha":"34fedd113b861988dd19020401bb5c2d41b46e3b","bytes":22450},{"path":"voice/Fem3/slf.att.sensors.WAV","sha":"2dabe97987c1bc7f025b09b89c3b2a5b3050b743","bytes":23178},{"path":"voice/Fem3/slf.att.turrets.WAV","sha":"d4db8b910632c6d3ff527af0fd52d674ee66b8da","bytes":22610},{"path":"voice/Fem3/slf.att.vehicle.WAV","sha":"ee33d9bf96427091b210de35a495553430371322","bytes":27800},{"path":"voice/Fem3/slf.def.base.WAV","sha":"72094fdac043cfed4102eaf29acf89a30a4f7ead","bytes":18970},{"path":"voice/Fem3/slf.def.defend.WAV","sha":"a9c48ba385adb0e4578baf28a843a8ac4184ac95","bytes":17446},{"path":"voice/Fem3/slf.def.flag.WAV","sha":"243745dcf1effa6d0692b594fa6c4b715cf2a942","bytes":20714},{"path":"voice/Fem3/slf.def.generator.WAV","sha":"7f6477a0a9756d84513df63cac4166ba87d7563a","bytes":20912},{"path":"voice/Fem3/slf.def.nexus.WAV","sha":"e07efaa78efa7b5700cec55fcf27c5a6b7e5f759","bytes":19836},{"path":"voice/Fem3/slf.def.sensors.WAV","sha":"b2248d1677cd71887d245e0f4a840727b34fc71f","bytes":22494},{"path":"voice/Fem3/slf.def.turrets.WAV","sha":"f4ea5cd3ecc66ade2c73bdec77e8ccfee206f2fc","bytes":21126},{"path":"voice/Fem3/slf.def.vehicle.WAV","sha":"0c0fe29dea3705a0e3f6a001753088dc74442dc5","bytes":24434},{"path":"voice/Fem3/slf.rep.base.WAV","sha":"ab1d239ed6d0681f9d8134c76883aae00f962709","bytes":20016},{"path":"voice/Fem3/slf.rep.equipment.WAV","sha":"7d0413349d542c7feb12167ef472a5221e380ac8","bytes":19704},{"path":"voice/Fem3/slf.rep.generator.WAV","sha":"c64a7117acd12a0ca03167e21136c7aa7ccacf9b","bytes":19750},{"path":"voice/Fem3/slf.rep.repairing.WAV","sha":"7db849da691f7e9f247f78c3124e8e201e751142","bytes":18456},{"path":"voice/Fem3/slf.rep.sensors.WAV","sha":"3354401bb0d77711b94946e63a63e39a031f7265","bytes":20408},{"path":"voice/Fem3/slf.rep.turrets.WAV","sha":"1651ea4e36c22b885cd4e595a512a640eda50b51","bytes":17526},{"path":"voice/Fem3/slf.rep.vehicle.WAV","sha":"21d51023e412c7f08c88b908db506cd09fd0b14c","bytes":23848},{"path":"voice/Fem3/slf.tsk.cover.WAV","sha":"66dbfb46d9248692279c67c28962ef1f00f3de76","bytes":13962},{"path":"voice/Fem3/slf.tsk.defense.WAV","sha":"e785c3f0882b17bda4041f8512b9e33291346694","bytes":20184},{"path":"voice/Fem3/slf.tsk.generic.WAV","sha":"ed4ee24d1dba0e16f2d15c84c17ae984c5f609c2","bytes":12504},{"path":"voice/Fem3/slf.tsk.remotes.WAV","sha":"30355bc0e2757c59c45c1e2e019e56d956fb17de","bytes":20060},{"path":"voice/Fem3/slf.tsk.sensors.WAV","sha":"44040efeb35d611633b768d29913e21bbe6ef52d","bytes":19320},{"path":"voice/Fem3/slf.tsk.turrets.WAV","sha":"3cd497c6b8a9fecaf85214627739b5f7c124976e","bytes":18262},{"path":"voice/Fem3/slf.tsk.vehicle.WAV","sha":"d5b79251cf081e568c3f009cdfcffc2fe034d07d","bytes":17328},{"path":"voice/Fem3/tgt.acquired.WAV","sha":"43fef3af95f86b442fca2acefddbc65ae45dcdf2","bytes":20342},{"path":"voice/Fem3/tgt.base.WAV","sha":"dd6bcd12a370391277b5d11aae1ae8760fc29598","bytes":28314},{"path":"voice/Fem3/tgt.destroyed.WAV","sha":"ca05f4954f0f1af893d98e3868ffb30a0e62d84e","bytes":23172},{"path":"voice/Fem3/tgt.flag.WAV","sha":"881772c66b503d4563acea54ee7490ada30e164f","bytes":23440},{"path":"voice/Fem3/tgt.my.WAV","sha":"8b427cda80a289954bc6f7e0de3ab840472c28f0","bytes":18622},{"path":"voice/Fem3/tgt.need.WAV","sha":"d58f122213fb803823a758b09768c3c3ef5c5fe4","bytes":17240},{"path":"voice/Fem3/tgt.sensors.WAV","sha":"41f45356b8800b76ddb37773a93f003163025eef","bytes":22802},{"path":"voice/Fem3/tgt.turret.WAV","sha":"15e8bca2c7ddfef2d36ecff3a96b2ab77ff63405","bytes":24806},{"path":"voice/Fem3/tgt.wait.WAV","sha":"80cec07da2bfa624ad150ff703764fee3354d1ff","bytes":24114},{"path":"voice/Fem3/vqk.anytime.WAV","sha":"83f71ec0af01e04a0575d9a5887752e3881ff980","bytes":4774},{"path":"voice/Fem3/vqk.base.WAV","sha":"daf8578856128d42d8a808c3245a1c49e3f64e46","bytes":11162},{"path":"voice/Fem3/vqk.ceasefire.WAV","sha":"14640c3efaeb4f32318dab3c0b42ce9391496bc9","bytes":9020},{"path":"voice/Fem3/vqk.dunno.WAV","sha":"faecb7072efdf23f37286cab09dea97d48e9cd94","bytes":6204},{"path":"voice/Fem3/vqk.help.WAV","sha":"e1d345208609d1efb495abcc7cedab724b819399","bytes":7322},{"path":"voice/Fem3/vqk.move.WAV","sha":"193fe31a0bc1aff883c09d0ff4bd30288104aa17","bytes":7288},{"path":"voice/Fem3/vqk.no.WAV","sha":"3aa8593fde53f7f7fc9054309b2c2dabfcafd48c","bytes":6218},{"path":"voice/Fem3/vqk.quiet.WAV","sha":"39680a4fcddea5a761b4f3b5ac01b3fb9a5d54eb","bytes":11324},{"path":"voice/Fem3/vqk.sorry.WAV","sha":"3b837f48beebf0462969318f045c22f5c4e240b4","bytes":5970},{"path":"voice/Fem3/vqk.thanks.WAV","sha":"50f303016949cc75d1197ee0acb379db006d5357","bytes":7168},{"path":"voice/Fem3/vqk.wait.WAV","sha":"4db9ffa90c6294ffe1cada1d7f34d54d9b68a412","bytes":6516},{"path":"voice/Fem3/vqk.yes.WAV","sha":"7678e6a693120095c980b91c8409297a259ab905","bytes":7014},{"path":"voice/Fem3/wrn.bomber.WAV","sha":"04be6df6904e88cef66bafcf51c0824061ba2cdb","bytes":17016},{"path":"voice/Fem3/wrn.enemy.WAV","sha":"83bf7965ff21f1bdbd58b18eaaf19133611ae0d5","bytes":18906},{"path":"voice/Fem3/wrn.vehicles.WAV","sha":"e542c56112b8efc98246216ec8eea3831fb1065b","bytes":16916},{"path":"voice/Fem3/wrn.watchit.WAV","sha":"f96b74a322b796f0ac04ff96bb9df1ac95e6b192","bytes":19078},{"path":"voice/Fem4/att.attack.WAV","sha":"a4e5acafad8355f60fcc917d6b0badf93d622214","bytes":12498},{"path":"voice/Fem4/att.base.WAV","sha":"739879e79ad3480b2554141bab106c10fb07645a","bytes":20818},{"path":"voice/Fem4/att.chase.WAV","sha":"8e6ed613c8829eefdc4764950ade1f85a36d074f","bytes":20318},{"path":"voice/Fem4/att.distract.WAV","sha":"be2c9480bcd318685ea2c2e71bd151c949d4d0c8","bytes":26276},{"path":"voice/Fem4/att.flag.WAV","sha":"1f5f79295f9dc2824dee2336f264111d4fd8c0f4","bytes":19948},{"path":"voice/Fem4/att.generator.WAV","sha":"ecd6942742b5f3d186778c7f2ed0b91970ec224d","bytes":25634},{"path":"voice/Fem4/att.objective.WAV","sha":"2382cc4272171b9b51978ac9702f7434ccf6657a","bytes":19524},{"path":"voice/Fem4/att.reinforcements.WAV","sha":"9cd44682f64275c834e3fc3621dde74b022cc591","bytes":23420},{"path":"voice/Fem4/att.sensors.WAV","sha":"1ef08eea571dd16f643f14e54e71c7d169d585f2","bytes":24196},{"path":"voice/Fem4/att.turrets.WAV","sha":"5d892b672d48731cb2ffc0715a14cf662b7e47cb","bytes":23022},{"path":"voice/Fem4/att.vehicle.WAV","sha":"a46dfa9dc4985842ad51fd29e01e11005ea54155","bytes":26170},{"path":"voice/Fem4/att.wait.WAV","sha":"a568c0b8204aea3b12f07d5607821751f25a5582","bytes":30702},{"path":"voice/Fem4/avo.deathcry_01.WAV","sha":"4d9a003dc96f4dead5b5683c88154f66b0670461","bytes":11324},{"path":"voice/Fem4/avo.deathcry_02.WAV","sha":"dda872e90ea6d6055be3dabbffdcfe106f93e3d1","bytes":10812},{"path":"voice/Fem4/avo.grunt.WAV","sha":"4b7190c30f0387dd2ea887a8eac119a3a76abc1e","bytes":5814},{"path":"voice/Fem4/avo.pain.WAV","sha":"fd78d6e24daef79de65ace5308496f87b39b2980","bytes":4646},{"path":"voice/Fem4/bas.clear.WAV","sha":"a14c6caee02ce5873cd968fc3d480dd9b408e9e2","bytes":21582},{"path":"voice/Fem4/bas.enemy.WAV","sha":"ec722bd7c7983b5629bad3b83964d9cfb82fec3f","bytes":24590},{"path":"voice/Fem4/bas.retake.WAV","sha":"cdce3b28e72c145844a73f02bae2520fe7dc75fa","bytes":23182},{"path":"voice/Fem4/bas.secure.WAV","sha":"f09495bbb007688cc10cca889e38ffbb496b4769","bytes":19012},{"path":"voice/Fem4/bas.taken.WAV","sha":"129abe2734c76a1a6fdf4c9108a4ca092ffb248e","bytes":20436},{"path":"voice/Fem4/cmd.acknowledge.WAV","sha":"53e5ba661d17ff3f3a88a9032ac0ff39b8801c41","bytes":19886},{"path":"voice/Fem4/cmd.bot.WAV","sha":"31dcd592104491d563a12d7cfa7df821620e714c","bytes":19670},{"path":"voice/Fem4/cmd.completed.WAV","sha":"8e7ca7705d495a1a3c9e1538e23d65392a243146","bytes":19276},{"path":"voice/Fem4/cmd.decline.WAV","sha":"b4de46df3775dbebf282a55bc0fff179e6851743","bytes":19366},{"path":"voice/Fem4/def.base.WAV","sha":"372cccb975bcbfb2ba3c1423e54dd5d3a27481e9","bytes":21758},{"path":"voice/Fem4/def.carrier.WAV","sha":"0d3b5d38be5d27c5fd7746f973d7430b2610b1b0","bytes":22750},{"path":"voice/Fem4/def.entrances.WAV","sha":"70c16d977fcd5f2c70443eb89692ef79b875e399","bytes":24044},{"path":"voice/Fem4/def.flag.WAV","sha":"5874d12f80b2a3c1cffceee8b0e802b26121019b","bytes":21582},{"path":"voice/Fem4/def.generator.WAV","sha":"fa020652426c4aa3fc2e6ae34dd878977596dc89","bytes":22474},{"path":"voice/Fem4/def.me.WAV","sha":"089511c301d4fe2c4897d754ba56cc22ae1f50d5","bytes":12462},{"path":"voice/Fem4/def.nexus.WAV","sha":"538cc31ee84dd1b36daec2906b33789b3f1aad14","bytes":22022},{"path":"voice/Fem4/def.objective.WAV","sha":"e01a1df6070e0eda5f8c0844daa734172befb889","bytes":21860},{"path":"voice/Fem4/def.reinforce.WAV","sha":"210afb851efd0aa27fb3d5690a8ad3c0081ce674","bytes":27448},{"path":"voice/Fem4/def.sensors.WAV","sha":"b558bb30d13292f1c6c1148022e19302f745cb46","bytes":24656},{"path":"voice/Fem4/def.turrets.WAV","sha":"a9d888022dd15f7d262e51cdd850a422e802fa3e","bytes":22632},{"path":"voice/Fem4/def.vehicle.WAV","sha":"c7fac25d2078dc9ca27e7dce140f47499b36a826","bytes":19138},{"path":"voice/Fem4/ene.base.WAV","sha":"5ef58bb295d15b54d1bf32203ca16bf4d2e12349","bytes":22174},{"path":"voice/Fem4/ene.disarray.WAV","sha":"dc1e71fbe541d1da406b6ec25e4c99c0fd0fdb54","bytes":30448},{"path":"voice/Fem4/ene.generator.WAV","sha":"8baa478fd0fb4fd5fd05d3a947c5662b7ec9672c","bytes":29014},{"path":"voice/Fem4/ene.remotes.WAV","sha":"07a00a1333cf61a80443e5c97cedec8507c171c6","bytes":31812},{"path":"voice/Fem4/ene.sensors.WAV","sha":"769ae23a1195cb176a5b7fddfd886fa29a304fc4","bytes":27336},{"path":"voice/Fem4/ene.turrets.WAV","sha":"c4c1665f644df0e1191652a2ae0a9e3058cc13e0","bytes":26300},{"path":"voice/Fem4/ene.vehicle.WAV","sha":"fd63c079eed4ff6e8929f1b972a551bee2e6780d","bytes":32466},{"path":"voice/Fem4/flg.flag.WAV","sha":"7650d35298adea8c70b7bd5cdf0e0c4f77544692","bytes":23300},{"path":"voice/Fem4/flg.give.WAV","sha":"63e2d0e8c5059e40f90286b028e5194ab7dc2bcc","bytes":21038},{"path":"voice/Fem4/flg.huntergive.WAV","sha":"bcef33ecfb29b90238e81c08b91763526821d2cf","bytes":21262},{"path":"voice/Fem4/flg.huntertake.WAV","sha":"b0edae918090828f68ac25c15444695dffd713ac","bytes":20360},{"path":"voice/Fem4/flg.retrieve.WAV","sha":"66b5f7c045e881ef0daa112903d6e5d4c1e8812f","bytes":23152},{"path":"voice/Fem4/flg.secure.WAV","sha":"f48cdd499fa51b7bdcf14ca36214e5146161d7f8","bytes":23178},{"path":"voice/Fem4/flg.take.WAV","sha":"b02493012b62eb5b7d04d460785bc395990398b1","bytes":17998},{"path":"voice/Fem4/gbl.anytime.WAV","sha":"02243a949ad57dc012621a60405e9e317252c3cd","bytes":13416},{"path":"voice/Fem4/gbl.awesome.WAV","sha":"887e34f3fea206262b9d7c6a37cc1a7a852ea47c","bytes":16924},{"path":"voice/Fem4/gbl.aww.WAV","sha":"fdc9e44a7f6f461f6be3f1e4c93859dec8d6282d","bytes":43068},{"path":"voice/Fem4/gbl.brag.WAV","sha":"5cfde11547f4cb5f0758ba755a36c75f331b6f72","bytes":23428},{"path":"voice/Fem4/gbl.bye.WAV","sha":"ff39c05871a787b41687acd2aae822b36588ca25","bytes":9746},{"path":"voice/Fem4/gbl.dunno.WAV","sha":"19b5c6178ed24b90840b985d5375c4b9a452aefe","bytes":13920},{"path":"voice/Fem4/gbl.goodgame.WAV","sha":"18081ca4aa3f078b8599580958cbf121de5c6595","bytes":18068},{"path":"voice/Fem4/gbl.hi.WAV","sha":"1ac8f8f3328410bd11cf76e13a86b87dee337f9f","bytes":9848},{"path":"voice/Fem4/gbl.nice.WAV","sha":"69c989eb83d12b68d6eed5598b9c7965a696a64a","bytes":16036},{"path":"voice/Fem4/gbl.no.WAV","sha":"cd75657b95b1e4dd026b04d0527d4e5d95e8711f","bytes":9494},{"path":"voice/Fem4/gbl.obnoxious.WAV","sha":"dcdd03016ee98e625f908803ddf195a60b487c54","bytes":29004},{"path":"voice/Fem4/gbl.oops.WAV","sha":"cd0dd46ebcc7be4760e6e135ee6c1130585d64d1","bytes":9928},{"path":"voice/Fem4/gbl.quiet.WAV","sha":"1e67f38b28cd750bd846b1255b036fa03192a330","bytes":12332},{"path":"voice/Fem4/gbl.rock.WAV","sha":"fabfd93b255df451e9005ca01aa3b15b7f937b27","bytes":18128},{"path":"voice/Fem4/gbl.sarcasm.WAV","sha":"6680f0415419a41aca4f144cc352459c6b4b3ecb","bytes":23340},{"path":"voice/Fem4/gbl.shazbot.WAV","sha":"e8ca99b2148d52f22f64fdd18f4ebfb9b3e88e97","bytes":16474},{"path":"voice/Fem4/gbl.shooting.WAV","sha":"2867e19803f73eae4596599cfaf70a623e6d0d87","bytes":12550},{"path":"voice/Fem4/gbl.thanks.WAV","sha":"530f32518390a4821b804a1b872399f9478e66a5","bytes":11078},{"path":"voice/Fem4/gbl.wait.WAV","sha":"d9996b1f525f6c73236fed0b69941b47d02f274c","bytes":14192},{"path":"voice/Fem4/gbl.when.WAV","sha":"4d0a27dc193bdebe6cb7f38749de4086e36c60be","bytes":32388},{"path":"voice/Fem4/gbl.woohoo.WAV","sha":"076f0a2b2ec6e64c6c586c93646b094aa626f5d6","bytes":13444},{"path":"voice/Fem4/gbl.yes.WAV","sha":"bbf31df3ff8d34a0208f3957eb1a56811e754935","bytes":10364},{"path":"voice/Fem4/need.bombardier.WAV","sha":"a834482a7f8f978e1dfcc427af17b7a2ea2f6696","bytes":19024},{"path":"voice/Fem4/need.cover.WAV","sha":"a440a66c02f94f0165ba61a0ba06d68b880f4394","bytes":20248},{"path":"voice/Fem4/need.driver.WAV","sha":"cf32048828e534b0bf02d2005f6fef2d1c2e52d3","bytes":26520},{"path":"voice/Fem4/need.escort.WAV","sha":"0ef011ef078a6c5eed2a825affdf342f27c82a40","bytes":23218},{"path":"voice/Fem4/need.flyer.WAV","sha":"1c9eec8fa7dfbe081c6e8df9059cdd3835e4bab2","bytes":26012},{"path":"voice/Fem4/need.gunship.WAV","sha":"1163ee3dc2fd190c7d6413cb609ee530410bb718","bytes":26300},{"path":"voice/Fem4/need.hold.WAV","sha":"4040ef0d9893dc9ab485adb8b105c85e80c5201f","bytes":24034},{"path":"voice/Fem4/need.ride.WAV","sha":"75b33bff187acede2fcc0594bc5304111c95f953","bytes":16420},{"path":"voice/Fem4/need.support.WAV","sha":"242cde7f91580f28e0400973302b0085b17ef6f6","bytes":20796},{"path":"voice/Fem4/need.tailgunner.WAV","sha":"e2be213cf6a72f636ad4ff017d7dfdc7a201c9a0","bytes":15902},{"path":"voice/Fem4/need.where.WAV","sha":"696dee69f26f3c6b0cd7832f2cdeae7803c37c67","bytes":12716},{"path":"voice/Fem4/rep.base.WAV","sha":"7d149aa4fa46acff47bab31be86521785efecad3","bytes":20682},{"path":"voice/Fem4/rep.generator.WAV","sha":"8ff1d0454da08de89e95945a3291baae5fd96f89","bytes":22496},{"path":"voice/Fem4/rep.me.WAV","sha":"b29086814a6c3bf2df5af58501edc8abc4ad6d7f","bytes":15814},{"path":"voice/Fem4/rep.sensors.WAV","sha":"3e793dd57b9f16f2a03c14e515076b776874d157","bytes":22802},{"path":"voice/Fem4/rep.turrets.WAV","sha":"c0df06878377d1d18f8256867c812fdfa636a441","bytes":20672},{"path":"voice/Fem4/rep.vehicle.WAV","sha":"f741447ddd277b1cc46d35478dbbed83b1c9c14c","bytes":25984},{"path":"voice/Fem4/slf.att.attack.WAV","sha":"344ce88e31651155b5480c84b2ad4ec08df86651","bytes":17018},{"path":"voice/Fem4/slf.att.base.WAV","sha":"a45821731797f38a61234097bdcd6b6190071fcc","bytes":26572},{"path":"voice/Fem4/slf.att.flag.WAV","sha":"327448f5050389f2b899f4fc79cb67807dee75f2","bytes":24862},{"path":"voice/Fem4/slf.att.generator.WAV","sha":"2e1a1b529ab44b64e2dde2155c13a3f91f063aff","bytes":25346},{"path":"voice/Fem4/slf.att.sensors.WAV","sha":"c048a7b775346c850d3ad2c36a824a264ee7313c","bytes":25190},{"path":"voice/Fem4/slf.att.turrets.WAV","sha":"b74adedbf58ff3601a50fe8f2a5515e87a230836","bytes":22890},{"path":"voice/Fem4/slf.att.vehicle.WAV","sha":"e123a9af9c220577e40339e4d829fd324cf9d8d9","bytes":28266},{"path":"voice/Fem4/slf.def.base.WAV","sha":"889d5b8b60416c165fef78a8f632bf5ee81fd98e","bytes":24432},{"path":"voice/Fem4/slf.def.defend.WAV","sha":"a8c1e948695e60f0c682a16d4de8863abd31b3a6","bytes":18384},{"path":"voice/Fem4/slf.def.flag.WAV","sha":"fd65184f0009df1b0d69dc40b343e145f079fee9","bytes":23170},{"path":"voice/Fem4/slf.def.generator.WAV","sha":"b2ec890780940ddc20a08fdc70a1c0b72fde6bf2","bytes":26494},{"path":"voice/Fem4/slf.def.nexus.WAV","sha":"df63936d1211991d51f89a9929ca59ab21adadc3","bytes":21744},{"path":"voice/Fem4/slf.def.sensors.WAV","sha":"54c5eaa5da4b80d7ffb9e883c1b0269ee32777f8","bytes":25196},{"path":"voice/Fem4/slf.def.turrets.WAV","sha":"49b87b2bd90704b3b19c302bbde6552cae80ab78","bytes":24400},{"path":"voice/Fem4/slf.def.vehicle.WAV","sha":"d710078c4765dda6235e02efd0654298562e0ec2","bytes":28964},{"path":"voice/Fem4/slf.rep.base.WAV","sha":"3a5f89643e648d511091faa5f87b0a21f84b2573","bytes":21552},{"path":"voice/Fem4/slf.rep.equipment.WAV","sha":"76699162f319928e35cde6970ecce82dc50b9018","bytes":24808},{"path":"voice/Fem4/slf.rep.generator.WAV","sha":"edeb0bada9c5f816e06eff2108ceca6588b8fc8a","bytes":26622},{"path":"voice/Fem4/slf.rep.repairing.WAV","sha":"8ec60671b67388dbbc2c442d5b87fb6431b50f8b","bytes":19764},{"path":"voice/Fem4/slf.rep.sensors.WAV","sha":"606c2ba0b46fe40c837bc65bb51b97d460fa1499","bytes":24812},{"path":"voice/Fem4/slf.rep.turrets.WAV","sha":"96bf6a68fbf1d367727ce6ea600c5a62c3fd63ce","bytes":23746},{"path":"voice/Fem4/slf.rep.vehicle.WAV","sha":"7bb123b76f17d4c0281aa82176d69629c798ade3","bytes":26276},{"path":"voice/Fem4/slf.tsk.cover.WAV","sha":"06fd59d0153c8d34657501ad6adac75260c83c11","bytes":15622},{"path":"voice/Fem4/slf.tsk.defense.WAV","sha":"d4d75d4ff2ba8cb9f2845afc4c778f62e3e9d153","bytes":22416},{"path":"voice/Fem4/slf.tsk.generic.WAV","sha":"1a0e5922b3faf3eaaf193d1cddfd8da9e30cc270","bytes":13980},{"path":"voice/Fem4/slf.tsk.remotes.WAV","sha":"6f229523fee1a9642a21158a71b1ccec1ccd879d","bytes":25886},{"path":"voice/Fem4/slf.tsk.sensors.WAV","sha":"d499e7038eab864872f8b06a82eaeea6a16f0ede","bytes":22942},{"path":"voice/Fem4/slf.tsk.turrets.WAV","sha":"a05cae1158bcff276e0618459df548746e5e5785","bytes":22140},{"path":"voice/Fem4/slf.tsk.vehicle.WAV","sha":"59e99c7cd51f82315ddccc1e02816ec4fb876bb9","bytes":20628},{"path":"voice/Fem4/tgt.acquired.WAV","sha":"51c8223ae7af3cff5a91a57a5e3982ba5e23e437","bytes":19142},{"path":"voice/Fem4/tgt.base.WAV","sha":"852bbd12d401a541ca3a52c722dc0a774960f2c9","bytes":35868},{"path":"voice/Fem4/tgt.destroyed.WAV","sha":"f0f7d37dbe6398ff645331bbbfdb129b5dd6be91","bytes":22494},{"path":"voice/Fem4/tgt.flag.WAV","sha":"dd05a4f35d7f437ccd10d5478b4fa776d7a64139","bytes":30136},{"path":"voice/Fem4/tgt.my.WAV","sha":"e4ab488ab04cb5642a2a6673ce949fd7293f127d","bytes":23306},{"path":"voice/Fem4/tgt.need.WAV","sha":"1c0efd4a4bb81df7c9bdd3a3bd34b14e0c7f09c5","bytes":20292},{"path":"voice/Fem4/tgt.sensors.WAV","sha":"655bb59a5180488a581e2050785c539ec992344a","bytes":30190},{"path":"voice/Fem4/tgt.turret.WAV","sha":"330bf80bd1b244ae7697a34ba00ead1fa78809da","bytes":30514},{"path":"voice/Fem4/tgt.wait.WAV","sha":"839433293818717643eda1491f162546ef4a132e","bytes":28290},{"path":"voice/Fem4/vqk.anytime.WAV","sha":"6cab22478c99430bc38bcebbfd3d86defea7333a","bytes":13522},{"path":"voice/Fem4/vqk.base.WAV","sha":"b63b51ae0ea9d40908728936071b4d8d70b557e9","bytes":12100},{"path":"voice/Fem4/vqk.ceasefire.WAV","sha":"8e3950c541e52fe552bd16e868c7fd864ac7d9c9","bytes":11086},{"path":"voice/Fem4/vqk.dunno.WAV","sha":"3b949be0e2b9d6dd590553457e50711c40581547","bytes":5540},{"path":"voice/Fem4/vqk.help.WAV","sha":"cf852c0c8c4d432f7c83ba8fd69a165449df981e","bytes":6918},{"path":"voice/Fem4/vqk.move.WAV","sha":"a12034493b824b36b1bb90522ed09e0c130ae329","bytes":13036},{"path":"voice/Fem4/vqk.no.WAV","sha":"be63c3d5c48a76df8bbe904f08d50669baf83fd1","bytes":9088},{"path":"voice/Fem4/vqk.quiet.WAV","sha":"3484e9610e787759c549d05a185dd7b77adfc013","bytes":11338},{"path":"voice/Fem4/vqk.sorry.WAV","sha":"8b5a0a79179191cc3545ab75de1a70f818a69604","bytes":6310},{"path":"voice/Fem4/vqk.thanks.WAV","sha":"969c3260d36b2db67e2ccfe32c731550bf18049f","bytes":7280},{"path":"voice/Fem4/vqk.wait.WAV","sha":"c5dc9e7870fd644a820abe3cf4361d6f53d6ae57","bytes":15896},{"path":"voice/Fem4/vqk.yes.WAV","sha":"8b884428ec8c9f8e3b9b5329f883ab453eedcff3","bytes":14490},{"path":"voice/Fem4/wrn.bomber.WAV","sha":"c274eec434fb1f5ce44c46037d0986cd802b8221","bytes":20820},{"path":"voice/Fem4/wrn.enemy.WAV","sha":"eae4c7dc67329742572479d8aedcaeab834e7b82","bytes":23270},{"path":"voice/Fem4/wrn.vehicles.WAV","sha":"cabf1568020f1d83e5e8f2a480b98d02f8b2bcaf","bytes":22994},{"path":"voice/Fem4/wrn.watchit.WAV","sha":"e31b46752f2887eae120540bd3701f1c7fe7826e","bytes":22088},{"path":"voice/Fem5/att.attack.WAV","sha":"80ab812a1039036057b0c5a9210d9463b30f22d2","bytes":11858},{"path":"voice/Fem5/att.base.WAV","sha":"0f9e3462d34e491aa134e753913370912f8a888a","bytes":21110},{"path":"voice/Fem5/att.chase.WAV","sha":"a2fc9a22faad1a42e6c1be434cc61f27681b82d1","bytes":17118},{"path":"voice/Fem5/att.distract.WAV","sha":"a8da2981865c3be06548ae1ee96b9e9f2a800c05","bytes":22968},{"path":"voice/Fem5/att.flag.WAV","sha":"254472fa8bf1332a36dd316e71cf6a3533d74574","bytes":17464},{"path":"voice/Fem5/att.generator.WAV","sha":"19fd83e6e6dc98b5fc46fa80acbff993d2c7f658","bytes":22576},{"path":"voice/Fem5/att.objective.WAV","sha":"5a13ee43bfc593d048e40ac9776d07a7f2692fd5","bytes":17542},{"path":"voice/Fem5/att.reinforcements.WAV","sha":"72ee07906496b46a24ef0e251ff189f58a569028","bytes":19610},{"path":"voice/Fem5/att.sensors.WAV","sha":"84b81410c35f84a86f466e549ebcbf859077967d","bytes":20860},{"path":"voice/Fem5/att.turrets.WAV","sha":"31f88870c0a1a07237fa0e1013d4163325e5dc2a","bytes":19388},{"path":"voice/Fem5/att.vehicle.WAV","sha":"92fd19636be7e829073133c5d8359d179d833aa4","bytes":20488},{"path":"voice/Fem5/att.wait.WAV","sha":"c1da251be6c79c0d346082de47c4e07b76b61ff2","bytes":25490},{"path":"voice/Fem5/avo.deathcry_01.WAV","sha":"4cbfd1e63146dfbe542cef8e95f361e6140c16cf","bytes":13138},{"path":"voice/Fem5/avo.deathcry_02.WAV","sha":"0a58bb1fb63647bbdc6c42af5976a009d4f680f5","bytes":13344},{"path":"voice/Fem5/avo.grunt.WAV","sha":"19f1750963acdff4b842637068170ddbb8f281c9","bytes":4152},{"path":"voice/Fem5/avo.pain.WAV","sha":"0e9dbb2ea0bf5ac1ee4db3dc11363e56f774ec81","bytes":7710},{"path":"voice/Fem5/bas.clear.WAV","sha":"6c593f3f4e4ee9007726c269bd10b6925855b232","bytes":16878},{"path":"voice/Fem5/bas.enemy.WAV","sha":"2ae07ba30820a06ed7da302123e225d8416f319a","bytes":18502},{"path":"voice/Fem5/bas.retake.WAV","sha":"23f2e8e432e9dd46fd941c3b70900a069aabbd5f","bytes":17994},{"path":"voice/Fem5/bas.secure.WAV","sha":"b6583910227e97b7059e607d7052f9c9be9378a0","bytes":17404},{"path":"voice/Fem5/bas.taken.WAV","sha":"464798dfa7975dd72d7e942cc6bef8705e9f3552","bytes":18218},{"path":"voice/Fem5/cmd.acknowledge.WAV","sha":"731877233980c5f73454e4b8518aa62770faf751","bytes":16742},{"path":"voice/Fem5/cmd.bot.WAV","sha":"b77c3954cc599cdd962f2024ea6db9c7c2887934","bytes":10108},{"path":"voice/Fem5/cmd.completed.WAV","sha":"9fae37db2068866c32bc79433a6fb6f3e2568e24","bytes":15086},{"path":"voice/Fem5/cmd.decline.WAV","sha":"858b48a7ae7115fb24ef0d8316fbe1561849625f","bytes":18518},{"path":"voice/Fem5/def.base.WAV","sha":"de575b9995c7d138af7e69af4a4f4a15291f1b6e","bytes":17514},{"path":"voice/Fem5/def.carrier.WAV","sha":"f0b76815191d22a93a24afd6cb6c6759edd0be6e","bytes":19722},{"path":"voice/Fem5/def.entrances.WAV","sha":"9f64e2f684b839ab77ccfcf9f4f0000e6aa5cb1f","bytes":19770},{"path":"voice/Fem5/def.flag.WAV","sha":"b637ad5cf3147659c3eecb8229dd98798ac59bf9","bytes":17480},{"path":"voice/Fem5/def.generator.WAV","sha":"a7c58550106e7953d77c1029b1462db8f7cc507f","bytes":18398},{"path":"voice/Fem5/def.me.WAV","sha":"64be383944cd91fb9141238500c2ec3fc367dc23","bytes":11028},{"path":"voice/Fem5/def.nexus.WAV","sha":"81706052f0c39ce5cb28f41f8670a95cb7fda050","bytes":17402},{"path":"voice/Fem5/def.objective.WAV","sha":"c6aa800aece53fc6c9871741faa0ce677307c011","bytes":17460},{"path":"voice/Fem5/def.reinforce.WAV","sha":"14ee6ae4033cf2d19bfe79c25e5430c079ef1790","bytes":20148},{"path":"voice/Fem5/def.sensors.WAV","sha":"c86d902decea0b764087ec1f33c973744566a4cc","bytes":18392},{"path":"voice/Fem5/def.turrets.WAV","sha":"7194e3d06ae67b19bf5a5599f9f8b97a6b5bc4ca","bytes":17176},{"path":"voice/Fem5/def.vehicle.WAV","sha":"e70bd2e3d17235632149cc76c4bcceb58892d2e5","bytes":16558},{"path":"voice/Fem5/ene.base.WAV","sha":"8f4d7b7457024c8025e337fa1611c8cb888b2b3a","bytes":19568},{"path":"voice/Fem5/ene.disarray.WAV","sha":"7141564848b75c2aaf218ee0c07f5cd7abab760e","bytes":23704},{"path":"voice/Fem5/ene.generator.WAV","sha":"a6d8cb8f96019bbede41901a78d39bbe2067874d","bytes":22288},{"path":"voice/Fem5/ene.remotes.WAV","sha":"f7f37605ed99fdadc89b46ae4c575d13ecf91e0b","bytes":23316},{"path":"voice/Fem5/ene.sensors.WAV","sha":"3dccee4601244643e184fc29fe521f1de4bc19f5","bytes":20654},{"path":"voice/Fem5/ene.turrets.WAV","sha":"273ad20d5b6c923b2c783f5d6225978ffa3ac61a","bytes":20612},{"path":"voice/Fem5/ene.vehicle.WAV","sha":"efbe201e5a12bf26477147b2fbec3e8820e27ff4","bytes":25644},{"path":"voice/Fem5/flg.flag.WAV","sha":"6562c3a7d5b8e775e0b8c28a7a98bae0d706ac82","bytes":21898},{"path":"voice/Fem5/flg.give.WAV","sha":"a4dd5e9dbb437b802d510956b65c168be203e2bd","bytes":16428},{"path":"voice/Fem5/flg.huntergive.WAV","sha":"f5aebd2c686b9b77550e00e7152e5a8c8585fcef","bytes":17996},{"path":"voice/Fem5/flg.huntertake.WAV","sha":"96d47f9600c8db53b793215227fdc671ef2788dd","bytes":16100},{"path":"voice/Fem5/flg.retrieve.WAV","sha":"a542b18c8413aeac41a41ec72cffffedb3bc87cf","bytes":18274},{"path":"voice/Fem5/flg.secure.WAV","sha":"80911c2ddfb069ba86b9ef97c9f1bbac3a0ffccd","bytes":18142},{"path":"voice/Fem5/flg.take.WAV","sha":"78cb96f0efcd3dff79a51f4d6d1895d3dac8c549","bytes":17606},{"path":"voice/Fem5/gbl.anytime.WAV","sha":"b6e044f0af20523e9c7a2dd8404c737501903011","bytes":14048},{"path":"voice/Fem5/gbl.awesome.WAV","sha":"6c84d20667a25dd49e2d1db192c12af6d2fb4e93","bytes":17764},{"path":"voice/Fem5/gbl.aww.WAV","sha":"dc5480b47e54dde565cdb64aff33b223044da73f","bytes":24962},{"path":"voice/Fem5/gbl.brag.WAV","sha":"ae8de448426dc1efd057955f934bb7d2063ac3d9","bytes":25830},{"path":"voice/Fem5/gbl.bye.WAV","sha":"5089ad26892c884b34822634fad3be00318271a9","bytes":11506},{"path":"voice/Fem5/gbl.dunno.WAV","sha":"1e60329a9248ed2cb15d28320460a78b3ba68fbd","bytes":12966},{"path":"voice/Fem5/gbl.goodgame.WAV","sha":"09001a6cb84c9452e9e4d866d006663bd6f13b55","bytes":16712},{"path":"voice/Fem5/gbl.hi.WAV","sha":"21f57e0b87d345ae9fcefcfbe2b6d844f24726d8","bytes":11368},{"path":"voice/Fem5/gbl.nice.WAV","sha":"fb6b662f7ce32c2e3189ff75e918e53d88e64051","bytes":17996},{"path":"voice/Fem5/gbl.no.WAV","sha":"2237a6e4b66ec2a672b30b46f68348049c1f8972","bytes":12316},{"path":"voice/Fem5/gbl.obnoxious.WAV","sha":"3f291d1a5567f14ab69198cd30da6ccdf39e283a","bytes":23730},{"path":"voice/Fem5/gbl.oops.WAV","sha":"3dbbd222f0b266ec07d8950327a83f045e0af094","bytes":11190},{"path":"voice/Fem5/gbl.quiet.WAV","sha":"f32db889439bb5b4214e460fd31f8dcebc9e8cec","bytes":11046},{"path":"voice/Fem5/gbl.rock.WAV","sha":"32a40e47842cf33aeab55e50e0b4c13ab78c632f","bytes":16994},{"path":"voice/Fem5/gbl.sarcasm.WAV","sha":"82a85baea7c62d9a27c5d7614a603d348258b4cd","bytes":25406},{"path":"voice/Fem5/gbl.shazbot.WAV","sha":"d5d9ee8ea2fed124fc324b53d2fd7c79139274fe","bytes":15960},{"path":"voice/Fem5/gbl.shooting.WAV","sha":"9283f76db3c234ad2cbf7c5a5c743c1168108380","bytes":15326},{"path":"voice/Fem5/gbl.thanks.WAV","sha":"103e678d0b33f672b7f9d2cb92d32d0d8553d136","bytes":12780},{"path":"voice/Fem5/gbl.wait.WAV","sha":"6146d4d246f2ef39d5ae0f1db8f69934d0c8e9cc","bytes":12956},{"path":"voice/Fem5/gbl.when.WAV","sha":"6c12ea32b72f6613109efbef4d70e4b86fb4a077","bytes":30332},{"path":"voice/Fem5/gbl.woohoo.WAV","sha":"5424b2be96612147922848d67e8105db294832f5","bytes":15604},{"path":"voice/Fem5/gbl.yes.WAV","sha":"bc98c317729ab3aa93199aeed2c480d0c17dafaf","bytes":12304},{"path":"voice/Fem5/need.bombardier.WAV","sha":"da501c61f3035109594dc5394fa755d17fa96fc8","bytes":19336},{"path":"voice/Fem5/need.cover.WAV","sha":"4a917bde6ecb8cca37f67e3207cdc1414dd5fa0b","bytes":20674},{"path":"voice/Fem5/need.driver.WAV","sha":"2562a1881435eb415e7846e1b3517e8014709145","bytes":22034},{"path":"voice/Fem5/need.escort.WAV","sha":"b56779bdbb322477077253c46891a0e8d9826895","bytes":18806},{"path":"voice/Fem5/need.flyer.WAV","sha":"7fea3e8e1ca77d5a582309c1a9333b4cfa488ae0","bytes":22580},{"path":"voice/Fem5/need.gunship.WAV","sha":"25a88eb631224bd73e69dee8dbc8045d7c4007f8","bytes":23338},{"path":"voice/Fem5/need.hold.WAV","sha":"ea83a9fb2f42caec957e8a840512b84ed89cdce8","bytes":25222},{"path":"voice/Fem5/need.ride.WAV","sha":"3b4f4f021c138f46851feb941c75766b8925db53","bytes":16180},{"path":"voice/Fem5/need.support.WAV","sha":"94308ccc31025c13f9700731e7acd80b41751f08","bytes":18474},{"path":"voice/Fem5/need.tailgunner.WAV","sha":"7ec1f53b47c3a602a47e7f232b9af0207e085fc7","bytes":16442},{"path":"voice/Fem5/need.where.WAV","sha":"26a2641c911f245f25b7b5d3ac99d951b98e9b7b","bytes":13302},{"path":"voice/Fem5/rep.base.WAV","sha":"74bcab68666f88ea17eb60fc68547c5150573cd5","bytes":18650},{"path":"voice/Fem5/rep.generator.WAV","sha":"8f183c76893cb3871fbc1925c1c5860323dbda22","bytes":20210},{"path":"voice/Fem5/rep.me.WAV","sha":"354525dcecc077eb561899e5bf7a846da013c31a","bytes":15126},{"path":"voice/Fem5/rep.sensors.WAV","sha":"182a160c38871f97fcd3a051667000c72cb38154","bytes":19812},{"path":"voice/Fem5/rep.turrets.WAV","sha":"163797ce5f75289b4b593887903f7d03a50f06e8","bytes":18580},{"path":"voice/Fem5/rep.vehicle.WAV","sha":"e9a0e314878424cd8fbbff309697969f882c5a41","bytes":21546},{"path":"voice/Fem5/slf.att.attack.WAV","sha":"68cf91d754b66dfa421d4a16bc34702ea2f233bb","bytes":13032},{"path":"voice/Fem5/slf.att.base.WAV","sha":"1dac95c48c08ae73c0e43fdd3e1010eacbba2390","bytes":23560},{"path":"voice/Fem5/slf.att.flag.WAV","sha":"e6679b8f4f86737f70abc8bc3dcfcbe58d26b959","bytes":22204},{"path":"voice/Fem5/slf.att.generator.WAV","sha":"24c5964d6656afe96d5270660e53f29c6b143102","bytes":22458},{"path":"voice/Fem5/slf.att.sensors.WAV","sha":"5984bf2c90737411ac06cb3035b68e09e2ce6c41","bytes":22310},{"path":"voice/Fem5/slf.att.turrets.WAV","sha":"dc33fc25110517a65e5606f177bd49095ce57018","bytes":20430},{"path":"voice/Fem5/slf.att.vehicle.WAV","sha":"c06984b50fc000e484b1d6dea183498984b9a5b7","bytes":24806},{"path":"voice/Fem5/slf.def.base.WAV","sha":"da24a817a3c059139484dbd4c4d4d170a66d43c8","bytes":17862},{"path":"voice/Fem5/slf.def.defend.WAV","sha":"2d993b0c3fbd9ba1725c619ee8d2dc35a1aab6fc","bytes":14498},{"path":"voice/Fem5/slf.def.flag.WAV","sha":"f639e1c8f83d956119a7d6d980916b00afb87083","bytes":17960},{"path":"voice/Fem5/slf.def.generator.WAV","sha":"f99476ee87818119cc5bb56412db29307fc9591e","bytes":19092},{"path":"voice/Fem5/slf.def.nexus.WAV","sha":"5f31149a0c682326e35ed8a4effee1b865162aae","bytes":18004},{"path":"voice/Fem5/slf.def.sensors.WAV","sha":"cd471d615e0062c836c384323b1bdad9eab89ff2","bytes":19408},{"path":"voice/Fem5/slf.def.turrets.WAV","sha":"2ba1509a80a6f1090826307aade39d670f31b166","bytes":18814},{"path":"voice/Fem5/slf.def.vehicle.WAV","sha":"6fd08e57924ac1de40a373ebe26d1a4d7a2c6457","bytes":20596},{"path":"voice/Fem5/slf.rep.base.WAV","sha":"ab8281b282c7583e63d293562ed16a8aaf0f8034","bytes":18120},{"path":"voice/Fem5/slf.rep.equipment.WAV","sha":"b9f9dd6723ed128bfab0ef47b6bdf16052599308","bytes":17698},{"path":"voice/Fem5/slf.rep.generator.WAV","sha":"1879580315c0ded809398ba69c476daf594df36f","bytes":19012},{"path":"voice/Fem5/slf.rep.repairing.WAV","sha":"b3fc79b6622a0df55e3f04159ad27f595f0fe376","bytes":15924},{"path":"voice/Fem5/slf.rep.sensors.WAV","sha":"6768c44dee3aa5229ddcdc0634e8f523c415e450","bytes":18570},{"path":"voice/Fem5/slf.rep.turrets.WAV","sha":"42276081fb4df2b057fccbc7c60d618b435bbbda","bytes":18204},{"path":"voice/Fem5/slf.rep.vehicle.WAV","sha":"4c5d54142cc931fe9c8b58104e44f3dfbd0d79ec","bytes":22010},{"path":"voice/Fem5/slf.tsk.cover.WAV","sha":"a1c07674bd28be54a0a4cb97441e41acc2a75123","bytes":14366},{"path":"voice/Fem5/slf.tsk.defense.WAV","sha":"95a7a3fde64de0582dedd0985bdee13041d9b9dc","bytes":19858},{"path":"voice/Fem5/slf.tsk.generic.WAV","sha":"69f10974919e44e24f956c2eeeae44371ae741d1","bytes":11836},{"path":"voice/Fem5/slf.tsk.remotes.WAV","sha":"9f7bf7e673883da28be5275e82c6f5c53d039eb3","bytes":20118},{"path":"voice/Fem5/slf.tsk.sensors.WAV","sha":"2579ac321803dec9ecda55a5906e26362185ba3f","bytes":17356},{"path":"voice/Fem5/slf.tsk.turrets.WAV","sha":"4469464db79a169937a25a3d1cc6be253fa12c9c","bytes":17436},{"path":"voice/Fem5/slf.tsk.vehicle.WAV","sha":"9c2be35a0d9c78c53c7a749e66d90b4d0119a2a8","bytes":16978},{"path":"voice/Fem5/tgt.acquired.WAV","sha":"5a30307e05c61c21a6e2e13d8ed0da6d8f66f959","bytes":16634},{"path":"voice/Fem5/tgt.base.WAV","sha":"feb5844889418feb8c1e54935128f2a7a0a4a89e","bytes":27224},{"path":"voice/Fem5/tgt.destroyed.WAV","sha":"04c058af1339301a6dca922c3f37cfbbe9928fab","bytes":18794},{"path":"voice/Fem5/tgt.flag.WAV","sha":"bfb72f8780860af78c98ac6c495c25b856011fc8","bytes":26806},{"path":"voice/Fem5/tgt.my.WAV","sha":"29df8c6d1f99fa1db5fa432815840321e866f542","bytes":18110},{"path":"voice/Fem5/tgt.need.WAV","sha":"9fe8ae6c0c0cc2c0bec94a80eda1c72668772908","bytes":17888},{"path":"voice/Fem5/tgt.sensors.WAV","sha":"48e47938fc71bd30749217ea7e01f97f84a494a9","bytes":28650},{"path":"voice/Fem5/tgt.turret.WAV","sha":"7c300f3fd8a762758685332d4fdf6d14dfb4b1df","bytes":26684},{"path":"voice/Fem5/tgt.wait.WAV","sha":"a8c63ed734a853ecc3c9e9cea52d90a2274fff70","bytes":21894},{"path":"voice/Fem5/vqk.anytime.WAV","sha":"8eb981c1fa5edca761dd722813b3ae489feb6242","bytes":7166},{"path":"voice/Fem5/vqk.base.WAV","sha":"1a0692f68e8c51891b379503ba939831ed89978f","bytes":11248},{"path":"voice/Fem5/vqk.ceasefire.WAV","sha":"ec08885d033a6c81c31278cfedb7132877665a96","bytes":9706},{"path":"voice/Fem5/vqk.dunno.WAV","sha":"a2f8601a8edee22666d242b6f253de0f3397bdbd","bytes":6114},{"path":"voice/Fem5/vqk.help.WAV","sha":"88ebafd50b460afdf0f41138c6b42c9522ebdff4","bytes":6902},{"path":"voice/Fem5/vqk.move.WAV","sha":"712f6e841d3b9bb71dc435d1b377633940c6120e","bytes":8434},{"path":"voice/Fem5/vqk.no.WAV","sha":"28aca54ac648caa7bc54cd55e45f79b11165d6f7","bytes":9296},{"path":"voice/Fem5/vqk.quiet.WAV","sha":"1772abb24b58b45b2cc3f418687dea108823f8ce","bytes":10328},{"path":"voice/Fem5/vqk.sorry.WAV","sha":"90e24d4ea6f9d8ea7ca5fad7a9c1f1140dec2397","bytes":9314},{"path":"voice/Fem5/vqk.thanks.WAV","sha":"735e9cd67ee6db50882caf12ee37d37cfa151b04","bytes":7990},{"path":"voice/Fem5/vqk.wait.WAV","sha":"3bdb44e78bda08a2aa9018191776d3c5ba03fbd1","bytes":8334},{"path":"voice/Fem5/vqk.yes.WAV","sha":"becb11ca238738a1971f777877fff56c6a7a5bf1","bytes":9490},{"path":"voice/Fem5/wrn.bomber.WAV","sha":"bec3d52a7915291ab3597b2d369fedecef000beb","bytes":18006},{"path":"voice/Fem5/wrn.enemy.WAV","sha":"5bd6c826e7391529b8b4d150303ae5685bc8b768","bytes":19594},{"path":"voice/Fem5/wrn.vehicles.WAV","sha":"b5fec4225c0c465f7eb4dfab7d90fe31a97e5216","bytes":19666},{"path":"voice/Fem5/wrn.watchit.WAV","sha":"8b9d1bd3726aa18abbe0860de17e0379bb8f2729","bytes":18700},{"path":"voice/Male1/att.attack.wav","sha":"6567fd337175b492db360e7fda9c506bf7b9dc1b","bytes":10572},{"path":"voice/Male1/att.base.wav","sha":"009e94944412df4bd28d7e59ce1173e5253b7515","bytes":20402},{"path":"voice/Male1/att.chase.wav","sha":"6a4ee3ef89ec9b339e31a3edf628f20126a63f95","bytes":16386},{"path":"voice/Male1/att.distract.wav","sha":"c8eddba839bcf3900685104237fd19b315eff70e","bytes":21108},{"path":"voice/Male1/att.flag.wav","sha":"66c4f476ad854ccae79651da09119696732709ac","bytes":16910},{"path":"voice/Male1/att.generator.wav","sha":"89412dd07e6bfdaf585cbe141181c17a516cfce8","bytes":24210},{"path":"voice/Male1/att.objective.wav","sha":"b601d97d03cf0e969b3e0f959542ef5a1ee6f110","bytes":17902},{"path":"voice/Male1/att.reinforcements.wav","sha":"7c030d8c8914ff08fbbb2135902adab24cf48022","bytes":21482},{"path":"voice/Male1/att.sensors.wav","sha":"45468e57d653cbcc74ba3f3c8d073248d22c9a8c","bytes":21744},{"path":"voice/Male1/att.turrets.wav","sha":"d96bbc042374679551717601d3f7218f57e7f0ee","bytes":21472},{"path":"voice/Male1/att.vehicle.wav","sha":"e8aa64139441096acb2a02f7f0049f165dbb2096","bytes":21694},{"path":"voice/Male1/att.wait.wav","sha":"19af25c5bc33e0e42a05f91ab63ef626137eff05","bytes":23942},{"path":"voice/Male1/avo.deathcry_01.wav","sha":"14ce3b02e7314ca1c41737f634b811fcfaba3a98","bytes":11836},{"path":"voice/Male1/avo.deathcry_02.wav","sha":"bc8b973041dd4137ec79ff01110aa0514e8f8045","bytes":15428},{"path":"voice/Male1/avo.grunt.wav","sha":"904aec356b7621669b4617259664d5913d59073c","bytes":5720},{"path":"voice/Male1/avo.pain.wav","sha":"c564ba9e9a15b40f134d20612ab4c4cc91a520e4","bytes":6860},{"path":"voice/Male1/bas.clear.wav","sha":"fd236f634c47575ac37069850a12488a313492d4","bytes":17078},{"path":"voice/Male1/bas.enemy.WAV","sha":"79616be2673024284af4a8a3abdc4b2703dc8b41","bytes":21580},{"path":"voice/Male1/bas.retake.wav","sha":"5fd0251208edddf40e0d79bfe3c01d9228139f80","bytes":16198},{"path":"voice/Male1/bas.secure.wav","sha":"d50f922374b0d1ada64855184aff169d2bab22ac","bytes":17410},{"path":"voice/Male1/bas.taken.wav","sha":"7c92ece0f0614b7d9c7cf4243f34c191fc7936c7","bytes":22446},{"path":"voice/Male1/cmd.acknowledge.wav","sha":"d8904cc74c46eee33d20661f4ba97d3f0d3857e5","bytes":15896},{"path":"voice/Male1/cmd.bot.WAV","sha":"162a60e44f722d93b167c267c1d27b9823ee11ae","bytes":13988},{"path":"voice/Male1/cmd.completed.wav","sha":"304917d1142d80f5e58f110144b891aa0d203247","bytes":15208},{"path":"voice/Male1/cmd.decline.wav","sha":"e3de21730d350e74046c948ed95b7046b79a0e68","bytes":19404},{"path":"voice/Male1/def.base.wav","sha":"bd1b70a02b922567a1b035b7f1af33fe3777bf08","bytes":20194},{"path":"voice/Male1/def.carrier.wav","sha":"e9631438053169fe583fa4e8ff7739ac4358a354","bytes":22644},{"path":"voice/Male1/def.entrances.wav","sha":"b3035410063dd9a6c85f6f4945bb3dbd8d2d712d","bytes":20728},{"path":"voice/Male1/def.flag.wav","sha":"626fa58e61005fdbc3c8c4c47a1aa1b1a5efa620","bytes":15652},{"path":"voice/Male1/def.generator.wav","sha":"825bc2955396fc51c2cfb995950d7208eb4ab6c0","bytes":20152},{"path":"voice/Male1/def.me.wav","sha":"9f36ff0ecb3204ca5f4a9e815db6086f299978fb","bytes":11418},{"path":"voice/Male1/def.nexus.wav","sha":"fc6c46a6c2b3b66d8b7117fb57982ce7836eefab","bytes":15892},{"path":"voice/Male1/def.objective.wav","sha":"72cb2e58c957a2b8bf49fdf63d4db2cb76f6d79a","bytes":18860},{"path":"voice/Male1/def.reinforce.wav","sha":"9f9b0c31f49080ede5f5c9ecba55b85d6a20745d","bytes":22388},{"path":"voice/Male1/def.sensors.wav","sha":"14e1de4357f90b8298b7adc325402446283fcf4f","bytes":18904},{"path":"voice/Male1/def.turrets.wav","sha":"7c30c372a7d5ca59c757cce3e1a362e90a52f976","bytes":21284},{"path":"voice/Male1/def.vehicle.wav","sha":"2029a6ed394f0611a943c5dbc6bf8f316dd11db1","bytes":16518},{"path":"voice/Male1/ene.base.wav","sha":"b286ed91705f7a2bb98557a17e5ee5604657b970","bytes":20814},{"path":"voice/Male1/ene.disarray.wav","sha":"b54591c1df8487ad15c7eb2b9c6e3a47122a9253","bytes":28650},{"path":"voice/Male1/ene.generator.wav","sha":"049d1ebc5fbbb97d02970a5be63d8918e7812744","bytes":24024},{"path":"voice/Male1/ene.remotes.wav","sha":"335d7ff82b5b3c469c5196feffc7bc025640a886","bytes":29402},{"path":"voice/Male1/ene.sensors.wav","sha":"dd1b43cec1cae7c0768874448f71406c98d951d2","bytes":23026},{"path":"voice/Male1/ene.turrets.wav","sha":"624c76c7f4aec97dd979c01f7e8132bddc1930a3","bytes":22034},{"path":"voice/Male1/ene.vehicle.wav","sha":"9b015be70a54cdb5386ccd7b2314b27adc0f2e27","bytes":29638},{"path":"voice/Male1/flg.flag.WAV","sha":"2f35104d07c836c6e0a548773ab12b68dea454c3","bytes":23150},{"path":"voice/Male1/flg.give.wav","sha":"140faf217986352fdb3e8041ae665874e08a22e8","bytes":17350},{"path":"voice/Male1/flg.huntergive.wav","sha":"b6e56da74a1fa7084b38364be74dbde414847efd","bytes":22038},{"path":"voice/Male1/flg.huntertake.wav","sha":"bf8848742c0863b6d5e04b72c3332d4b6f0f2e03","bytes":18702},{"path":"voice/Male1/flg.retrieve.WAV","sha":"fbde385221da53c9cbaee838aa0cef37ade146e0","bytes":18246},{"path":"voice/Male1/flg.secure.wav","sha":"21a930fd9b91e7a7e18b9e83cd017c7f39a55ae9","bytes":20408},{"path":"voice/Male1/flg.take.wav","sha":"1cff6c0d95137ebc63b2548ac6c99800da28fffd","bytes":17890},{"path":"voice/Male1/gbl.anytime.wav","sha":"44499fd290026c1b749e2c54bcddb088b591c299","bytes":12872},{"path":"voice/Male1/gbl.awesome.wav","sha":"e80bdeb79ad18b865839b50a260748ba56ad0f0e","bytes":16384},{"path":"voice/Male1/gbl.aww.wav","sha":"85114b01bf5ba729fac5e431cf3876cf4d632883","bytes":35744},{"path":"voice/Male1/gbl.brag.WAV","sha":"f7b832cf1314e51f6300b6ecf6f01d92901e1a0a","bytes":26220},{"path":"voice/Male1/gbl.bye.wav","sha":"663ee3c6709ccc8b8133669de50258e2270c0746","bytes":10598},{"path":"voice/Male1/gbl.dunno.wav","sha":"7417c25a5dbba903f93abc444666ce766e588fe8","bytes":14850},{"path":"voice/Male1/gbl.goodgame.wav","sha":"e615aa888a41f7f092ecda501797d34e52dd8ae9","bytes":18080},{"path":"voice/Male1/gbl.hi.wav","sha":"7813965edc18ccafe621c3d970f5230c4c20e460","bytes":9974},{"path":"voice/Male1/gbl.nice.wav","sha":"6456c865397032ce142bb4d7d727db068e5951b7","bytes":25586},{"path":"voice/Male1/gbl.no.wav","sha":"a001e90a1da30e859fe3b689935a8b914dc9bb94","bytes":9510},{"path":"voice/Male1/gbl.obnoxious.WAV","sha":"ef43c7521fbdd589b69b624edd103f8a45e502b9","bytes":32066},{"path":"voice/Male1/gbl.oops.wav","sha":"27465151b5239e6073e8c99df6f6a3312ab7382a","bytes":12860},{"path":"voice/Male1/gbl.quiet.WAV","sha":"f6362cb9c2c4233dcce032852c9d1f434e6639a6","bytes":14062},{"path":"voice/Male1/gbl.rock.WAV","sha":"4df9e9026f0e63d0de8bb7a3999f248d05040ff2","bytes":15692},{"path":"voice/Male1/gbl.sarcasm.WAV","sha":"ca86eca1d012d9186fa38360e8118beb2d6f5305","bytes":23508},{"path":"voice/Male1/gbl.shazbot.wav","sha":"f73cedb9cdc650cc7f4aa01db9fe30ee88327555","bytes":17188},{"path":"voice/Male1/gbl.shooting.WAV","sha":"b99e0715c47c682ea3b87c19a9fc8d70ff60e25c","bytes":17338},{"path":"voice/Male1/gbl.thanks.wav","sha":"2ff345ed23cd1a25bf52919a09804ce8617dd8b2","bytes":11168},{"path":"voice/Male1/gbl.wait.wav","sha":"5d023b99a1e558f982c4b015bac9f44ce6d5aa67","bytes":10782},{"path":"voice/Male1/gbl.when.wav","sha":"1c29d9f40ec5305a8201f4a6ad63a299dc82d7a8","bytes":23592},{"path":"voice/Male1/gbl.woohoo.WAV","sha":"4b3272d0e1de1c56da74df5afe9d353a664a9f47","bytes":23560},{"path":"voice/Male1/gbl.yes.wav","sha":"a84ca2f772bb04448d4607848c5aadc644a75293","bytes":14880},{"path":"voice/Male1/need.bombardier.wav","sha":"06aa7120870c5ff2c6d43aefe1fe9214ff630ad6","bytes":15278},{"path":"voice/Male1/need.cover.WAV","sha":"40028d72a5e6a015744f15bfcef777555516068c","bytes":17068},{"path":"voice/Male1/need.driver.wav","sha":"977a6eca4d790c9049e0d20f85e7f9bf67fe22df","bytes":24574},{"path":"voice/Male1/need.escort.wav","sha":"1c9c5e4e56abe4e5ccb1fbac4cc7497e3241f1c9","bytes":21066},{"path":"voice/Male1/need.flyer.WAV","sha":"ab473191ac51d41a4b50d45cafab5afa4266c066","bytes":22610},{"path":"voice/Male1/need.gunship.WAV","sha":"81f4fbc596e23cc0b23b473a52faf15918d527d3","bytes":24108},{"path":"voice/Male1/need.hold.wav","sha":"b40682a142961a8cb2534dbff129cd5ec3b003d1","bytes":24540},{"path":"voice/Male1/need.ride.WAV","sha":"be98324361bacbcbd8017308a35dd7aaf57be094","bytes":16516},{"path":"voice/Male1/need.support.wav","sha":"144e9274031da1aa8972f136bf1fcd29757e2a71","bytes":22296},{"path":"voice/Male1/need.tailgunner.wav","sha":"dbde03479c8fb8e6556c2e2b0269335070472075","bytes":16770},{"path":"voice/Male1/need.where.wav","sha":"898391791c1d0e4e3bbdd1f8060fe16f0f8952e4","bytes":15008},{"path":"voice/Male1/rep.base.wav","sha":"e00842695c9660c3c14772f14cc181363b7a7e72","bytes":16104},{"path":"voice/Male1/rep.generator.wav","sha":"2b5f45b46aaa6c904c363883db95ebc94fd954d9","bytes":21306},{"path":"voice/Male1/rep.me.wav","sha":"118217def542cac8f697d8498af7e171715e7d24","bytes":16978},{"path":"voice/Male1/rep.sensors.wav","sha":"3dc28b24434ee45123680ef65f60483276128b27","bytes":19254},{"path":"voice/Male1/rep.turrets.wav","sha":"6a02f38b09248672410c53c0cac0953fa6f1f2c2","bytes":23874},{"path":"voice/Male1/rep.vehicle.wav","sha":"7a2717a7ecfb8dd3ca4f83dfdb0ebda21e55c7b4","bytes":26466},{"path":"voice/Male1/slf.att.attack.WAV","sha":"09d490743f099818af1f496d49cbec43ce099a35","bytes":16548},{"path":"voice/Male1/slf.att.base.WAV","sha":"ff23c4cbd4d32dad57fa1a0b5cd35872e6875807","bytes":21330},{"path":"voice/Male1/slf.att.flag.WAV","sha":"92118c8b1ceb39c1e280da98c768193661b9ffa7","bytes":21364},{"path":"voice/Male1/slf.att.generator.WAV","sha":"738e36b56b052f85e752f032d22de9f9e2914db7","bytes":22378},{"path":"voice/Male1/slf.att.sensors.WAV","sha":"c1838cd81fb69f2c7272094883f92d6ccec8c3d3","bytes":27404},{"path":"voice/Male1/slf.att.turrets.WAV","sha":"23831dd4d6edaab7938d5860bb4c0affcb3b5ac3","bytes":23406},{"path":"voice/Male1/slf.att.vehicle.WAV","sha":"45fa17dc7ef7877a7480181a661d405c447eb1a4","bytes":25798},{"path":"voice/Male1/slf.def.base.WAV","sha":"3a69321b77a9572494f2f788902a5e574af7de59","bytes":19754},{"path":"voice/Male1/slf.def.defend.WAV","sha":"000fafd106125471fad41c899e22871d98cf51af","bytes":15252},{"path":"voice/Male1/slf.def.flag.WAV","sha":"e568d554fb48d8a6ea14b17a8d668ee5479b5401","bytes":21118},{"path":"voice/Male1/slf.def.generator.WAV","sha":"23a0ebe39bd8dd9e9045c9b3f0eed6dd2bf8015a","bytes":19324},{"path":"voice/Male1/slf.def.nexus.WAV","sha":"0a23e9146727c3686b6fb7d01cd5b3e0adc181ee","bytes":21644},{"path":"voice/Male1/slf.def.sensors.WAV","sha":"da6c21f261551325d19b853527e572eaa77a4352","bytes":23398},{"path":"voice/Male1/slf.def.turrets.WAV","sha":"4b4f12ed4e0302e11eff54d78bbbff70dfa0dad5","bytes":19892},{"path":"voice/Male1/slf.def.vehicle.WAV","sha":"3ac02b0983f3f8864c3d617bed34c881e7c15900","bytes":20540},{"path":"voice/Male1/slf.rep.base.WAV","sha":"3b33f145764db8794442a8e9e27dd53ef1402747","bytes":18810},{"path":"voice/Male1/slf.rep.equipment.WAV","sha":"bc44706e35922f355d63249aef26731cf2042761","bytes":19146},{"path":"voice/Male1/slf.rep.generator.WAV","sha":"dce691c40fccdceeb80c2617c9b106cab43eeba8","bytes":18160},{"path":"voice/Male1/slf.rep.repairing.WAV","sha":"b792dabb5206b87492fce2ac2e4f408988c69172","bytes":18832},{"path":"voice/Male1/slf.rep.sensors.WAV","sha":"ef345073045cb2f93ecdde1d795ac7bcf5408d0b","bytes":19866},{"path":"voice/Male1/slf.rep.turrets.WAV","sha":"0840feb8e075556fd2328ac58dacda32c9764f27","bytes":23080},{"path":"voice/Male1/slf.rep.vehicle.WAV","sha":"1bfd72e5914e4e822b531e269cc46242f7c581d7","bytes":26252},{"path":"voice/Male1/slf.tsk.cover.WAV","sha":"4cb8b228f8c0f69304d058853ef9fd08f9884dd4","bytes":13086},{"path":"voice/Male1/slf.tsk.defense.WAV","sha":"59078f4f88d17d67ed7fdb894aa7fa8f6a1957b5","bytes":21108},{"path":"voice/Male1/slf.tsk.generic.WAV","sha":"a93c8ecd3eda1161c48b1b5ed114d89df348e22f","bytes":13754},{"path":"voice/Male1/slf.tsk.remotes.WAV","sha":"04a12746fda400341c9c15242075b51ba299bff6","bytes":24364},{"path":"voice/Male1/slf.tsk.sensors.WAV","sha":"16b2514ac64f07321e76549869e7ff42f26fd40f","bytes":19928},{"path":"voice/Male1/slf.tsk.turrets.WAV","sha":"094129654dd416a178a1571735499746e3666ceb","bytes":16362},{"path":"voice/Male1/slf.tsk.vehicle.WAV","sha":"3d242aeba08c23814168892243fd509758add075","bytes":18302},{"path":"voice/Male1/tgt.acquired.wav","sha":"f3fc59b4a14f10217b71453eab763091877ab979","bytes":20424},{"path":"voice/Male1/tgt.base.wav","sha":"8db87c33617f7058d82e480afb0da482b40e9492","bytes":38652},{"path":"voice/Male1/tgt.destroyed.wav","sha":"95b8eef8a75cef30551ad91ffe966eef9b844848","bytes":18116},{"path":"voice/Male1/tgt.flag.wav","sha":"955b98a531f117a4a3caa0f842ce86118cccf56b","bytes":28960},{"path":"voice/Male1/tgt.my.WAV","sha":"0ec74c43fe18680ee37ffda917c4b450cffd908a","bytes":18884},{"path":"voice/Male1/tgt.need.wav","sha":"78d92a9eef17d4cdb0d8b3451513659c74cbb097","bytes":16350},{"path":"voice/Male1/tgt.sensors.wav","sha":"12a10b15e24a9d90ac1616f2b25fa93befbf10bb","bytes":28486},{"path":"voice/Male1/tgt.turret.wav","sha":"e4727e733ee5ecb999b21151eb338f3bbc6afc1a","bytes":28016},{"path":"voice/Male1/tgt.wait.wav","sha":"95847a01cde9ea2362fcf429dd905c7076d3f4bd","bytes":21558},{"path":"voice/Male1/vqk.anytime.wav","sha":"6639fd74098f79478d610836400e5bb1ed0b408c","bytes":10856},{"path":"voice/Male1/vqk.base.WAV","sha":"9103c561b8b778fc1f293bbfccd4262419be61f2","bytes":11710},{"path":"voice/Male1/vqk.ceasefire.wav","sha":"c3594c3e2c18a25b9d11dab0fcb880e02981a928","bytes":8948},{"path":"voice/Male1/vqk.dunno.wav","sha":"94e429fdb6811ac9735dcd54aa4292e38856c8e4","bytes":12604},{"path":"voice/Male1/vqk.help.WAV","sha":"f46d8d09d2811f4795b8cd182dc82cf63a649ac5","bytes":8072},{"path":"voice/Male1/vqk.move.wav","sha":"6232b283252340f23bc490fd85cb2486d641ca3d","bytes":10954},{"path":"voice/Male1/vqk.no.wav","sha":"439968a98a36324debeada33838048b2793cf7ab","bytes":5840},{"path":"voice/Male1/vqk.quiet.WAV","sha":"66858d1436c33188949b85a2a32639950d3a3967","bytes":10612},{"path":"voice/Male1/vqk.sorry.wav","sha":"0da73a20747aff078e2d8bddee8ae4066ee1deb2","bytes":5728},{"path":"voice/Male1/vqk.thanks.wav","sha":"5a90906c91abdd42b0f3ae687695276c1243a019","bytes":7138},{"path":"voice/Male1/vqk.wait.wav","sha":"db5df4d891a360237fff7d2a4cd00c8ed429a783","bytes":12748},{"path":"voice/Male1/vqk.yes.wav","sha":"929277feb10d43ed1c11cc473bcaa49cbcc95542","bytes":8160},{"path":"voice/Male1/wrn.bomber.wav","sha":"e95a05cbaa931c55334f548be2fe25a8231caa88","bytes":16222},{"path":"voice/Male1/wrn.enemy.wav","sha":"fb5c89ee457d45ae002bdf45946d090f21ab12ce","bytes":19012},{"path":"voice/Male1/wrn.vehicles.wav","sha":"5cf610e5007b7431fbb567218b58c287d6c05e59","bytes":17292},{"path":"voice/Male1/wrn.watchit.wav","sha":"689d408d9f0359d6d672c6b17e0ca6074f314097","bytes":20892},{"path":"voice/Male2/att.attack.WAV","sha":"a91c14a32d8fefb65e4b8cfd65e1048ec30d5453","bytes":13516},{"path":"voice/Male2/att.base.WAV","sha":"5eca0e4e99d87536df670a49d937c867c818c459","bytes":19952},{"path":"voice/Male2/att.chase.WAV","sha":"1de42f14587352656eee07c23bdd2ac853f8168c","bytes":20460},{"path":"voice/Male2/att.distract.WAV","sha":"d0a0922d3ff4c8f44015cc492c8822fd7d26fd11","bytes":25642},{"path":"voice/Male2/att.flag.WAV","sha":"c76eef19e53fdbabbec526ebbd2d1d7c43b5138e","bytes":18612},{"path":"voice/Male2/att.generator.WAV","sha":"a342fee8db3b11d3f212ec8590f2fc807c843004","bytes":24626},{"path":"voice/Male2/att.objective.WAV","sha":"fc50ab1395bc83b4780e8c8a9f624c6564514c2f","bytes":19094},{"path":"voice/Male2/att.reinforcements.WAV","sha":"f5fc9cf9a550a171455a25c54c135fb9103e6a45","bytes":21478},{"path":"voice/Male2/att.sensors.WAV","sha":"e0e2185d1fd1cce351d927becac55819ed8894b8","bytes":23992},{"path":"voice/Male2/att.turrets.WAV","sha":"d87c01da6cfef95e74b1872cfa31531d0ae70a91","bytes":23380},{"path":"voice/Male2/att.vehicle.WAV","sha":"7de77bb71ce3653e45ae274f2fd8fb9766c7c6ef","bytes":21526},{"path":"voice/Male2/att.wait.WAV","sha":"6cc68b3bf8b4d9bd272c398690fe8907dcbd90d2","bytes":26662},{"path":"voice/Male2/avo.deathcry_01.WAV","sha":"d3dbacb1acd86cda3420de09ee78244251335d4d","bytes":13884},{"path":"voice/Male2/avo.deathcry_02.WAV","sha":"78d2d48dce54b7d0cfbd43d18ec54b11c1a8703a","bytes":16444},{"path":"voice/Male2/avo.grunt.WAV","sha":"6fca4312b78271ce1ea0b47308866cf8362d4ce7","bytes":4354},{"path":"voice/Male2/avo.pain.WAV","sha":"597c13b92fb503af7a9a721883f7a29d7ce3e7d5","bytes":6024},{"path":"voice/Male2/bas.clear.WAV","sha":"1c4a1fbaabcdd8df800500de7d374ba2b92ea275","bytes":16670},{"path":"voice/Male2/bas.enemy.WAV","sha":"2eb88d492f3f2b7a93af9eedc60c80f067053e12","bytes":20084},{"path":"voice/Male2/bas.retake.WAV","sha":"25d55c9c7ad8ef77d671ddf9e9a22b41a6eb3c48","bytes":17224},{"path":"voice/Male2/bas.secure.WAV","sha":"784c1312ba592ee7093d06590db67146e20d7b40","bytes":18984},{"path":"voice/Male2/bas.taken.WAV","sha":"b7f3a04fcfe9438021490312c414f8af59dc9bac","bytes":19474},{"path":"voice/Male2/cmd.acknowledge.WAV","sha":"031e95cd10bbb4b0f48f4ec9e2f7308c024b3e4d","bytes":17392},{"path":"voice/Male2/cmd.bot.WAV","sha":"15f3afbfc5b3f2510526021126e75b75ac2ff089","bytes":16608},{"path":"voice/Male2/cmd.completed.WAV","sha":"de1283652362903047502eb343c21456219b59eb","bytes":15474},{"path":"voice/Male2/cmd.decline.WAV","sha":"1318cde44128dee3a8ab9b489ce0fa7cc40f2cae","bytes":16770},{"path":"voice/Male2/def.base.WAV","sha":"e60476672ffd99b821b770b80dc8cf838d6ae083","bytes":19642},{"path":"voice/Male2/def.carrier.WAV","sha":"4021e3b5fd091a63eca769542fa2b3a09b9cf35a","bytes":19512},{"path":"voice/Male2/def.entrances.WAV","sha":"515344d3fc8184562c2578de691b45c269c07c2c","bytes":20538},{"path":"voice/Male2/def.flag.WAV","sha":"600258a80953685bf8598008b7237942b7b5f66c","bytes":18854},{"path":"voice/Male2/def.generator.WAV","sha":"a4a0eacc4f8148745fad2ff05c60e7b4a8e522a1","bytes":17516},{"path":"voice/Male2/def.me.WAV","sha":"8c9a11cdd7912d91b51e31a91bb8d0b549f100c8","bytes":11582},{"path":"voice/Male2/def.nexus.WAV","sha":"a5bd80df79824514daffb4edba8aa41b0a15e347","bytes":18316},{"path":"voice/Male2/def.objective.WAV","sha":"87a8d920113814fe6aadbbf6f0e73ee2b229f788","bytes":19344},{"path":"voice/Male2/def.reinforce.WAV","sha":"6bd4f5da3ea718269e3a6da3c4cf59d1d07b063f","bytes":22142},{"path":"voice/Male2/def.sensors.WAV","sha":"92a502ee56d9635834f67a16f408df96c126983f","bytes":19128},{"path":"voice/Male2/def.turrets.WAV","sha":"70df5b735b8ad40eae6d1f42e050901edc83552e","bytes":18484},{"path":"voice/Male2/def.vehicle.WAV","sha":"0c0f33976d5cbb13323e3abcaf55eb8278c6b3e5","bytes":21684},{"path":"voice/Male2/ene.base.WAV","sha":"99aac2115fc4a1d97b95ad4562a876ffe017db19","bytes":20004},{"path":"voice/Male2/ene.disarray.WAV","sha":"6c85e0213558aee5e8eb1394175ffeb03d1ae629","bytes":24530},{"path":"voice/Male2/ene.generator.WAV","sha":"5c669fff76de477bc5d8fb7bd71b5e8e7bbdc568","bytes":23824},{"path":"voice/Male2/ene.remotes.WAV","sha":"b7498b6f64e6188da0423797e4ab42529a32128c","bytes":26456},{"path":"voice/Male2/ene.sensors.WAV","sha":"0f3eacaa302c2a0f2c4f9817ea945cb69c48aaa2","bytes":21340},{"path":"voice/Male2/ene.turrets.WAV","sha":"4bafa3b249f41dd509fd747fd0d72d571f4c759e","bytes":22604},{"path":"voice/Male2/ene.vehicle.WAV","sha":"c24c6da5e6f52db167b4c2d9a4d7e898cd89382e","bytes":29136},{"path":"voice/Male2/flg.flag.WAV","sha":"b61452b2fec9055dc9ff8f3e96abb4e9b5234e74","bytes":19722},{"path":"voice/Male2/flg.give.WAV","sha":"785129f9f2677564bafe184c2b94106cd0260108","bytes":16366},{"path":"voice/Male2/flg.huntergive.WAV","sha":"724a59d3ed7300ce1abe4cac55f7aa0dbfc56c0b","bytes":17964},{"path":"voice/Male2/flg.huntertake.WAV","sha":"cc38bf2ef7ba7a88625c3dfb30fdb36e3fcb4da5","bytes":17354},{"path":"voice/Male2/flg.retrieve.WAV","sha":"d901eb2cfcb2c8f1c556ebbee1780278e354e260","bytes":20484},{"path":"voice/Male2/flg.secure.WAV","sha":"eae6796652be207bd74a798d25ebf0af39166d2b","bytes":18866},{"path":"voice/Male2/flg.take.WAV","sha":"d5eaa63a65b9af04c4907451605ec59e0cddbaf4","bytes":17098},{"path":"voice/Male2/gbl.anytime.WAV","sha":"8c772c86cca53cca1f24307526df2f46be4e8704","bytes":15488},{"path":"voice/Male2/gbl.awesome.WAV","sha":"18392d8551a069dacb5b61e169606a721d2a85b4","bytes":17528},{"path":"voice/Male2/gbl.aww.WAV","sha":"e9888294b64fa8b5821ebcb6c6cb67f7f7482ddd","bytes":23856},{"path":"voice/Male2/gbl.brag.WAV","sha":"778d6eda4cb5e184402c74087861a34da9028ff5","bytes":26670},{"path":"voice/Male2/gbl.bye.WAV","sha":"da0d38622865176e34c225489f187ea39c7cde68","bytes":12014},{"path":"voice/Male2/gbl.dunno.WAV","sha":"d9fb05119229b2364dbf6df1d08d30e3ce9c1e73","bytes":19684},{"path":"voice/Male2/gbl.goodgame.WAV","sha":"63660b14a0abb3edf2eba15ad140cc106342e013","bytes":16186},{"path":"voice/Male2/gbl.hi.WAV","sha":"35230c2ef553c0b36dfec58909bf6128f0630b8b","bytes":10430},{"path":"voice/Male2/gbl.nice.WAV","sha":"106500e9d9256653a455ac9a09f58a47f6f435ab","bytes":16596},{"path":"voice/Male2/gbl.no.WAV","sha":"928cb1a45cfe72c27a133cf15d2ffa819f43d6c3","bytes":14010},{"path":"voice/Male2/gbl.obnoxious.WAV","sha":"dccc768d9627b589576dafa85528ec89a8e2e0fd","bytes":33566},{"path":"voice/Male2/gbl.oops.WAV","sha":"86449ee02d30eabb05f89fe0839b74844887988d","bytes":11746},{"path":"voice/Male2/gbl.quiet.WAV","sha":"aae350dc443188588f3d0585897ffa3145f8f4b5","bytes":12726},{"path":"voice/Male2/gbl.rock.WAV","sha":"1f6a8a792a7746af910ecf35fe37aed99e2f2826","bytes":18026},{"path":"voice/Male2/gbl.sarcasm.WAV","sha":"f1bdaffb9dccaeb68fee48e8ea890ede24631325","bytes":33238},{"path":"voice/Male2/gbl.shazbot.WAV","sha":"50638c954f8315bf090b3abfb4da08b9962f7d56","bytes":18624},{"path":"voice/Male2/gbl.shooting.WAV","sha":"8bb49f6ad648455e9b8df45c17fe157ccc346a11","bytes":16436},{"path":"voice/Male2/gbl.thanks.WAV","sha":"0e5765037d9572fde4105913870e1ef1a5b30498","bytes":10660},{"path":"voice/Male2/gbl.wait.WAV","sha":"06451a18ea50931aa28d3de7256588b2ab4bfaea","bytes":13012},{"path":"voice/Male2/gbl.when.WAV","sha":"b2c715942f7cea250fa1387cc9f14d834fa9daa2","bytes":16652},{"path":"voice/Male2/gbl.woohoo.WAV","sha":"81faed78d2298253123c376ad3340f08a640058b","bytes":21454},{"path":"voice/Male2/gbl.yes.WAV","sha":"72477bc6eb2d32e092e0f202d0f6d7d8e4e5d6b4","bytes":15238},{"path":"voice/Male2/need.bombardier.WAV","sha":"78f05e775694752b5df15263655a0824e11d1745","bytes":18608},{"path":"voice/Male2/need.cover.WAV","sha":"533d2b220af5fbe6bf7d730dece14e102639b1fc","bytes":22032},{"path":"voice/Male2/need.driver.WAV","sha":"cbe43ccb333f0f5edd1dc4750ae877ddc4248460","bytes":24958},{"path":"voice/Male2/need.escort.WAV","sha":"8e28eaa9f9a82173495f76e064e8283799d55db3","bytes":22888},{"path":"voice/Male2/need.flyer.WAV","sha":"ebe5925dbf0bbec209ffc113238b58ee32fc1984","bytes":24364},{"path":"voice/Male2/need.gunship.WAV","sha":"16c06ae2bbc8c991df0f3459e7dfff8374efa834","bytes":32002},{"path":"voice/Male2/need.hold.WAV","sha":"c96d2ec883ce53149c2f5c7c91e9fee0b66192ed","bytes":24012},{"path":"voice/Male2/need.ride.WAV","sha":"6bbb8e4a0b599230069deb604b8b1374d9964a33","bytes":16786},{"path":"voice/Male2/need.support.WAV","sha":"2bcb6b79bba6b744efc705a7c376124c595d75ec","bytes":22462},{"path":"voice/Male2/need.tailgunner.WAV","sha":"10e73081daa103bce5264b037a4979c97f3dd9ea","bytes":18662},{"path":"voice/Male2/need.where.WAV","sha":"93d59b9b675c65f5746e9326f00d80ac6b654cf4","bytes":11394},{"path":"voice/Male2/rep.base.WAV","sha":"8c498554ba6d2c04bab18ea56f1e680675d22c65","bytes":24820},{"path":"voice/Male2/rep.generator.WAV","sha":"2041adcc508ef5c368a97f56d65e0d36af5b0c6c","bytes":19620},{"path":"voice/Male2/rep.me.WAV","sha":"f43d5068912c5803fd58dda088f642ca5b56a3d5","bytes":15168},{"path":"voice/Male2/rep.sensors.WAV","sha":"2a59027cc8e971535612c80bd6881c39fa5d39b2","bytes":23746},{"path":"voice/Male2/rep.turrets.WAV","sha":"4a91b0a2a8eaa8e98b1d3b4c6f79f07795f7c9eb","bytes":21050},{"path":"voice/Male2/rep.vehicle.WAV","sha":"f1c292365f1135307371155162d944b68fdeea06","bytes":22092},{"path":"voice/Male2/slf.att.attack.WAV","sha":"ab093990fd3774a2781a1a2a8c44ec5a23afe8ef","bytes":13202},{"path":"voice/Male2/slf.att.base.WAV","sha":"2bacf239e90a8df3a2b4495698bc1b17e99e2675","bytes":22016},{"path":"voice/Male2/slf.att.flag.WAV","sha":"0292d693d44ffa45d4338ad59ce26b65a4e4f038","bytes":24872},{"path":"voice/Male2/slf.att.generator.WAV","sha":"68c9b7387d4095bca381212d4c59ac776d784ae3","bytes":22964},{"path":"voice/Male2/slf.att.sensors.WAV","sha":"b78070a9334060b34300469a2c776fcd8d917d97","bytes":23884},{"path":"voice/Male2/slf.att.turrets.WAV","sha":"605efe97ae191aeebee948e6b2ddb202f58656b0","bytes":23654},{"path":"voice/Male2/slf.att.vehicle.WAV","sha":"c46dcb7ff3bd253b46eb58e63437c8f71ae327d1","bytes":28924},{"path":"voice/Male2/slf.def.base.WAV","sha":"d01ce54aec61a86aaa0f22e0aa1c3dc8f74aad87","bytes":23984},{"path":"voice/Male2/slf.def.defend.WAV","sha":"d3a83f730767aab92d0c3a8dfceb274c441b6992","bytes":17668},{"path":"voice/Male2/slf.def.flag.WAV","sha":"351a26562315b6d529d11dadfad0e27bb22724e0","bytes":20776},{"path":"voice/Male2/slf.def.generator.WAV","sha":"437473393aa9d6653481bd59b99fadd50c1125c6","bytes":21074},{"path":"voice/Male2/slf.def.nexus.WAV","sha":"bee486bfe796d12fab82c2def59702c113454a09","bytes":22348},{"path":"voice/Male2/slf.def.sensors.WAV","sha":"6dd780290f66ab8fec03da82ac13e33235daace4","bytes":22186},{"path":"voice/Male2/slf.def.turrets.WAV","sha":"ef8913d7483762ade276e2a9b48bf97f0d8c22a6","bytes":25302},{"path":"voice/Male2/slf.def.vehicle.WAV","sha":"2b6445f96b703e74b7d79ae610bddc6bf04339d6","bytes":27342},{"path":"voice/Male2/slf.rep.base.WAV","sha":"bf08dc32424496f2ca0443fed5aebfcfc27b2819","bytes":23460},{"path":"voice/Male2/slf.rep.equipment.WAV","sha":"f3c9d3a49ed3d8ff28127009e993f1631d6d4c1b","bytes":25196},{"path":"voice/Male2/slf.rep.generator.WAV","sha":"bdaf045dc84fac5b0d9cae95370ba82a550f8668","bytes":20532},{"path":"voice/Male2/slf.rep.repairing.WAV","sha":"32f79d8902dbcdd83cc26facc8d13036d4dcde37","bytes":19674},{"path":"voice/Male2/slf.rep.sensors.WAV","sha":"23918dee42023cf16bf62a7c4377466557c25ba1","bytes":23248},{"path":"voice/Male2/slf.rep.turrets.WAV","sha":"08a36ff8dfc74b0a56346457029be11955f3351c","bytes":26602},{"path":"voice/Male2/slf.rep.vehicle.WAV","sha":"25c56094ed36b7e734d5e895a4ae18fbed71e785","bytes":28430},{"path":"voice/Male2/slf.tsk.cover.WAV","sha":"5f1c10886f0c76731b972b18b8c8fbaf53e98a25","bytes":14348},{"path":"voice/Male2/slf.tsk.defense.WAV","sha":"22ca73d184239fee19d7812896872d98109622d9","bytes":24852},{"path":"voice/Male2/slf.tsk.generic.WAV","sha":"74577fb2f1a85c794c905e04548f628829026cab","bytes":10856},{"path":"voice/Male2/slf.tsk.remotes.WAV","sha":"03db8377ed30f0c4b65507f5793eb741e970db12","bytes":24628},{"path":"voice/Male2/slf.tsk.sensors.WAV","sha":"f752a5c3a44682a37b21d5973ec5977d106ca3a0","bytes":26088},{"path":"voice/Male2/slf.tsk.turrets.WAV","sha":"b12e379e74f93bd46209f6abd7e7610e217e5c1b","bytes":20232},{"path":"voice/Male2/slf.tsk.vehicle.WAV","sha":"048291cf30b5ee7ecc3830b816a850d3d4ba511e","bytes":20160},{"path":"voice/Male2/tgt.acquired.WAV","sha":"5f02990c27f84005f38aa690575fe270df9896c4","bytes":16572},{"path":"voice/Male2/tgt.base.WAV","sha":"f2b0f7546eb1a7e38b79a98cde6dc358d386100f","bytes":27052},{"path":"voice/Male2/tgt.destroyed.WAV","sha":"b0b42275764874014ee0cf71a34859b621d41feb","bytes":16842},{"path":"voice/Male2/tgt.flag.WAV","sha":"f4b22b96bf063241270311283f269cd1c945697a","bytes":25642},{"path":"voice/Male2/tgt.my.WAV","sha":"329c14bebc2ca28295fcfce6c7c4da9c33b43d9a","bytes":17468},{"path":"voice/Male2/tgt.need.WAV","sha":"f84f3e965d4accda0ba3737be719c74abf9bd561","bytes":19586},{"path":"voice/Male2/tgt.sensors.WAV","sha":"13be368c46f7d3b60d686c145e24c709d8eec184","bytes":24430},{"path":"voice/Male2/tgt.turret.WAV","sha":"b67ac66ceabce194a0833b5f2343b9a2504ddf36","bytes":25408},{"path":"voice/Male2/tgt.wait.WAV","sha":"2add10cd1f2217f6b522df9eed37243b3ef92245","bytes":25434},{"path":"voice/Male2/vqk.anytime.WAV","sha":"d8dda61a5d7255ebe171d8f9e5f851ac6ec63f54","bytes":14304},{"path":"voice/Male2/vqk.base.WAV","sha":"41dd4e94778c1e7ab1da954abe971a12e32f0cb7","bytes":12848},{"path":"voice/Male2/vqk.ceasefire.WAV","sha":"3e3ff3f8ebdad8a6c13aa6e1c84dc8a1952d699f","bytes":18010},{"path":"voice/Male2/vqk.dunno.WAV","sha":"1c19f015479a9b1962fab61761e8c637b020a39c","bytes":9292},{"path":"voice/Male2/vqk.help.WAV","sha":"dbffd197e51c11f9f50eae7c99b95dcd952ae077","bytes":5346},{"path":"voice/Male2/vqk.move.WAV","sha":"4bec259cc9a5f3ad54d5d9a23cc811729f99c75f","bytes":6870},{"path":"voice/Male2/vqk.no.WAV","sha":"030dd1ec1845a11466317b74c2a9c134fa40502a","bytes":5326},{"path":"voice/Male2/vqk.quiet.WAV","sha":"1a83a03aef99666ef72acb244ee7a212262078f8","bytes":7986},{"path":"voice/Male2/vqk.sorry.WAV","sha":"8c59c79952c3710be70e04f46ee74e1c87da47d5","bytes":11014},{"path":"voice/Male2/vqk.thanks.WAV","sha":"124dbbfa2bacb28edaefc09731eeebe62bcaa7d8","bytes":7926},{"path":"voice/Male2/vqk.wait.WAV","sha":"db388ee3bd4b4078ce4deef97efe2163adc66082","bytes":4032},{"path":"voice/Male2/vqk.yes.WAV","sha":"1a25ca44d7a41a6b3ed6481912b480432c8a12d5","bytes":6848},{"path":"voice/Male2/wrn.bomber.WAV","sha":"b9fb19ad3e10b00f70d5567196c961e9e0d76c30","bytes":17244},{"path":"voice/Male2/wrn.enemy.WAV","sha":"36d55683bac60ddfd22b7e51b8036398fa640b5d","bytes":20090},{"path":"voice/Male2/wrn.vehicles.WAV","sha":"1e84d8a658cdc3ce5fd50cb7def5b5e124a6026d","bytes":18926},{"path":"voice/Male2/wrn.watchit.WAV","sha":"b519e36552ded8e92bb5d7e70be20547835cf52b","bytes":17452},{"path":"voice/Male3/att.attack.WAV","sha":"61c1b79fa28f0c35700caffa58442cd59691e7af","bytes":10524},{"path":"voice/Male3/att.base.WAV","sha":"fbd6a8f0ee7e589ac77328757201a5ec4916633b","bytes":18342},{"path":"voice/Male3/att.chase.WAV","sha":"e773154888683506e234831d9711fa1464375eaa","bytes":15020},{"path":"voice/Male3/att.distract.WAV","sha":"8ff3457a1fc3bbbc0018a75559e5e1ac5d9b98b1","bytes":19766},{"path":"voice/Male3/att.flag.WAV","sha":"efcec7f377ade0191e1760cbd869d1019f04113f","bytes":15766},{"path":"voice/Male3/att.generator.WAV","sha":"dcde0a231df8541445e6d7450fbc57a4c8c68470","bytes":22412},{"path":"voice/Male3/att.objective.WAV","sha":"35839af1551bdea983d3538ed6a495b027e27179","bytes":18498},{"path":"voice/Male3/att.reinforcements.WAV","sha":"9698345c8a267406cc0f96229652e7779df7c0fb","bytes":19730},{"path":"voice/Male3/att.sensors.WAV","sha":"99ef711b3c512207d39361c78f5c2b00d87d913a","bytes":22022},{"path":"voice/Male3/att.turrets.WAV","sha":"a1553a2fcd589f797e361200cc27385e2f807526","bytes":21476},{"path":"voice/Male3/att.vehicle.WAV","sha":"43c5a4133ead6fdf18e242ad2809a077c7744299","bytes":19956},{"path":"voice/Male3/att.wait.WAV","sha":"2980983dc9d03ed83e355cedc34b74a680c96f94","bytes":27364},{"path":"voice/Male3/avo.deathcry_01.WAV","sha":"007599f5026b8cadf767f0c692f3b7aa6bca7e5f","bytes":13380},{"path":"voice/Male3/avo.deathcry_02.WAV","sha":"5c4f75abf2e3d66832c9ae3c47b49ba9563d9674","bytes":15420},{"path":"voice/Male3/avo.grunt.WAV","sha":"b19cdac1a151aa83ea31fd1dc243d8458923da0e","bytes":2770},{"path":"voice/Male3/avo.pain.WAV","sha":"b80c8c237b97eb9f0d23528774f34b5a86ba2576","bytes":6304},{"path":"voice/Male3/bas.clear.WAV","sha":"86e071fda68c74ec25acdaf4dbd3d61d47172212","bytes":17704},{"path":"voice/Male3/bas.enemy.WAV","sha":"5837394e5cc16d1bebabdb6223b0cd2e8744ae24","bytes":19504},{"path":"voice/Male3/bas.retake.WAV","sha":"baeb32cf8c7029381f3b3e5b5dbbf53338dd5b21","bytes":17418},{"path":"voice/Male3/bas.secure.WAV","sha":"8fadf7db37cc8701176d1544b83e79b4d6308c24","bytes":16474},{"path":"voice/Male3/bas.taken.WAV","sha":"067daa016cbf41981574650ed4792048a6a0a201","bytes":16764},{"path":"voice/Male3/cmd.acknowledge.WAV","sha":"b5a6872d452c85698aefa8b348b563b31f1434cb","bytes":18288},{"path":"voice/Male3/cmd.bot.WAV","sha":"21cc9ce188cac60d8456bacbffa4cef22155ebd2","bytes":18928},{"path":"voice/Male3/cmd.completed.WAV","sha":"46951f3dbf32dc675fb0ae3c3b295477077794b3","bytes":20122},{"path":"voice/Male3/cmd.decline.WAV","sha":"47de4bc0b75463aa2c66d5e91483420419d7d7b7","bytes":23436},{"path":"voice/Male3/def.base.WAV","sha":"db9b89d66520a5fc8f513e8536647c8d29681486","bytes":16792},{"path":"voice/Male3/def.carrier.WAV","sha":"53345278fd9366b692ef0c54013c5a9e6247a9cd","bytes":19516},{"path":"voice/Male3/def.entrances.WAV","sha":"0aa7f99bd2d34a438665dbd121a66ec60407b2f2","bytes":22154},{"path":"voice/Male3/def.flag.WAV","sha":"1161d858704215cf2567c2f3ee6b5e3a0b0641c8","bytes":18700},{"path":"voice/Male3/def.generator.WAV","sha":"c1907dd9b750500a4defab141e615182156958a5","bytes":21720},{"path":"voice/Male3/def.me.WAV","sha":"63fd52b666b1a6a2db5e83028e473b42ed2f56c2","bytes":10080},{"path":"voice/Male3/def.nexus.WAV","sha":"854fc31b422341c7fe03ba7d4b8d995d61cb45ab","bytes":21942},{"path":"voice/Male3/def.objective.WAV","sha":"e02b3e77fa781679a5278045ebcfea667e79dcd5","bytes":16228},{"path":"voice/Male3/def.reinforce.WAV","sha":"7fdd49362baf05e3dd413b7a882c52ae06269e64","bytes":21730},{"path":"voice/Male3/def.sensors.WAV","sha":"55d2ba5ad872989966354df5a48f6ee0b4721ab6","bytes":16594},{"path":"voice/Male3/def.turrets.WAV","sha":"da78f93e5b0b9dff624dc3301dedc9535bef04ab","bytes":21552},{"path":"voice/Male3/def.vehicle.WAV","sha":"8525e9df43a448ef6a3ede7ea2c38c12eb6f797b","bytes":15252},{"path":"voice/Male3/ene.base.WAV","sha":"d267f95528b4b041848a022000f7d2c08de23231","bytes":26964},{"path":"voice/Male3/ene.disarray.WAV","sha":"5830cc6395e4a5e98648f8ade0d5d85b26407d6e","bytes":31086},{"path":"voice/Male3/ene.generator.WAV","sha":"c282f789a46741c0c3d373ba63824428ef4b9600","bytes":25370},{"path":"voice/Male3/ene.remotes.WAV","sha":"8642d65d9b2c6f9aace48f62a015cf26224291a6","bytes":28516},{"path":"voice/Male3/ene.sensors.WAV","sha":"67b8c4781f677c90c98dc6986516a6886d737f65","bytes":23458},{"path":"voice/Male3/ene.turrets.WAV","sha":"1b41b8be1a9cb9b0d0bbf97e4fe668122e8edd2c","bytes":24450},{"path":"voice/Male3/ene.vehicle.WAV","sha":"99d909c4707f2fe53bba1b74b0834bb819520861","bytes":32062},{"path":"voice/Male3/flg.flag.WAV","sha":"77d56d1fdbf7262f511748fa272be2a7499841bc","bytes":23150},{"path":"voice/Male3/flg.give.WAV","sha":"8a812fd54ce2239587a9f0550aaa46283d630945","bytes":14238},{"path":"voice/Male3/flg.huntergive.WAV","sha":"401dd74a14f317c5e5bdfd1d6848250f866d6a69","bytes":25422},{"path":"voice/Male3/flg.huntertake.WAV","sha":"619adfc4df3cf999c16b05dedf74d14572d3e192","bytes":19840},{"path":"voice/Male3/flg.retrieve.WAV","sha":"4f5d27b66fd0317bcea1dbef5e391eb677cc7ea8","bytes":15884},{"path":"voice/Male3/flg.secure.WAV","sha":"ab0d5a5206e6d596c58d52d959c63b18072221dd","bytes":23840},{"path":"voice/Male3/flg.take.WAV","sha":"f44aaeb700cfa9b2d8a54d0f6e640991c97f30ee","bytes":22012},{"path":"voice/Male3/gbl.anytime.WAV","sha":"ba43036c0cb15e3bfb7ba4e0f3945fe26ed98f0e","bytes":23908},{"path":"voice/Male3/gbl.awesome.WAV","sha":"d3bdf3cd10a95c467a7ff9e1f1338ed35ae78a44","bytes":14136},{"path":"voice/Male3/gbl.aww.WAV","sha":"e2f86278a4f8fe5af3de3f9e17f67236bebc6c5a","bytes":50520},{"path":"voice/Male3/gbl.brag.WAV","sha":"811721e5db3d4d6b3d1c8b2aeb16cad66f977d29","bytes":29676},{"path":"voice/Male3/gbl.bye.WAV","sha":"a09196996805846df046153a6e0ca906b48a1c9e","bytes":10862},{"path":"voice/Male3/gbl.dunno.WAV","sha":"e8998e33d36033b1d129697f01b2f4654c834516","bytes":18198},{"path":"voice/Male3/gbl.goodgame.WAV","sha":"d8fcac4d2c5ff0ecc6ff41f70f773a463fa6a1b5","bytes":12090},{"path":"voice/Male3/gbl.hi.WAV","sha":"39dd1a29974de7ca8f2189d7ff736acb9c3d08ff","bytes":18276},{"path":"voice/Male3/gbl.nice.WAV","sha":"30b659a4685cfea83d75a8e13f64bcebace7c61c","bytes":25502},{"path":"voice/Male3/gbl.no.WAV","sha":"18314808631ad3744f7156990dd7f129f11f36f5","bytes":9324},{"path":"voice/Male3/gbl.obnoxious.WAV","sha":"218afba4df978fe718b131a11e375281c801650a","bytes":26220},{"path":"voice/Male3/gbl.oops.WAV","sha":"3c149c32066a48ac68983478bf6af89bbe34f3bd","bytes":12388},{"path":"voice/Male3/gbl.quiet.WAV","sha":"c2d19c9408fa22e821934c243f075d173a8b25ed","bytes":11886},{"path":"voice/Male3/gbl.rock.WAV","sha":"191956b911a1f1a00d70d0a8936b431adb3d6257","bytes":24602},{"path":"voice/Male3/gbl.sarcasm.WAV","sha":"89e1389c3faf1c31cbaeda577d73811d8ec10d8d","bytes":26040},{"path":"voice/Male3/gbl.shazbot.WAV","sha":"fee0d0ee8637db02639557ba6b1f1dca2a0a2ace","bytes":18478},{"path":"voice/Male3/gbl.shooting.WAV","sha":"0b026c43da8bf7f11030086f3091e4c831da7989","bytes":12810},{"path":"voice/Male3/gbl.thanks.WAV","sha":"35352ca3f9276668dadbb41837254b3f892ab012","bytes":12392},{"path":"voice/Male3/gbl.wait.WAV","sha":"c3a293a3177af9f88df125cd29cf4fc6f1f92442","bytes":19970},{"path":"voice/Male3/gbl.when.WAV","sha":"36b6e1abbd57a83c9b96e1f84c9e58fdf6f6f5cf","bytes":30650},{"path":"voice/Male3/gbl.woohoo.WAV","sha":"c30a814de43de183e0627920fa3c10b7e90dbd5e","bytes":25566},{"path":"voice/Male3/gbl.yes.WAV","sha":"44517a4ea1ba56a66aee60a08aaf7625e3c2b9b2","bytes":24864},{"path":"voice/Male3/need.bombardier.WAV","sha":"cd4ff851c6aa5e3f644f34681adb8ce1eccb861d","bytes":16238},{"path":"voice/Male3/need.cover.WAV","sha":"f38e32fb3978e4e8f8e102086c52c471bb3e9fe1","bytes":22444},{"path":"voice/Male3/need.driver.WAV","sha":"ca0e4c328320d21373ad6a067a237394404661a0","bytes":22760},{"path":"voice/Male3/need.escort.WAV","sha":"95b3f663c231bab6f15fc49c67015e30844a795b","bytes":20026},{"path":"voice/Male3/need.flyer.WAV","sha":"13432e73f952c69249ae0446f21653bc94ac6216","bytes":21754},{"path":"voice/Male3/need.gunship.WAV","sha":"9633c78393bb16a8c2aa60cebafb6bfb21e08876","bytes":24346},{"path":"voice/Male3/need.hold.WAV","sha":"cacf92ec6331f52842e5e709e0030e7722b12d69","bytes":25064},{"path":"voice/Male3/need.ride.WAV","sha":"eeaff10b90ce98dca18dbc90518fb26baceac9a2","bytes":15130},{"path":"voice/Male3/need.support.WAV","sha":"c40e416177c92c581abf8c890a7c48a9ef1177c4","bytes":19442},{"path":"voice/Male3/need.tailgunner.WAV","sha":"08c12affa629d8b6705c6eb34cb796f48b2f260a","bytes":18338},{"path":"voice/Male3/need.where.WAV","sha":"bb382667deee123d0ae920fea2d64ab4144d1ebf","bytes":10316},{"path":"voice/Male3/rep.base.WAV","sha":"aa3643ecf04117ea7c08cce162e4621dc3138b02","bytes":18776},{"path":"voice/Male3/rep.generator.WAV","sha":"0cd47df4c4ea1515d4c986d33abc7773a3690ebc","bytes":19740},{"path":"voice/Male3/rep.me.WAV","sha":"1bfeff4fd70ed97403d279ab7c6bb4f598633e97","bytes":15566},{"path":"voice/Male3/rep.sensors.WAV","sha":"b6896578cf0c6086dfa18e4ebdfc95ab6b08b593","bytes":17066},{"path":"voice/Male3/rep.turrets.WAV","sha":"e6163c6e35ed72b00b62371a4c94b33d64a9525d","bytes":16850},{"path":"voice/Male3/rep.vehicle.WAV","sha":"c5db77573ba9e638db6726efb2dfea459e12af08","bytes":21040},{"path":"voice/Male3/slf.att.attack.WAV","sha":"8b472bc6784963ec443610a2a680946134a2bc01","bytes":12028},{"path":"voice/Male3/slf.att.base.WAV","sha":"c130dc0270b8d1ded98c4cedf6c944b74883fb88","bytes":20728},{"path":"voice/Male3/slf.att.flag.WAV","sha":"9a4fd5fa49de4e53bc1cb692888246b2d282146c","bytes":19210},{"path":"voice/Male3/slf.att.generator.WAV","sha":"0ff7c27d5167bac83e7816892cf26cc8eb272fb7","bytes":21294},{"path":"voice/Male3/slf.att.sensors.WAV","sha":"42ed92e8c38a76a5d998d3303b43512c1d3330cf","bytes":22964},{"path":"voice/Male3/slf.att.turrets.WAV","sha":"f07a508512c8158fe71fb30a77922e2fe554850a","bytes":21842},{"path":"voice/Male3/slf.att.vehicle.WAV","sha":"59de91c3b85a8aa1edb06a40922d68874d33d529","bytes":24924},{"path":"voice/Male3/slf.def.base.WAV","sha":"48563983d83f0ef4ce2f97339fc6dc3b5871b8a9","bytes":17802},{"path":"voice/Male3/slf.def.defend.WAV","sha":"32860bea00ea2c62a1be9ac607fc56744dd17fc2","bytes":14840},{"path":"voice/Male3/slf.def.flag.WAV","sha":"02b8db6f522d774cce8e8b2f32bb0bcaa2634f82","bytes":19764},{"path":"voice/Male3/slf.def.generator.WAV","sha":"d575e9c5c23fa61d03e2e417e2cb762b1ac7a02f","bytes":18608},{"path":"voice/Male3/slf.def.nexus.WAV","sha":"f27b6bfc76c3e03ed496b54a0bbc07e6cd8c2c57","bytes":18250},{"path":"voice/Male3/slf.def.sensors.WAV","sha":"6e9693c2c4e5678e7adfb2fe6d59733dbe94e39e","bytes":18944},{"path":"voice/Male3/slf.def.turrets.WAV","sha":"0847d8ffbd97cd0bbe91af1defacf67fb0ed789d","bytes":18970},{"path":"voice/Male3/slf.def.vehicle.WAV","sha":"e3c457e89b34a26f7473c81fdab7ca1f71c46d78","bytes":20186},{"path":"voice/Male3/slf.rep.base.WAV","sha":"0575eeb1084d6d7a5deb05235be356e261c6732d","bytes":18240},{"path":"voice/Male3/slf.rep.equipment.WAV","sha":"507dcfb16aea7885b4e0fec0e9d04bbf65c3e287","bytes":19994},{"path":"voice/Male3/slf.rep.generator.WAV","sha":"d55b935c0e574897d1568f7dca0d5dc34cdfec28","bytes":19422},{"path":"voice/Male3/slf.rep.repairing.WAV","sha":"3e3c02f1ec83475141b2cdd649a2eba4c4b1e059","bytes":18864},{"path":"voice/Male3/slf.rep.sensors.WAV","sha":"e05ac9a04f994a632b4e299ebafe8184b18b5f54","bytes":18080},{"path":"voice/Male3/slf.rep.turrets.WAV","sha":"d064ab204a1c5bbbcd16f1b7ce00d4a69f6520f4","bytes":17630},{"path":"voice/Male3/slf.rep.vehicle.WAV","sha":"83dca9e4ad246c000c5b8741884beb965cf40f33","bytes":24448},{"path":"voice/Male3/slf.tsk.cover.WAV","sha":"e08e795e3b0f4186f0cd857bb98f37ca8a348180","bytes":13462},{"path":"voice/Male3/slf.tsk.defense.WAV","sha":"eefbe3931ecd8d26ecaadddc51556e7936e9df2f","bytes":19292},{"path":"voice/Male3/slf.tsk.generic.WAV","sha":"af96ffacc1f5d1ce22a3b7c8853414e83bbfd443","bytes":11648},{"path":"voice/Male3/slf.tsk.remotes.WAV","sha":"ff1c4a30355f23ed6067d456e1d122139e85f2ac","bytes":21350},{"path":"voice/Male3/slf.tsk.sensors.WAV","sha":"fdda6eb9c1b55258edbb791231d2e1f78ebf0047","bytes":18154},{"path":"voice/Male3/slf.tsk.turrets.WAV","sha":"d7511f0662fc9b66ded12a74702cf661f48560c7","bytes":18680},{"path":"voice/Male3/slf.tsk.vehicle.WAV","sha":"a4d3fa2139123a70c63e0fa7f54030c376c863c8","bytes":17130},{"path":"voice/Male3/tgt.acquired.WAV","sha":"63ab19b66ef5fe7bc45f0054962c42609e9c6e12","bytes":19882},{"path":"voice/Male3/tgt.base.WAV","sha":"2ad952beb6cd8999cda3b93bbacd1c67b3d04311","bytes":36250},{"path":"voice/Male3/tgt.destroyed.WAV","sha":"bc16aaf44f10da81d7f9c5a4668292a9ce1f12b3","bytes":25160},{"path":"voice/Male3/tgt.flag.WAV","sha":"5c8c2abae3d1d82925e73f8bbb95a034c97b5ebf","bytes":25908},{"path":"voice/Male3/tgt.my.WAV","sha":"53facbaf723e3a57c4ed591359bab96be2adf9c4","bytes":14904},{"path":"voice/Male3/tgt.need.WAV","sha":"6363b319287a8be1dd39829e860d18f9a5be0313","bytes":18408},{"path":"voice/Male3/tgt.sensors.WAV","sha":"510d2eabe8590ac7e50b5000974d1d9b45fbd98c","bytes":33072},{"path":"voice/Male3/tgt.turret.WAV","sha":"f3c1d56e9ce3ecdc894cdd91a67668991f159480","bytes":32840},{"path":"voice/Male3/tgt.wait.WAV","sha":"8615bdaa05362c7d71bbfb88882709058b45aea4","bytes":31560},{"path":"voice/Male3/vqk.anytime.WAV","sha":"010329b1d2b78345e61b0da1146a019a1f6544ec","bytes":13976},{"path":"voice/Male3/vqk.base.WAV","sha":"57619a1984a73ce2b92abafaffa3cae3beb9445b","bytes":13670},{"path":"voice/Male3/vqk.ceasefire.WAV","sha":"63be8f65ae4c60182c4a65f364b1a2f02a16b518","bytes":14700},{"path":"voice/Male3/vqk.dunno.WAV","sha":"cb69e1e63c54f44462818d0144df8a9b81c05529","bytes":10178},{"path":"voice/Male3/vqk.help.WAV","sha":"da381e2a84f0c8422bbbfe4193d4d408eefc1453","bytes":6068},{"path":"voice/Male3/vqk.move.WAV","sha":"4666e56d6d2ec4af8f31fbf1ff93a18022e0f5ef","bytes":15408},{"path":"voice/Male3/vqk.no.WAV","sha":"099f89ec1d2819451d69049717e94a616fff0971","bytes":9664},{"path":"voice/Male3/vqk.quiet.WAV","sha":"474456dc01193905bc2ea55521f5e6c197b998a5","bytes":14470},{"path":"voice/Male3/vqk.sorry.WAV","sha":"a3917ca96ed9ce9b9d521562d47196fadd3041c6","bytes":6098},{"path":"voice/Male3/vqk.thanks.WAV","sha":"3c982447ae7f7fc63e0f7178ba103a4a41cd0e23","bytes":9126},{"path":"voice/Male3/vqk.wait.WAV","sha":"c408e0a1e2a754e6b8bfc4e87cf93f6b88030757","bytes":13298},{"path":"voice/Male3/vqk.yes.WAV","sha":"26aca9cc568a15c1f9dcab3951afc16c3e6c2180","bytes":13518},{"path":"voice/Male3/wrn.bomber.WAV","sha":"51d177c024c27731d4ae9b1b960fd1b12a9352e8","bytes":16898},{"path":"voice/Male3/wrn.enemy.WAV","sha":"7b8120c54b1c19a5efe96352d5141bb17ac3d971","bytes":17562},{"path":"voice/Male3/wrn.vehicles.WAV","sha":"11cde6c29ce051e681ac9f2412d107f0950f13a1","bytes":22640},{"path":"voice/Male3/wrn.watchit.WAV","sha":"de70c18ed1582f649c61bd4c2f8f9fe217828a76","bytes":20980},{"path":"voice/Male4/att.attack.WAV","sha":"1d804f699942ffb9bf35ae215e373dd5b92a8137","bytes":13982},{"path":"voice/Male4/att.base.WAV","sha":"8cc54c726f315ba3d859dc475205ca2db848591d","bytes":19996},{"path":"voice/Male4/att.chase.WAV","sha":"75a658aa97ab1f6de320b723267f7161a7c77971","bytes":20228},{"path":"voice/Male4/att.distract.WAV","sha":"de5041fa5079776e6fb320b3005640cb7a640e69","bytes":21004},{"path":"voice/Male4/att.flag.WAV","sha":"625eeb123ad9838bf897b98e17c906a131baf25c","bytes":21178},{"path":"voice/Male4/att.generator.WAV","sha":"34934f699cae97206538bdab91bae3881f14a678","bytes":23620},{"path":"voice/Male4/att.objective.WAV","sha":"9124e7845ca445f86fd29e3cb5bf5cff3b501be6","bytes":19744},{"path":"voice/Male4/att.reinforcements.WAV","sha":"e22521b6b682ac37a48aaee6644a6bbf31b1daa0","bytes":20086},{"path":"voice/Male4/att.sensors.WAV","sha":"2d0ae35797822ee1b5d8ef2226e1153c28907da9","bytes":23254},{"path":"voice/Male4/att.turrets.WAV","sha":"fdd506e3f7d7d794e95c82b278b36e13541e75f6","bytes":21258},{"path":"voice/Male4/att.vehicle.WAV","sha":"f03d93eec8e986096a373babc83fecab46b4e115","bytes":21030},{"path":"voice/Male4/att.wait.WAV","sha":"abc2a584f76cdc2f97cf7c0e49805f714cbad834","bytes":24968},{"path":"voice/Male4/avo.deathcry_01.WAV","sha":"76e7f58972474ecaacaa352f69d816dd1ead31f6","bytes":15940},{"path":"voice/Male4/avo.deathcry_02.WAV","sha":"180b95222357b99e915fafc1cf4dd47327e189ec","bytes":14908},{"path":"voice/Male4/avo.grunt.WAV","sha":"6891973f3d7831958e052f967190fa4797a099cb","bytes":4690},{"path":"voice/Male4/avo.pain.WAV","sha":"674804a317170b4f59b3e470932ae6ddb1226cde","bytes":6522},{"path":"voice/Male4/bas.clear.WAV","sha":"e5c1bd87a204d981f2422d07fb2e4f27dbe336bd","bytes":16940},{"path":"voice/Male4/bas.enemy.WAV","sha":"9eeee130e2d4642404ededb543829caee7925143","bytes":18442},{"path":"voice/Male4/bas.retake.WAV","sha":"be33583e8db7d50e6b453f109891808b748b8cbb","bytes":16990},{"path":"voice/Male4/bas.secure.WAV","sha":"3b83a1da8c6b9f9360dbf5e57ca90082906754ea","bytes":17028},{"path":"voice/Male4/bas.taken.WAV","sha":"a6b5d16a30a4aa9af682597233045dc05b29efd7","bytes":18968},{"path":"voice/Male4/cmd.acknowledge.WAV","sha":"87c1c529b854c9a5e03a6fe4f3e2ecbd0cf6cdf3","bytes":13708},{"path":"voice/Male4/cmd.bot.WAV","sha":"d4ffe3157f9baf89ef68c1928fad4812653471a6","bytes":15106},{"path":"voice/Male4/cmd.completed.WAV","sha":"dcb77dc4562812473e63ddd73d6e83a9d1778bf5","bytes":13384},{"path":"voice/Male4/cmd.decline.WAV","sha":"cc11af5084753c068b5ce4dffd10212860b65d57","bytes":16232},{"path":"voice/Male4/def.base.WAV","sha":"50b1320e08559ac5701009c67beb50f8ea86693f","bytes":17818},{"path":"voice/Male4/def.carrier.WAV","sha":"235817371462655f23e381dd06846ed8ab2f5527","bytes":19180},{"path":"voice/Male4/def.entrances.WAV","sha":"ee58c17783b52ba9c8de68c72ffcd07498fb8fd5","bytes":18024},{"path":"voice/Male4/def.flag.WAV","sha":"229c062cb46332ee5ab349b232d81c40140a2125","bytes":18122},{"path":"voice/Male4/def.generator.WAV","sha":"561b172307e7c37598405d1bc4818b9ea35f3bb0","bytes":16776},{"path":"voice/Male4/def.me.WAV","sha":"c32eee3214cf8894ac39754c5df5111c64059cb1","bytes":11176},{"path":"voice/Male4/def.nexus.WAV","sha":"bf34463d034856cfa952a90b8788baf084f2e6bb","bytes":17618},{"path":"voice/Male4/def.objective.WAV","sha":"0ae33df2bc8bf70ef21000eded5aacc29f108dfe","bytes":18272},{"path":"voice/Male4/def.reinforce.WAV","sha":"dcb5d5ec003abb650d7c9766221ba758365fe579","bytes":18842},{"path":"voice/Male4/def.sensors.WAV","sha":"87a7980fcb952a937918db642a1e53c46895837a","bytes":18376},{"path":"voice/Male4/def.turrets.WAV","sha":"41a654cd19550406b7365c04bd8f489c1d391ccc","bytes":17324},{"path":"voice/Male4/def.vehicle.WAV","sha":"86f581616f36023c82cd6be8b2c274c1d2ad9469","bytes":17392},{"path":"voice/Male4/ene.base.WAV","sha":"c8b095361a8c03c91d4c228b4e67bf8833108f53","bytes":21562},{"path":"voice/Male4/ene.disarray.WAV","sha":"e5f42f1180a80deeacf20f6be496abc9f40a4051","bytes":26344},{"path":"voice/Male4/ene.generator.WAV","sha":"43fbcff0e655bd88b4e8d2b647a094c44fb2074c","bytes":23858},{"path":"voice/Male4/ene.remotes.WAV","sha":"03f87e871ac76f2da74e400970325c941244f81d","bytes":28884},{"path":"voice/Male4/ene.sensors.WAV","sha":"f756bbe7efb8c8461da76de0dd040ef2a0fa3f78","bytes":21666},{"path":"voice/Male4/ene.turrets.WAV","sha":"6df626187bc44e408da6d7381e49557bab049612","bytes":19218},{"path":"voice/Male4/ene.vehicle.WAV","sha":"99a0b90d261305b5ddd2d01b2f13823fb2badff2","bytes":23378},{"path":"voice/Male4/flg.flag.WAV","sha":"97191400965511dcbbe9fbb9d340b3e4131376b4","bytes":20328},{"path":"voice/Male4/flg.give.WAV","sha":"9ea900b0be820cffd2d101eb1f756d54aeb1a8c1","bytes":15160},{"path":"voice/Male4/flg.huntergive.WAV","sha":"2b8e535757e1eac0d1589ae1c131d499c6733bd2","bytes":18096},{"path":"voice/Male4/flg.huntertake.WAV","sha":"db5b70e06e8232387f1253f2fb44be7760e4dfeb","bytes":17722},{"path":"voice/Male4/flg.retrieve.WAV","sha":"2d5ec092f4ff4f9864a882ece5eebb90f96fa251","bytes":19494},{"path":"voice/Male4/flg.secure.WAV","sha":"51a107f1add54d2c2ae5bc0041045e645496c01d","bytes":17592},{"path":"voice/Male4/flg.take.WAV","sha":"49cba85c0d7826a31da2bc37a6002834e2301866","bytes":16842},{"path":"voice/Male4/gbl.anytime.WAV","sha":"687327ff765953d2d477ff5bbebcff773f3813cd","bytes":13540},{"path":"voice/Male4/gbl.awesome.WAV","sha":"631d752bd8bbb6cff7139360a3da88844167fa77","bytes":11606},{"path":"voice/Male4/gbl.aww.WAV","sha":"2394e5c3378923d62f72c4b1b1062d5b4f202ce2","bytes":30768},{"path":"voice/Male4/gbl.brag.WAV","sha":"ca97f7f46b67517046b38687177ed97606d4e7c6","bytes":21534},{"path":"voice/Male4/gbl.bye.WAV","sha":"6678ceb437adc6b281c7c4a2815a0d9c8924c3a6","bytes":10154},{"path":"voice/Male4/gbl.dunno.WAV","sha":"8647f8b23d203e5bbcf1247ae0dd0a9e8f2d7501","bytes":12104},{"path":"voice/Male4/gbl.goodgame.WAV","sha":"c908c2157613ed025d6009904c12a0f3865956d3","bytes":15054},{"path":"voice/Male4/gbl.hi.WAV","sha":"0dec98b83f67ffc702c0934a070e202e63df8e94","bytes":10712},{"path":"voice/Male4/gbl.nice.WAV","sha":"ec85205b8adeb781c2b9d0692f4c970204680be4","bytes":18442},{"path":"voice/Male4/gbl.no.WAV","sha":"082308d4bfd740871ce1c6c41b4eef46ac6f4b70","bytes":10308},{"path":"voice/Male4/gbl.obnoxious.WAV","sha":"6d7b61d24a1e7fd1aa4a062f5c17105960737127","bytes":27610},{"path":"voice/Male4/gbl.oops.WAV","sha":"e9064067f5a1e3b203e6d873891200458d98a510","bytes":12348},{"path":"voice/Male4/gbl.quiet.WAV","sha":"67ab91e26b88a250e8faaffa752fed0ae7889d3d","bytes":11330},{"path":"voice/Male4/gbl.rock.WAV","sha":"ff83163f2a05ba69e8b3fbc223ff13d367c5dc1e","bytes":22160},{"path":"voice/Male4/gbl.sarcasm.WAV","sha":"55fff90a6a58453b78c7c9ddc97a79efc3852f0f","bytes":21260},{"path":"voice/Male4/gbl.shazbot.WAV","sha":"b07592362b62426db3b0b15b8a1806ef55fa4fd1","bytes":16912},{"path":"voice/Male4/gbl.shooting.WAV","sha":"fd73cb9ecc661448d3de5e51355f21a6db15e70c","bytes":17398},{"path":"voice/Male4/gbl.thanks.WAV","sha":"3137cd0c0c7a263d928d3b2a5b22069c7e7afef8","bytes":11092},{"path":"voice/Male4/gbl.wait.WAV","sha":"aeaea452e17fbfb0b872a0557ee72f2f3592cbf9","bytes":14038},{"path":"voice/Male4/gbl.when.WAV","sha":"71b08c8bf29cb03768d0badf2558e986d3000401","bytes":22464},{"path":"voice/Male4/gbl.woohoo.WAV","sha":"357768656db8bdbc93002979c0e9f8c9998405bb","bytes":20338},{"path":"voice/Male4/gbl.yes.WAV","sha":"ab6cacf0a5ca60c3100db24427f68d4e157dbe44","bytes":12960},{"path":"voice/Male4/need.bombardier.WAV","sha":"7c37cf64f2984fe750998010a96ffc58a66c5ead","bytes":17486},{"path":"voice/Male4/need.cover.WAV","sha":"166612240f5558a0213895ad5eed2e37d730a914","bytes":17708},{"path":"voice/Male4/need.driver.WAV","sha":"9ba59c6aa26aa279f816e846753c0921866eb39e","bytes":21222},{"path":"voice/Male4/need.escort.WAV","sha":"0e2af1fc21a1ae4bcaf2c1046ae19edbbda7e06a","bytes":17554},{"path":"voice/Male4/need.flyer.WAV","sha":"af217cbce644c7cd617376dc54fdeb658ed078c9","bytes":21800},{"path":"voice/Male4/need.gunship.WAV","sha":"ed698fd45146de13d2847e78488622455ddafc13","bytes":23540},{"path":"voice/Male4/need.hold.WAV","sha":"0051f5550f88f1859f91d3c816a8567e2343a792","bytes":20550},{"path":"voice/Male4/need.ride.WAV","sha":"4b4b2292a8e236aec25bf01c28138f2de13f36e8","bytes":21848},{"path":"voice/Male4/need.support.WAV","sha":"7b90520d4527cedc7b7ff0b81ffc6a8b0002dea2","bytes":18006},{"path":"voice/Male4/need.tailgunner.WAV","sha":"4b7ef5ab9054a6cef12622854194d26c864d9625","bytes":14826},{"path":"voice/Male4/need.where.WAV","sha":"c938604ecafa3a8ff0a5ca027c37dd30a2fa8bf0","bytes":13232},{"path":"voice/Male4/rep.base.WAV","sha":"044f420cbc4deccc9dbcb9717c96da3fd27bf3f0","bytes":15098},{"path":"voice/Male4/rep.generator.WAV","sha":"98e6ccfa9c0808ed801b0985e8109a61c63d84ca","bytes":17248},{"path":"voice/Male4/rep.me.WAV","sha":"d87cbce4d12ad67b54fa08940c1109738f82995a","bytes":13684},{"path":"voice/Male4/rep.sensors.WAV","sha":"ee2b3d4cdae1005e90df45f1d1443e6b1832688d","bytes":16878},{"path":"voice/Male4/rep.turrets.WAV","sha":"d298eefaf8fda03c1259fe1a50028aa698850b92","bytes":16442},{"path":"voice/Male4/rep.vehicle.WAV","sha":"09b828428b65c31f2869a8348ee51675ce3b0950","bytes":20516},{"path":"voice/Male4/slf.att.attack.WAV","sha":"39c8d4700343da64713819b9e7a34e66de897e76","bytes":12746},{"path":"voice/Male4/slf.att.base.WAV","sha":"63b614a9187a4134c176d2960362ff1722aaeec6","bytes":19312},{"path":"voice/Male4/slf.att.flag.WAV","sha":"1e3177871fe0a4756b0ca048d87574e8362881a7","bytes":20858},{"path":"voice/Male4/slf.att.generator.WAV","sha":"110e6832a079871d3bf5d9c6e82083eb13263ea2","bytes":20374},{"path":"voice/Male4/slf.att.sensors.WAV","sha":"e2551bd4100cbe8e47f03f15e8370e080ff317e2","bytes":20780},{"path":"voice/Male4/slf.att.turrets.WAV","sha":"cf882cdd4c1f194e3d8deec83851367fc4a4a295","bytes":21504},{"path":"voice/Male4/slf.att.vehicles.WAV","sha":"f68233738fc70b73f66d6722e60b550570902055","bytes":23840},{"path":"voice/Male4/slf.def.base.WAV","sha":"cf257844afcea4af4c0d0df4a6c8c9351b6cb545","bytes":17970},{"path":"voice/Male4/slf.def.defend.WAV","sha":"d2f76c9fc7c7f489e3a1204a00840b911bcf9975","bytes":13628},{"path":"voice/Male4/slf.def.flag.WAV","sha":"d2313ad703408e5484c808f3e6c703f3435186e9","bytes":19996},{"path":"voice/Male4/slf.def.generator.WAV","sha":"19778a2a911c75d449c8e1225e8813e328bb26b2","bytes":17960},{"path":"voice/Male4/slf.def.nexus.WAV","sha":"0da740973f96d3c2d6130049350e1e3afedceec1","bytes":18838},{"path":"voice/Male4/slf.def.sensors.WAV","sha":"cb1a2ce87d021d0baebddaeb37028ef7b238fefd","bytes":18214},{"path":"voice/Male4/slf.def.turrets.WAV","sha":"71e5de95e1f13c040385209bc2bcffed6ffc3e9d","bytes":18354},{"path":"voice/Male4/slf.def.vehicle.WAV","sha":"e9afefeb65b4fae44442a51459cff2b1f7f29d15","bytes":19646},{"path":"voice/Male4/slf.rep.base.WAV","sha":"f266064dacba78af367c38fa2501ab5b657e00e0","bytes":16388},{"path":"voice/Male4/slf.rep.equipment.WAV","sha":"1889562131d60e2edd751862cfb04b8a04c585c3","bytes":18996},{"path":"voice/Male4/slf.rep.generator.WAV","sha":"680ec48a29e3294aaf697df1f073ffa778c64e70","bytes":18150},{"path":"voice/Male4/slf.rep.repairing.WAV","sha":"06e56f581369991858fa05b4276565fecf383c76","bytes":17244},{"path":"voice/Male4/slf.rep.sensors.WAV","sha":"8fd4d1851789b28e146b63eac71bd0ed02dc817d","bytes":19856},{"path":"voice/Male4/slf.rep.turrets.WAV","sha":"3e111d00d3c7db79f35884491e078141d0017eca","bytes":19076},{"path":"voice/Male4/slf.rep.vehicle.WAV","sha":"443548998a86ff570c3ad75875d9b19595d088f3","bytes":22432},{"path":"voice/Male4/slf.tsk.cover.WAV","sha":"09266e5d6a0f2489278381786e128d79b6a6c347","bytes":14278},{"path":"voice/Male4/slf.tsk.defense.WAV","sha":"9836163139f5d553c0415f313d7527de38a139b2","bytes":17580},{"path":"voice/Male4/slf.tsk.generic.WAV","sha":"46c180b8672281fa05c7874ab3fd5cfd3439b4ec","bytes":11226},{"path":"voice/Male4/slf.tsk.remotes.WAV","sha":"fd7fd6ff1adb6ec12a447e51a5898e3f6f15cbda","bytes":20842},{"path":"voice/Male4/slf.tsk.sensors.WAV","sha":"4c52aa3ad05bf6becbf9a7f79e2f29287b5cc2e3","bytes":16996},{"path":"voice/Male4/slf.tsk.turrets.WAV","sha":"aa3619c7734b88f0ac4e350295e8e525add95f4f","bytes":16896},{"path":"voice/Male4/slf.tsk.vehicle.WAV","sha":"b6db552d5e07fc36326c7407318f1cc1607a8f35","bytes":16696},{"path":"voice/Male4/tgt.acquired.WAV","sha":"92a75b31b7e387570e50b5fe739a23a255daa3fb","bytes":17142},{"path":"voice/Male4/tgt.base.WAV","sha":"8f84917cb1807211b4a1ec0565dec8be8467c0cb","bytes":28396},{"path":"voice/Male4/tgt.destroyed.WAV","sha":"3e9e5efd6567c186200409cf2a6844dfb36d832a","bytes":25274},{"path":"voice/Male4/tgt.flag.WAV","sha":"a1a4d81b751942f6c6b26571f70e46b68fa5c996","bytes":25212},{"path":"voice/Male4/tgt.my.WAV","sha":"088b6dc0c56ca77c27fe46bbf94f2c07d742f886","bytes":16822},{"path":"voice/Male4/tgt.need.WAV","sha":"453cbde9a2f0c6c53ca90a4165df093ab5d1e727","bytes":17848},{"path":"voice/Male4/tgt.sensors.WAV","sha":"355012bd6021fc0691432b45b3dedc04da91727a","bytes":26898},{"path":"voice/Male4/tgt.turret.WAV","sha":"326cfe4c35e06ad1eb9c303bc04130a5e2e8d620","bytes":25596},{"path":"voice/Male4/tgt.wait.WAV","sha":"03dd855207563fa8aeab39ad2da778ba0e6d629a","bytes":24870},{"path":"voice/Male4/vqk.anytime.WAV","sha":"f334c4b360646563df09f9802a679def950c9519","bytes":12074},{"path":"voice/Male4/vqk.base.WAV","sha":"af8ce5aeacd85b3acf418cb10f5b2add6e66faba","bytes":14586},{"path":"voice/Male4/vqk.ceasefire.WAV","sha":"c3c8b7074d6091dd68bd39eb009e789a4a7ff4d9","bytes":9616},{"path":"voice/Male4/vqk.dunno.WAV","sha":"8c8a2a0c9f6de74c52c61f866e37feba0981e580","bytes":6878},{"path":"voice/Male4/vqk.help.WAV","sha":"8820518daa75f9975ab713a17c6b17341a8dda57","bytes":7000},{"path":"voice/Male4/vqk.move.WAV","sha":"b8c8642089a90081efc52906019f6d32755825ba","bytes":15364},{"path":"voice/Male4/vqk.no.WAV","sha":"f900f2a88b99c7e0e83049290a1458e8e4247148","bytes":11342},{"path":"voice/Male4/vqk.quiet.WAV","sha":"13a4145f296c4eb967c85079b3395ec551ea499e","bytes":10212},{"path":"voice/Male4/vqk.sorry.WAV","sha":"93cb9e73836f843d927bcac1da3bf1495629b99f","bytes":5924},{"path":"voice/Male4/vqk.thanks.WAV","sha":"ece31b7749fcdc8fc5a037cc682decae0b8507f3","bytes":6296},{"path":"voice/Male4/vqk.wait.WAV","sha":"66e7957ed7c53ec421b843b2b555a79a2a0a6043","bytes":4956},{"path":"voice/Male4/vqk.yes.WAV","sha":"a188178fc7d201fccfd01f65d47971efe06fdd42","bytes":3670},{"path":"voice/Male4/wrn.bomber.WAV","sha":"95d25753229f9dfe7c8a3fe9f800272b9274bff2","bytes":16980},{"path":"voice/Male4/wrn.enemy.WAV","sha":"7f98ce346024a9aa893300017154293c479c7406","bytes":21572},{"path":"voice/Male4/wrn.vehicles.WAV","sha":"f0561bd2027c4aab039057dd14afe781fc30c87e","bytes":21794},{"path":"voice/Male4/wrn.watchit.WAV","sha":"b5e6a7f6e5b533f2721f35cbe25cb226b1e98f94","bytes":18946},{"path":"voice/Male5/att.attack.WAV","sha":"003c3953babdbc7ec6d34cbbe2f8d06441b985eb","bytes":10842},{"path":"voice/Male5/att.base.WAV","sha":"22dc0bb04d882eb1a6ec067c6a2fdd881a604096","bytes":18158},{"path":"voice/Male5/att.chase.WAV","sha":"886d3f4af0b05f9f685e59a4ceb3c377b9a74e2f","bytes":20124},{"path":"voice/Male5/att.distract.WAV","sha":"f1fd7aa722da543c73161cd6cf1b900851afaaae","bytes":22180},{"path":"voice/Male5/att.flag.WAV","sha":"49d9d0a5a887af9a49b9dc86700133e8ac7faa97","bytes":19288},{"path":"voice/Male5/att.generator.WAV","sha":"5d04f1fbb87dda297385e6bf4ac8b4868b75e1ad","bytes":22840},{"path":"voice/Male5/att.objective.WAV","sha":"00c61457949ad5715882a6ea5fe8bb2f24c5b45b","bytes":19542},{"path":"voice/Male5/att.reinforcements.WAV","sha":"970cbdc6a7d251a3c8e50159bba67c58bead2cbd","bytes":21004},{"path":"voice/Male5/att.sensors.WAV","sha":"3b88be3a9595e4e09e7eb7ecb510f406aa0aacb3","bytes":25116},{"path":"voice/Male5/att.turrets.WAV","sha":"e58a6fdb10fa826bad8ca955a9df3b093b6bb220","bytes":23426},{"path":"voice/Male5/att.vehicle.WAV","sha":"80b6aa1ef7fe6bbe94a33867aec9653ce09b91e9","bytes":22320},{"path":"voice/Male5/att.wait.WAV","sha":"b08c3a74f5b7e86268ab35038ea6d55e678d684f","bytes":28442},{"path":"voice/Male5/avo.deathcry_01.WAV","sha":"e4ea66ebad5f316928da6510e019681e8607bfc7","bytes":11876},{"path":"voice/Male5/avo.deathcry_02.WAV","sha":"bca67e6e98e5c4b15968f88cc31a3b7db54a0d85","bytes":10300},{"path":"voice/Male5/avo.grunt.WAV","sha":"baadc539a832453d33a3ef714ab52015fe2842ea","bytes":3294},{"path":"voice/Male5/avo.pain.WAV","sha":"4421bcd9e5a9b12b77ccb03f844aad5da7c42dca","bytes":3480},{"path":"voice/Male5/bas.clear.WAV","sha":"8b86eeb1251b5a5d0262122f29d11ed0947e2618","bytes":19078},{"path":"voice/Male5/bas.enemy.WAV","sha":"0918f3cb88a5e0c2e2c354e08f2c5661a1ce10a2","bytes":19020},{"path":"voice/Male5/bas.retake.WAV","sha":"eb331ac457dcd3371fa83ba03074e60aa9a807da","bytes":20356},{"path":"voice/Male5/bas.secure.WAV","sha":"ddce9a16ca1b1f0fadff9ed86dbb820f5eca5fdb","bytes":18418},{"path":"voice/Male5/bas.taken.WAV","sha":"c67e0e95fa77c9fa7c1c6f0ff9097b35bd8eca19","bytes":19958},{"path":"voice/Male5/cmd.acknowledge.WAV","sha":"c9bac40f26912605d721ba6071e5b5e53b2d41cd","bytes":16708},{"path":"voice/Male5/cmd.bot.WAV","sha":"106e83b41731ba57b720619c509b136f15c225b8","bytes":16390},{"path":"voice/Male5/cmd.completed.WAV","sha":"c0e6b4c54dcd087278462f09794e6414de7d5f4f","bytes":16494},{"path":"voice/Male5/cmd.decline.WAV","sha":"a77012d359c416ebe3b48a0b7a9f93e314004db5","bytes":22396},{"path":"voice/Male5/def.base.WAV","sha":"c8bc70963ceeb8f07dda16cc583167827456f8a3","bytes":18908},{"path":"voice/Male5/def.carrier.WAV","sha":"dba2cef2c27ffff635cbb859dd4f263c7d0e38dd","bytes":21560},{"path":"voice/Male5/def.entrances.WAV","sha":"218484b8ffc7b7e73537d54943e694f5d4771998","bytes":16356},{"path":"voice/Male5/def.flag.WAV","sha":"f13de016533d9146229593a3464cdabb0c841574","bytes":18998},{"path":"voice/Male5/def.generator.WAV","sha":"f7fe2a647acc3acda834031ae6e4072d30ecaac2","bytes":17692},{"path":"voice/Male5/def.me.WAV","sha":"da89000faa945cfba0af2f0e4fbba387f3a83b6b","bytes":12160},{"path":"voice/Male5/def.nexus.WAV","sha":"6f20d9d206abe79e5a7fd914d4850942cb1f7cca","bytes":16816},{"path":"voice/Male5/def.objective.WAV","sha":"fe077ca34e6be67045436381a059aaf39881f567","bytes":18370},{"path":"voice/Male5/def.reinforce.WAV","sha":"ac1be945239ae6b9caa23dad8b8743c3242a932d","bytes":23592},{"path":"voice/Male5/def.sensors.WAV","sha":"e8806c58949742e87a2c7f7b64850d06918fc0c9","bytes":22760},{"path":"voice/Male5/def.turrets.WAV","sha":"4f037e56a2ffe9d0cade738d2c8db5bee103dad6","bytes":20964},{"path":"voice/Male5/def.vehicle.WAV","sha":"ad5fd1ef8701eb7d717eaa66233346cd128ad347","bytes":21792},{"path":"voice/Male5/ene.base.WAV","sha":"d41972c987dfbc037935c472c23ed9cb352b1c7d","bytes":19152},{"path":"voice/Male5/ene.disarray.WAV","sha":"29568435c22ef1fff32d909f1a90e05fb2443be0","bytes":28022},{"path":"voice/Male5/ene.generator.WAV","sha":"fa00125a571785d67da954ff7a4deba7305d12fa","bytes":19530},{"path":"voice/Male5/ene.remotes.WAV","sha":"2608e973506fa1cb3b7ccd19bb08b9624c36a357","bytes":27772},{"path":"voice/Male5/ene.sensors.WAV","sha":"3941ae94af39be7d51c1d7172a5c1d7862f2db69","bytes":25074},{"path":"voice/Male5/ene.turrets.WAV","sha":"81943da70490e89b87b33c7c9ea3fcf97e921427","bytes":19980},{"path":"voice/Male5/ene.vehicle.WAV","sha":"206520e222d2f0945ef1f95ab01a618cd6ec185f","bytes":25454},{"path":"voice/Male5/flg.flag.WAV","sha":"021820a0dd2ca707cbc919b777d3a8dd578a857b","bytes":20092},{"path":"voice/Male5/flg.give.WAV","sha":"192483826d705dea180bac73afefc5838c40090c","bytes":17090},{"path":"voice/Male5/flg.huntergive.WAV","sha":"c277d332fa147cbeaeda6c63d024fbdedab58c9e","bytes":20144},{"path":"voice/Male5/flg.huntertake.WAV","sha":"83aa8bcc94a0a8525bd48497c5bed36e8fa7f16b","bytes":16142},{"path":"voice/Male5/flg.retrieve.WAV","sha":"5d837481db1624676d45c7905771a541f72d01d4","bytes":22628},{"path":"voice/Male5/flg.secure.WAV","sha":"18eaa9c85adb1e2d5a8d72965bb70fde0ea08fab","bytes":18124},{"path":"voice/Male5/flg.take.WAV","sha":"1ad3968637144b464e7a2806616fc36d22fa66fe","bytes":17662},{"path":"voice/Male5/gbl.anytime.WAV","sha":"cde04b14ef99db2e5e2165af24c23d63b5c06a99","bytes":11754},{"path":"voice/Male5/gbl.awesome.WAV","sha":"f1032294f4f6532b684b8c413a422a79d5d0145f","bytes":18118},{"path":"voice/Male5/gbl.aww.WAV","sha":"22536d3d65b63e7611d61840d24efcfaaaf23650","bytes":31626},{"path":"voice/Male5/gbl.brag.WAV","sha":"40139b7fbc103d395cd0b996e9fd71877e5d832d","bytes":36434},{"path":"voice/Male5/gbl.bye.WAV","sha":"cd13c7141361ba13e32ca705fc6e7ff0b948d2e2","bytes":10048},{"path":"voice/Male5/gbl.dunno.WAV","sha":"735fea8abddd995399789f103675aaf4583ed451","bytes":16922},{"path":"voice/Male5/gbl.goodgame.WAV","sha":"ba4e22d79ba320cd8f85772b844ee81d1ac9a22d","bytes":17136},{"path":"voice/Male5/gbl.hi.WAV","sha":"e8bec3e1fcfdf9e964316f561cde3fa801a777dc","bytes":8790},{"path":"voice/Male5/gbl.nice.WAV","sha":"c609f06400d114f0d225f92dd3ece4ac41238fdb","bytes":22356},{"path":"voice/Male5/gbl.no.WAV","sha":"3cd0d803a79fcf58e05779b45d1d78a7a0096791","bytes":7496},{"path":"voice/Male5/gbl.obnoxious.WAV","sha":"979966aa10854b2282a41bccec5c2186baef3674","bytes":25938},{"path":"voice/Male5/gbl.oops.WAV","sha":"84c93f5daf614fb02496aaa83759559655784e0d","bytes":10162},{"path":"voice/Male5/gbl.quiet.WAV","sha":"353e0aaaee476ed532a79b1dc7391b1b85f1a985","bytes":11708},{"path":"voice/Male5/gbl.rock.WAV","sha":"f7ad3cfacdd173266e5713c6ed619f1a7c1a6f18","bytes":16324},{"path":"voice/Male5/gbl.sarcasm.WAV","sha":"b0c0b8eeb2f1c301891ea0bcdfd35751a4375af6","bytes":23166},{"path":"voice/Male5/gbl.shazbot.WAV","sha":"fe3320280a9ecfd498aecf7af2b063792cc8c94e","bytes":13598},{"path":"voice/Male5/gbl.shooting.WAV","sha":"a0334860e8ccab3a14071817f0f4565079d3410a","bytes":25562},{"path":"voice/Male5/gbl.thanks.WAV","sha":"05fac766f1d40e36e870a4003fed37e011e43964","bytes":9454},{"path":"voice/Male5/gbl.wait.WAV","sha":"7b6306a5e413cf5bb4f1c6a82a428e7e8104115c","bytes":11108},{"path":"voice/Male5/gbl.when.WAV","sha":"aca17b37df2768def3228a9fc1884a9d56c28ad3","bytes":24074},{"path":"voice/Male5/gbl.woohoo.WAV","sha":"27d75c30f0e2e05b88173a26c307e3c248a20417","bytes":16164},{"path":"voice/Male5/gbl.yes.WAV","sha":"4b0675607d6fe0f69c1648723d1143815ebe0456","bytes":9318},{"path":"voice/Male5/need.bombardier.WAV","sha":"24d0dbc97aa0436154d3f0abd0cdfd0e820dc26e","bytes":20162},{"path":"voice/Male5/need.cover.WAV","sha":"fa94be96ca342ddefe44fe64bcc82ce3bdc6d690","bytes":18650},{"path":"voice/Male5/need.driver.WAV","sha":"a24dcfcdbfe991062368afdc45873fdc2b528f7c","bytes":23998},{"path":"voice/Male5/need.escort.WAV","sha":"310e68aae17fa612c26b69fddaa67ebbe7b3f4a0","bytes":21774},{"path":"voice/Male5/need.flyer.WAV","sha":"7f7fe36c101f08052833f09a736de0e1013c4374","bytes":25620},{"path":"voice/Male5/need.gunship.WAV","sha":"b6471ad04ed3952956c5797da65722f45ec945d4","bytes":28812},{"path":"voice/Male5/need.hold.WAV","sha":"a01b9c843ba955d26f13109ae5fb5f6f7f3b369d","bytes":30628},{"path":"voice/Male5/need.ride.WAV","sha":"1d703a486b1d3f1f6785d1f620cef37596c2fb95","bytes":15014},{"path":"voice/Male5/need.support.WAV","sha":"ae248aaa629356fd4e7c11b2ed535441d63d543f","bytes":19740},{"path":"voice/Male5/need.tailgunner.WAV","sha":"397e755f3184191831b5e9021b6977df67a9d599","bytes":19622},{"path":"voice/Male5/need.where.WAV","sha":"ecc9adcb7838536df88447712f3673411f8e3089","bytes":11800},{"path":"voice/Male5/rep.base.WAV","sha":"2f2feeb2f0d4c23a78645791c7ead30f1d7e4a5c","bytes":19652},{"path":"voice/Male5/rep.generator.WAV","sha":"05c83ee7669f1827fd4ac9d3b188dd471e625d79","bytes":22548},{"path":"voice/Male5/rep.me.WAV","sha":"1a7beb8b9bf65a6f0d32abb0fbaf2467fd1b64fb","bytes":14598},{"path":"voice/Male5/rep.sensors.WAV","sha":"4e82b68fc7aa80ce1675c6d45ab02f0acef07a60","bytes":22794},{"path":"voice/Male5/rep.turrets.WAV","sha":"a20b4267b76edaa59a61137a5315d4589a958fea","bytes":21368},{"path":"voice/Male5/rep.vehicle.WAV","sha":"3bf34bb2f9253796ab964d121bed19bf44480440","bytes":27396},{"path":"voice/Male5/slf.att.attack.WAV","sha":"76b739c7e077cc49253e842aaa634c52ef2228cf","bytes":13844},{"path":"voice/Male5/slf.att.base.WAV","sha":"20c20cb516e6aef61a0657b3f435d82a008ae7fc","bytes":20514},{"path":"voice/Male5/slf.att.flag.WAV","sha":"23b06cc57191a0a7eeeb6e139b44538dc57da326","bytes":21036},{"path":"voice/Male5/slf.att.generator.WAV","sha":"9bba29862270b8d9a45d60c96a31f87894169da0","bytes":23090},{"path":"voice/Male5/slf.att.sensors.WAV","sha":"89b991792f24e70ddb3717b0579a5c8de5616e70","bytes":25048},{"path":"voice/Male5/slf.att.turrets.WAV","sha":"43e9a5f193693003f0f85fe2494b439e5028bb1f","bytes":22602},{"path":"voice/Male5/slf.att.vehicle.WAV","sha":"d9bb2d27d975c669a20718a11b23a5d57b4a16e0","bytes":26290},{"path":"voice/Male5/slf.def.base.WAV","sha":"4040f9abe1c9c3d2706c93bfa7e67769b3c03529","bytes":23680},{"path":"voice/Male5/slf.def.defend.WAV","sha":"63faa03e3ebc83d9848f26f1174b7b78d47b1ec6","bytes":17198},{"path":"voice/Male5/slf.def.flag.WAV","sha":"e5585b0392ee5fd178f60c39126b73ff66418cb5","bytes":20640},{"path":"voice/Male5/slf.def.generator.WAV","sha":"f402a62e52138580636de1730b31cefeb157a54b","bytes":20512},{"path":"voice/Male5/slf.def.nexus.WAV","sha":"e3fa14e4f498ed4937190284b93ee3793569e4f1","bytes":19094},{"path":"voice/Male5/slf.def.sensors.WAV","sha":"fadd70bcb040a031ee62efdfa958cacfe5b495d5","bytes":22708},{"path":"voice/Male5/slf.def.turrets.WAV","sha":"12262959880a04adf888ab5127bd967d5d5562ae","bytes":23284},{"path":"voice/Male5/slf.def.vehicle.WAV","sha":"38a133e078e5f4be86795a739c5b12adc848ab4c","bytes":23826},{"path":"voice/Male5/slf.rep.base.WAV","sha":"a122d3f424ef1a30ed279813a94e2f407c21a188","bytes":21834},{"path":"voice/Male5/slf.rep.equipment.WAV","sha":"36d5c357aa24b09e5979935df070389ccddde918","bytes":23840},{"path":"voice/Male5/slf.rep.generator.WAV","sha":"6131a45d3267804477f19e2dfa6f33925a3a93a5","bytes":23780},{"path":"voice/Male5/slf.rep.repairing.WAV","sha":"8505f1e3cdb96a0f51d6884eab291b4e2ee6fc7a","bytes":20124},{"path":"voice/Male5/slf.rep.sensors.WAV","sha":"8d96061276b3c032a3a799795036e08723d8f0ca","bytes":25014},{"path":"voice/Male5/slf.rep.turrets.WAV","sha":"6b00e1be88b72817fd3a3084cf06d42ea1d836f7","bytes":23836},{"path":"voice/Male5/slf.rep.vehicle.WAV","sha":"4e03400dca4a3db0ddcd406cade3fbf34b8e5a6f","bytes":28020},{"path":"voice/Male5/slf.tsk.cover.WAV","sha":"58672cb387a4befef9d9a71771e96edf41e7a96f","bytes":15226},{"path":"voice/Male5/slf.tsk.defense.WAV","sha":"108114130610fe40849093f6c4bc03b33b1e4127","bytes":21600},{"path":"voice/Male5/slf.tsk.generic.WAV","sha":"465cedcd572780641d445024ed84e2ce12c2d516","bytes":13184},{"path":"voice/Male5/slf.tsk.remotes.WAV","sha":"2334f874be0ddc8955136df43e3736fd28c1938d","bytes":24316},{"path":"voice/Male5/slf.tsk.sensors.WAV","sha":"3191656d71bfb9d9942ec5d38667e9bb207a3c45","bytes":19004},{"path":"voice/Male5/slf.tsk.turrets.WAV","sha":"5546b0c268e473fff13bba9c4069d1d2dfb91c6a","bytes":20232},{"path":"voice/Male5/slf.tsk.vehicle.WAV","sha":"b2f4510395f96313056f92a9c9349ad3a51e5a8d","bytes":21136},{"path":"voice/Male5/tgt.acquired.WAV","sha":"47fbb167418f7308c0b3277c49b2cd878a2e81ba","bytes":18040},{"path":"voice/Male5/tgt.base.WAV","sha":"1ea824c5aa1cce5b5e76272aff5efd6dfcd9c255","bytes":33436},{"path":"voice/Male5/tgt.destroyed.WAV","sha":"331a4a25f0f028b04ed9acfb4dc28b9cca78528b","bytes":19308},{"path":"voice/Male5/tgt.flag.WAV","sha":"053bb1447d3b61bcff4eb52098409f424e3a2685","bytes":28666},{"path":"voice/Male5/tgt.my.WAV","sha":"9ee9e72f3ed6e69edeee8def7c3bd3049d4ae767","bytes":20016},{"path":"voice/Male5/tgt.need.WAV","sha":"15a51dfcf69e1a8bdbcc868dc057718bcc8b52f8","bytes":22532},{"path":"voice/Male5/tgt.sensors.WAV","sha":"1c8e67963a110a306c3e2f724679e6a9adec86cb","bytes":33612},{"path":"voice/Male5/tgt.turret.WAV","sha":"d80974a6d91f0d4acf678cbf46512bba832e8f73","bytes":31000},{"path":"voice/Male5/tgt.wait.WAV","sha":"2a743f950edbac0abc03543cf509dd9e25e44171","bytes":25262},{"path":"voice/Male5/vqk.anytime.WAV","sha":"a7645bc54738920fd5030f9871bcb392a834ee3b","bytes":13644},{"path":"voice/Male5/vqk.base.WAV","sha":"ea14a120a5e6d680582ac53c2fe7f85143b24c01","bytes":11690},{"path":"voice/Male5/vqk.ceasefire.WAV","sha":"b7cc9aed0da70b37f41afc105eb754284e37e8c3","bytes":16754},{"path":"voice/Male5/vqk.dunno.WAV","sha":"4ab1b95991b4aafba7571e9fb2b6596b7c1ae640","bytes":9862},{"path":"voice/Male5/vqk.help.WAV","sha":"c2505d5504f5ef68467430fccfbb4a6a6d3ae8e5","bytes":7890},{"path":"voice/Male5/vqk.move.WAV","sha":"e491166fc9f1b70c8c9aa3b64360d154bbfcddc6","bytes":15076},{"path":"voice/Male5/vqk.no.WAV","sha":"cd4caf70aa7a75a43a5a7ec2a02d5520b0c9cab9","bytes":5612},{"path":"voice/Male5/vqk.quiet.WAV","sha":"eb1fe877832c50442cd2a747f995fc475714f93e","bytes":19138},{"path":"voice/Male5/vqk.sorry.WAV","sha":"d0459f0061415e52ccc8a275f48741e14deb48d2","bytes":6624},{"path":"voice/Male5/vqk.thanks.WAV","sha":"ad90375833f87983f8ffa9f38ea55b6075eb4689","bytes":5308},{"path":"voice/Male5/vqk.wait.WAV","sha":"7d195fee117702dcb9053b69cb4f9ba9bfebe2b7","bytes":5838},{"path":"voice/Male5/vqk.yes.WAV","sha":"1a6a4d99f74ec248d25b4f7655cf841c6b2d7f19","bytes":4690},{"path":"voice/Male5/wrn.bomber.WAV","sha":"b433fa1fa8887cb2b140c8e1685d84bc3f211cd9","bytes":21876},{"path":"voice/Male5/wrn.enemy.WAV","sha":"369934e58baefb2e4808c80459dbf4481a79f0bf","bytes":24506},{"path":"voice/Male5/wrn.vehicles.WAV","sha":"9cc50888811f9d380981638fb7e8a076bdc05322","bytes":23832},{"path":"voice/Male5/wrn.watchit.WAV","sha":"50057dd49a4f52e812791e5b1ccc4bfbb64ccc67","bytes":21538},{"path":"voice/Training/Any/ANY.abort.WAV","sha":"2ea999358e8d9e4405c91f80aa51b4a0bd75204b","bytes":25496},{"path":"voice/Training/Any/ANY.abortsoon.WAV","sha":"90d547bbe6a78b32fa7b651549c42dcd70992cae","bytes":41972},{"path":"voice/Training/Any/ANY.abortwarn.WAV","sha":"67fab4be74de20b5affe24c36f4be5c724a13ae8","bytes":40930},{"path":"voice/Training/Any/ANY.alright.WAV","sha":"a51b5d6137febcec33203d35ca294375920d9754","bytes":10116},{"path":"voice/Training/Any/ANY.blowoff01.WAV","sha":"566979276ea92ce9170bcc931364e15a66bfaa71","bytes":39294},{"path":"voice/Training/Any/ANY.blowoff02.WAV","sha":"59ec5b0307e51bc72ed2d08f5626677ef01749c8","bytes":34674},{"path":"voice/Training/Any/ANY.blowoff03.WAV","sha":"dd212a1402bfacb5dac0b8bdfa31161f66984e38","bytes":23314},{"path":"voice/Training/Any/ANY.careful.WAV","sha":"8034e7b754c8ef305b84249d949b745b1b26220d","bytes":12532},{"path":"voice/Training/Any/ANY.check01.WAV","sha":"18d05ba46758cb79e1dfb09970f9933aea8c1e24","bytes":9136},{"path":"voice/Training/Any/ANY.check02.WAV","sha":"b5308917f28dacc8e69529234c05e337dc14ee3a","bytes":8838},{"path":"voice/Training/Any/ANY.good.WAV","sha":"109099e66e6509600b146e356f5243075831d84e","bytes":10796},{"path":"voice/Training/Any/ANY.healthkit.WAV","sha":"4fbab5d681e1177f8e24f8f43df11781afc31de2","bytes":30220},{"path":"voice/Training/Any/ANY.hunting.WAV","sha":"1c2ca3067dabafb9e02010f78ac22ac6c2f226bf","bytes":12572},{"path":"voice/Training/Any/ANY.incoming_msg.WAV","sha":"4006e26421494915f379073292765df69aa26f4c","bytes":19392},{"path":"voice/Training/Any/ANY.jingo01.WAV","sha":"e4b0cbd4d49f4336824d71fde197d1b5dc5b5a9a","bytes":22202},{"path":"voice/Training/Any/ANY.jingo02.WAV","sha":"21cbdc23c12698aecae053aa96d0fcf5e67f986c","bytes":27014},{"path":"voice/Training/Any/ANY.jingo03.WAV","sha":"5f0c29f0e973dd803b8f2d7819c0bc5a95b21167","bytes":27488},{"path":"voice/Training/Any/ANY.kudo01.WAV","sha":"cdcdbe77425b662e18bd800a1b822d31dab365ee","bytes":14040},{"path":"voice/Training/Any/ANY.kudo02.WAV","sha":"a8cf979a9fad3f17201ef4b3026b9536851bc158","bytes":11644},{"path":"voice/Training/Any/ANY.kudo03.WAV","sha":"d4c53719bfcb63d439084db838a043f6c10f8d10","bytes":20676},{"path":"voice/Training/Any/ANY.kudo04.WAV","sha":"9d508e8d935ce8bfc88cd06e719329affa25f946","bytes":15468},{"path":"voice/Training/Any/ANY.obj_complete_01.WAV","sha":"82d4b4959fab38549a00c538958dff08e70a9efb","bytes":18366},{"path":"voice/Training/Any/ANY.obj_complete_02.WAV","sha":"98946e3d399cde32f99486151b06c7761ec9e8b8","bytes":19732},{"path":"voice/Training/Any/ANY.offcourse.WAV","sha":"c332e1c14360560ef4f8bb3deb1d1ab76f1b94fd","bytes":18118},{"path":"voice/Training/Any/ANY.prompt01.WAV","sha":"9115e0c77ad65c7855b44f8bc2bc19a866d2e4c1","bytes":26676},{"path":"voice/Training/Any/ANY.prompt02.WAV","sha":"033e18f707518a3637c60b7a7ecc0cdef90789cc","bytes":13218},{"path":"voice/Training/Any/ANY.prompt03.WAV","sha":"37b573828b8e85eb70192850abc1dff95d95e57c","bytes":29488},{"path":"voice/Training/Any/ANY.prompt04.WAV","sha":"e45469d90828b9dae1ebfb13e7dda15373a03c86","bytes":23870},{"path":"voice/Training/Any/ANY.prompt05.WAV","sha":"10013dbbbfd4e1883c12362068d6cd5a66e943c9","bytes":11756},{"path":"voice/Training/Any/ANY.prompt06.WAV","sha":"2ee1455d01dd6ead1c1517977d63fd09c36bb94e","bytes":25632},{"path":"voice/Training/Any/ANY.prompt07.WAV","sha":"fcb965fde82a832245668e36713cc74bb18e8521","bytes":18902},{"path":"voice/Training/Any/ANY.timer01.WAV","sha":"c9584ca4f50aef578490c891ff0609f899423528","bytes":17084},{"path":"voice/Training/Any/ANY.timer02.WAV","sha":"dd54cdd2ea1a57e173866b8537196fd99089c694","bytes":13216},{"path":"voice/Training/Any/ANY.tip.scavenge01.WAV","sha":"d8e01346607f27f09f541ea57678f6a46a0aac92","bytes":29100},{"path":"voice/Training/Any/ANY.tip.scavenge02.WAV","sha":"78650353b7b61e17d02775cf3c5350240eca5850","bytes":26990},{"path":"voice/Training/Any/ANY.tip.skiing.WAV","sha":"00e1b3d45be239e66dee44f7f44c2fe32235369c","bytes":56544},{"path":"voice/Training/Any/ANY.tip_now01.WAV","sha":"9754eb3ccdfe8b676c9c6e4729be886eb32430be","bytes":14580},{"path":"voice/Training/Any/ANY.tip_now02.WAV","sha":"88e82a3c0cfeebc02ed9da64b1bde0a9d9caedb7","bytes":15046},{"path":"voice/Training/Any/ANY.warning01.WAV","sha":"6693ee63b92120a6a65bf25e9c689c5223f526db","bytes":28270},{"path":"voice/Training/Any/ANY.warning02.WAV","sha":"6651210e76e8a685a4a9cbe935b5b0dfd8f6993d","bytes":40080},{"path":"voice/Training/Any/ANY.warning03.WAV","sha":"6094312c637cb151d63d24d69c1b1ec10a35f892","bytes":17244},{"path":"voice/Training/Any/ANY.warning04.WAV","sha":"90580d2294f2e61beda6e93cb243ba81c4aa74ea","bytes":12338},{"path":"voice/Training/Any/ANY.warning05.WAV","sha":"05acb6e903f1b63bbd782f44eb422ba4742c2634","bytes":33234},{"path":"voice/Training/Any/ANY.warning06.WAV","sha":"23a08e928126354835151fbe3bf06ee3d04e960c","bytes":17972},{"path":"voice/Training/Any/ANY.warning07.WAV","sha":"bd3262c3b705c73854f323faa4b40413f8aa5f6a","bytes":22152},{"path":"voice/Training/Any/ANY.waypoint01.WAV","sha":"2ed93afc349bb2ced66520055db9118a54169031","bytes":27940},{"path":"voice/Training/Any/ANY.waypoint02.WAV","sha":"7609aedcb35f3d2b0b9d4afa7952fce8c7864774","bytes":19876},{"path":"voice/Training/Any/ANY.waypoint03.WAV","sha":"e068a0f8bf589f7506ff3faff0502e7dc37a0b05","bytes":17134},{"path":"voice/Training/Briefings/T1.brief01.WAV","sha":"df3b55ec255018c224156c07096a7dd6043d8fe9","bytes":172044},{"path":"voice/Training/Briefings/T1.brief02.WAV","sha":"57a74614dfa67f7641fc71ecb0b79ca24a05d6e7","bytes":84316},{"path":"voice/Training/Briefings/T1.brief03.WAV","sha":"e59ac41830058d82b9838ddee20357e83ef0afcf","bytes":153926},{"path":"voice/Training/Briefings/T1.brief04.WAV","sha":"510f61cfe76d38bd2438610976638c35d19db07c","bytes":75956},{"path":"voice/Training/Briefings/T1.brief05.WAV","sha":"6d048e6a912b1a244a1d9a7aabe762ca7cb73aec","bytes":48986},{"path":"voice/Training/Briefings/T1.brief06.WAV","sha":"72b91816706a9b952cf4fe2eaceea4b51df14985","bytes":17924},{"path":"voice/Training/Briefings/T2.brief01.WAV","sha":"5c7ce4f789d557a4ef1e8beb0171f43fdfa560de","bytes":118278},{"path":"voice/Training/Briefings/T2.brief02.WAV","sha":"241324a8426433f95cbfdeeecf275114bc3ba2ea","bytes":128096},{"path":"voice/Training/Briefings/T2.brief03.WAV","sha":"e3327da6eb93e28323ffdcdd6c5b7e7c13e3b79a","bytes":57136},{"path":"voice/Training/Briefings/T2.brief04.WAV","sha":"31baf21d309d2a3f9e7e11af3dba88fc99ca311c","bytes":159626},{"path":"voice/Training/Briefings/T2.brief05.WAV","sha":"181c15dfe952459d5037021f53bb9c0d3d6837fd","bytes":101782},{"path":"voice/Training/Briefings/T2.brief06.WAV","sha":"fc868717b7b0e4a63428102739d1543589b1f107","bytes":25076},{"path":"voice/Training/Briefings/T3.brief01.WAV","sha":"069e64f4c9785c24ab69f0cedbda39e7b54ce8dc","bytes":99204},{"path":"voice/Training/Briefings/T3.brief02.WAV","sha":"5e951e170c9b93640e560f474a6c073685af1e50","bytes":124790},{"path":"voice/Training/Briefings/T3.brief03.WAV","sha":"08a3a4f1f5024dcddfc710fce4c8ba2ec06143e2","bytes":178756},{"path":"voice/Training/Briefings/T3.brief04.WAV","sha":"5d251503b0236448a078bd12e98d4b363889fcd5","bytes":125090},{"path":"voice/Training/Briefings/T3.brief05.WAV","sha":"093651bd80bf9a7e24535433e6868d9c41c3cd18","bytes":133658},{"path":"voice/Training/Briefings/T3.brief06.WAV","sha":"6bb3a60f79cb5a790782905a2ea702405d15ba3d","bytes":48100},{"path":"voice/Training/Briefings/T4.brief01.WAV","sha":"2cda82d7cdb36006825e41d745b3c125f2735349","bytes":105720},{"path":"voice/Training/Briefings/T4.brief02.WAV","sha":"9a37047f790a208e5090907f5c002936fb0374ae","bytes":146474},{"path":"voice/Training/Briefings/T4.brief03.WAV","sha":"8e331322296ed2fa3ca7a66b19adb492d941d422","bytes":124610},{"path":"voice/Training/Briefings/T4.brief04.WAV","sha":"9436663771be2606515044d2497e2eb109f4ff3a","bytes":129714},{"path":"voice/Training/Briefings/T4.brief05.WAV","sha":"7e7df5a57e1f5814c60a364204d9eb4b09fbcdfb","bytes":96772},{"path":"voice/Training/Briefings/T4.brief06.WAV","sha":"ca30357d9511c9685cca79f6196d3eca26b38040","bytes":56248},{"path":"voice/Training/Briefings/T5.brief01.WAV","sha":"aa1b2617ae0f8bce942fc4d9c08f52ab94b0e660","bytes":106578},{"path":"voice/Training/Briefings/T5.brief02.WAV","sha":"9cf65dae394fdea3e77d44daf5c99ad10ab55eaa","bytes":171840},{"path":"voice/Training/Briefings/T5.brief03.WAV","sha":"5f555574f273f0ae713213a7e8e4251be3fa8336","bytes":137986},{"path":"voice/Training/Briefings/T5.brief04.WAV","sha":"5c83b51b2d373cff272b473cbf2784db74066195","bytes":135740},{"path":"voice/Training/Briefings/T5.brief05.WAV","sha":"7ee7a620ca31d06899e2c727ad4b7956694be774","bytes":143952},{"path":"voice/Training/Briefings/T5.brief06.WAV","sha":"3265cc67d9b7becb270a88108193d4704d3562c3","bytes":64660},{"path":"voice/Training/Briefings/T6.brief01.WAV","sha":"1dbda4d68bb5cbd17d703964143adca445a78171","bytes":166182},{"path":"voice/Training/Briefings/T6.brief02.WAV","sha":"556171c90e4f19520c3162a5b058e9595d8cfa50","bytes":142526},{"path":"voice/Training/Briefings/T6.brief03.WAV","sha":"b44afb6733cd0762d818ebb1d5ac2d2552e318cc","bytes":194030},{"path":"voice/Training/Briefings/T6.brief04.WAV","sha":"3602e6694511b9ae668ce98c9c13a93680d53a76","bytes":116734},{"path":"voice/Training/Briefings/T6.brief05.WAV","sha":"db7b78f8f923d99c69b388989ea11d3e29285ac5","bytes":117248},{"path":"voice/Training/Briefings/T6.brief06.WAV","sha":"b6a57501d68f0c486b94a19cb2331688b9dd0b77","bytes":60100},{"path":"voice/Training/Mission1/ANY.move.WAV","sha":"4177a8307a6dea096e33bfac6f38fd46a3e64f63","bytes":11502},{"path":"voice/Training/Mission1/ANY.practice.WAV","sha":"e12918382b48e334615f0dd585441ac9014d525c","bytes":35302},{"path":"voice/Training/Mission1/ANY.practice1.WAV","sha":"57a76af45dc793a8568ed08f18c7809d08b2a39f","bytes":35570},{"path":"voice/Training/Mission1/ANY.warning06.WAV","sha":"c722024d55c907fbfd2757600ffc4ca5dbd68c57","bytes":19018},{"path":"voice/Training/Mission1/T1.tip.IFF.WAV","sha":"8fcb8aca7b06d38011d9610eaef848acbf823e0d","bytes":92402},{"path":"voice/Training/Mission1/T1.tip.blaster01.WAV","sha":"e44e0e567abbd5e080407e63e8d4e4f835394efd","bytes":65566},{"path":"voice/Training/Mission1/T1.tip.chaingun.WAV","sha":"1773a45883ac72fdabcc9d89d27c19894cdf6079","bytes":59144},{"path":"voice/Training/Mission1/T1.tip.energy.WAV","sha":"02f44ea94df8c0d56a9e723977c0dd0b4be5d4f4","bytes":54092},{"path":"voice/Training/Mission1/T1.tip.jets01.WAV","sha":"1183fa7d6632ce11bb801b3d4930647b918fce42","bytes":39818},{"path":"voice/Training/Mission1/T1.tip.jets02.WAV","sha":"cc578055a084119031e778e7e1c247802bbd655f","bytes":42680},{"path":"voice/Training/Mission1/T1.tip.jets03.WAV","sha":"bd34f85a3ad86f1f8980b0462f4813585f102e5c","bytes":38714},{"path":"voice/Training/Mission1/T1.tip.pack01.WAV","sha":"ff09381787ce69b53adce3c33d4899eacb89cdfc","bytes":71208},{"path":"voice/Training/Mission1/T1.tip.pack02.WAV","sha":"811c6f507bcbc256f646ebab6afe4374aed9c7d3","bytes":50558},{"path":"voice/Training/Mission1/T1.tip.skiing01.WAV","sha":"1acb573f46d8a265d0ead40c6752d2f78c4854da","bytes":40838},{"path":"voice/Training/Mission1/T1.tip.skiing02.WAV","sha":"5ef3d0ed9a79a6ed16744b0d53cb61ae78c0284f","bytes":40518},{"path":"voice/Training/Mission1/T1.tip.skiing02a.WAV","sha":"302caa5eaaa2d2366b8c4573c7b5309ad61ff5ab","bytes":36660},{"path":"voice/Training/Mission1/T1.tip.skiing03.WAV","sha":"db01e54e0ea14774cbd9237b8cf9612a29aeb6d7","bytes":37298},{"path":"voice/Training/Mission1/T1.tip.sniper02.WAV","sha":"96e12662eea499fdb7d34e107b7363aee002e132","bytes":71720},{"path":"voice/Training/Mission1/T1.tip.sniper03.WAV","sha":"86fee15416600a8ae8a0b82ec9b5f87b32355cf2","bytes":41164},{"path":"voice/Training/Mission1/T1.tip.sniper04.WAV","sha":"979862dbbc47d36738a13ca8e252f238720a8e3c","bytes":52352},{"path":"voice/Training/Mission1/T1.tip.spinfusor.WAV","sha":"4ff174ee659a6030460718ef5ac930af365017b0","bytes":63480},{"path":"voice/Training/Mission1/T1.tip.tactics.WAV","sha":"02310d8c97f0aa5e5204046650a02d5ca27e3610","bytes":47624},{"path":"voice/Training/Mission1/T1.tip.tactics02.WAV","sha":"0019e3745bc484635360d2709d6658ac7f36018f","bytes":52454},{"path":"voice/Training/Mission1/T1_01.WAV","sha":"aa65074e896ce0e9caf547a2dda72b71aa940335","bytes":37284},{"path":"voice/Training/Mission1/T1_01a.WAV","sha":"00222154e9aff6e6cb012836752114ddfdf328e0","bytes":31240},{"path":"voice/Training/Mission1/T1_01b.WAV","sha":"b38f8b5464b85205decf6f16b6f28eba304caa0a","bytes":37242},{"path":"voice/Training/Mission1/T1_01c.WAV","sha":"b93debefa6514fffdf6349ae1dec08875eeb7527","bytes":17740},{"path":"voice/Training/Mission1/T1_02.WAV","sha":"bcade3b89bcc12cfae45cec1c0dc224d8e94baff","bytes":37760},{"path":"voice/Training/Mission1/T1_02a.WAV","sha":"2496e55fde0bfd2764cbad1273a05aaea762bd5c","bytes":22626},{"path":"voice/Training/Mission1/T1_03.WAV","sha":"2077881728ade218a751ad092f0b10dcba5e6ffb","bytes":47376},{"path":"voice/Training/Mission1/T1_03a.WAV","sha":"6e23a79bef0942874c443d84b5301066c7f21874","bytes":34672},{"path":"voice/Training/Mission1/T1_03b.WAV","sha":"f8de84b439861d4c76ef565ea4de5de30b6c5c02","bytes":60008},{"path":"voice/Training/Mission1/T1_03c.WAV","sha":"387983dd28dc2e32b545c92e165c2fc122363646","bytes":38792},{"path":"voice/Training/Mission1/T1_04.WAV","sha":"766dc8002c707d20ff40226a347bac14815f347e","bytes":9968},{"path":"voice/Training/Mission1/T1_05.WAV","sha":"436e47dd5f40d0f8ac79cca720f41ed112eef412","bytes":13574},{"path":"voice/Training/Mission1/T1_06.WAV","sha":"0d9bf2a6508b79a60a51ce1709a8d1e97cedd21a","bytes":15408},{"path":"voice/Training/Mission1/T1_08.WAV","sha":"48800a06a11d5c387372cfaeaf3f9e20a0e28615","bytes":55180},{"path":"voice/Training/Mission1/T1_09.WAV","sha":"5c46804f8dc8358a0269e24e253195ab498f997e","bytes":17610},{"path":"voice/Training/Mission1/T1_10.WAV","sha":"9f01edd6760988147abd001c52395d3fbba8d233","bytes":19966},{"path":"voice/Training/Mission1/T1_10a.WAV","sha":"9fe5040f703b2640c784dd17d2d48403a6320539","bytes":59452},{"path":"voice/Training/Mission1/T1_10b.WAV","sha":"7f923c8d88e05f61e80cc664deb9445cfb41de0d","bytes":59036},{"path":"voice/Training/Mission1/T1_11.WAV","sha":"b3e66d1d2bbb3828519d5090b4d51eca0d66d567","bytes":64890},{"path":"voice/Training/Mission1/T1_12a.WAV","sha":"cf58d09077c26533826cc25e6171393ac50ba912","bytes":12104},{"path":"voice/Training/Mission1/T1_12b.WAV","sha":"b388e4faa95860028b79fcf8b5d0c573f277e535","bytes":54086},{"path":"voice/Training/Mission1/T1_13.WAV","sha":"4a6e17c8102fbf9787396c8c7e6e2461284ee092","bytes":21142},{"path":"voice/Training/Mission1/T1_14.WAV","sha":"5e73a8451e56f6f16ac9b64c5f2a9615e853494d","bytes":12376},{"path":"voice/Training/Mission1/T1_15.WAV","sha":"7439ee3d7b86e502c346926afb3e7a2a7ba1ccda","bytes":11842},{"path":"voice/Training/Mission1/T1_16.WAV","sha":"707e3cf66ce9999043d9dc934ebf23de569759e3","bytes":16840},{"path":"voice/Training/Mission1/T1_17.WAV","sha":"bf35d64c0ce3cbad3ad869f735b2843d20981557","bytes":59616},{"path":"voice/Training/Mission1/T1_18.WAV","sha":"9ac7aed0998a021b9a7b4c07cbccbb84f513be51","bytes":47164},{"path":"voice/Training/Mission1/T1_21.WAV","sha":"3117026342b3431905fe9c4e7f2b3aaa79babadf","bytes":49440},{"path":"voice/Training/Mission1/T1_22.WAV","sha":"55a0331c28464c50e0ce74c25c9c89955974ae2c","bytes":37684},{"path":"voice/Training/Mission1/T1_22a.WAV","sha":"64800400a1b49036ae27f06c5c2d7f8c212e179b","bytes":83912},{"path":"voice/Training/Mission1/T1_23.WAV","sha":"154112c5569e84e866749ddf00c8fb8e64a81be9","bytes":16526},{"path":"voice/Training/Mission1/T1_23a.WAV","sha":"e14b4f713300ffdc8a9244079c5b5ccd3c2b6ea1","bytes":38542},{"path":"voice/Training/Mission1/T1_23b.WAV","sha":"981e606a5c874cc33fa0d4963401f17d02b3a926","bytes":36902},{"path":"voice/Training/Mission1/T1_24.WAV","sha":"a9ce3048916afdacd7a686db3c1ca4d5a67f76fd","bytes":43000},{"path":"voice/Training/Mission1/T1_24a.WAV","sha":"de6736784fdfb8d34324909c8f79c6a459e00a0e","bytes":22580},{"path":"voice/Training/Mission1/T1_25.WAV","sha":"0a34953797e01e342fac6bee8dffa987fe770d85","bytes":41142},{"path":"voice/Training/Mission1/T1_25a.WAV","sha":"95f794a265a6926b017648cc434a45c53606cf6d","bytes":35356},{"path":"voice/Training/Mission1/T1_26.WAV","sha":"27ad9d417521db8b723090feeca6e9e11a6c7e3b","bytes":45858},{"path":"voice/Training/Mission1/T1_27.WAV","sha":"0bb80412100e358760961e65d17780038a522982","bytes":56874},{"path":"voice/Training/Mission1/T1_27a.WAV","sha":"1c4ca531fcc10deccbb0e8ef21137e73a9163754","bytes":53900},{"path":"voice/Training/Mission1/T1_27b.WAV","sha":"c385ab7a5e01dc3ce936268a9af4f26eb56a688d","bytes":17426},{"path":"voice/Training/Mission1/T1_28.WAV","sha":"3d932ebb6f51404084a6efa05ff3f96fea4ec57a","bytes":65604},{"path":"voice/Training/Mission1/T1_29.WAV","sha":"e2b8fec1294d98c5c3f8544cadbbd7d0f585a2f6","bytes":51120},{"path":"voice/Training/Mission1/T1_29a.WAV","sha":"97eb5a135ad098c79a9fa6c122a95aaa73aa4edd","bytes":37178},{"path":"voice/Training/Mission1/T1_30.WAV","sha":"1ef060844cd0def7cffe4c54f95abda7d0df1028","bytes":37698},{"path":"voice/Training/Mission2/T2.cya01.WAV","sha":"66e6035b608c10612b5c068f13be5ebaf76dfb76","bytes":43594},{"path":"voice/Training/Mission2/T2.inventory01.WAV","sha":"cf5a9024c3ba4ab2de4c1a05db17a95ca4d88dc7","bytes":43226},{"path":"voice/Training/Mission2/T2.repairpack.WAV","sha":"c0811a989ec6931d53ef346a96c3e52655b07a2c","bytes":19526},{"path":"voice/Training/Mission2/T2.tip.defense01.WAV","sha":"ed345b9442676a9f84413b1288374c7609b50e0b","bytes":49658},{"path":"voice/Training/Mission2/T2.tip.defense02.WAV","sha":"e0145ed824695e27fdd1b231e9ea836a8fc0259e","bytes":67984},{"path":"voice/Training/Mission2/T2.tip.defense03.WAV","sha":"167ba18b196c4a02a1078314b14a8fa0eb7c0774","bytes":72784},{"path":"voice/Training/Mission2/T2.tip.defense05.WAV","sha":"f0109e0bb3f6664a4baee58c6f6995fe643ba30a","bytes":30236},{"path":"voice/Training/Mission2/T2.tip.defense05a.WAV","sha":"e814b461fac630751fbc94b79c5bc55ec2aff49c","bytes":20348},{"path":"voice/Training/Mission2/T2.tip.defense06.WAV","sha":"30bb8f0113ca5fcdcaa8696ab9bd90ff92359e08","bytes":39608},{"path":"voice/Training/Mission2/T2.tip.defense07.WAV","sha":"afbf065c62aab9729d9c93040d2b507ff3a02264","bytes":36822},{"path":"voice/Training/Mission2/T2.tip.dropit.WAV","sha":"76eb448adb85cd73b5987050ffc0d98062bdf0a0","bytes":42330},{"path":"voice/Training/Mission2/T2.tip.elf.WAV","sha":"90ace14586ffc28a41c31eaf232326236a3f90b3","bytes":89214},{"path":"voice/Training/Mission2/T2.tip.evading.WAV","sha":"087618a7aada27ea08eb34b9c2b6a4fcce112e35","bytes":77850},{"path":"voice/Training/Mission2/T2.tip.gens01.WAV","sha":"d5a2940d342623311692478861133eccd92af938","bytes":49700},{"path":"voice/Training/Mission2/T2.tip.inventory.WAV","sha":"318c639bfce29e8243e39b9108c1959bb5c3042d","bytes":76146},{"path":"voice/Training/Mission2/T2.tip.inventory01.WAV","sha":"5b8d6c07cd51759922af9f90a6ef921a29981f0c","bytes":61062},{"path":"voice/Training/Mission2/T2.tip.inventory03.WAV","sha":"8fec979b02132c0d98686ab1205a96e688ba3f56","bytes":62468},{"path":"voice/Training/Mission2/T2.tip.missile.WAV","sha":"f23ddf619ed7ab4c2b873f33e1fd9a00e9254e33","bytes":74576},{"path":"voice/Training/Mission2/T2.tip.missile02.WAV","sha":"b7c73e11d504d8e3daa6ebf6621b269643242320","bytes":40384},{"path":"voice/Training/Mission2/T2.tip.missile03.WAV","sha":"9fba7377a4a65eb1b076485981d0e7b36375e773","bytes":65036},{"path":"voice/Training/Mission2/T2.tip.plasma.WAV","sha":"54ef6a1f273e1ab6f56a757278e3bbdc00ae6423","bytes":72414},{"path":"voice/Training/Mission2/T2.tip.repair01.WAV","sha":"8b452dbf72175a0b5deaee3d73724bccb5f7e5d8","bytes":41688},{"path":"voice/Training/Mission2/T2.tip.repair02.WAV","sha":"9de4cb180394e553758a705a1963a81f56434549","bytes":24978},{"path":"voice/Training/Mission2/T2.tip.repair03.WAV","sha":"1a98737d5e4c915aff070eb3d0ddb98531da2279","bytes":55638},{"path":"voice/Training/Mission2/T2.tip.scanned.WAV","sha":"2987b6d7c644dc110e1cf955fb77ca4c4446e5b9","bytes":49700},{"path":"voice/Training/Mission2/T2.tip.shieldpack.WAV","sha":"a5126f5020deb4d38377bef7996da3c8d37d0ec1","bytes":76942},{"path":"voice/Training/Mission2/T2.tip.tlaser.WAV","sha":"d736b7be9ab0086642c54d9c27cb958143f98626","bytes":75570},{"path":"voice/Training/Mission2/T2.tip.turret01.WAV","sha":"29791dbf1fed71235721a927db82b226be80a561","bytes":65874},{"path":"voice/Training/Mission2/T2.tip.turret02.WAV","sha":"c3d46f489b4e05b924564a8b7c19e68e63732859","bytes":105998},{"path":"voice/Training/Mission2/T2_01.WAV","sha":"d70aac1fee40b8c4877267a9e214888949b6bf67","bytes":23852},{"path":"voice/Training/Mission2/T2_01a.WAV","sha":"0333434dc20cd2f0076fc0aff4828b8e8dbb8256","bytes":48686},{"path":"voice/Training/Mission2/T2_01b.WAV","sha":"7aa37c3953544a157ab7968e97ad4a3f4436af1f","bytes":49820},{"path":"voice/Training/Mission2/T2_02.WAV","sha":"1e05cd200c50742524a6cf8f2718362bf4a21ae2","bytes":40044},{"path":"voice/Training/Mission2/T2_03.WAV","sha":"831235e55e84cebb53ec6e9228308dc8ecb623e1","bytes":28024},{"path":"voice/Training/Mission2/T2_04.WAV","sha":"c2e61e7d34fee0cae43534293be11763f831af24","bytes":61296},{"path":"voice/Training/Mission2/T2_04a.WAV","sha":"f0cc114743fc4246f230591e879dd2389dda1126","bytes":47850},{"path":"voice/Training/Mission2/T2_05.WAV","sha":"5c43becd6611ba057e30b4a8b8255ea29997de9c","bytes":67184},{"path":"voice/Training/Mission2/T2_05a.WAV","sha":"48b76e877eb5bf4a0db5d341db6b70d111c483c3","bytes":15186},{"path":"voice/Training/Mission2/T2_05b.WAV","sha":"47f685ee34a1e0ff2163b61db4dd5967289ef5e9","bytes":40406},{"path":"voice/Training/Mission2/T2_05c.WAV","sha":"0665b95579850fa78b3ec6440bebaed54f340e9a","bytes":33618},{"path":"voice/Training/Mission2/T2_07.WAV","sha":"059de39a2dbae83aae8e96c186fc2453fcda90de","bytes":71294},{"path":"voice/Training/Mission2/T2_08.WAV","sha":"20b31e33017e44c464c1cc47c0d2df99d8ad9af6","bytes":30176},{"path":"voice/Training/Mission2/T2_09.WAV","sha":"2d3353b5b54ed776588da13a212f8ae58f598036","bytes":19016},{"path":"voice/Training/Mission2/T2_09a.WAV","sha":"170d1cff497b62fbc328726e3914d4ee240fd961","bytes":100056},{"path":"voice/Training/Mission2/T2_09b.WAV","sha":"cd24fd3784e244a44889d187954b6863281c7c37","bytes":70386},{"path":"voice/Training/Mission2/T2_10.WAV","sha":"70a196e95323ab769d92e3161a835b93738e3cd5","bytes":50610},{"path":"voice/Training/Mission2/T2_10a.WAV","sha":"137111308e2878bfcc46a0fa10706122cae35004","bytes":54208},{"path":"voice/Training/Mission2/T2_11.WAV","sha":"8cc1cfecc03646e41584d550b207bdfaea0791f8","bytes":39936},{"path":"voice/Training/Mission2/T2_12.WAV","sha":"dfba575831a87f662cef30fc115abb5c0b60a8b0","bytes":27588},{"path":"voice/Training/Mission2/T2_13.WAV","sha":"05074f70c6f5fa1ba17ae4a76941f8bd33e84b48","bytes":49744},{"path":"voice/Training/Mission3/T3.cloaking.WAV","sha":"62cea4a298ebe4b649c5ccb4a18379b3d1d15cb7","bytes":26522},{"path":"voice/Training/Mission3/T3.tip.cloaking01.WAV","sha":"e97b9a81c72bd636ebe87407179bbf0be9c002e7","bytes":87152},{"path":"voice/Training/Mission3/T3.tip.cloaking02.WAV","sha":"f7909ece40e29f0f2a7d3d9c209ecf7f206262e4","bytes":36894},{"path":"voice/Training/Mission3/T3.tip.cloaking03.WAV","sha":"30f06c4f8ada957e94dd199ecf2d6da5cd99e2dd","bytes":45058},{"path":"voice/Training/Mission3/T3.tip.equipment01.WAV","sha":"d0b9f5f9d0740e907a340fc02da67eeb8aab0510","bytes":50270},{"path":"voice/Training/Mission3/T3.tip.equipment02.WAV","sha":"9773e8c03ab5899d585286e5dc5852496d313418","bytes":71804},{"path":"voice/Training/Mission3/T3.tip.freelook.WAV","sha":"869099f980ba77deefaf4ecad830f8e5b61a03f4","bytes":37810},{"path":"voice/Training/Mission3/T3.tip.piloting01.WAV","sha":"ea82dcfeeecb33a3fbdc74ef9fe3b0378bae3342","bytes":77332},{"path":"voice/Training/Mission3/T3.tip.piloting02.WAV","sha":"99bebb3daddf56f3cd983a8624de847fb09d61c4","bytes":69382},{"path":"voice/Training/Mission3/T3.tip.piloting03.WAV","sha":"a985e065dc85e089faf668799d5dc5e6590840b1","bytes":59496},{"path":"voice/Training/Mission3/T3.tip.piloting04.WAV","sha":"5b13402a703837534e33eff2e0f751e4fbef7498","bytes":66072},{"path":"voice/Training/Mission3/T3.tip.shocklance.WAV","sha":"68be7e03a092b7a080a219f94304db20f6da995b","bytes":44092},{"path":"voice/Training/Mission3/T3.tip.underwater01.WAV","sha":"84af1e706882c0f781b4ee96aaf8aa0cd6f052b9","bytes":29498},{"path":"voice/Training/Mission3/T3.tip.underwater02.WAV","sha":"ee0da6e4f09dbbc6194e2f23c124408ac9c87f8f","bytes":70630},{"path":"voice/Training/Mission3/T3.warning01.WAV","sha":"1b6c140a2a7914f4363e41407142a86c14ad2a68","bytes":27274},{"path":"voice/Training/Mission3/T3.warning02.WAV","sha":"678438618eedafe49787a356bec3c64b23d9ce0d","bytes":13490},{"path":"voice/Training/Mission3/T3.warning03.WAV","sha":"98ab6a12975cda0cb841bb5a9ccba72001607d74","bytes":38476},{"path":"voice/Training/Mission3/T3_01.WAV","sha":"002175b004f66e5af0e6909926311f1c465d3859","bytes":39322},{"path":"voice/Training/Mission3/T3_02.WAV","sha":"7628c125e4d3f2647b6fbcb4a56719f360e3d810","bytes":66852},{"path":"voice/Training/Mission3/T3_03.WAV","sha":"0888449744bc00b7c947a004b48402f8b2fa9f37","bytes":59740},{"path":"voice/Training/Mission3/T3_04.WAV","sha":"c6fad3eb2750c41c3b13fa6e62ec6c10d2181997","bytes":34610},{"path":"voice/Training/Mission3/T3_05.WAV","sha":"a3e9abe7b8a664d3e4e27db9f574e556d3138e8e","bytes":60206},{"path":"voice/Training/Mission3/T3_06.WAV","sha":"bf825d13e5d96b54cd897fd62e952a811d6bdc44","bytes":45718},{"path":"voice/Training/Mission3/T3_07.WAV","sha":"8b95d928e16212841f5f578f42279c46ae52dea8","bytes":45800},{"path":"voice/Training/Mission3/T3_07a.WAV","sha":"49751f4c6ad212526ae3cbc8535fb7e3df9ce6f2","bytes":33498},{"path":"voice/Training/Mission3/T3_07b.WAV","sha":"5cc28ea94a3f9cd4e71ec38b09dda76d00ed7294","bytes":35806},{"path":"voice/Training/Mission3/T3_08.WAV","sha":"2fc46274afd32db72e39f084b4160a1375012b53","bytes":25426},{"path":"voice/Training/Mission3/T3_08a.WAV","sha":"c3964a709c261adacebd3c8b899daf8de3fb4817","bytes":16670},{"path":"voice/Training/Mission3/T3_08b.WAV","sha":"6ca9fc08293a3045dc858b865c41af3d96fde07a","bytes":50790},{"path":"voice/Training/Mission3/T3_09.WAV","sha":"d141190093b0eca3b93d0e29c1bc5e9967390cfb","bytes":25908},{"path":"voice/Training/Mission3/T3_09a.WAV","sha":"e6673a263bb1d612224f87ec6249edcb813481a9","bytes":65952},{"path":"voice/Training/Mission3/T3_10.WAV","sha":"1dc54932e598eb25115d7e241d5d77f5f474e81f","bytes":44186},{"path":"voice/Training/Mission3/T3_11.WAV","sha":"309575857d77d471ce9bc75d9b0ea31f821553de","bytes":42850},{"path":"voice/Training/Mission3/T3_12.WAV","sha":"aefb35d25cbcb500f9ecb209ae9f47ee93f09662","bytes":66038},{"path":"voice/Training/Mission3/T3_12a.WAV","sha":"ad25d2f7747f098ffcc86c5fe66a98a12cd16fcc","bytes":33194},{"path":"voice/Training/Mission3/T3_13.WAV","sha":"9340e426e053cbf194f11eb99f024d31323157d9","bytes":43606},{"path":"voice/Training/Mission4/T4.defense01.WAV","sha":"e3d3c09b4df6e60ce87fdedba70ca756029ea259","bytes":53370},{"path":"voice/Training/Mission4/T4.ff_gendown01.WAV","sha":"ef52bea9dbbe21f220198ee3c471db3b63eaf834","bytes":23232},{"path":"voice/Training/Mission4/T4.ff_gendown02.WAV","sha":"d05fd63b629b118a33d5c20af6f1853514046d5e","bytes":24966},{"path":"voice/Training/Mission4/T4.fieldsUP_01.WAV","sha":"20d0e9c15f1598578d86db8920cf770efdaad589","bytes":23004},{"path":"voice/Training/Mission4/T4.fieldsUP_02.WAV","sha":"6bcb497427bbc4f7b0f030b79ad7fd7e46888e57","bytes":18522},{"path":"voice/Training/Mission4/T4.forcefields01.WAV","sha":"42d30e121c62c2b52f1235add8b03a610471ac31","bytes":20690},{"path":"voice/Training/Mission4/T4.forcefields02.WAV","sha":"b41eb770983c7edc0c714f5a2c38ec94bae5c83c","bytes":20490},{"path":"voice/Training/Mission4/T4.gendown02.WAV","sha":"928835dfaac70856c5064951b44042227fceed89","bytes":22046},{"path":"voice/Training/Mission4/T4.gendown02a.WAV","sha":"19834b698ebe7df80d988e548df32e917da21ff2","bytes":20446},{"path":"voice/Training/Mission4/T4.genup.WAV","sha":"ba6f7314eed436f940c1d15de704e6bd636a9de1","bytes":19734},{"path":"voice/Training/Mission4/T4.genup02.WAV","sha":"8dba1faebee7f2203c3059ca5c034d478eab8b30","bytes":22016},{"path":"voice/Training/Mission4/T4.genup02a.WAV","sha":"a4857d15460853803fb314fa817bd929a4cbe9ba","bytes":23236},{"path":"voice/Training/Mission4/T4.repgen.WAV","sha":"34b78453f9d52e1471bf4e31229dd4f4c62530d8","bytes":18474},{"path":"voice/Training/Mission4/T4.tip.camera01.WAV","sha":"97fb0c5dcda28bb710ff5e29f0d3310bdea033c0","bytes":93030},{"path":"voice/Training/Mission4/T4.tip.camera02.WAV","sha":"0e39c0133eeab5812886ed6f7c2fc8a246a05705","bytes":35046},{"path":"voice/Training/Mission4/T4.tip.camera03.WAV","sha":"1c73ff25ee5e609b6fe259ca36a8cbfa6c3c86db","bytes":48506},{"path":"voice/Training/Mission4/T4.tip.camera04.WAV","sha":"10208eae0582f3145a00c45239d95b162814d5d3","bytes":64820},{"path":"voice/Training/Mission4/T4.tip.defense02.WAV","sha":"56c8378e26748a3193c83ff25d6e75802a5574b5","bytes":62064},{"path":"voice/Training/Mission4/T4.tip.defense03.WAV","sha":"2b28ae07b1fcd904e0107d1d9e06c62ec60c2fbb","bytes":69514},{"path":"voice/Training/Mission4/T4.tip.defense05.WAV","sha":"c28316a89c617ef74dc7c7222980993ba5da6b42","bytes":73070},{"path":"voice/Training/Mission4/T4.tip.defense06.WAV","sha":"f2d55d4647a32359fd0e63207462200a3d27145e","bytes":46464},{"path":"voice/Training/Mission4/T4.tip.defense07.WAV","sha":"30504daaf5c938423b9683cd8cbeebf89aa820c3","bytes":80172},{"path":"voice/Training/Mission4/T4.tip.defense08.WAV","sha":"57d6fbc9328657eaeffb9776a766a1d9a2e4fc8f","bytes":38818},{"path":"voice/Training/Mission4/T4.tip.defense09.WAV","sha":"e4cc0a9ad3a3811f72b4566008abc0dd0352070f","bytes":41422},{"path":"voice/Training/Mission4/T4.tip.deploy.WAV","sha":"37b72c41a2da914675a4aba6ba1b3b694b11ddfa","bytes":72632},{"path":"voice/Training/Mission4/T4.tip.deploy01.WAV","sha":"84393d9caf2a5facb6a0566b7e589cc60e48d3b4","bytes":37760},{"path":"voice/Training/Mission4/T4.tip.deploy02.WAV","sha":"ca93cc0df8dc743f610e6d726ecb13b32d6a591f","bytes":48466},{"path":"voice/Training/Mission4/T4.tip.depturret.WAV","sha":"16bbc6750836f3b9fd0837e8f286ba4171acce54","bytes":51232},{"path":"voice/Training/Mission4/T4.tip.generator01.WAV","sha":"2573c8207b1615e70d4adee0ea1b70293d10db47","bytes":74684},{"path":"voice/Training/Mission4/T4.tip.generator01a.WAV","sha":"0bfc5bb463097ff61b1828c6981e568547927b66","bytes":54160},{"path":"voice/Training/Mission4/T4.tip.generator01b.WAV","sha":"a58f676a8dcbbe383a985ac502182afc36fb3bf5","bytes":45402},{"path":"voice/Training/Mission4/T4.tip.generator02.WAV","sha":"a5eb1f6d0d08397c94057ab9c2004a5c393cb184","bytes":53978},{"path":"voice/Training/Mission4/T4.tip.mortar.WAV","sha":"23d5448ae4065b1344f6366f33a205f5fe31191d","bytes":70018},{"path":"voice/Training/Mission4/T4.tip.mortar_02.WAV","sha":"8a4d27c7fe18d08c445ba32a8f08bdc5c2e85980","bytes":51346},{"path":"voice/Training/Mission4/T4.tip.objects.WAV","sha":"66f389a0bd58dec29fddf11f321dedebaba5aa5a","bytes":83884},{"path":"voice/Training/Mission4/T4.warning01.WAV","sha":"815a0a48e00b643cc650ab6c7cf93f2ade7d3477","bytes":16088},{"path":"voice/Training/Mission4/T4.warning02.WAV","sha":"b3ae4f91e6451db4594839b381eb61928ffa9cf3","bytes":25116},{"path":"voice/Training/Mission4/T4_01.WAV","sha":"7b812c28478ccc78417fae30d8fdfbbba07a5d2f","bytes":60768},{"path":"voice/Training/Mission4/T4_01a.WAV","sha":"25bfc5ae3ad7ee4006d0d1fb41c1e4f8d699ae35","bytes":21522},{"path":"voice/Training/Mission4/T4_01b.WAV","sha":"db055d10960433eb54e8341dee484cf8cec24be5","bytes":25694},{"path":"voice/Training/Mission4/T4_01c.WAV","sha":"62ce5ab45d90878b245715897131c28cdd69a8a3","bytes":91574},{"path":"voice/Training/Mission4/T4_02.WAV","sha":"6cdcd612334280c977c6111ab3f1ec3bfc83eb78","bytes":61408},{"path":"voice/Training/Mission4/T4_02a.WAV","sha":"8f91b1e17c305ddd2f6a389400489106b01a8e1c","bytes":27346},{"path":"voice/Training/Mission4/T4_02b.WAV","sha":"be79a4b83b7e5acd6fa541aa0f63cf2efe676a22","bytes":35786},{"path":"voice/Training/Mission4/T4_03.WAV","sha":"cceb6ea79c9558de0e990ea0e7ea447ea2830ff2","bytes":70126},{"path":"voice/Training/Mission4/T4_03a.WAV","sha":"bda00c0bb06b219059fd9f8a3c63a6b2df6cf402","bytes":68614},{"path":"voice/Training/Mission4/T4_03b.WAV","sha":"43c5376319e6d2c3f6cd433b7baa2ab8bdde1bee","bytes":15758},{"path":"voice/Training/Mission4/T4_03c.WAV","sha":"23e10e4c146e150442c088a7545fa4dfa10b3d10","bytes":21178},{"path":"voice/Training/Mission4/T4_03d.WAV","sha":"3f6a622b1f991e1d941ba8bc13f22bf6b0db04e8","bytes":21696},{"path":"voice/Training/Mission4/T4_03e.WAV","sha":"63d4fc2bd5172b185e4b63339755cd052b45389e","bytes":18504},{"path":"voice/Training/Mission4/T4_03f.WAV","sha":"69fd0fdce7bb07259b6bf6f69214bb3e499a7964","bytes":17174},{"path":"voice/Training/Mission4/T4_03g.WAV","sha":"75e2e57d5d1afbab040c4d0c088a056bf9e47519","bytes":51640},{"path":"voice/Training/Mission4/T4_03h.WAV","sha":"24bbca0a0d856075bf15432465e18dc26862cee1","bytes":31414},{"path":"voice/Training/Mission4/T4_03i.WAV","sha":"871ce991743c2442f023f6d6892f1ab4a724648d","bytes":43798},{"path":"voice/Training/Mission4/T4_03j.WAV","sha":"3d52530d1fb8cd16ae73d23865b58c87676eb0b4","bytes":31024},{"path":"voice/Training/Mission4/T4_03k.WAV","sha":"bd6ce92048ffbfbc42e8340663e91716dc916a3c","bytes":39196},{"path":"voice/Training/Mission4/T4_04.WAV","sha":"567c5f25ce9eeb3b832039341fb704d1dd4cbe7d","bytes":52468},{"path":"voice/Training/Mission4/T4_04a.WAV","sha":"54c89d01c50c710597d62ca2af54c48c7cbebcd8","bytes":65820},{"path":"voice/Training/Mission4/T4_04b.WAV","sha":"c3d877173c91db2249332942d59de7d82953f0f5","bytes":74616},{"path":"voice/Training/Mission4/T4_05.WAV","sha":"d6af8ec13f34eeeb6c9b3d748979d36b0776f6fd","bytes":14250},{"path":"voice/Training/Mission4/T4_06.WAV","sha":"c9fcea5103e0e07bf8c82ee3d1c73a82948eaf10","bytes":33408},{"path":"voice/Training/Mission4/T4_07.WAV","sha":"ce07a3d3cc44c6204ec2cff7a3b19b3454703d1a","bytes":16852},{"path":"voice/Training/Mission4/T4_07a.WAV","sha":"c0fd71b84c7afede8565c4221eeb5fc90eb6171b","bytes":16798},{"path":"voice/Training/Mission4/T4_08.WAV","sha":"fe10610c6299a3ecce1e33ece151f707a6ae40c9","bytes":46928},{"path":"voice/Training/Mission4/T4_09.WAV","sha":"18dca9a60b12f0f79b689af48541bd88ca2ee78c","bytes":24900},{"path":"voice/Training/Mission4/T4_10.WAV","sha":"fb21abd94437462eb0555b4e85641b5357a823c0","bytes":26902},{"path":"voice/Training/Mission4/T4_11.WAV","sha":"1dbc843108f5bc05b63b028debe2c7bc3852078d","bytes":71786},{"path":"voice/Training/Mission4/T4_CCend.WAV","sha":"24496e02b2636ffcf7d7010bcd07aa1532ff8751","bytes":65242},{"path":"voice/Training/Mission4/T4_controlTurret.WAV","sha":"f7e589376cc11c49fcd2f11b05f4f081da5b2906","bytes":62024},{"path":"voice/Training/Mission5/T5.failure01.WAV","sha":"431590b0943c9dcd2e259b5da284b77c3b2a519c","bytes":40966},{"path":"voice/Training/Mission5/T5.failure02.WAV","sha":"b042ce238164d4637af386dc50bb8caa6acdce7b","bytes":70414},{"path":"voice/Training/Mission5/T5.tip.firepower.WAV","sha":"a35594ddea12c6900bdd6a44889979ce0d05d280","bytes":83228},{"path":"voice/Training/Mission5/T5.tip.satchel01.WAV","sha":"df26f8b44443073b34c14b5571ede2e2f236bc46","bytes":50542},{"path":"voice/Training/Mission5/T5.tip.stations01.WAV","sha":"7a45e29c823fd397f29e8f14d30a7b6c66684bb1","bytes":59776},{"path":"voice/Training/Mission5/T5.tip.stations02.WAV","sha":"a93dadfe371609654c54cf3e719c5840ba89d443","bytes":45780},{"path":"voice/Training/Mission5/T5_01.WAV","sha":"89a5d3daa61dcc87e2b0c74588a421fe4f1eaee3","bytes":69196},{"path":"voice/Training/Mission5/T5_02.WAV","sha":"a4dd7bdaa090fc0f33195777ea7af27f2cc63831","bytes":60908},{"path":"voice/Training/Mission5/T5_03.WAV","sha":"bc98f38a32fb3d7acef73a610cb9bfd5289eca6d","bytes":54116},{"path":"voice/Training/Mission5/T5_04.WAV","sha":"5f30a0edc444f4f50c21a00ccc8a9ffb56bc0ae3","bytes":97618},{"path":"voice/Training/Mission5/T5_05.WAV","sha":"518b91d9921d92542ad2cc59080defeb8af36a6e","bytes":42556},{"path":"voice/Training/Mission5/T5_05a.WAV","sha":"0024661887defb060e155559c1743b3d2a1b51dc","bytes":30268},{"path":"voice/Training/Mission5/T5_05b.WAV","sha":"c51515535a7b125c411b2425e208f810fafbe288","bytes":27128},{"path":"voice/Training/Mission5/T5_06.WAV","sha":"6a7baacdb0d3d5a3d4a1d7ecfe41b08601f6ee53","bytes":25050},{"path":"voice/Training/Mission5/T5_06a.WAV","sha":"c919495cdf32fda24bfdcda732969ff68aeeebb7","bytes":24158},{"path":"voice/Training/Mission5/T5_06b.WAV","sha":"ed17412c10cb493beae096068ed77a6a162dac72","bytes":18068},{"path":"voice/Training/Mission5/T5_06c.WAV","sha":"a75222b9153ac4aa4e5adff78960dab7a509abf0","bytes":20246},{"path":"voice/Training/Mission5/T5_06d.WAV","sha":"405dfb66df5ae985b86aff38ce164de7362de94a","bytes":13136},{"path":"voice/Training/Mission5/T5_07.WAV","sha":"ccc21058b1740ef7a913e8a4b9ad6b533fb34560","bytes":15124},{"path":"voice/Training/Mission5/T5_07a.WAV","sha":"96186e475dc92a552c935dff5ecdd8b6feada12c","bytes":14566},{"path":"voice/Training/Mission5/T5_08.WAV","sha":"704ee7953ab2ee673934c26bc88d8a1578421937","bytes":58142},{"path":"voice/Training/Mission5/T5_08_urgent.WAV","sha":"834dfce2d51cc57d482c55deb3c7ae559c7c0c99","bytes":50164},{"path":"voice/Training/Mission5/T5_08a.WAV","sha":"becc281297938b5fc30d04c3a57e3462a92b8848","bytes":12328},{"path":"voice/Training/Mission5/T5_08b.WAV","sha":"f5605fba3412511f3cf3f576ff3998b20c47740c","bytes":11580},{"path":"voice/Training/Mission5/T5_08c.WAV","sha":"745741ef76a7a6f75a2e07e9416eb1177c7c8b33","bytes":10706},{"path":"voice/Training/Mission5/T5_08d.WAV","sha":"38e742bf9ee446656f022dc70e6ca82b1c5edbdf","bytes":11956},{"path":"voice/Training/Mission5/T5_09.WAV","sha":"f29967137fd07fb8cee53278c52f1d445cc30dca","bytes":62982}]}\n', 'prep/voice_lab.py': '"""Offline preparation helpers; no neural inference is used by a game server."""\nfrom __future__ import annotations\nimport csv\nimport hashlib\nimport io\nimport json\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport urllib.request\nimport wave\nfrom collections import Counter\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nfrom pathlib import Path\nfrom urllib.parse import quote\n\nROOT = Path(__file__).resolve().parent\n\ndef read_manifest():\n    return json.loads((ROOT / \'assets.json\').read_text())\n\ndef groups():\n    return Counter(x[\'path\'].split(\'/\')[1] for x in read_manifest()[\'files\'])\n\ndef git_sha(data):\n    return hashlib.sha1(b\'blob \' + str(len(data)).encode() + b\'\\0\' + data).hexdigest()\n\ndef download_assets(destination, selected=None, workers=4):\n    """Pinned originals only; resume only when bytes match the recorded Git blob."""\n    destination = Path(destination)\n    manifest = read_manifest()\n    entries = [x for x in manifest[\'files\'] if selected is None or x[\'path\'].split(\'/\')[1] in selected]\n    if not entries:\n        raise ValueError(\'No source files match the selected groups.\')\n    def one(row):\n        relative = Path(row[\'path\'])\n        if relative.is_absolute() or \'..\' in relative.parts:\n            raise ValueError(\'Unsafe source path\')\n        target = destination / relative\n        target.parent.mkdir(parents=True, exist_ok=True)\n        if target.exists() and git_sha(target.read_bytes()) == row[\'sha\']:\n            return \'cached\'\n        url = \'https://raw.githubusercontent.com/\' + manifest[\'repository\'] + \'/\' + manifest[\'revision\'] + \'/\' + quote(manifest[\'voice_root\'] + \'/\' + row[\'path\'], safe=\'/\')\n        for attempt in range(3):\n            try:\n                with urllib.request.urlopen(urllib.request.Request(url, headers={\'User-Agent\':\'Tribes2-Voice-Lab\'}), timeout=40) as response:\n                    data = response.read()\n                if git_sha(data) != row[\'sha\']:\n                    raise ValueError(\'Source hash mismatch for \' + row[\'path\'])\n                temporary = target.with_name(target.name + \'.part\')\n                temporary.write_bytes(data)\n                temporary.replace(target)\n                return \'downloaded\'\n            except Exception:\n                if attempt == 2:\n                    raise\n                time.sleep(attempt + 1)\n    failures, counts = [], Counter()\n    with ThreadPoolExecutor(max_workers=max(1, min(workers, 8))) as pool:\n        futures = {pool.submit(one, row):row for row in entries}\n        for done, future in enumerate(as_completed(futures), 1):\n            try:\n                counts[future.result()] += 1\n            except Exception as exc:\n                failures.append({\'path\':futures[future][\'path\'], \'error\':str(exc)})\n            if done % 100 == 0 or done == len(entries):\n                print(f\'{done}/{len(entries)}: {dict(counts)}, failed={len(failures)}\', flush=True)\n    (destination / \'download_failures.json\').write_text(json.dumps(failures, indent=2))\n    if failures:\n        raise RuntimeError(f\'{len(failures)} files failed; inspect download_failures.json and rerun to resume.\')\n    return dict(counts)\n\ndef write_csv(path, rows, columns):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open(\'w\', newline=\'\', encoding=\'utf-8\') as f:\n        w = csv.DictWriter(f, fieldnames=columns)\n        w.writeheader()\n        w.writerows(rows)\n\ndef transcript_seeds():\n    text = (ROOT / \'cannedChatItems.cs\').read_text()\n    pattern = r\'\'\'installChatItem\\(\\s*\'[^\']+\'\\s*,\\s*"((?:\\\\.|[^"\\\\])*)"\\s*,\\s*"([^"\\n]+)"\'\'\'\n    return {name.casefold():words for words, name in re.findall(pattern, text) if words.strip()}\n\ndef build_review(path):\n    """Do not infer speaker from Training subdirectory, or overwrite human review."""\n    path = Path(path)\n    previous = {}\n    if path.exists():\n        with path.open(newline=\'\', encoding=\'utf-8\') as f:\n            previous = {x[\'path\']:x for x in csv.DictReader(f)}\n    seeds = transcript_seeds()\n    rows = []\n    for entry in read_manifest()[\'files\']:\n        p = Path(entry[\'path\'])\n        group = p.parts[1]\n        row = {\'path\':entry[\'path\'], \'group\':group,\n               \'speaker_id\': \'\' if group == \'Training\' else group,\n               \'speaker_verified\':\'false\',\n               \'text\':seeds.get(p.stem.casefold(), \'\') if group not in (\'Announcer\',\'Training\') else \'\',\n               \'transcript_verified\':\'false\', \'use_as_reference\':\'false\', \'notes\':\'\'}\n        row.update(previous.get(entry[\'path\'], {}))\n        rows.append(row)\n    write_csv(path, rows, list(rows[0]))\n    return rows\n\ndef decode_pcm(source, destination, sample_rate=24000):\n    if not shutil.which(\'ffmpeg\'):\n        raise RuntimeError(\'FFmpeg is required to decode the original game WAV codecs.\')\n    destination = Path(destination)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    temporary = destination.with_name(destination.stem + \'.tmp.wav\')\n    command = [\'ffmpeg\',\'-nostdin\',\'-hide_banner\',\'-loglevel\',\'error\',\'-y\',\'-i\',str(source),\n               \'-ac\',\'1\',\'-ar\',str(sample_rate),\'-c:a\',\'pcm_s16le\',str(temporary)]\n    result = subprocess.run(command, capture_output=True, text=True)\n    if result.returncode:\n        temporary.unlink(missing_ok=True)\n        raise RuntimeError(result.stderr[-1500:])\n    temporary.replace(destination)\n    return destination\n\ndef preview_path(raw_root, pcm_root, relative):\n    source = Path(raw_root) / relative\n    if not source.is_file():\n        raise FileNotFoundError(\'Download the pack first: \' + str(source))\n    return decode_pcm(source, Path(pcm_root) / relative)\n\ndef make_reference(review_csv, raw_root, pcm_root, output, speaker_id):\n    """Concatenate reviewed excerpts of ONE speaker; transcript must match ALL speech."""\n    with Path(review_csv).open(newline=\'\', encoding=\'utf-8\') as f:\n        rows = [x for x in csv.DictReader(f) if x[\'speaker_id\'] == speaker_id and x[\'use_as_reference\'].lower() == \'true\']\n    if not rows:\n        raise ValueError(\'No reference clips selected for \' + speaker_id)\n    if any(x[\'speaker_verified\'].lower() != \'true\' or x[\'transcript_verified\'].lower() != \'true\' or not x[\'text\'].strip() for x in rows):\n        raise ValueError(\'Verify speaker and complete transcript for every selected reference clip.\')\n    frames = []\n    rate = 24000\n    for row in rows:\n        path = preview_path(raw_root, pcm_root, row[\'path\'])\n        with wave.open(str(path), \'rb\') as w:\n            if (w.getnchannels(), w.getsampwidth(), w.getframerate()) != (1,2,rate):\n                raise ValueError(\'Reference must be mono 24 kHz PCM16\')\n            frames.append(w.readframes(w.getnframes()))\n    data = (b\'\\0\\0\' * int(rate * .15)).join(frames)\n    seconds = len(data) / (rate * 2)\n    if not 3 <= seconds <= 30:\n        raise ValueError(f\'Reference is {seconds:.1f}s; select roughly 8–15 seconds of clean consistent speech (3–30s limit here).\')\n    output = Path(output)\n    output.parent.mkdir(parents=True, exist_ok=True)\n    with wave.open(str(output),\'wb\') as w:\n        w.setnchannels(1); w.setsampwidth(2); w.setframerate(rate); w.writeframes(data)\n    output.with_suffix(\'.txt\').write_text(\' \'.join(x[\'text\'].strip() for x in rows))\n    output.with_suffix(\'.json\').write_text(json.dumps({\'speaker_id\':speaker_id,\'sources\':[x[\'path\'] for x in rows], \'seconds\':seconds}, indent=2))\n    return output\n\ndef install_breeze(destination):\n    """Separate interpreter keeps official pinned dependencies away from Kaggle\'s kernel."""\n    destination = Path(destination)\n    source = destination / \'breeze-tts\'\n    python = destination / \'venv\' / \'bin\' / \'python\'\n    destination.mkdir(parents=True, exist_ok=True)\n    if not source.exists():\n        subprocess.run([\'git\',\'clone\',\'--depth\',\'1\',\'https://github.com/breezeblue-ai/breeze-tts.git\',str(source)], check=True)\n    if not python.exists():\n        subprocess.run([sys.executable,\'-m\',\'venv\',str(destination / \'venv\')], check=True)\n    subprocess.run([str(python),\'-m\',\'pip\',\'install\',\'--upgrade\',\'pip\'], check=True)\n    subprocess.run([str(python),\'-m\',\'pip\',\'install\',\'-r\',str(source / \'requirements.txt\')], check=True)\n    # Record actual source/dependencies used. A rerun reuses this checkout rather than pulling changes.\n    revision = subprocess.check_output([\'git\',\'-C\',str(source),\'rev-parse\',\'HEAD\'], text=True).strip()\n    (destination / \'source_revision.txt\').write_text(revision + \'\\n\')\n    freeze = subprocess.check_output([str(python),\'-m\',\'pip\',\'freeze\'], text=True)\n    (destination / \'requirements-resolved.txt\').write_text(freeze)\n    return python, source\n\ndef clone_preview(destination, reference, text, output, seed=42):\n    destination, reference, output = map(Path, (destination, reference, output))\n    python = destination / \'venv\' / \'bin\' / \'python\'\n    source = destination / \'breeze-tts\'\n    model_path = destination / \'model\'\n    if not python.is_file():\n        raise RuntimeError(\'Install Breeze first.\')\n    subprocess.run([str(python),\'-c\',\'import torch; assert torch.cuda.is_available(), "No CUDA GPU available"\'], check=True)\n    # infer.py expects a local directory, not a model ID.\n    subprocess.run([str(python),\'-c\',\n        \'from huggingface_hub import snapshot_download; import sys; snapshot_download("BreezeBlue/Breeze-TTS-2", local_dir=sys.argv[1])\',str(model_path)],check=True)\n    transcript = reference.with_suffix(\'.txt\').read_text().strip()\n    if not transcript:\n        raise ValueError(\'Reference transcript is empty.\')\n    output.parent.mkdir(parents=True, exist_ok=True)\n    command = [str(python), str(source / \'infer.py\'), str(model_path),\n        \'--ref-audio\', str(reference), \'--ref-text\', transcript, \'--text\', text,\n        \'--output\', str(output), \'--seed\', str(seed), \'--cfg-scale\',\'1\',\'--no-fast-all\']\n    # Eager mode avoids fast-path compilation and extra graph memory. GPU compatibility remains to be tested.\n    subprocess.run(command, cwd=source, check=True)\n    return output\n', 'pronunciation-overrides.json': '{\n  "aliases": {"CodeMasterCody3D": "Cody", "Cody_42": "Cody forty two"},\n  "phones": {\n    "cody": ["K", "OW1", "D", "IY0"],\n    "derm": ["D", "ER1", "M"],\n    "bioderm": ["B", "AY1", "OW0", "D", "ER0", "M"],\n    "spinfusor": ["S", "P", "IH1", "N", "F", "Y", "UW0", "Z", "ER0"]\n  }\n}\n', 'toolchain-lock.json': '{\n  "tested_python": "3.13.16",\n  "game_revision": "b9dda9eaf153d706e94fc71f242866d2f2709394",\n  "breeze_source_revision": "daa6cb2efe5471f09369229f322d20bb956cb3c3",\n  "breeze_model": {"repo": "BreezeBlue/Breeze-TTS-2", "revision": "3e28c5151381a722f1d8661b4118c298caa77aa4"},\n  "asr_model": {"repo": "Systran/faster-whisper-large-v3", "revision": "edaa852ec7e145841d8ffdb056a99866b5f0a478"},\n  "speaker_model": {"repo": "Wespeaker/wespeaker-voxceleb-resnet34-LM", "revision": "f0c48c298fd835726c27956a5d617bad7115627e"},\n  "mfa_version": "3.3.9",\n  "mfa_acoustic_sha256": "d35ce271ded357d833d2f4b8d1041dc3748b9538567ba13f2c697f4e4126711b",\n  "mfa_dictionary_sha256": "e8c6c7b036ae2b7c78d2768b8dc6b1f9359175b842956d00b48c53c9c332e6b0",\n  "kaldi_native_fbank": "1.22.5"\n}\n', 'requirements-breeze.lock.txt': 'accelerate==1.12.0\nannotated-doc==0.0.5\nannotated-types==0.8.0\nanyio==4.16.0\naudioop-lts==0.2.2\nbrotli==1.2.1\ncertifi==2026.10.23\ncffi==2.1.1\ncharset-normalizer==3.5.2\nclick==8.5.0\ncloudpickle==3.1.2\ndecorator==5.3.1\neinops==0.8.2\nfastapi==0.162.0\nfilelock==4.2.7\nflatbuffers==25.12.19\nfsspec==2026.9.0\ngradio==6.17.3\ngradio_client==2.5.0\ngroovy==0.1.2\nh11==0.16.0\nhf-gradio==0.4.1\nhf-xet==1.8.0\nhttpcore==1.0.9\nhttpx==0.28.1\nhuggingface_hub==0.36.2\nidna==3.20\niniconfig==2.3.1\nJinja2==3.1.6\njoblib==1.6.0\nlazy-loader==0.6\nlibrosa==1.0.0\nllvmlite==0.50.0\nmarkdown-it-py==4.2.0\nMarkupSafe==3.0.4\nmdurl==0.1.2\nmpmath==1.3.0\nmsgpack==1.2.4\nnarwhals==2.28.0\nnetworkx==3.7\nnumba==0.68.1\nnumpy==2.5.4\nnvidia-cublas-cu12==12.8.4.1\nnvidia-cuda-cupti-cu12==12.8.90\nnvidia-cuda-nvrtc-cu12==12.8.93\nnvidia-cuda-runtime-cu12==12.8.90\nnvidia-cudnn-cu12==9.10.2.21\nnvidia-cufft-cu12==11.3.3.83\nnvidia-cufile-cu12==1.13.1.3\nnvidia-curand-cu12==10.3.9.90\nnvidia-cusolver-cu12==11.7.3.90\nnvidia-cusparse-cu12==12.5.8.93\nnvidia-cusparselt-cu12==0.7.1\nnvidia-nccl-cu12==2.27.5\nnvidia-nvjitlink-cu12==12.8.93\nnvidia-nvshmem-cu12==3.3.20\nnvidia-nvtx-cu12==12.8.90\nonnxruntime==1.31.1\nopentelemetry-api==1.45.2\norjson==3.13.0\npackaging==26.3\npandas==3.1.0\npillow==12.3.0\nplatformdirs==4.14.3\npluggy==1.7.0\npooch==1.9.0\nprotobuf==7.36.2\npsutil==7.2.2\npycparser==3.12\npydantic==2.14.2\npydantic_core==2.50.2\npydub==0.25.1\nPygments==2.21.0\npytest==9.2.0\npython-dateutil==4.0.0\npython-multipart==0.0.33\npytz==2026.6\nPyYAML==6.0.3\nqwen-tts==0.1.1\nregex==2026.9.29\nrequests==2.34.2\nrich==15.0.0\nruff==0.17.3\nsafehttpx==0.1.7\nsafetensors==0.8.0\nscikit-learn==1.9.1\nscipy==1.18.1\nsemantic-version==2.10.0\nsetuptools==84.2.1\nshellingham==1.5.4\nsoundfile==0.14.0\nsox==1.5.0\nsoxr==1.1.0\nstarlette==1.10.0\nsympy==1.14.0\nthreadpoolctl==3.7.0\ntokenizers==0.22.2\ntomlkit==0.14.0\ntorch==2.9.1\ntorchaudio==2.9.1\ntqdm==4.70.1\ntransformers==4.57.3\ntriton==3.5.1\ntyper==0.27.3\ntyping-inspection==0.4.4\ntyping_extensions==4.16.0\ntzdata==2026.6\nurllib3==2.8.0\nuvicorn==0.54.0\n', 'rights-attestation.example.json': '{\n  "performer_consent_and_recording_rights_confirmed": false,\n  "voice_ids": [],\n  "scope": "private research and noncommercial testing",\n  "documents_reviewed_by_ai": false,\n  "public_production_or_commercial_clearance": "not established",\n  "license": "BreezeBlue Research and Non-Commercial License 1.1, sections 5(c), 5(d), 1.7 and 3",\n  "instructions": "Create artifacts/permissions/rights-attestation.json only after the required performer consents and reference-recording rights are actually secured. List the permitted voice IDs. This example grants no permission. Production/public deployment needs a separate licensing assessment."\n}\n'}
for name,text in FILES.items():
    path=ROOT/name
    path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists() and path.read_text()!=text:
        old=path.read_bytes()
        vault=ROOT/'code_vault'/f'{path.name}.{hashlib.sha256(old).hexdigest()[:16]}'
        vault.parent.mkdir(exist_ok=True); vault.write_bytes(old)
    path.write_text(text)
def run(*args):
    subprocess.run([str(x) for x in args],cwd=ROOT,check=True)
print('Materialized',len(FILES),'files')
run('nvidia-smi','--query-gpu=name,memory.total,memory.free','--format=csv')
run('ffmpeg','-version')


## Original audio: pinned download, preserved source and FFmpeg decode

In [ ]:
run(sys.executable,'-u',ROOT/'remote_prepare.py')

## Transcription candidates: isolated environment and resumable ASR

In [ ]:
run(sys.executable,'-u',ROOT/'setup_remote.py','asr')
ASR=ROOT/'environments/asr/bin/python'
run(ASR,'-u',ROOT/'asr_entry.py','--asr','--all-asr','--skip-prep')
run(ASR,ROOT/'build_reports.py')

## Speaker comparisons and review
Compare training recordings individually with enrolled candidate voices.
Cosine scores are not identity probabilities. Review the original audio and
matching complete transcript, including any ASR errors, before certifying a source.
The game mission scripts identify Kenzie and Akanruth roles; acoustic assignments
and numbered-pack to in-game display-name mappings remain separate evidence.


In [ ]:
run(ASR,'-m','pip','install','kaldi-native-fbank==1.22.5')
run(ASR,'-u',ROOT/'speaker_audit.py')
run(ASR,ROOT/'pilot.py','references','--character-packs')

## Breeze pilot: three independent voices, one model load
These small auditions explicitly use provisional references, NOT production banks.
Listen to each reference and clone together; the process does not mark them verified.
Dependencies, model/source revisions, memory and generation times are saved.


In [ ]:
RIGHTS_CONFIRMED=False  # Set only after the required permissions are actually secured.
permission_path=ROOT/'artifacts/permissions/rights-attestation.json'
if not permission_path.exists():
    if not RIGHTS_CONFIRMED:
        raise PermissionError('Breeze sections 5(c) and 5(d) require explicit performer consent and reference-recording rights; noncommercial use alone is not consent.')
    permission=json.loads((ROOT/'rights-attestation.example.json').read_text())
    permission['performer_consent_and_recording_rights_confirmed']=True
    permission['voice_ids']=['Announcer','TrainingFemale','BriefingMale',
        *(f'Male{i}' for i in range(1,6)),*(f'Fem{i}' for i in range(1,6)),
        *(f'Derm{i}' for i in range(1,4)),'Bot1']
    permission_path.parent.mkdir(parents=True,exist_ok=True)
    permission_path.write_text(json.dumps(permission,indent=2))
run(sys.executable,'-u',ROOT/'setup_remote.py','breeze')
BREEZE=ROOT/'environments/breeze/bin/python'
run(BREEZE,'-u',ROOT/'pilot.py','generate','--speakers','Announcer','TrainingFemale','BriefingMale','--allow-provisional-reference')


In [ ]:
from IPython.display import Audio,display
for voice in ('Announcer','TrainingFemale','BriefingMale'):
    print(voice,'ORIGINAL REFERENCE — provisional transcript/speaker review')
    display(Audio(filename=str(ROOT/'artifacts/references-provisional'/f'{voice}.wav')))
    print(voice,'BREEZE GENERATED AUDITION — unreviewed')
    display(Audio(filename=str(ROOT/'artifacts/pilot'/f'{voice}_neutral.wav')))


## Character-pack neutral pilots
The same short test sentence is rendered for the fourteen numbered packs.
This takes about four minutes on the tested T4, with one model load.
Numbered pack IDs are not verified in-game display-name mappings.


In [ ]:
CHARACTERS=[*(f'Male{i}' for i in range(1,6)),*(f'Fem{i}' for i in range(1,6)),*(f'Derm{i}' for i in range(1,4)),'Bot1']
run(BREEZE,'-u',ROOT/'pilot.py','generate','--speakers',*CHARACTERS,'--allow-provisional-reference')
run(ASR,'-u',ROOT/'validate_pilots.py')
run(sys.executable,ROOT/'build_auditions.py')

## Emotion and event auditions (review gate)
Five emotions use separate Voice Direction jobs. The documented English cues
are laugh, sigh, cough and clears throat in parentheses. Generated carrier speech
must be isolated away from a reviewed event; unsupported cues are never assumed.
Stock grunts, pain and cheers remain candidates until classified as spoken or wordless.


In [ ]:
RUN_EXPRESSIVE_AUDITIONS=False  # compute control, not automatic quality approval
if RUN_EXPRESSIVE_AUDITIONS:
    VOICES=['Announcer','TrainingFemale','BriefingMale',*CHARACTERS]
    run(BREEZE,'-u',ROOT/'pilot.py','generate','--speakers',*VOICES,'--expressions','--no-events','--allow-provisional-reference')
    run(BREEZE,'-u',ROOT/'pilot.py','generate','--speakers','Announcer','--expressions','--allow-provisional-reference')
else:
    print('Optional style sweep is disabled: approximately 35–40 minutes for the non-neutral speech auditions on the tested T4.')


## Compact gap material
The declared announcement workload has 333 predicted stress-sensitive transitions.
The set-cover selector chose 32 sentences. This is not all English or all names.
The announcer-only neutral material takes about five minutes on the tested T4.
For an existing bank, use the incremental planner with explicit voices and styles.
Its optional --workload-priority favors missing transitions shared by more workload
messages. Predicted gains must still pass generation, word checks and actual alignment.
Keep failed attempts and earlier banks; give parallel GPU shards different output tags.


In [ ]:
RUN_GAP_MATERIAL=False
run(sys.executable,ROOT/'gap_jobs.py','--speaker','Announcer')
if RUN_GAP_MATERIAL:
    run(BREEZE,'-u',ROOT/'pilot.py','generate','--speakers','Announcer',
        '--jobs',ROOT/'artifacts/gap-jobs-Announcer.json','--allow-provisional-reference')
run(ASR,'-u',ROOT/'validate_pilots.py')


## Phone alignment and candidate bank
MFA emits phone intervals. ASR word times must never be used as diphone boundaries.
Unknown pronunciations are refused before alignment. Review stops in waveform or
spectrogram: mark an anchor inside the closure, before the full protected release.
Stress variants are retained; allophones require explicit annotation. The automatic
bank is intentionally unverified, and the production renderer rejects it.


In [ ]:
RUN_ALIGNMENT=False
if RUN_ALIGNMENT:
    run(sys.executable,'-u',ROOT/'setup_remote.py','align')
    run(sys.executable,ROOT/'make_alignment_jobs.py','--stock-announcer','--pilots')
    run(ASR,'-u',ROOT/'alignment.py','--jobs',ROOT/'artifacts/alignment-jobs.json',
        '--lexicon',ROOT/'artifacts/reports/cmudict.json','--overrides',ROOT/'pronunciation-overrides.json',
        '--out',ROOT/'artifacts/phone-alignment','--mfa',ROOT/'environments/mfa/bin/mfa')
else:
    print('Alignment stage ready; enable after the short pilot.')


In [ ]:
if RUN_ALIGNMENT:
    run(sys.executable,ROOT/'render_auditions.py','--bank',ROOT/'artifacts/phone-alignment/candidate-bank','--bank-pronunciations')
    run(sys.executable,'-S',ROOT/'render_voice_matrix.py','--bank-pronunciations')
    run(ASR,ROOT/'build_reports.py')


## Sample-only runtime tests and measured fixture benchmark
The tests use synthetic fixtures, not a claim about voice naturalness or phonetic
coverage. Production rendering requires a bank whose actual reviews have passed.
Unknown words, missing units/events and missing emotion banks raise explicit errors.
Optional neutral/stress fallback must be selected explicitly and is reported.


In [ ]:
run(sys.executable,ROOT/'test_engine.py')
run(ASR,ROOT/'test_preparation.py')
print((ROOT/'test-results.json').read_text())

## Export durable private artifacts
The archive excludes environments and model weights. Stock and decoded audio,
reports, references, pilot WAVs and sample banks are included. The VL2 packager
only accepts reviewed rendered phrases. Every native client must have the bytes;
server-side filenames do not transfer audio. Live client playback is not certified.


In [ ]:
run(sys.executable,ROOT/'build_auditions.py')
run(sys.executable,ROOT/'build_event_auditions.py')
run(sys.executable,ROOT/'export_review.py')
run(sys.executable,ROOT/'record_environment.py')
run(sys.executable,ROOT/'archive_artifacts.py')
print('/kaggle/working/tribes2-voice-artifacts.tar.gz')
print('========== DONE ==========')
